# Residual dust in quiescent galaxies: the "ISM equilibrium at large radii" hypothesis

**Working hypothesis.** Some quiescent galaxies keep a residual dust reservoir in
their ISM because that ISM stayed in *equilibrium at large radii* and was **never
funnelled inward to feed the AGN**. Without a strong inflow → no AGN-driven gas
heating → no thermal/sputtering dust destruction → dust survives long after
quenching.

**Refined scenario under test (2026-06).** Galaxies with
$10.25 < \log_{10}M_\star < 11.25$ sitting in **overmassive halos** (positive offset
from the SHMR ridge, §8a) may be **funneling HI from the CGM** into a **large rotating
H$_2$ disk**; the disk feeds the BH until jets ignite and quench the galaxy rapidly,
while the **extended dust survives unless the AGN couples to the ISM** (in SIMBA the
only ISM-coupled AGN channel is X-ray feedback, active when the BH is in jet mode
**and** $f_{\rm gas} = M_{\rm gas}/M_\star < 0.2$). The link-by-link test is **§8j**;
supporting diagnostics: §7b–§7d (cold CGM reservoir), §5d-gas (HI/H$_2$ extension),
§8i (gas rotational support & spin misalignment), §8c (event ordering env → AGN → quench).

### Sample & method

- **Sample:** every galaxy *passive at z = 0*, defined by $\mathrm{sSFR}(z{=}0) < 0.2/t_H$
  (the same $0.2/t$ threshold used to mark the quenching time `QT`), with
  $\log_{10}(M_\star/M_\odot) > 9.5$ (`MASS_FLOOR`), faceted throughout into the z=0
  stellar-mass bins 9.5–10.5 / 10.5–11 / $\geq$11. Residual dust is a *measured
  outcome*, never a selection cut.
- **Per-galaxy evolutionary anchors** (not absolute snapshots). For each galaxy we
  trace its progenitors and locate, on its own track:
  `SF-peak` (max sSFR) · `ssfr-min` · `SFT` (crossing of $1/t$) · `QT` (crossing of
  $0.2/t$) · `post-quench` (persistence end) · `dust-peak` · `dust-min` · `gas-min` · `z=0`.
- **"Star-forming" reference:** the *same* galaxies sampled at their **SFT**, i.e.
  when they were still forming stars (no separate control sample).
- **Fast vs slow:** $\tau_q = t_{QT} - t_{SFT}$, normalized by the age of the universe
  at quenching: **fast** if $\log_{10}(\tau_q/t_H(z_{QT})) < -1.5$ (quenched in
  $\lesssim 3\%$ of the then-current Hubble time; `TAU_SPLIT_LOG` in §0), else **slow**.
  `SPLIT_MODE` in §0 can swap this for a split on any catalogue property.

### Structure

Cells are grouped into thematic **Parts** organised by physical domain; the legacy
section ids (§0…§10) are kept in every header as stable anchors, so all "Needs §…"
cross-references remain valid.

| Part | Content | sections |
|---|---|---|
| 0 | setup & simulation-level products | §0, §0b, §1, §5·pre, §1z |
| 1 | sanity checks, sample & general statistics | §2, §2·diag1–3, §3, §3b, §7·setup, S1, S2 |
| 2 | when & how these galaxies quench (timing) | §3c, §3d, §3e |
| 3 | cached products & per-stage diagnostics (build once) | §5, §4b·load, §7a, §7g, §8c·pre, §4 (DIAGS) |
| 4 | stellar properties (concentration / size ratios) | stellar+gas r20/r80, r20/r50 |
| 5 | ISM: reservoirs & dust survival | §4a–§4d, §3f–§3j |
| 6 | CGM reservoir, temperature & profiles | §7b–§7h, §7p |
| 7 | kinematics, morphology & ISM distribution | §5c–§5e, dust/HI/H₂/star r20/r80, r20/r50 |
| 8 | environment | §7i, §7L, §7N |
| 9 | primary axis? fast/slow vs halo mass vs AGN mode | §4b, §7j, §8a–§8n, η² synthesis |
| 10 | interpretation against the hypothesis | §9 |
| 11 | observables: abundances & sub-mm/IR number counts | §10 |
| App | superseded, negative & QC material | §4e, §4b-iii, §7k |

**New in this pass.** Size-ratio (concentration) diagnostics — $r_{20}/r_{80}$ and
$r_{20}/r_{50}$ for **stars, gas, dust, HI and H$_2$** — added to the catalogue grids
(Part 4), the particle-profile product (Part 7, exact percentile radii from the rebuilt
§5a) and the population-character grid (§8l); plus an η² test (Part 9) of which axis the
population is really organised by.

> **Run environment.** This repository's data live on the cluster; the heavy steps
> (catalogue loads, progenitor build, particle extraction) are gated behind
> `BUILD_*` / `RUN_*` flags so the notebook can be re-run incrementally there.


# Part 0 — Setup & simulation-level products

Configuration, plotting style, the progenitor tracks + property history, the progenitor-index matrix and the multi-redshift selection anchors. Everything here depends only on the simulation, not on the selected sample.

## I.1 (§0) · Setup & configuration

In [ ]:
import os
import gc
import numpy as np
import numpy.lib.recfunctions as rfn
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from collections import defaultdict

from astropy.io import fits
from astropy import units as u
from astropy import constants as const
from astropy.cosmology import Planck15 as COSMO   # matches the repo's quenching batch

from scipy.stats import spearmanr, ks_2samp, fisher_exact, mannwhitneyu, wilcoxon
from scipy.spatial import cKDTree

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen, extract_particles
from simbanator.analysis.quenching import find_quenching_times
from simbanator.utils.geometry import shrink_center, principal_axes, rotate_to_frame
from simbanator.utils.conversions import Dust_to_Metal, Z_to_OH12

# ── configuration ──────────────────────────────────────────────────────────
SIM_NAME      = "cis100"          # configured simulation name (~/.simbanator/config.json)
BOX           = 100               # Mpc/h, used for sanity only
START_SNAP    = 150               # z ~ 0 anchor for the tracks (snap 151 == z=0; 150 ~ z<0.01)
END_SNAP      = 44                # z ~ 4.8, far enough back to capture SF-peak/SFT
PROG_RANGE    = range(END_SNAP, START_SNAP + 1)   # snaps used to build the progenitor tree

MASS_FLOOR    = 9.5               # log10(M*/Msun) floor for the z=0 sample
MASS_BIN_EDGES  = [9.5, 10.5, 11.0, np.inf]   # log10(M*/Msun) bin edges (binned at z=0)
MASS_BIN_LABELS = ["9.5-10.5", "10.5-11", ">=11"]
PASSIVE_FACTOR = 0.2             # passive if sSFR < PASSIVE_FACTOR / t_H  (== QT threshold)
TAU_SPLIT_LOG = -1.25            # fast/slow cut on log10(tau_q / t_H(z_QT)): fast if below
                                 #   (-1.5 -> quenched in <~3% of the age of the universe at QT)

# ── sample-split configuration ───────────────────────────────────────────────
# Every "fast vs slow" comparison in this notebook is driven by ONE split chosen
# here, so the whole analysis can be re-run on a *different* parameter without
# editing any plotting cell. Saved figures go to PLOTDIR/<SPLIT_TAG>/ (set in §3),
# so different splits never overwrite each other and the folder records the split.
#
#   SPLIT_MODE = "fast_slow"  -> original tau_q split (default; reproduces old results)
#   SPLIT_MODE = "property"   -> split any catalogue property at a chosen stage by a
#                               critical value (group A = value above the cut)
# SPLIT_PROP accepts any DIAGS key (§4): "dust/M*", "H2/M*", "HI/M*", "sSFR",
# "dust/gas", "HI/H2", "Rgas/Rstar", "Mdust", "MH2", "MHI", "Mstar", "Mgas", "Rgas".
# SPLIT_STAGE is any evolutionary anchor in STAGES (e.g. "post_quench","qt","z0","sft").
SPLIT_MODE        = "fast_slow"     # "fast_slow" | "property"
SPLIT_PROP        = "dust/M*"       # property mode: what to split on
SPLIT_STAGE       = "post_quench"   # property mode: stage at which to measure it
SPLIT_THRESH      = -5.0            # property mode: critical value (group A = value > this)
SPLIT_LOG         = True            # property mode: compare log10(value) to SPLIT_THRESH
SPLIT_THRESH_MODE = "fixed"         # "fixed" -> SPLIT_THRESH ; "median" -> balanced groups
SPLIT_LABELS      = ("high", "low") # legend labels for (group A above cut, group B below)
SPLIT_COLORS      = ("C3", "C0")    # plot colors for (group A, group B)
NGAS_MIN      = 5               # min gas particles for catalogue reliability / particle work
CORRUPT_SNAPS = set()            # snapshot numbers known to be truncated/corrupt; auto-grows at runtime

# heavy-step switches (set True the first time, then reuse cached products)
BUILD_PROGEN  = False            # build the progenitor FITS (needs all caesar catalogs)
BUILD_HISTORY = False            # build the property history HDF5 (needs all caesar catalogs)
RUN_PARTICLES = True             # intent flag: profiles are built on the CLUSTER (hours) via the
#                                  # SLURM job (§5a·plan -> submit_profiles.sh); the notebook only loads.
BUILD_BH      = True             # build the BH accretion / AGN-energy history (reads all catalogs)
PRESELECT_BUILD = True           # §1b: build history only for massive+quenched-at-z0 targets (False -> all z~0)

# cached product locations (under ./output/<sim>/...)
PROG_FILE     = "progenitors_most_mass.fits"
HIST_FILE     = "history_passive_z0.hdf5"     # written under output/<sim>/caesar_sfh/

sim   = Simulation(SIM_NAME)
OUT   = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR = os.path.join(OUT, "caesar_sfh")
PLOTDIR = os.path.join(OUT, "plots", "residual_dust")
os.makedirs(PLOTDIR, exist_ok=True)
HIST_PATH = os.path.join(SFHDIR, HIST_FILE)

# ── output routing: topical subfolders under PLOTDIR + a tables dir ──────────
# figpath(name) sends each figure into a topical subfolder of PLOTDIR (which already
# carries the <SPLIT_TAG> set in §3), chosen by the filename prefix — one helper keeps
# every savefig uniform and stops plots piling up flat or leaking into the repo root.
TABLEDIR = os.path.join(OUT, "tables")
os.makedirs(TABLEDIR, exist_ok=True)
_FIG_ROUTER = [
    ("diag", "00_diagnostics"),
    ("co_agn", "06_observables"), ("agn_persistence", "06_observables"),
    ("AGN_fraction", "06_observables"), ("dq_", "06_observables"),
    ("quiescent_grid", "06_observables"), ("tdep_", "06_observables"),
    ("Environment_dependent", "06_observables"), ("Redshift_dependent", "06_observables"),
    ("scenario_", "04_drivers_agn"), ("sinks_", "04_drivers_agn"), ("shmr_", "04_drivers_agn"),
    ("matched_twins", "04_drivers_agn"), ("mhalo_event", "04_drivers_agn"),
    ("event_", "04_drivers_agn"), ("causal_", "04_drivers_agn"),
    ("branch_character", "04_drivers_agn"), ("postqt_agn", "04_drivers_agn"),
    ("agn_feeding", "04_drivers_agn"), ("grid_", "04_drivers_agn"), ("bh_", "04_drivers_agn"),
    ("cgmT", "05_drivers_cgm_env"), ("cgm_", "05_drivers_cgm_env"), ("sat_", "05_drivers_cgm_env"),
    ("env_", "05_drivers_cgm_env"), ("cluster_infall", "05_drivers_cgm_env"),
    ("pdusty", "02_dust_survival"), ("tau_dust", "02_dust_survival"), ("dust_", "02_dust_survival"),
    ("halobin", "03_stage_distributions"), ("track_", "03_stage_distributions"),
    ("dist_", "03_stage_distributions"),
    ("critical_point", "01_quench_timing"), ("process_timescales", "01_quench_timing"),
    ("phase_occupancy", "01_quench_timing"), ("crossover", "01_quench_timing"),
    ("sfpeak_premise", "01_quench_timing"),
]
def figpath(name):
    """Absolute path for figure `name`, routed into a topical subfolder of PLOTDIR
    (created on demand) by filename prefix; unmatched names fall to 99_misc."""
    base = os.path.basename(name)
    sub = next((s for pre, s in _FIG_ROUTER if base.startswith(pre)), "99_misc")
    d = os.path.join(PLOTDIR, sub); os.makedirs(d, exist_ok=True)
    return os.path.join(d, base)

print("simulation :", SIM_NAME)
print("track snaps:", START_SNAP, "->", END_SNAP, f"(z {sim.get_z_from_snap(START_SNAP):.3f} -> {sim.get_z_from_snap(END_SNAP):.3f})")
print("z=0 t_H    :", f"{COSMO.age(0).value:.3f} Gyr  -> passive cut sSFR < {PASSIVE_FACTOR/ (COSMO.age(0).value*1e9):.2e} /yr")

## I.2 (§0b) · Global plot style

Big, consistent fonts and larger figures for every plot below. Tune `PLOT_FIG_SCALE`, `PLOT_FONT_BASE`, `PLOT_FONT_FLOOR` in one place; the rest of the notebook is unchanged.

In [ ]:
# ── global plot style: big consistent fonts + larger figures (tune the 3 knobs) ───────────────
# Applied once here so every figure below inherits it WITHOUT editing each cell. plt.subplots /
# plt.figure are wrapped to enlarge all (even hard-coded) figsizes; explicit tiny `fontsize=`
# args are floored so nothing stays unreadable. Re-running this cell is safe (idempotent).
import functools
import matplotlib as mpl
from matplotlib.axes import Axes
from matplotlib.figure import Figure

PLOT_FIG_SCALE  = 1.45   # multiply every figure's width & height
PLOT_FONT_BASE  = 15     # base size (pt) for text that has no explicit size
PLOT_FONT_FLOOR = 14     # explicit per-call fontsizes below this are raised to it

mpl.rcParams.update({
    "font.size":        PLOT_FONT_BASE,
    "axes.titlesize":   PLOT_FONT_BASE + 2,
    "axes.labelsize":   PLOT_FONT_BASE + 1,
    "xtick.labelsize":  PLOT_FONT_BASE,
    "ytick.labelsize":  PLOT_FONT_BASE,
    "legend.fontsize":  PLOT_FONT_BASE,
    "figure.titlesize": PLOT_FONT_BASE + 4,
    "axes.titleweight": "bold",
    "figure.dpi":       110,
    "lines.linewidth":  2.0,
})

def _bump(fs):
    """Raise a small numeric font size to the floor; leave strings ('small', 'large') & big ones."""
    if isinstance(fs, (int, float)) and not isinstance(fs, bool) and fs < PLOT_FONT_FLOOR:
        return PLOT_FONT_FLOOR
    return fs

def _patch_fontsize(cls, names):
    for nm in names:
        orig = getattr(cls, nm, None)
        if orig is None or getattr(orig, "_fontfloor", False):
            continue
        @functools.wraps(orig)
        def wrapper(*a, __orig=orig, **kw):
            if "fontsize" in kw: kw["fontsize"] = _bump(kw["fontsize"])
            if "size" in kw:     kw["size"]     = _bump(kw["size"])
            return __orig(*a, **kw)
        wrapper._fontfloor = True
        setattr(cls, nm, wrapper)

_patch_fontsize(Axes, ["set_title", "set_xlabel", "set_ylabel", "legend", "text", "annotate",
                       "set_xticklabels", "set_yticklabels"])
_patch_fontsize(Figure, ["suptitle", "legend"])
try:                                            # secondary_xaxis labels carry their own fontsize=
    from matplotlib.axes._secondary_axes import SecondaryAxis
    _patch_fontsize(SecondaryAxis, ["set_xlabel", "set_ylabel"])
except Exception:
    pass
for _nm in ["title", "suptitle", "xlabel", "ylabel", "legend", "text", "annotate"]:
    _o = getattr(plt, _nm, None)
    if _o is not None and not getattr(_o, "_fontfloor", False):
        @functools.wraps(_o)
        def _w(*a, __o=_o, **kw):
            if "fontsize" in kw: kw["fontsize"] = _bump(kw["fontsize"])
            if "size" in kw:     kw["size"]     = _bump(kw["size"])
            return __o(*a, **kw)
        _w._fontfloor = True
        setattr(plt, _nm, _w)

# NB: plt.subplots delegates to plt.figure internally, so patching figure() alone
# scales BOTH (patching both would double-scale).
if not getattr(plt.figure, "_figscale", False):
    _orig_figure = plt.figure
    @functools.wraps(_orig_figure)
    def _figure(*a, **kw):
        base = kw.get("figsize") or mpl.rcParams["figure.figsize"]
        kw["figsize"] = (base[0] * PLOT_FIG_SCALE, base[1] * PLOT_FIG_SCALE)
        return _orig_figure(*a, **kw)
    _figure._figscale = True
    plt.figure = _figure

print(f"plot style: fonts>={PLOT_FONT_FLOOR}pt (base {PLOT_FONT_BASE}), figures x{PLOT_FIG_SCALE}")


## I.3 (§1) · Build / load progenitor tracks and the property history

We follow the repository's established pipeline
(`caesar_read_progen` → `HDF5BuildHistory`). The progenitor table is built for **all**
z≈0 galaxies; we then read the full property history and select the passive,
massive subset from the first time-row (§2). `dust`, `H2`, `HI`, gas, stellar mass,
both half-mass radii and positions are all pulled in one pass.

In [ ]:
# ── 1a. progenitor tree (optional, heavy) ──────────────────────────────────
if BUILD_PROGEN:
    cs0 = sim.load_catalog(snap=START_SNAP)
    all_ids = [g.GroupID for g in cs0.galaxies]
    caesar_read_progen(all_ids, PROG_FILE, PROG_RANGE, sim, output_dir=None)
    del cs0; gc.collect()
    print("progenitor table written.")
else:
    print("BUILD_PROGEN=False -> using existing", os.path.join(OUT, "progenitors", PROG_FILE))

In [ ]:
# ── 1b. property history (optional, heavy) ─────────────────────────────────
# NOTE: confirm these keys against the §0.1 probe. 'masses.HI' / 'masses.H2' are
# the standard CAESAR-SIMBA names; change here if your build differs.
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust",
        "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "radii.stellar_r20", "radii.stellar_r80", "radii.gas_r20", "radii.gas_r80",
        "pos", "ngas", "nstar", "ages.mass_weighted", "metallicities.mass_weighted", "metallicities.mass_weighted_cgm", 
        "metallicities.stellar", "rotation.total_L", "rotation.total_kappa_rot",
        "rotation.stellar_kappa_rot", "rotation.gas_kappa_rot",   # disc vs pressure support (kappa_rot)
        "rotation.stellar_BoverT", "rotation.gas_BoverT",         # bulge/total morphology
        "rotation.stellar_L", "rotation.gas_L",                   # angular momentum (specific AM, misalignment)
        "rotation.stellar_ALPHA", "rotation.stellar_BETA",        # spin-axis orientation (misalignment fallback)
        "rotation.gas_ALPHA", "rotation.gas_BETA",
        "velocity_dispersions.stellar", "velocity_dispersions.gas",  # dispersion (pressure) support [km/s]
        "temperatures.mass_weighted",
    ],
    # halo_data feeds the §7 CGM diagnostics. Full HDF5 paths keep the output
    # keys unique vs the galaxy_data names (otherwise masses.gas would collide).
    # The robust read below drops any path this catalog build does not have.
    "halo_data": [
        "masses.total",
        "halo_data/dicts/masses.gas",
        "halo_data/dicts/masses.stellar",
        "halo_data/dicts/masses.HI",
        "halo_data/dicts/masses.H2",
        "halo_data/dicts/masses.dust",
        "halo_data/dicts/metallicities.mass_weighted",
        "halo_data/dicts/temperatures.mass_weighted_cgm",
        "halo_data/dicts/virial_quantities.temperature",
        "halo_data/dicts/virial_quantities.m200c",
        "halo_data/dicts/virial_quantities.spin_param",
    ],
}

if BUILD_HISTORY:
    cs0 = sim.load_catalog(snap=START_SNAP)
    hist = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
    with fits.open(hist.progen_file) as hdul:
        valid_ids = np.asarray(hdul[1].data["GroupID"])
        if PRESELECT_BUILD:
            # pre-select the build sample at the z~0 anchor (START_SNAP): massive + fully quenched.
            # Mirrors passive_sample_mask (§1z) minus the ngas reliability cut, so gas-poor quenched
            # galaxies are NOT dropped from the build. Keep MASS_FLOOR / PASSIVE_FACTOR in sync with §0.
            _tH0_yr = COSMO.age(float(sim.get_z_from_snap(START_SNAP))).value * 1e9   # cosmic time at z~0 [yr]
            _gid0 = np.array([g.GroupID for g in cs0.galaxies])
            _ms0  = np.array([float(g.masses["stellar"]) for g in cs0.galaxies])    # M* [Msun]
            _sfr0 = np.array([float(g.sfr) for g in cs0.galaxies])                  # SFR [Msun/yr]
            with np.errstate(all="ignore"):
                _ssfr0  = np.where(_ms0 > 0, _sfr0 / _ms0, np.nan)                  # sSFR [1/yr]
                _presel = ((np.log10(np.where(_ms0 > 0, _ms0, np.nan)) >= MASS_FLOOR)
                           & (_ssfr0 < PASSIVE_FACTOR / _tH0_yr))                  # massive & quenched
            _keep = {int(g) for g in _gid0[_presel]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"[pre-select] massive (logM*>={MASS_FLOOR}) + quenched "
                  f"(sSFR < {PASSIVE_FACTOR}/t_H = {PASSIVE_FACTOR / _tH0_yr:.2e}/yr) at "
                  f"z~{float(sim.get_z_from_snap(START_SNAP)):.3f}: "
                  f"{len(valid_ids)}/{len(_gid0)} galaxies selected for the history build")
            del _gid0, _ms0, _sfr0, _ssfr0, _presel, _keep
    hist.get_history_indx(valid_ids, START_SNAP, END_SNAP)
    # robust read: drop any property the catalog build doesn't have (e.g. a missing
    # masses.HI or radii.gas_half_mass) and retry, instead of failing the whole build.
    props_try = {k: list(v) for k, v in PROPS.items()}
    while True:
        try:
            z_dict, props = hist.get_property_history(props_try, verbose=1)
            break
        except KeyError as e:
            msg = str(e); dropped = False
            for fam, plist in props_try.items():
                for pr in list(plist):
                    if pr in msg or pr.split("/")[-1] in msg:
                        plist.remove(pr); print("  [history] dropping missing property:", pr); dropped = True
            if not dropped:
                raise
    hist.save_history_to_hdf5(HIST_FILE)
    del cs0; gc.collect()
    print("history written to", HIST_PATH)

In [ ]:
# ── 1c. load the cached history ────────────────────────────────────────────
with h5py.File(HIST_PATH, "r") as f:
    galaxy_ids = f["metadata/galaxy_ids"][:]                 # (n_gal,) GroupID at START_SNAP
    snaps_arr  = f["metadata/snapshots"][:]                  # (n_snap,) snapshot numbers
    redshift   = f["redshift/Redshift"][:]                   # (n_snap,)
    # recurse: property names with "/" (the halo_data/dicts/... keys) are stored as
    # nested groups, so walk all datasets and key them by their full path under properties.
    P = {}
    f["properties"].visititems(
        lambda name, obj: P.__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)

# arrays are ordered from START_SNAP (row 0, ~z=0) down to END_SNAP
t_cosmic_yr = COSMO.age(redshift).value * 1e9                # (n_snap,) cosmic time [yr]

n_snap, n_gal = P["sfr"].shape
print(f"history: {n_snap} snapshots x {n_gal} galaxies")
print("row 0  -> snap", snaps_arr[0], f"z={redshift[0]:.4f}")
print("row -1 -> snap", snaps_arr[-1], f"z={redshift[-1]:.4f}")
print("properties:", list(P.keys()))

In [ ]:
# 5·pre — progenitor-index matrix aligned to the history (one group-catalogue load).
#   Trace between the HIGHEST and LOWEST snapshots actually present in the cached history (snaps_arr),
#   not §0's END_SNAP: the cache can predate a config change (e.g. END_SNAP 43->44) and this
#   matrix must line up row-for-row with snaps_arr (top AND bottom), else the vstack KeyErrors
#   on the missing snap (e.g. cache top snap 151 while START_SNAP=150).
cs0 = sim.load_catalog(snap=START_SNAP)
_hP = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
_hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
_PROG_INDEX = np.vstack([_hP.history_indx[str(s)] for s in snaps_arr])   # (n_snap, n_gal)
del cs0; gc.collect()
print("_PROG_INDEX:", _PROG_INDEX.shape)

In [ ]:
BH_HIST_PATH = os.path.join(SFHDIR, "bh_history_passive_z0.hdf5")
EPS_R = 0.1   # radiative/feedback efficiency for the AGN energy proxy

BH_CANDIDATES = {
    "bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
    "bh_mdot": ["bhmdot", "bh_mdot"],
    "bh_fedd": ["bh_fedd", "bhfedd", "fedd"],
}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

if BUILD_BH:
    cs0 = sim.load_catalog(snap=START_SNAP)
    _hb = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
    _hb.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    pidx = np.vstack([_hb.history_indx[str(s)] for s in snaps_arr])   # (n_snap, n_gal)
    del cs0; gc.collect()

    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    resolved = {}
    bh_skipped = []
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            bh_skipped.append(snap); continue        # leave NaN for this row
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri])
                cols_valid = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    resolved[k] = p
                    if p is not None:
                        BH[k][ri, cols_valid] = f[p][:][vi]
        except (OSError, KeyError) as e:
            # truncated/corrupt catalog -> record it, leave NaN row, keep going
            print(f"  [skip] snap {snap}: {type(e).__name__}: {e}")
            CORRUPT_SNAPS.add(snap); bh_skipped.append(snap); continue
    with h5py.File(BH_HIST_PATH, "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print("BH history written:", BH_HIST_PATH)
    print("resolved keys:", resolved)
    if bh_skipped:
        print(f"skipped {len(bh_skipped)} corrupt snapshot(s): {sorted(set(bh_skipped))}")
else:
    print("BUILD_BH=False -> expecting", BH_HIST_PATH)

## I.4 (§1z) · Multi-redshift selection anchors — progenitor tracks per selection epoch

The z=0-selected sample (§2) only contains **survivors**: every quenched galaxy that later
disappears by merging into a more massive system is invisible to a z=0 selection, so the
backtracked history is conditioned on survival. To remove that bias we repeat the whole
selection at several anchor redshifts (z = 0, 0.4, 0.7, 1, 2, 3, 4). For each anchor:

1. take the snapshot nearest the target redshift;
2. build the progenitor table and the property history back to ~`TRACK_AGE_FRAC` of the
   cosmic age at selection (clamped to the snapshots that actually have catalogs) —
   exactly mirroring §1a/§1b, so the table layout is identical to the z=0 product;
3. save one HDF5 table per anchor (`history_anchor_<tag>.hdf5` under `caesar_sfh/`), each
   stamped with its anchor metadata, so every population's history can be studied
   independently of this notebook's z=0 flow.

The **selection cuts live in one function**, `passive_sample_mask` (passive at the anchor
epoch: sSFR < `PASSIVE_FACTOR`/t_H(z_anchor), log M* > `MASS_FLOOR`, ngas ≥ `NGAS_MIN`):
§2 calls it for the z=0 flow and the §1z loader applies the same cuts at row 0 of every
anchor table. The z=0 anchor reuses the existing §1 products — nothing is rebuilt.

Build is gated by `BUILD_MULTI_Z` (heavy; cluster). The loader cell prints a per-anchor
summary (n_tracked, n_passive) for every table already on disk.


In [ ]:
# 1z·config — multi-redshift selection anchors (z=0 reuses the §1 z~0 products).
SELECTION_REDSHIFTS = [0.0, 0.4, 0.7, 1.0, 2.0, 3.0, 4.0]
BUILD_MULTI_Z   = False    # heavy: progenitor FITS + history HDF5 per anchor (cluster)
TRACK_AGE_FRAC  = 0.09     # track back to ~ this fraction of the cosmic age at selection
                           # (matches §1: the z=0 track spans 13.8 -> 1.2 Gyr ~ 9%)
ANCHOR_END_OVERRIDE = {}   # e.g. {4.0: 20} to force a specific end snapshot for one anchor

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

# selection cuts as ONE function: §2 (z=0) and every §1z anchor use this same definition.
def passive_sample_mask(P_, t_cosmic_yr_):
    """Passive + massive + gas-reliable cuts evaluated at row 0 (the anchor epoch).
    Returns (mask, dict of the individual cuts)."""
    mstar0 = P_["masses.stellar"][0]
    sfr0, ngas0 = P_["sfr"][0], P_["ngas"][0]
    ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
    cuts = {
        "passive":   ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr_[0]),
        "massive":   np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
        "enoughgas": ngas0 >= NGAS_MIN,
    }
    return cuts["passive"] & cuts["massive"] & cuts["enoughgas"], cuts

# snap <-> z <-> age grids over every snapshot the mapping knows about
_sall, _zall = [], []
for _s in range(0, START_SNAP + 1):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value                                   # [Gyr], ascending with snap

ANCHORS = {}
for _zt in SELECTION_REDSHIFTS:
    if _zt == 0.0:                                  # the §1 products ARE the z=0 anchor
        ANCHORS[_zt] = dict(tag=_ztag(_zt), snap=START_SNAP,
                            z=float(sim.get_z_from_snap(START_SNAP)), end_snap=END_SNAP,
                            prog_file=PROG_FILE, hist_path=HIST_PATH)
        continue
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(_sall[np.searchsorted(_aall, _age_end)])          # first snap older than the target age
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, _end))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(tag=_tag, snap=_snap, z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"))

print(f"{'target z':>9s} {'snap':>5s} {'z(snap)':>8s} {'end':>5s} {'z(end)':>8s}   history file")
for _zt, _A in ANCHORS.items():
    _have = "[ok]" if os.path.exists(_A["hist_path"]) else "[missing]"
    print(f"{_zt:9.1f} {_A['snap']:5d} {_A['z']:8.3f} {_A['end_snap']:5d} "
          f"{float(sim.get_z_from_snap(_A['end_snap'])):8.3f}   {_have} {os.path.basename(_A['hist_path'])}")


In [ ]:
# 1z·build — per-anchor progenitor table + property history (gated; cluster).
# Mirrors §1a/§1b exactly: the FITS keys the tracks, the history covers every FITS row,
# and the passive selection is applied at LOAD time with passive_sample_mask (as §2 does).
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if _zt == 0.0:
            print(f"[{A['tag']}] reuses the §1 products ({os.path.basename(A['hist_path'])}) — skip")
            continue
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {A['hist_path']}")
            continue
        # clamp the track end upward to the first snapshot that actually has a catalog
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        if end != A["end_snap"]:
            print(f"[{A['tag']}] end snap {A['end_snap']} -> {end} (first with a usable catalog)")
            A["end_snap"] = end

        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        ids_a = [g.GroupID for g in cs_a.galaxies]
        caesar_read_progen(ids_a, A["prog_file"], range(end, A["snap"] + 1), sim, output_dir=None)

        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            if PRESELECT_BUILD:
                # pre-select at THIS anchor epoch (A["snap"], z=A["z"]): massive + fully quenched.
                # Same cuts as §1b, evaluated at the anchor redshift (sSFR < PASSIVE_FACTOR/t_H(z)).
                _tHa_yr = COSMO.age(float(A["z"])).value * 1e9                       # cosmic time at anchor [yr]
                _gida = np.array([g.GroupID for g in cs_a.galaxies])
                _msa  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])   # M* [Msun]
                _sfra = np.array([float(g.sfr) for g in cs_a.galaxies])                 # SFR [Msun/yr]
                with np.errstate(all="ignore"):
                    _ssfra   = np.where(_msa > 0, _sfra / _msa, np.nan)                 # sSFR [1/yr]
                    _presela = ((np.log10(np.where(_msa > 0, _msa, np.nan)) >= MASS_FLOOR)
                                & (_ssfra < PASSIVE_FACTOR / _tHa_yr))                  # massive & quenched
                _keepa = {int(g) for g in _gida[_presela]}
                valid_ids = np.asarray([i for i in valid_ids if int(i) in _keepa], dtype=valid_ids.dtype)
                print(f"  [pre-select] {A['tag']}: massive (logM*>={MASS_FLOOR}) + quenched "
                      f"(sSFR < {PASSIVE_FACTOR}/t_H = {PASSIVE_FACTOR / _tHa_yr:.2e}/yr) at "
                      f"z={A['z']:.2f}: {len(valid_ids)}/{len(_gida)} galaxies for the history build")
                del _gida, _msa, _sfra, _ssfra, _presela, _keepa
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:                                  # same robust drop-and-retry as §1b
            try:
                z_dict, props = hist.get_property_history(props_try, verbose=0)
                break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [history] dropping missing property:", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        with h5py.File(A["hist_path"], "r+") as f:   # stamp anchor metadata -> self-describing table
            f["metadata"].attrs.update({
                "anchor_z_target": _zt, "anchor_snap": A["snap"], "anchor_z": A["z"],
                "end_snap": end, "passive_factor": PASSIVE_FACTOR,
                "mass_floor": MASS_FLOOR, "ngas_min": NGAS_MIN,
            })
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history written -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)


In [ ]:
# 1z·load — generic anchor loader + per-anchor passive sample (entry point for the
# independent per-epoch studies; this notebook's main flow stays on the z=0 anchor).
def load_anchor_history(ztarget):
    """Load one anchor's history table -> dict with keys
    galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P, anchor_z, anchor_snap.
    Row 0 of every property array is the anchor epoch (z ~ ztarget)."""
    A = ANCHORS[ztarget]
    H = {"anchor_z": A["z"], "anchor_snap": A["snap"], "P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

ANCHOR_SAMPLES = {}                                  # {z_target: {mask, gids, n}}
print(f"{'target z':>9s} {'snap':>5s} {'n_snap':>7s} {'n_tracked':>10s} {'n_passive':>10s}")
for _zt, _A in ANCHORS.items():
    if not os.path.exists(_A["hist_path"]):
        print(f"{_zt:9.1f} {_A['snap']:5d}    -- missing: set BUILD_MULTI_Z=True and run 1z·build --")
        continue
    _H = load_anchor_history(_zt)
    _smask, _ = passive_sample_mask(_H["P"], _H["t_cosmic_yr"])
    ANCHOR_SAMPLES[_zt] = dict(mask=_smask, gids=_H["galaxy_ids"][_smask], n=int(_smask.sum()))
    print(f"{_zt:9.1f} {_A['snap']:5d} {len(_H['snaps_arr']):7d} {len(_H['galaxy_ids']):10d} "
          f"{int(_smask.sum()):10d}")
    del _H; gc.collect()


# Part 1 — Sanity checks, sample & general statistics

Define the z=0 passive sample, then verify the machinery before any science: is the sample anchored at z≈0, is the progenitor chain spatially continuous, is each galaxy attached to a sensible parent halo? Then the critical-point / fast–slow split and the headcount behind every later facet.

## II.1 (§2) · Define the z=0 passive, massive sample

Selection uses the first time-row of the history (so indices stay aligned with the
tracks). Passive ≡ $\mathrm{sSFR} < 0.2/t_H$ at that epoch; massive ≡
$\log_{10}M_\star > 9.5$ (= `MASS_FLOOR`; the 10.5–11 and $\geq$11 bins isolate the massive subset); plus a gas-particle floor so later particle work is meaningful.

In [ ]:
mstar0 = P["masses.stellar"][0]
sfr0   = P["sfr"][0]
ngas0  = P["ngas"][0]
ssfr0  = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)

tH0 = t_cosmic_yr[0]                             # cosmic time at the z~0 anchor [yr]
# selection cuts: ONE definition (passive_sample_mask, §1z) shared with every anchor
sample_mask, _cuts = passive_sample_mask(P, t_cosmic_yr)
passive, massive, enoughgas = _cuts["passive"], _cuts["massive"], _cuts["enoughgas"]
sample_cols = np.where(sample_mask)[0]           # column indices into the history arrays
sample_gids = galaxy_ids[sample_cols]            # GroupIDs at START_SNAP

print(f"passive            : {np.nansum(passive)}")
print(f"+ logM* > {MASS_FLOOR}     : {np.nansum(passive & massive)}")
print(f"+ ngas >= {NGAS_MIN}      : {sample_mask.sum()}  <- final sample")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(np.log10(mstar0), np.log10(ssfr0), s=4, c="0.7", label="all z~0")
ax[0].scatter(np.log10(mstar0[sample_cols]), np.log10(ssfr0[sample_cols]), s=8, c="C3", label="sample")
ax[0].axhline(np.log10(PASSIVE_FACTOR/tH0), ls="--", c="k")
for _e in MASS_BIN_EDGES[:-1]:
    ax[0].axvline(_e, ls=":", c="k", lw=1)        # mass-bin boundaries (9.5, 10.5, 11)
ax[0].set_xlabel(r"$\log_{10} M_\star$"); ax[0].set_ylabel(r"$\log_{10}$ sSFR [yr$^{-1}$]"); ax[0].legend()
dts = P["masses.dust"][0] / np.where(mstar0 > 0, mstar0, np.nan)
ax[1].hist(np.log10(dts[sample_cols][dts[sample_cols] > 0]), bins=25, color="C3", alpha=0.8)
ax[1].set_xlabel(r"$\log_{10}(M_{\rm dust}/M_\star)$ at z$\approx$0"); ax[1].set_ylabel("N")
ax[1].set_title("residual dust in the passive sample")
plt.tight_layout(); plt.show()

### II.1·diag (§2·diag) · Sample-anchor sanity check

Confirms the tracked z=0 sample is actually anchored at **START_SNAP (z≈0)** and not, via `caesar_read_progen`'s `np.sort(snaplist)[0]` base, at **END_SNAP (z≈4.8)** — which would silently restrict the sample to high-z survivors (massive only, low-mass end missing). Reports the FITS base anchor, whether `galaxy_ids` line up with the z≈0 catalogue, and overlays the tracked-sample stellar-mass function on the full z≈0 catalogue. Read-only.

In [ ]:
# §2·diag — IS THE TRACKED SAMPLE ANCHORED AT z~0 (START_SNAP) OR MISTAKENLY AT END_SNAP?
#   caesar_read_progen keys the progenitor FITS at np.sort(snaplist)[0] = END_SNAP (z~4.8),
#   while get_history_indx treats those rows as the START_SNAP sample. If so, the sample is
#   the set of galaxies that already existed at high z -> biased massive, low-mass end missing.
#   This cell is read-only and self-contained (no _gfield2 / build needed).
def _ms_at(snap):
    """stellar-mass array at a snapshot's catalogue (None if unreadable)."""
    try:
        with h5py.File(sim.get_caesar_file(int(snap)), "r") as f:
            for p in ("galaxy_data/dicts/masses.stellar", "galaxy_data/masses.stellar"):
                if p in f:
                    return np.asarray(f[p][:], dtype=np.float64)
    except (OSError, KeyError):
        pass
    return None

_ms_start, _ms_end = _ms_at(START_SNAP), _ms_at(END_SNAP)
_Ns = len(_ms_start) if _ms_start is not None else -1
_Ne = len(_ms_end)   if _ms_end   is not None else -1
_gid = np.asarray(galaxy_ids, dtype=np.int64)
print(f"row 0 of the history is z = {redshift[0]:.3f}  (snap {int(snaps_arr[0])})  -> should be ~0")
print(f"galaxies @START_SNAP {START_SNAP} (z~{sim.get_z_from_snap(START_SNAP):.2f}): {_Ns}")
print(f"galaxies @END_SNAP   {END_SNAP} (z~{sim.get_z_from_snap(END_SNAP):.2f}): {_Ne}")
print(f"tracked galaxy_ids   : N={len(_gid)}  min={_gid.min()}  max={_gid.max()}")

_progf = os.path.join(OUT, "progenitors", PROG_FILE)
if os.path.exists(_progf):
    with fits.open(_progf) as _h:
        _fg = np.asarray(_h[1].data["GroupID"], dtype=np.int64)
    print(f"progenitor FITS      : {_progf}")
    print(f"   rows={len(_fg)}  max GroupID={_fg.max()}")
    # GroupID is the 0-based galaxy index at the FITS base snapshot, so max+1 ~ Ngal there.
    if _Ns > 0 and _Ne > 0:
        d_start, d_end = abs((_fg.max() + 1) - _Ns), abs((_fg.max() + 1) - _Ne)
        anchor = "START_SNAP (z~0)" if d_start < d_end else "END_SNAP (z~4.8)"
        verdict = "OK" if d_start < d_end else "BUG -> sample = high-z survivors (massive only, low-mass missing)"
        print(f">>> FITS base looks anchored at {anchor}  [{verdict}]")
        print(f"    (max GroupID+1={_fg.max()+1} vs N_start={_Ns}, N_end={_Ne})")
else:
    print(f"progenitor FITS not found at {_progf} (skipping anchor test)")

# decisive cross-check: do the tracked sample's z~0 masses match the START_SNAP catalogue
# at the SAME galaxy_ids? if get_history_indx mis-anchored, these will NOT line up.
if _ms_start is not None:
    _ok = (_gid >= 0) & (_gid < _Ns)
    with np.errstate(all="ignore"):
        _rel = np.abs(P["masses.stellar"][0][_ok] - _ms_start[_gid[_ok]]) / np.where(
            _ms_start[_gid[_ok]] > 0, _ms_start[_gid[_ok]], np.nan)
    _md = float(np.nanmedian(_rel))
    print(f"row-0 mass vs START_SNAP catalogue @ galaxy_ids: median rel.diff = {_md:.2e}  "
          f"-> {'ALIGNED (galaxy_ids ARE z~0 indices)' if _md < 1e-3 else 'MISALIGNED (galaxy_ids are NOT z~0 indices!)'}")

# visual: tracked-sample completeness vs the full z~0 catalogue mass function
with np.errstate(all="ignore"):
    _lm_samp = np.log10(P["masses.stellar"][0][P["masses.stellar"][0] > 0])
    _lm_cat  = np.log10(_ms_start[_ms_start > 0]) if _ms_start is not None else None
fig, axd = plt.subplots(figsize=(7.2, 4.6))
_bb = np.linspace(8.5, 12.5, 41)
if _lm_cat is not None:
    axd.hist(_lm_cat, bins=_bb, histtype="step", lw=2.2, color="0.5",
             label=f"all @snap {START_SNAP} (N={len(_lm_cat)})")
axd.hist(_lm_samp, bins=_bb, histtype="stepfilled", alpha=0.5, color="C3",
         label=f"tracked history (N={len(_lm_samp)})")
for _e in MASS_BIN_EDGES[:-1]:
    axd.axvline(_e, ls=":", c="k", lw=1)
axd.axvline(MASS_FLOOR, ls="--", c="C0", lw=1.5, label=f"MASS_FLOOR={MASS_FLOOR}")
axd.set_xlabel(r"$\log_{10} M_\star$ at $z\approx0$"); axd.set_ylabel("N"); axd.set_yscale("log")
axd.set_title("tracked-sample completeness vs the full $z\\approx0$ catalogue")
axd.legend(fontsize=10)
plt.tight_layout()
plt.savefig(figpath("diag_sample_completeness.png"), dpi=130, bbox_inches="tight")
plt.show()


### II.1·diag2 (§2·diag2) · Backward-chain sanity check — spatial continuity

Validates the progenitor tracking by **spatial continuity** (the strongest test): a real galaxy moves continuously through the periodic box, so its minimum-image step between adjacent snapshots is a tiny fraction of the box; a broken link teleports onto an unrelated galaxy → box-scale jumps. Positions are taken to **comoving** coordinates — physical-vs-comoving is detected from whether the *full-catalogue* box scales with $a(z)$, and the box length comes from the full catalogue (the tracked subsample clusters at high $z$ and under-fills the box). Plots are honest: the step-size distribution and median/90th-pct step **vs redshift** (a spike at particular snapshots points to a tree/catalogue problem there; a flat low level means clean tracking). No unwrapped trajectories — unwrapping would force even a teleport to look continuous. Saved to `00_diagnostics/`. Read-only.

In [ ]:
# §2·diag2 — CHAIN SANITY via SPATIAL continuity. A real galaxy moves continuously through the
#   box; a broken progenitor link teleports onto an unrelated galaxy. We work in COMOVING
#   coordinates: positions are detected as physical vs comoving (does the FULL-catalogue box
#   scale with a(z)?) and converted if needed, and the box length is taken from the full
#   catalogue (the tracked subsample clusters at high z and under-fills the box, so its own
#   extent is not a safe box estimate). Read-only.
def _full_box(snap):
    """per-dim extent (max-min) of ALL galaxy positions at `snap`, or NaN."""
    try:
        with h5py.File(sim.get_caesar_file(int(snap)), "r") as f:
            for p in ("galaxy_data/pos", "galaxy_data/dicts/pos"):
                if p in f:
                    a = np.asarray(f[p][:], dtype=np.float64)
                    return float(np.nanmax(a) - np.nanmin(a))
    except (OSError, KeyError):
        pass
    return np.nan

if "pos" not in P:
    print("history has no 'pos' field -> spatial chain check unavailable (add 'pos' to PROPS).")
else:
    _pos = np.asarray(P["pos"], dtype=np.float64)
    if _pos.ndim != 3 or _pos.shape[-1] != 3:
        print(f"unexpected pos shape {_pos.shape}; expected (n_snap, n_gal, 3) -> skipping.")
    else:
        _z = np.asarray(redshift, dtype=np.float64)
        _bs, _be = _full_box(snaps_arr[0]), _full_box(snaps_arr[-1])
        _as, _ae = 1.0 / (1.0 + _z[0]), 1.0 / (1.0 + _z[-1])
        # physical -> full-box scales with a(z); comoving -> roughly constant box
        _phys = (np.isfinite(_bs) and np.isfinite(_be)
                 and abs(_be / _bs - _ae / _as) < abs(_be / _bs - 1.0))
        if _phys:
            _posc = _pos * (1.0 + _z)[:, None, None]          # physical -> comoving
            L = _bs / _as if np.isfinite(_bs) else np.nanpercentile(_posc[np.isfinite(_posc)], 99.99)
            print(f"positions look PHYSICAL: full box {snaps_arr[0]}->{snaps_arr[-1]} = "
                  f"{_bs:.4g}->{_be:.4g} (ratio {_be/_bs:.2f} ~ a-ratio {_ae/_as:.2f}); "
                  f"converted to comoving via x(1+z).")
        else:
            _posc = _pos
            L = _bs if np.isfinite(_bs) else np.nanpercentile(_posc[np.isfinite(_posc)], 99.99)
            print(f"positions look COMOVING: full box {snaps_arr[0]}->{snaps_arr[-1]} = "
                  f"{_bs:.4g}->{_be:.4g} (ratio {_be/_bs if np.isfinite(_be/_bs) else float('nan'):.2f}).")
        print(f"comoving box length used: L = {L:.4g} (pos units)")

        _d = np.diff(_posc, axis=0); _d -= L * np.round(_d / L)   # min-image in comoving coords
        _disp = np.sqrt(np.sum(_d ** 2, axis=2)) / L              # step as a FRACTION of the box
        # normalise by the time between adjacent history ROWS: a skipped/corrupt snapshot makes
        # two rows span a larger dt, so a real galaxy moves farther -> a spike that is NOT a
        # broken link. dividing by dt gives a gap-immune "comoving speed" (box-fraction / Gyr).
        _t   = np.asarray(t_cosmic_yr, dtype=np.float64)
        _dt  = np.abs(np.diff(_t)) / 1e9                          # Gyr between adjacent rows
        _dsn = np.abs(np.diff(np.asarray(snaps_arr, dtype=np.int64)))
        _speed = _disp / np.where(_dt > 0, _dt, np.nan)[:, None]  # box-fraction per Gyr
        _fin = np.isfinite(_disp); _v = _disp[_fin]
        _sv = _speed[np.isfinite(_speed)]
        JUMP_FRAC = 0.05                                          # step >5% of box ~ teleport
        SPEED_HI  = 0.05                                          # box-fraction / Gyr deemed implausible
        _med = float(np.median(_v)) if _v.size else np.nan
        _p99 = float(np.percentile(_v, 99)) if _v.size else np.nan
        _fbig = float(np.mean(_v > JUMP_FRAC)) if _v.size else np.nan
        _fspd = float(np.mean(_sv > SPEED_HI)) if _sv.size else np.nan
        print("spatial continuity (min-image step / box):")
        print(f"  median step={_med:.2e} | 99th pct={_p99:.2e} | steps > {JUMP_FRAC:.0%} box = {_fbig:.2%}")
        print(f"  dt-normalised: steps > {SPEED_HI:g} box/Gyr = {_fspd:.2%}  (gap-immune)")
        with np.errstate(all="ignore"):
            _galbad = np.nansum(_speed > SPEED_HI, axis=0) / np.maximum(np.sum(np.isfinite(_speed), axis=0), 1)
        _tracked = np.sum(_fin, axis=0) >= 3
        _broken = _tracked & (_galbad > 0.2)
        print(f"  galaxies judged (>=3 valid steps): {int(_tracked.sum())}/{_pos.shape[1]} | "
              f"suspect tracks (>20% over-speed steps): {int(_broken.sum())} "
              f"({(np.mean(_broken[_tracked]) if _tracked.any() else 0.0):.1%})")

        # per-transition table: locate the spike and show whether it sits on a snapshot GAP
        _p90s = np.array([np.nanpercentile(_disp[k][np.isfinite(_disp[k])], 90)
                          if np.isfinite(_disp[k]).any() else np.nan for k in range(_disp.shape[0])])
        _p90v = np.array([np.nanpercentile(_speed[k][np.isfinite(_speed[k])], 90)
                          if np.isfinite(_speed[k]).any() else np.nan for k in range(_speed.shape[0])])
        _ngap = int(np.sum(_dsn > 1))
        print(f"snapshot gaps in the history (dsnap>1): {_ngap}"
              + (f" -> {[(int(snaps_arr[k]), int(snaps_arr[k+1])) for k in np.where(_dsn>1)[0]][:10]}" if _ngap else ""))
        print("worst transitions by 90th-pct step (a GAP inflates step but not step/Gyr):")
        print(f"  {'snap':>11s} {'dsnap':>5s} {'dt[Gyr]':>8s} {'90th step':>10s} {'90th/Gyr':>10s}")
        for k in sorted(np.argsort(-np.nan_to_num(_p90s))[:6]):
            tag = "  <-GAP" if _dsn[k] > 1 else ("  <-spike" if _p90v[k] > SPEED_HI else "")
            print(f"  {int(snaps_arr[k]):>4d}->{int(snaps_arr[k+1]):<4d} {_dsn[k]:>5d} {_dt[k]:>8.3f} "
                  f"{_p90s[k]:>10.2e} {_p90v[k]:>10.2e}{tag}")
        print(">>> " + ("CHAIN OK (continuous once dt-normalised)" if (_fspd < 0.02 and np.nanmedian(_p90v) < SPEED_HI)
                        else "CHAIN SUSPECT -> over-speed steps remain after dt-normalisation: "
                             "real broken links at the flagged snapshot(s)"))

        # honest plots: (a) step/box distribution; (b) 90th-pct step AND step/Gyr vs z, gaps marked
        _zmid = 0.5 * (_z[:-1] + _z[1:])
        fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.8))
        if _v.size:
            axa.hist(np.log10(np.clip(_v, 1e-6, None)), bins=50, color="C0", alpha=0.85)
        axa.axvline(np.log10(JUMP_FRAC), ls="--", c="k", lw=1.5, label=f"teleport threshold {JUMP_FRAC:.0%}")
        axa.set_xlabel(r"$\log_{10}$(step / box)"); axa.set_ylabel("N steps"); axa.set_yscale("log")
        axa.set_title("per-step displacement distribution"); axa.legend()
        axb.plot(_zmid, _p90s, "-s", ms=4, color="C1", label="90th step/box")
        axb.plot(_zmid, _p90v, "-o", ms=4, color="C0", label="90th step/box per Gyr")
        for k in np.where(_dsn > 1)[0]:
            axb.axvline(_zmid[k], color="0.7", lw=1.0, zorder=0)
        axb.axhline(SPEED_HI, ls="--", c="k", lw=1.0)
        axb.set_yscale("log"); axb.set_xlabel("redshift (between adjacent snapshots)")
        axb.set_ylabel("90th-pct displacement"); axb.set_title("continuity vs epoch (grey = snapshot gap)")
        axb.legend(fontsize=9); axb.invert_xaxis()
        plt.tight_layout()
        plt.savefig(figpath("diag_chain_spatial.png"), dpi=130, bbox_inches="tight"); plt.show()
        print("Read: if the step spike sits on a grey GAP line and the step/Gyr curve stays flat,")
        print("      it is just a larger time step (no broken link). A step/Gyr spike = real problem.")


### II.1·diag3 (§2·diag3) · Spot-check the corrected progenitor chain

Before committing to a full cache rebuild, this validates the **proposed `get_history_indx` rewrite** on a small sample: it re-derives the most-massive-progenitor chain directly from `tree_data/progen_galaxy_star` (catalog index → its progenitor index at the previous snapshot) for ~10 currently-broken + ~10 currently-good galaxies, reads their **raw** positions at each snapshot, and checks the corrected chain is spatially continuous where the cached one teleports. If the cached-broken galaxies come out continuous (`FIXED`), the rewrite is the right fix. Read-only — reads caesar files, does **not** rebuild the cache.

In [ ]:
# §2·diag3 — SPOT-CHECK the corrected progenitor chain BEFORE any rebuild. Re-derives the
#   most-massive-progenitor chain directly from tree_data/progen_galaxy_star (the algorithm
#   proposed for get_history_indx) for a few currently-broken + currently-good galaxies, then
#   checks the corrected chain is spatially continuous where the cached one teleports.
#   Read-only: reads caesar files, does NOT rebuild the cache.
def _gd(snap, paths):
    with h5py.File(sim.get_caesar_file(int(snap)), "r") as f:
        for p in paths:
            if p in f:
                return np.asarray(f[p][:])
    return None

_POS = ("galaxy_data/pos", "galaxy_data/dicts/pos")
_p0 = _gd(START_SNAP, _POS)
if _p0 is None or "pos" not in P:
    print("need galaxy positions in the catalogue and history 'pos' field -> skip spot-check.")
else:
    Lc = float(np.nanmax(_p0) - np.nanmin(_p0))            # comoving box from the full catalogue
    _pc = np.asarray(P["pos"], float)
    _dd = np.diff(_pc, axis=0); _dd -= Lc * np.round(_dd / Lc)
    _cached_max = np.nanmax(np.sqrt(np.sum(_dd**2, axis=2)) / Lc, axis=0)   # per-galaxy worst step
    _bro = np.where(_cached_max > 0.05)[0]
    _gud = np.where(_cached_max < 5e-3)[0]
    _rng = np.random.RandomState(0)
    _sel = np.concatenate([_rng.choice(_bro, min(10, len(_bro)), replace=False),
                           _rng.choice(_gud, min(10, len(_gud)), replace=False)]) if len(_bro) else _gud[:20]
    _ids0 = np.asarray(galaxy_ids, np.int64)[_sel]          # catalog indices @ START_SNAP

    _snaps = list(range(int(START_SNAP), int(np.min(snaps_arr)) - 1, -1))
    _cur = _ids0.astype(float)
    _cpos = np.full((len(_snaps), len(_sel), 3), np.nan)
    for i, s in enumerate(_snaps):
        _ps = _gd(s, _POS)
        if _ps is None:
            break
        _ps = _ps.astype(float)
        _ok = np.isfinite(_cur); _ii = _cur[_ok].astype(np.int64)
        _vv = (_ii >= 0) & (_ii < len(_ps)); _row = np.where(_ok)[0][_vv]
        _cpos[i, _row, :] = _ps[_ii[_vv]]
        if i < len(_snaps) - 1:                            # advance: progen@s -> index@(s-1)
            _pg = _gd(s, ("tree_data/progen_galaxy_star",))
            if _pg is None:
                break
            _pg = np.asarray(_pg)[:, 0].astype(np.int64)
            _nx = np.full(len(_sel), np.nan)
            _nx[_row] = np.where(_pg[_ii[_vv]] >= 0, _pg[_ii[_vv]], np.nan)
            _cur = _nx
    _dc = np.diff(_cpos, axis=0); _dc -= Lc * np.round(_dc / Lc)
    _corr_max = np.nanmax(np.sqrt(np.sum(_dc**2, axis=2)) / Lc, axis=0)

    print(f"spot-check {len(_sel)} galaxies (comoving box L={Lc:.4g}); max step / box:")
    print(f"  {'col':>7s} {'GID@z0':>7s} {'cached':>10s} {'corrected':>10s}  verdict")
    for j in range(len(_sel)):
        cmx, km = _cached_max[_sel[j]], _corr_max[j]
        v = ("FIXED" if (cmx > 0.05 and km < 0.05) else
             "ok" if cmx <= 0.05 else "STILL-BAD")
        print(f"  {int(_sel[j]):>7d} {int(_ids0[j]):>7d} {cmx:>10.2e} {km:>10.2e}  {v}")
    _nb = int(np.sum(_cached_max[_sel] > 0.05))
    _nf = int(np.sum((_cached_max[_sel] > 0.05) & (_corr_max < 0.05)))
    print(f">>> {_nf}/{_nb} cached-broken galaxies become spatially continuous with the corrected "
          f"chain -> {'FIX CONFIRMED' if (_nb > 0 and _nf == _nb) else 'inspect remaining'}")


## II.2 (§3) · Per-galaxy critical evolutionary points & fast/slow classification

For each sampled galaxy we run `find_quenching_times` on its sSFR(t) track to get
**SFT**, **QT**, and the **persistence end** (post-quench). When several quench
events exist we keep the **latest** one (the one that leaves the galaxy passive at
z=0). We add **SF-peak** (max sSFR), **dust-peak**, and **dust-min** (minimum dust
before the track first drops to ~0). Each critical *time* is mapped to its nearest
snapshot row, and we record the galaxy's CAESAR index there for later particle work.

$\tau_q = t_{QT}-t_{SFT}$ → **fast** if $\log_{10}(\tau_q/t_H(z_{QT})) < -1.5$
(quenched in $\lesssim 3\%$ of the age of the universe at QT; `TAU_SPLIT_LOG` in §0), else **slow**.

In [ ]:
STAGES = ["sf_peak", "ssfr_min", "sft", "qt", "post_quench",
          "dust_peak", "dust_min", "gas_min", "z0"]

# Critical points are taken on *smoothed* tracks so that a single-snapshot spike
# can no longer masquerade as a global peak/trough. Minima (SFR/dust/gas) are
# restricted to the trough *before* the track collapses to its floor, so they
# capture the real dip rather than the trivial quenched/depleted floor near z=0.
SMOOTH_W = 3      # centered moving-median window (# snapshots) for extremum tracks

def _nearest_row(t_target):
    if not np.isfinite(t_target):
        return -1
    return int(np.argmin(np.abs(t_cosmic_yr - t_target)))

def _smooth(y, w=SMOOTH_W):
    '''Centered, NaN-aware moving median of a 1-D array (odd window in # samples).'''
    n = len(y); h = w // 2
    out = np.full(n, np.nan)
    for i in range(n):
        seg = y[max(0, i - h):min(n, i + h + 1)]
        seg = seg[np.isfinite(seg)]
        if seg.size:
            out[i] = np.median(seg)
    return out

def _ordered_positive(q, valid):
    '''Forward-in-time indices where (valid & q>0) plus the smoothed positive track.'''
    d = np.where(valid, q, np.nan).astype(float)
    pos = np.isfinite(d) & (d > 0)
    if pos.sum() < 2:
        return None, None
    order = np.where(pos)[0]
    order = order[np.argsort(t_cosmic_yr[order])]
    return order, _smooth(d[order])

def _peak_time(q, valid):
    '''Cosmic time of the GLOBAL maximum of the smoothed positive track
       (robust to local/noisy maxima).'''
    order, ds = _ordered_positive(q, valid)
    if order is None or not np.isfinite(ds).any():
        return np.nan
    return t_cosmic_yr[order[np.nanargmax(ds)]]

def _trough_before_floor(q, valid, floor_frac=1e-6):
    '''Cosmic time of the minimum of the smoothed track, restricted to before it
       first collapses to ~0 (its floor). Generalizes the old dust-min logic so the
       same definition serves dust, SFR and gas minima.'''
    order, ds = _ordered_positive(q, valid)
    if order is None or not np.isfinite(ds).any():
        return np.nan
    floor = floor_frac * np.nanmax(ds)
    below = np.isfinite(ds) & (ds <= floor)
    cut = int(np.argmax(below)) if below.any() else len(ds)   # first collapse to floor
    cut = max(cut, 2)
    seg = ds[:cut]
    if not np.isfinite(seg).any():
        return np.nan
    return t_cosmic_yr[order[np.nanargmin(seg)]]

def track_start_yr(col):
    '''Earliest cosmic time [yr] with a valid star-forming track (finite, positive sSFR)
       — the same 'valid' definition used for the anchors below; NaN if never valid.'''
    mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
    ssfr  = np.where(mstar > 0, sfr / mstar, np.nan)
    valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
    return float(np.nanmin(t_cosmic_yr[valid])) if valid.any() else np.nan

records = []
for col, gid in zip(sample_cols, sample_gids):
    mstar = P["masses.stellar"][:, col]
    sfr   = P["sfr"][:, col]
    dust  = P["masses.dust"][:, col]
    gas   = P["masses.gas"][:, col]
    ssfr  = np.where(mstar > 0, sfr / mstar, np.nan)
    valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
    if valid.sum() < 5:
        continue

    t = t_cosmic_yr[valid]; s = ssfr[valid]
    o = np.argsort(t); t, s = t[o], s[o]
    tu, ui = np.unique(t, return_index=True); su = s[ui]
    if len(tu) < 5:
        continue

    qts, sfts, _, dbg = find_quenching_times(
        tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True,
    )
    if len(qts) == 0:
        continue
    k = int(np.argmax(qts))                          # latest quench event -> the z=0 one
    t_qt, t_sft = qts[k], sfts[k]
    t_postq = dbg["persistence_end_times"][k]
    kf = int(np.argmin(qts))                          # earliest quench event -> main/initial quenching
    t_qt_first, t_sft_first = qts[kf], sfts[kf]

    # smoothed-track anchors (robust to single-snapshot spikes; see note above)
    t_sfpeak = _peak_time(ssfr, valid)                  # max specific SFR  (sSFR peak)
    t_ssfrmin = _trough_before_floor(ssfr, valid)        # sSFR trough before quench floor
    t_dpeak  = _peak_time(dust, valid)                  # max dust mass
    t_dmin   = _trough_before_floor(dust, valid)        # dust trough before collapse
    t_gasmin = _trough_before_floor(gas, valid)         # gas trough before depletion
    t_z0     = t_cosmic_yr[0]

    t_times = {"sf_peak": t_sfpeak, "ssfr_min": t_ssfrmin, "sft": t_sft, "qt": t_qt,
               "post_quench": t_postq, "dust_peak": t_dpeak, "dust_min": t_dmin,
               "gas_min": t_gasmin, "z0": t_z0}
    rows = {st: _nearest_row(t_times[st]) for st in STAGES}

    tau_q = t_qt - t_sft
    z_qt  = float(np.interp(t_qt, t_cosmic_yr[::-1], redshift[::-1]))
    tH_qt = COSMO.age(z_qt).value * 1e9
    records.append(dict(
        gid=int(gid), col=int(col),
        t_sf_peak=t_sfpeak, t_ssfr_min=t_ssfrmin, t_sft=t_sft, t_qt=t_qt, t_post_quench=t_postq,
        t_dust_peak=t_dpeak, t_dust_min=t_dmin, t_gas_min=t_gasmin, t_z0=t_z0,
        tau_q=tau_q, tau_q_over_tH=tau_q/tH_qt, z_qt=z_qt,
        t_qt_first=t_qt_first, t_sft_first=t_sft_first,
        **{f"row_{st}": rows[st] for st in STAGES},
    ))

print(f"galaxies with a usable quench event: {len(records)} / {len(sample_cols)}")

In [ ]:
# assemble a structured table of critical points + class
gids   = np.array([r["gid"] for r in records])
cols   = np.array([r["col"] for r in records])
tau_q  = np.array([r["tau_q"] for r in records])
tau_n  = np.array([r["tau_q_over_tH"] for r in records])
# ── build the active split (configured in §0 via SPLIT_MODE) ────────────────
# Group A == is_fast (name kept for continuity), group B == is_slow. Galaxies whose
# splitting value is undefined fall into NEITHER group (not lumped into B).
def _split_value(prop, stage):
    """Per-galaxy value of `prop` at evolutionary `stage` over `records` (NaN if missing).
       Mirrors the §4 DIAGS recipes but is available here, before DIAGS is built."""
    rws = np.array([r[f"row_{stage}"] for r in records])
    def _col(name):
        out = np.full(len(records), np.nan); a = P[name]
        for i, (rw, cl) in enumerate(zip(rws, cols)):
            if rw >= 0:
                out[i] = a[rw, cl]
        return out
    def _ratio(n, d):
        a, b = _col(n), _col(d); return np.where(b > 0, a / b, np.nan)
    recipe = {
        "Mdust": ("masses.dust",), "MH2": ("masses.H2",), "MHI": ("masses.HI",),
        "Mstar": ("masses.stellar",), "Mgas": ("masses.gas",), "Rgas": ("radii.gas_half_mass",),
        "sSFR": ("sfr", "masses.stellar"), "dust/M*": ("masses.dust", "masses.stellar"),
        "H2/M*": ("masses.H2", "masses.stellar"), "HI/M*": ("masses.HI", "masses.stellar"),
        "dust/gas": ("masses.dust", "masses.gas"), "HI/H2": ("masses.HI", "masses.H2"),
        "Rgas/Rstar": ("radii.gas_half_mass", "radii.stellar_half_mass"),
    }
    if prop not in recipe:
        raise KeyError(f"SPLIT_PROP {prop!r} unknown; choose one of {list(recipe)}")
    spec = recipe[prop]
    return _col(*spec) if len(spec) == 1 else _ratio(*spec)

def build_split():
    """Return (is_A, is_B, label_A, label_B, color_A, color_B, tag) for the configured split."""
    if SPLIT_MODE == "fast_slow":
        a = np.log10(tau_n) < TAU_SPLIT_LOG
        b = np.isfinite(tau_n) & ~a
        tag = f"fast_slow_tau{TAU_SPLIT_LOG:g}".replace("-", "m").replace(".", "p")
        return a, b, "fast", "slow", "C3", "C0", tag
    if SPLIT_MODE == "property":
        val = _split_value(SPLIT_PROP, SPLIT_STAGE)
        v = np.log10(np.where(val > 0, val, np.nan)) if SPLIT_LOG else np.asarray(val, float)
        defi = np.isfinite(v)
        thr = float(np.nanmedian(v)) if SPLIT_THRESH_MODE == "median" else float(SPLIT_THRESH)
        a = defi & (v > thr); b = defi & ~(v > thr)
        op = "log " if SPLIT_LOG else ""
        la = f"{SPLIT_LABELS[0]} ({op}{SPLIT_PROP}>{thr:.2g}@{SPLIT_STAGE})"
        lb = f"{SPLIT_LABELS[1]} (\u2264{thr:.2g})"
        safe = SPLIT_PROP.replace("/", "_").replace("*", "")
        tag = f"{safe}_{SPLIT_STAGE}_{SPLIT_THRESH_MODE}{thr:.2g}".replace("-", "m").replace(".", "p")
        return a, b, la, lb, SPLIT_COLORS[0], SPLIT_COLORS[1], tag
    raise ValueError(f"unknown SPLIT_MODE {SPLIT_MODE!r}")

is_fast, is_slow, LBL_A, LBL_B, COL_A, COL_B, SPLIT_TAG = build_split()
SPLIT_DESC = f"{LBL_A} vs {LBL_B}"
classes = np.where(is_fast, LBL_A, np.where(is_slow, LBL_B, "none"))

# route every saved figure into a per-split subfolder (folder name == which split)
PLOTDIR = os.path.join(OUT, "plots", "residual_dust", SPLIT_TAG)
os.makedirs(PLOTDIR, exist_ok=True)
print(f"SPLIT_MODE={SPLIT_MODE!r}  tag={SPLIT_TAG!r}")
print(f"  A [{LBL_A}]: {int(is_fast.sum())}   B [{LBL_B}]: {int(is_slow.sum())}"
      f"   undefined (neither): {len(records) - int(is_fast.sum()) - int(is_slow.sum())}")
print(f"  plots -> {PLOTDIR}")

# stellar-mass bin (at z=0) for every record -> used to facet ALL comparisons below
NBINS = len(MASS_BIN_LABELS)
logm_rec = np.log10(np.where(P["masses.stellar"][0, cols] > 0, P["masses.stellar"][0, cols], np.nan))
mbin = np.full(len(records), -1, dtype=int)
for _b in range(NBINS):
    inb = (logm_rec >= MASS_BIN_EDGES[_b]) & (logm_rec < MASS_BIN_EDGES[_b + 1])
    mbin[inb] = _b

print("mass bins (logM* at z=0):")
for _b in range(NBINS):
    bm = mbin == _b
    print(f"  {MASS_BIN_LABELS[_b]:9s}: {bm.sum():4d}  ({LBL_A} {np.sum(bm & is_fast)}, {LBL_B} {np.sum(bm & is_slow)})")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.log10(tau_q[tau_q > 0]), bins=20, color="C0", alpha=0.8)
ax[0].set_xlabel(r"$\log_{10}\,\tau_q$ [yr]"); ax[0].set_ylabel("N")
ax[1].hist(np.log10(tau_n[tau_n > 0]), bins=20, color="C1", alpha=0.8)
if SPLIT_MODE == "fast_slow":   # the fast/slow cut lives on log10(tau_q/t_H) -> THIS panel
    ax[1].axvline(TAU_SPLIT_LOG, ls="--", c="k",
                  label=fr"cut $\log_{{10}}(\tau_q/t_H)={TAU_SPLIT_LOG:g}$")
    ax[1].legend()
ax[1].set_xlabel(r"$\log_{10}\,(\tau_q / t_H(z_{QT}))$"); ax[1].set_ylabel("N")
plt.tight_layout(); plt.show()

In [ ]:
# Persist the critical-point catalogue for reuse / TOPCAT
def save_critical_table():
    names = (["GALAXY_ID", "COL", "CLASS", "TAU_Q_YR", "TAU_Q_OVER_TH", "Z_QT"]
             + [f"T_{s.upper()}" for s in STAGES]
             + [f"ROW_{s.upper()}" for s in STAGES])
    coldefs = [
        fits.Column(name="GALAXY_ID", format="K", array=gids),
        fits.Column(name="COL",       format="K", array=cols),
        fits.Column(name="CLASS",     format="8A", array=np.array(classes, dtype="S8")),
        fits.Column(name="TAU_Q_YR",  format="D", array=tau_q),
        fits.Column(name="TAU_Q_OVER_TH", format="D", array=tau_n),
        fits.Column(name="Z_QT",      format="D", array=np.array([r["z_qt"] for r in records])),
    ]
    for s in STAGES:
        coldefs.append(fits.Column(name=f"T_{s.upper()}", format="D",
                                   array=np.array([r[f"t_{s}"] for r in records])))
    for s in STAGES:
        coldefs.append(fits.Column(name=f"ROW_{s.upper()}", format="J",
                                   array=np.array([r[f"row_{s}"] for r in records], dtype=np.int32)))
    outp = os.path.join(TABLEDIR, "residual_dust_critical_points.fits")
    fits.HDUList([fits.PrimaryHDU(), fits.BinTableHDU.from_columns(coldefs, name="CRIT")]).writeto(outp, overwrite=True)
    return outp

print("saved:", save_critical_table())

### II.3 (§3b) · When do the critical points happen? Redshift / cosmic-time distributions

For each evolutionary anchor, compare the **redshift distribution** of fast vs slow
quenchers (with a matching **cosmic-time** axis on top). This shows *when* in cosmic
history each population reaches SF-peak, SFT, QT, etc. — e.g. whether slow quenchers
quench later (closer to z=0), consistent with retaining their reservoir longer.

In [ ]:
# redshift <-> cosmic-time mapping for the secondary axis (monotonic interpolation)
_zg = np.linspace(0, 8, 400)
_tg = COSMO.age(_zg).value                      # Gyr, decreasing in z
def _z_to_t(z): return np.interp(z, _zg, _tg)                 # z -> t [Gyr]
def _t_to_z(t): return np.interp(t, _tg[::-1], _zg[::-1])     # t -> z
def t_at_crit(stage):
    return np.array([r[f"t_{stage}"] for r in records]) / 1e9  # cosmic time [Gyr]

# common cosmic-time binning across panels
_allt = np.concatenate([t_at_crit(st) for st in STAGES])
_allt = _allt[np.isfinite(_allt)]
tlo, thi = np.nanpercentile(_allt, 2), np.nanpercentile(_allt, 98)
tbins = np.linspace(tlo, max(thi, tlo + 0.5), 16)

# rows = mass bin, cols = critical point; fast vs slow overlaid in each panel
ncol = len(STAGES)
fig, axes = plt.subplots(NBINS, ncol, figsize=(3.0 * ncol, 3.0 * NBINS), squeeze=False)
for bi in range(NBINS):
    binmask = mbin == bi
    for ai, st in enumerate(STAGES):
        ax = axes[bi, ai]
        tt = t_at_crit(st)
        for msk, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            v = tt[msk]; v = v[np.isfinite(v)]
            if len(v) == 0:
                continue
            ax.hist(v, bins=tbins, color=c, alpha=0.5, density=True, label=f"{lab} ({len(v)})")
            ax.axvline(np.median(v), color=c, ls="--", lw=1.2)
        if bi == 0:
            secax = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t))
            secax.set_xlabel(f"{st}\nredshift z", fontsize=8)
        if bi == NBINS - 1:
            ax.set_xlabel("cosmic time t [Gyr]")
        ax.legend(fontsize=6)
    axes[bi, 0].set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\npdf", fontsize=9)
fig.suptitle(f"Cosmic time (top: redshift) of each critical point — {SPLIT_DESC}, by mass bin", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "critical_point_redshift_dist.png"), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 7·setup — halo-mass bins (parallel to the z=0 stellar-mass facets) + generalized plotters.
# Halo mass = z=0 parent-halo total (P["masses.total"], row 0), aligned to galaxies.
# SINGLE SOURCE of the halo-mass binning: every later cell (§3d, §3f, §7, §8) reads these
# names — do not redefine HMASS_BIN_* downstream.
HMASS_BIN_EDGES  = [11.5, 12.0, 13.0, 14.5, np.inf]      # log10(Mhalo/Msun) edges
HMASS_BIN_LABELS = ["11.5-12", "12-13", "13-14.5", ">=14.5"]
NHBINS = len(HMASS_BIN_LABELS)

_mh0 = P["masses.total"][0, cols]
logmh_rec = np.log10(np.where(_mh0 > 0, _mh0, np.nan))
hmbin = np.full(len(records), -1, dtype=int)
for _b in range(NHBINS):
    inb = (logmh_rec >= HMASS_BIN_EDGES[_b]) & (logmh_rec < HMASS_BIN_EDGES[_b + 1])
    hmbin[inb] = _b
print("halo-mass bins (logMhalo at z=0):")
for _b in range(NHBINS):
    bm = hmbin == _b
    print(f"  {HMASS_BIN_LABELS[_b]:9s}: {bm.sum():4d}  ({LBL_A} {np.sum(bm & is_fast)}, {LBL_B} {np.sum(bm & is_slow)})")

# shared binomial-fraction CI (used by §3f fast-fraction bands and §10d AGN-CO contingency)
def _wilson(k, n, zc=1.0):
    """Wilson score interval for a binomial fraction (zc=1 -> ~68% band)."""
    if n == 0:
        return np.nan, np.nan, np.nan
    p = k / n; d = 1 + zc**2 / n
    centre = (p + zc**2 / (2 * n)) / d
    half = (zc / d) * np.sqrt(p * (1 - p) / n + zc**2 / (4 * n**2))
    # clamp the bounds to bracket p exactly: float rounding can make centre±half land a
    # hair beyond p, and matplotlib's errorbar rejects yerr<0 (even -1e-16) -> p-l / h-p >= 0.
    return p, min(max(centre - half, 0.0), p), max(min(centre + half, 1.0), p)

# generalized twins of the §4 plotters: take a diagnostics dict + bin assignment/labels
def violin_stage_g(diags, qty, binarr, binlabels, binname="logMhalo", logy=True, ylabel=None, fname=None):
    nbn = len(binlabels)
    fig, axs = plt.subplots(1, nbn, figsize=(5.0 * nbn, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(nbn):
        ax = axs[bi]; binmask = binarr == bi
        for j, st in enumerate(STAGES):
            for off, msk, c in [(-0.18, is_fast & binmask, COL_A), (0.18, is_slow & binmask, COL_B)]:
                v = diags[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    continue
                v = np.log10(v) if logy else v
                parts = ax.violinplot([v], positions=[j + off], widths=0.32, showmedians=True)
                for b in parts["bodies"]:
                    b.set_facecolor(c); b.set_alpha(0.6)
                for key in ("cmedians", "cbars", "cmins", "cmaxes"):
                    if key in parts:
                        parts[key].set_color(c)
        ax.set_xticks(range(len(STAGES))); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"{binname} {binlabels[bi]} (n={binmask.sum()})")
    axs[0].set_ylabel(ylabel or (f"log10 {qty}" if logy else qty))
    fig.suptitle(f"{qty} across stages  ({LBL_A} vs {LBL_B})", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath( fname), dpi=130, bbox_inches="tight")
    plt.show()

def median_track_g(diags, qty, binarr, binlabels, binname="logMhalo", logy=True, fname=None):
    xs = np.arange(len(STAGES)); nbn = len(binlabels)
    fig, axs = plt.subplots(1, nbn, figsize=(5.0 * nbn, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(nbn):
        ax = axs[bi]; binmask = binarr == bi
        for msk, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            med, lo, hi = [], [], []
            for st in STAGES:
                v = diags[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    med.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
                vv = np.log10(v) if logy else v
                med.append(np.median(vv)); lo.append(np.percentile(vv, 25)); hi.append(np.percentile(vv, 75))
            ax.plot(xs, med, "-o", color=c, label=lab); ax.fill_between(xs, lo, hi, color=c, alpha=0.15)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"{binname} {binlabels[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(f"median log10 {qty}" if logy else f"median {qty}")
    fig.suptitle(f"stage evolution of {qty}", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath( fname), dpi=130, bbox_inches="tight")
    plt.show()

### S1 · General sample statistics

Sample sizes, stellar/halo-mass coverage, redshift span and the fast/slow split — the
headcount behind every later facet. Read-only.

In [ ]:
# S1 · general sample statistics. Needs §3 (records, is_fast/is_slow, mbin, tau_n),
#   §7·setup (hmbin, HMASS_BIN_LABELS).
_ms0 = P["masses.stellar"][0, cols]                       # z=0 stellar mass, aligned to records
_logms0 = np.log10(np.where(_ms0 > 0, _ms0, np.nan))
print(f"tracked sample: {len(records)} galaxies over {len(snaps_arr)} snapshots, "
      f"z=[{redshift.min():.2f}, {redshift.max():.2f}]")
print(f"  classified: {LBL_A}={int(is_fast.sum())}  {LBL_B}={int(is_slow.sum())}  "
      f"unclassified={int((~(is_fast | is_slow)).sum())}")
print(f"  z=0 logM*: median={np.nanmedian(_logms0):.2f}  "
      f"range=[{np.nanmin(_logms0):.2f}, {np.nanmax(_logms0):.2f}]")
print("\n  stellar-mass facets (z=0):")
for b in range(NBINS):
    bm = mbin == b
    print(f"    logM* {MASS_BIN_LABELS[b]:9s}: {bm.sum():4d}  "
          f"({LBL_A} {int((bm & is_fast).sum())}, {LBL_B} {int((bm & is_slow).sum())})")
print("  halo-mass facets (z=0):")
for b in range(NHBINS):
    bm = hmbin == b
    print(f"    logMh {HMASS_BIN_LABELS[b]:9s}: {bm.sum():4d}  "
          f"({LBL_A} {int((bm & is_fast).sum())}, {LBL_B} {int((bm & is_slow).sum())})")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
_lm = _logms0[np.isfinite(_logms0)]
ax[0].hist(_lm, bins=20, color="0.7", label="all tracked")
ax[0].hist(_logms0[is_fast], bins=20, histtype="step", color=COL_A, lw=1.5, label=LBL_A)
ax[0].hist(_logms0[is_slow], bins=20, histtype="step", color=COL_B, lw=1.5, label=LBL_B)
ax[0].set_xlabel(r"$\log_{10} M_\star(z{=}0)\ [M_\odot]$"); ax[0].set_ylabel("N")
ax[0].legend(); ax[0].set_title("tracked stellar-mass distribution")
_tn = np.log10(tau_n[np.isfinite(tau_n) & (tau_n > 0)])
ax[1].hist(_tn, bins=25, color="0.7")
if globals().get("SPLIT_MODE") == "fast_slow" and "TAU_SPLIT_LOG" in globals():
    ax[1].axvline(TAU_SPLIT_LOG, ls="--", c="k", label=f"fast/slow cut ({TAU_SPLIT_LOG:g})"); ax[1].legend()
ax[1].set_xlabel(r"$\log_{10}(\tau_q/t_H)$"); ax[1].set_ylabel("N")
ax[1].set_title("quench-timescale distribution")
plt.tight_layout(); plt.show()

### S2 · Halo-association sanity

Is every galaxy attached to a sensible parent halo? Checks that the stellar-to-halo mass
ratio is physical ($M_\star/M_{\rm halo}\lesssim0.05$ expected; flag $>0.2$), that
$M_{\rm halo}\ge M_\star$, and that the SHMR has no teleport outliers. Read-only.

In [ ]:
# S2 · halo-association sanity. Needs §3 (records, cols, is_fast/is_slow), §7·setup (hmbin).
_ms0 = P["masses.stellar"][0, cols]
_mh0 = P["masses.total"][0, cols]
with np.errstate(all="ignore"):
    _shmr = np.where(_mh0 > 0, _ms0 / _mh0, np.nan)
_bad_ratio = np.isfinite(_shmr) & (_shmr > 0.2)                  # stellar > 20% of halo -> suspect
_bad_order = np.isfinite(_ms0) & np.isfinite(_mh0) & (_mh0 < _ms0)
_no_halo   = ~np.isfinite(_mh0) | (_mh0 <= 0)
print(f"halo-association checks over {len(records)} galaxies:")
print(f"  M*/Mhalo > 0.2 (suspicious): {int(_bad_ratio.sum())}")
print(f"  Mhalo < M*  (broken assoc):  {int(_bad_order.sum())}")
print(f"  no/zero parent-halo mass:    {int(_no_halo.sum())}")
print(f"  median M*/Mhalo = {np.nanmedian(_shmr):.4f}  (expect ~0.01-0.05 at these masses)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
lx = np.log10(np.where(_mh0 > 0, _mh0, np.nan)); ly = np.log10(np.where(_ms0 > 0, _ms0, np.nan))
ax[0].scatter(lx[is_slow], ly[is_slow], s=10, c=COL_B, alpha=.6, label=LBL_B)
ax[0].scatter(lx[is_fast], ly[is_fast], s=10, c=COL_A, alpha=.6, label=LBL_A)
_bad = _bad_ratio | _bad_order
if _bad.sum():
    ax[0].scatter(lx[_bad], ly[_bad], s=45, facecolors="none", edgecolors="k", label="flagged")
_xx = np.array([np.nanmin(lx), np.nanmax(lx)])
for fr in (0.05, 0.2):
    ax[0].plot(_xx, _xx + np.log10(fr), ls=":", c="0.5", lw=.8)
ax[0].set_xlabel(r"$\log_{10}M_{\rm halo}$"); ax[0].set_ylabel(r"$\log_{10}M_\star$")
ax[0].legend(); ax[0].set_title("SHMR sanity (z=0); dotted = 5%, 20%")
_sh = _shmr[np.isfinite(_shmr) & (_shmr > 0)]
ax[1].hist(np.log10(_sh), bins=25, color="0.7")
ax[1].axvline(np.log10(0.2), ls="--", c="k", label="0.2 flag")
ax[1].set_xlabel(r"$\log_{10}(M_\star/M_{\rm halo})$"); ax[1].set_ylabel("N")
ax[1].legend(); ax[1].set_title("stellar-to-halo mass ratio")
plt.tight_layout(); plt.show()

### II.1·diag4 (§2·diag4) · Halo-tracking sanity — does the parent halo evolve sensibly?

We do **not** build a separate halo merger tree: each galaxy is associated to its parent halo per snapshot, so “halo tracking” means *does the parent halo evolve continuously along the galaxy's progenitor chain?* Two light, cached checks — halo-**mass** continuity (the analogue of the galaxy spatial-continuity test) and example **position tracks** — plus an opt-in heavy check (`READ_HALO_POS`) that reads the true halo-centre positions and the galaxy-in-halo containment ($\mathrm{sep}/R_{\rm halo}$) from the catalogs. *Caveat:* a real major merger / infall makes $M_{\rm halo}$ jump and the galaxy switch halos, so a flagged step is **suspect**, not necessarily a broken link.

In [ ]:
# §2·diag4 — halo-tracking sanity (read-only). Needs §3 (records, cols, mbin), §1 (P, redshift,
#   t_cosmic_yr, snaps_arr), §5·pre (_PROG_INDEX). Reuses _full_box from §2·diag2.

# --- (1) halo-MASS continuity along the progenitor chain (cached P["masses.total"]) ---
if "masses.total" not in P:
    print("history has no 'masses.total' -> halo-mass continuity unavailable.")
else:
    _mh = np.asarray(P["masses.total"], float)[:, cols]                 # (n_snap, n_rec) halo mass
    with np.errstate(all="ignore"):
        _lmh = np.log10(np.where(_mh > 0, _mh, np.nan))
    _t  = np.asarray(t_cosmic_yr, float); _dt = np.abs(np.diff(_t)) / 1e9       # Gyr between rows
    _dlog = np.abs(np.diff(_lmh, axis=0))                              # |Δ log Mh| per step
    _rate = _dlog / np.where(_dt > 0, _dt, np.nan)[:, None]            # dex / Gyr (gap-immune)
    JUMP_DEX, RATE_HI = 0.5, 0.5
    _v = _dlog[np.isfinite(_dlog)]; _rv = _rate[np.isfinite(_rate)]
    print("halo-mass continuity along the progenitor chain:")
    if _v.size:
        print(f"  median |ΔlogMh|={np.median(_v):.3f} dex | 99th={np.percentile(_v,99):.3f} | "
              f"steps > {JUMP_DEX} dex = {np.mean(_v>JUMP_DEX):.2%}")
        print(f"  dt-normalised: steps > {RATE_HI:g} dex/Gyr = {np.mean(_rv>RATE_HI):.2%} (gap-immune)")
    with np.errstate(all="ignore"):
        _halobad = np.nansum(_rate > RATE_HI, axis=0) / np.maximum(np.sum(np.isfinite(_rate), axis=0), 1)
    _trk = np.sum(np.isfinite(_dlog), axis=0) >= 3
    _susp = _trk & (_halobad > 0.2)
    print(f"  galaxies (>=3 valid steps): {int(_trk.sum())} | suspect halo tracks "
          f"(>20% over-rate steps): {int(_susp.sum())} "
          f"({(np.mean(_susp[_trk]) if _trk.any() else 0):.1%})")
    print("  (a real major merger inflates one step -> 'suspect' != broken; cross-check the tracks below.)")

# --- (2) example "track of positions" (centrals ≈ halo centre), cached P["pos"], comoving ---
_have_sc = False
if "pos" in P and "_full_box" in globals():
    _pos = np.asarray(P["pos"], float); _z = np.asarray(redshift, float)
    _bs, _be = _full_box(snaps_arr[0]), _full_box(snaps_arr[-1])
    _as, _ae = 1.0/(1.0+_z[0]), 1.0/(1.0+_z[-1])
    _phys = (np.isfinite(_bs) and np.isfinite(_be) and abs(_be/_bs - _ae/_as) < abs(_be/_bs - 1.0))
    _posc = _pos*(1.0+_z)[:, None, None] if _phys else _pos
    L = (_bs/_as if _phys else _bs) if np.isfinite(_bs) else np.nanpercentile(_posc[np.isfinite(_posc)], 99.99)
    fig, axes = plt.subplots(1, NBINS, figsize=(5.0*NBINS, 4.6), squeeze=False)
    for bi in range(NBINS):
        ax = axes[0, bi]; cand = np.where(mbin == bi)[0]
        if len(cand):
            nfin = np.array([int(np.isfinite(_posc[:, cols[r], 0]).sum()) for r in cand])
            pick = cand[np.argsort(-nfin)[:3]]
        else:
            pick = []
        for r in pick:
            xy = _posc[:, cols[r], :2]; good = np.isfinite(xy[:, 0])
            if good.sum() < 3:
                continue
            xs, ys = xy[good, 0].copy(), xy[good, 1].copy()
            for arr in (xs, ys):                          # unwrap periodic box jumps -> continuous path
                d = np.diff(arr); d -= L*np.round(d/L); arr[1:] = arr[0] + np.cumsum(d)
            sc = ax.scatter(xs, ys, c=_t[good]/1e9, cmap="viridis", s=16, zorder=2); _have_sc = True
            ax.plot(xs, ys, "-", lw=0.7, color="0.6", zorder=1)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}"); ax.set_xlabel("x [comoving]")
        if bi == 0:
            ax.set_ylabel("y [comoving]")
    if _have_sc:
        fig.colorbar(sc, ax=axes[0, -1], label="cosmic time [Gyr]")
    fig.suptitle("progenitor position tracks (centrals ≈ halo centre); a continuous path = good tracking", y=1.02)
    plt.tight_layout(); plt.savefig(figpath("diag_halo_position_tracks.png"), dpi=130, bbox_inches="tight"); plt.show()
else:
    print("position tracks need P['pos'] and §2·diag2's _full_box -> skipped.")

# --- (3) OPTIONAL heavy: true halo-CENTRE continuity + galaxy-in-halo containment (reads catalogs) ---
READ_HALO_POS = False    # set True to validate the parent-halo centre directly (one catalog read / snapshot)
if READ_HALO_POS:
    _colrec = np.array([r["col"] for r in records]); _nrec = len(records)
    HPOS = np.full((len(snaps_arr), _nrec, 3), np.nan); SEP = np.full((len(snaps_arr), _nrec), np.nan)
    HR = np.full((len(snaps_arr), _nrec), np.nan)
    for si, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        gx_row = _PROG_INDEX[si, _colrec]; sel = np.where(np.isfinite(gx_row))[0]
        if not len(sel):
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                phi  = np.asarray(f["galaxy_data/parent_halo_index"][:], np.int64)
                gpos = np.asarray(f["galaxy_data/pos"][:], float)
                hpos = next((np.asarray(f[p][:], float) for p in ("halo_data/pos", "halo_data/dicts/pos") if p in f), None)
                hr   = next((np.asarray(f[p][:], float) for p in (
                              "halo_data/dicts/virial_quantities.r200c",
                              "halo_data/dicts/virial_quantities.radius",
                              "halo_data/dicts/radii.total") if p in f), None)
        except (OSError, KeyError):
            continue
        if hpos is None:
            continue
        for r in sel:
            gxi = int(gx_row[r]); h = int(phi[gxi]) if gxi < len(phi) else -1
            if h < 0 or h >= len(hpos):
                continue
            HPOS[si, r] = hpos[h]; SEP[si, r] = np.sqrt(np.sum((gpos[gxi] - hpos[h])**2))
            if hr is not None and h < len(hr):
                HR[si, r] = hr[h]
    with np.errstate(all="ignore"):
        _contain = (SEP / np.where(HR > 0, HR, np.nan))[np.isfinite(SEP) & (HR > 0)]
    if _contain.size:
        print(f"\ngalaxy-in-halo containment (separation / R_halo): median={np.median(_contain):.2f}, "
              f"frac > 1.5 = {np.mean(_contain>1.5):.2%}  (high -> mis-association)")
    # halo-centre spatial continuity (comoving min-image step / box), gap-immune
    _zc = np.asarray(redshift, float); _Hp = HPOS*(1.0+_zc)[:, None, None] if ("_phys" in dir() and _phys) else HPOS
    _Lh = L if "L" in dir() and np.isfinite(L) else np.nanpercentile(_Hp[np.isfinite(_Hp)], 99.99)
    _dH = np.diff(_Hp, axis=0); _dH -= _Lh*np.round(_dH/_Lh)
    _stepH = np.sqrt(np.sum(_dH**2, axis=2)) / _Lh
    _rateH = _stepH / np.where(_dt > 0, _dt, np.nan)[:, None]
    _sh = _rateH[np.isfinite(_rateH)]
    if _sh.size:
        print(f"halo-centre continuity: 99th step/box per Gyr = {np.percentile(_sh,99):.2e}; "
              f"steps > 0.05 box/Gyr = {np.mean(_sh>0.05):.2%}")
else:
    print("READ_HALO_POS=False -> skipping the heavy catalog-read halo-centre/containment check.")


# Part 2 — When & how do these galaxies quench?

The timing of the quench itself: process timescales (intervals between critical points), phase-duration distributions by stellar & halo mass, and phase occupancy over cosmic time. Descriptive timing that sets up the dust-survival and driver analyses below.

### III.1 (§3c) · Process timescales — the intervals *between* critical points

§3b showed *when* each anchor happens; here we measure the *durations between* them,
per galaxy, and compare fast vs slow (Mann–Whitney *p* per panel). Sign convention:
every interval is (later − earlier), so positive = forward in cosmic time.

- **approach** (track start → SF-peak): does a class take *longer to reach* its SF peak?
- **plateau** (SF-peak → SFT): decline before leaving the main sequence.
- **tau_q** (SFT → QT): the quench time itself (the fast/slow discriminator).
- **dust / gas survival** (QT → dust_min / gas_min): how long the reservoir persists *after* quenching — the residual-dust headline.
- **dust decline** (dust-peak → dust_min) and **dust–gas lag** (gas_min → dust_min, &gt;0 ⇒ dust outlives gas).

The second figure collapses each class to its *median* anchor times on a single
cosmic-time axis — a visual life-cycle that directly tests "slow rise, then immediate quench".

In [ ]:
# ── §3c. Process timescales: intervals between critical points (fast vs slow) ──
# Every interval below is (later anchor − earlier anchor), so a positive value means
# "forward in cosmic time". Units: Gyr. All anchors come from `records` (§3); the only
# extra ingredient is each galaxy's track-start time (earliest valid sSFR snapshot),
# recomputed here so we can ask "how long did it take to REACH the SF peak".

def _rt(key):  # anchor time [Gyr] across records
    return np.array([r[key] for r in records]) / 1e9

t_start = np.array([track_start_yr(r["col"]) for r in records]) / 1e9   # §3 helper
t_sfpk, t_sft_, t_qt_  = _rt("t_sf_peak"), _rt("t_sft"), _rt("t_qt")
t_dpk,  t_dmin, t_gmin = _rt("t_dust_peak"), _rt("t_dust_min"), _rt("t_gas_min")

# timescale registry: label -> (values[Gyr], one-line meaning)
INTERVALS = {
    "approach\n(start->SFpeak)":    (t_sfpk - t_start, "time to reach the sSFR peak"),
    "plateau\n(SFpeak->SFT)":       (t_sft_ - t_sfpk,  "decline before leaving the MS"),
    "tau_q\n(SFT->QT)":             (t_qt_  - t_sft_,  "quench time (MS -> passive)"),
    "dust survival\n(QT->dustmin)": (t_dmin - t_qt_,   "dust persistence after quenching"),
    "gas survival\n(QT->gasmin)":   (t_gmin - t_qt_,   "gas depletion after quenching"),
    "dust decline\n(dpeak->dmin)":  (t_dmin - t_dpk,   "duration of the dust drop"),
    "dust-gas lag\n(gasmin->dmin)": (t_dmin - t_gmin,  "does dust outlive gas? (>0 yes)"),
}
ikeys = list(INTERVALS)

# ── figure 1: faceted interval distributions, fast vs slow, Mann-Whitney p per panel ──
ncol = len(ikeys)
fig, axes = plt.subplots(NBINS, ncol, figsize=(2.7 * ncol, 2.9 * NBINS), squeeze=False)
for bi in range(NBINS):
    binmask = mbin == bi
    for ai, key in enumerate(ikeys):
        ax = axes[bi, ai]
        vals = INTERVALS[key][0]
        fa = vals[is_fast & binmask]; fa = fa[np.isfinite(fa)]
        sl = vals[is_slow & binmask]; sl = sl[np.isfinite(sl)]
        if len(fa) + len(sl) == 0:
            ax.set_axis_off(); continue
        lo, hi = np.nanpercentile(np.concatenate([fa, sl]), [2, 98])
        bins = np.linspace(min(lo, 0.0), max(hi, lo + 0.1), 18)
        for v, c, lab in [(fa, COL_A, LBL_A), (sl, COL_B, LBL_B)]:
            if len(v):
                ax.hist(v, bins=bins, color=c, alpha=0.5, density=True, label=f"{lab} ({len(v)})")
                ax.axvline(np.median(v), color=c, ls="--", lw=1.2)
        ax.axvline(0.0, color="k", lw=0.6, alpha=0.4)
        if len(fa) >= 3 and len(sl) >= 3:
            p = mannwhitneyu(fa, sl, alternative="two-sided").pvalue
            ax.text(0.97, 0.95, f"p={p:.1g}", transform=ax.transAxes, ha="right", va="top",
                    fontsize=7, bbox=dict(fc="w", ec="0.7", alpha=0.7, pad=1))
        if bi == 0:
            ax.set_title(key, fontsize=8)
        if bi == NBINS - 1:
            ax.set_xlabel("Gyr")
        ax.legend(fontsize=6, loc="upper left")
    axes[bi, 0].set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\npdf", fontsize=9)
fig.suptitle(f"Process timescales (intervals between critical points) — {SPLIT_DESC}, by mass bin", y=1.005)
plt.tight_layout(); plt.savefig(figpath( "process_timescales_intervals.png"), dpi=130, bbox_inches="tight"); plt.show()

# compact median table (Gyr) so the numbers are reusable in the write-up
print(f"median interval [Gyr]            {LBL_A:>8s} {LBL_B:>8s}    p(MWU)")
for key in ikeys:
    vals = INTERVALS[key][0]
    fa = vals[is_fast]; fa = fa[np.isfinite(fa)]
    sl = vals[is_slow]; sl = sl[np.isfinite(sl)]
    p = mannwhitneyu(fa, sl, alternative="two-sided").pvalue if (len(fa) >= 3 and len(sl) >= 3) else np.nan
    print(f"  {key.replace(chr(10),' '):28s} {np.median(fa):8.2f} {np.median(sl):8.2f}   {p:8.1g}")

# ── figure 2: median event timeline — the average life-cycle of each class ──
ANCHORS = ["sf_peak", "sft", "qt", "dust_min", "gas_min"]
A_T = {a: _rt(f"t_{a}") for a in ANCHORS}
fig, axes = plt.subplots(NBINS, 1, figsize=(8.5, 1.9 * NBINS + 0.5), squeeze=False)
for bi in range(NBINS):
    ax = axes[bi, 0]; binmask = mbin == bi
    for j, (msk, c, lab) in enumerate([(is_fast & binmask, COL_A, LBL_A),
                                        (is_slow & binmask, COL_B, LBL_B)]):
        y = 0.6 - 0.2 * j
        med = [np.nanmedian(A_T[a][msk]) if np.any(msk) else np.nan for a in ANCHORS]
        ax.plot(med, [y] * len(ANCHORS), "-", color=c, lw=2, alpha=0.7, zorder=1)
        ax.scatter(med, [y] * len(ANCHORS), color=c, s=45, zorder=2,
                   label=f"{lab} (N={int(np.sum(msk))})")
        if bi == 0 and j == 0:
            for a, m in zip(ANCHORS, med):
                ax.annotate(a, (m, y), textcoords="offset points", xytext=(0, 9),
                            ha="center", fontsize=7, rotation=30)
    ax.set_ylim(0.2, 0.85); ax.set_yticks([])
    ax.set_ylabel(f"logM*\n{MASS_BIN_LABELS[bi]}", fontsize=9)
    ax.legend(fontsize=7, loc="lower right")
    if bi == 0:
        sx = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t)); sx.set_xlabel("redshift z")
    if bi == NBINS - 1:
        ax.set_xlabel("median cosmic time of each anchor [Gyr]")
fig.suptitle(f"Median event timeline — {SPLIT_DESC}", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "process_timescales_timeline.png"), dpi=130, bbox_inches="tight"); plt.show()

### III.2 (§3d) · Phase-duration distributions by stellar & halo mass (all galaxies)

Durations of the four evolutionary phases (star-forming, quenching, post-quench, quenched), pooled over the whole sample and shown per **stellar-mass** bin (left) and per **halo-mass** bin (right) — **no** fast/slow split. Also shown **normalized by the cosmic time (age of the universe) at the end of each phase** — the same $t_H$ normalization as $\tau_q$ in §3.

In [ ]:
# 3d · Phase-DURATION distributions by stellar mass AND halo mass (ALL galaxies; no fast/slow split).
#   Phases follow §3e: star-forming (start->SFT), quenching (SFT->QT), post-quench (QT->post_q),
#   quenched (post_q->z0). Shown as absolute durations [Gyr] AND normalized by the cosmic time
#   (= age of universe) at the END of each phase. Anchor times here ARE cosmic times, so t_H(event)
#   equals the end anchor -> for the quenching phase the normalized value IS §3's tau_q_over_tH.
# halo-mass bins: canonical HMASS_BIN_EDGES/LABELS/NHBINS from §7·setup (Part II) — no local override
_g = lambda key: np.array([r[key] for r in records], float) / 1e9      # anchor time [Gyr]
_t_start = np.array([track_start_yr(r["col"]) for r in records]) / 1e9   # §3 helper
_t_sft, _t_qt = _g("t_sft"), _g("t_qt")
_t_pq = _g("t_post_quench")                                            # NaN where no persistence end
_t_z0 = _g("t_z0")
PHASE_DUR = {
    "star-forming\n(start->SFT)": _t_sft - _t_start,
    "quenching\n(SFT->QT)":       _t_qt  - _t_sft,
    "post-quench\n(QT->post_q)":   _t_pq  - _t_qt,
    "quenched\n(post_q->z0)":     _t_z0  - _t_pq,
}
# cosmic time (age of universe) at each phase's END -> duration/end = fraction of t_H elapsed
_PHASE_END = {"star-forming\n(start->SFT)": _t_sft, "quenching\n(SFT->QT)": _t_qt,
              "post-quench\n(QT->post_q)": _t_pq,   "quenched\n(post_q->z0)": _t_z0}
PHASE_DUR_NORM = {n: PHASE_DUR[n] / np.where(_PHASE_END[n] > 0, _PHASE_END[n], np.nan) for n in PHASE_DUR}

# stellar bins: §3 mbin (z=0). Halo bins: measured AT a critical point, so galaxies are
# compared at the same environment WHEN the phases happened (not the z=0 descendant halo).
HM_STAGE  = "z0"                   # anchor for the environment: "sft" | "qt" | "post_quench" | "z0"
HM_EDGES  = HMASS_BIN_EDGES        # reuse the canonical §7·setup edges (override locally if needed)
HM_LABELS = [f"{l} @{HM_STAGE}" for l in HMASS_BIN_LABELS]
_mh_st = np.full(len(records), np.nan)
for _i, _r in enumerate(records):
    _rw = _r[f"row_{HM_STAGE}"]
    if _rw >= 0:
        _mh_st[_i] = P["masses.total"][_rw, cols[_i]]
_lmh_st = np.log10(np.where(_mh_st > 0, _mh_st, np.nan))
_hmbin = np.full(len(records), -1, int)
for _b in range(len(HM_LABELS)):
    _hmbin[(_lmh_st >= HM_EDGES[_b]) & (_lmh_st < HM_EDGES[_b + 1])] = _b
print(f"§3d halo bins at @{HM_STAGE}:",
      {HM_LABELS[b]: int((_hmbin == b).sum()) for b in range(len(HM_LABELS))},
      "| unassigned:", int((_hmbin == -1).sum()))
_SPLITS = [(mbin,   MASS_BIN_LABELS, "stellar mass logM*", "viridis"),
           (_hmbin, HM_LABELS,       f"halo mass logMh @{HM_STAGE}", "plasma")]

def _plot_phase_dist(durdict, xlabel, suptitle, fname, xmax=None):
    nph = len(durdict)
    fig, axes = plt.subplots(nph, 2, figsize=(11, 2.8 * nph), squeeze=False, sharex="row")
    for ri, (name, dur) in enumerate(durdict.items()):
        fin_all = dur[np.isfinite(dur)]
        if len(fin_all) == 0:
            for ci in range(2):
                axes[ri, ci].set_axis_off()
            continue
        lo, hi = np.nanpercentile(fin_all, [1, 99])
        if xmax is not None:
            hi = xmax
        bins = np.linspace(min(lo, 0.0), max(hi, lo + 1e-3), 26)
        for ci, (binarr, labels, binname, cmap) in enumerate(_SPLITS):
            ax = axes[ri, ci]
            cols_c = [plt.get_cmap(cmap)(x) for x in np.linspace(0.15, 0.85, len(labels))]
            for bi in range(len(labels)):
                v = dur[(binarr == bi) & np.isfinite(dur)]
                if len(v) < 5:
                    continue
                ax.hist(v, bins=bins, histtype="step", density=True, lw=2, color=cols_c[bi],
                        label=f"{labels[bi]} (N={len(v)}, med {np.median(v):.2f})")
                ax.axvline(np.median(v), color=cols_c[bi], ls="--", lw=1.3)
            ax.axvline(0, color="k", lw=0.6, alpha=0.4)
            if ri == 0:
                ax.set_title(f"by {binname}")
            if ri == nph - 1:
                ax.set_xlabel(xlabel)
            if ci == 0:
                ax.set_ylabel(name)
            ax.legend()
    fig.suptitle(suptitle, y=1.0)
    plt.tight_layout()
    plt.savefig(figpath( fname), dpi=130, bbox_inches="tight")
    plt.show()

def _phase_table(durdict, header):
    print(header)
    for binarr, labels, binname, _ in _SPLITS:
        print(f"  by {binname}:")
        for bi in range(len(labels)):
            row = []
            for name, dur in durdict.items():
                v = dur[(binarr == bi) & np.isfinite(dur)]
                tag = name.splitlines()[0]
                row.append(f"{tag}={np.median(v):.2f}" if len(v) >= 5 else f"{tag}=--")
            print(f"    {labels[bi]:10s} (N={int(np.sum(binarr == bi))}): " + "  ".join(row))

_plot_phase_dist(PHASE_DUR, "phase duration [Gyr]",
                 "Phase-duration distributions [Gyr] — all galaxies, by stellar (left) & halo (right) mass",
                 "phase_duration_distributions.png")
_plot_phase_dist(PHASE_DUR_NORM, r"phase duration / $t_H$(phase end)",
                 r"Phase duration / $t_H$ at phase end — all galaxies, by stellar & halo mass",
                 "phase_duration_norm_distributions.png", xmax=1.0)
_phase_table(PHASE_DUR,      "median phase duration [Gyr] — all galaxies, by mass bin:")
_phase_table(PHASE_DUR_NORM, "median phase duration / t_H(end) (dimensionless) — all galaxies, by mass bin:")


### III.3 (§3e) · Phase occupancy over cosmic time — how many galaxies sit in each phase

Here is the
**instantaneous** census instead: at each cosmic time, how many galaxies are currently
**star-forming** (t &lt; SFT), **quenching** (SFT–QT), **post-quench** (QT–post_quench;
the transitional, still-dusty window) or **quenched** (t ≥ post_quench). A galaxy enters
the stack once its track starts and flows rightward into "quenched" by z=0, so the stacked
area shows the *flux* of the population through each phase. Faceted by stellar-mass bin and,
separately, by halo-mass bin (canonical §7·setup halo bins, defined in Part II).

In [ ]:
# ── §3e. Phase occupancy over cosmic time — FRACTION of galaxies in each phase ──
# (normalized) At each cosmic time the stacked bands sum to 1: the fraction of galaxies
# that have FORMED by t which currently sit in each evolutionary phase. Phases are bounded
# by the robust SF anchors (SFT, QT, post_quench), mutually exclusive & ordered in time:
#   star-forming : t < SFT  |  quenching : SFT<=t<QT  |  post-quench : QT<=t<post_q  |  quenched : t>=post_q
# A galaxy joins the (normalized) population once its track has started (t >= track-start).

_ts_e     = np.array([track_start_yr(r["col"]) for r in records])         # §3 helper [yr]
t_start_e = np.where(np.isfinite(_ts_e), _ts_e, t_cosmic_yr[-1]) / 1e9    # fallback: earliest snap
t_sft_e   = np.array([r["t_sft"] for r in records]) / 1e9
t_qt_e    = np.array([r["t_qt"]  for r in records]) / 1e9
t_pq_raw  = np.array([r["t_post_quench"] for r in records]) / 1e9
t_pq_e    = np.where(np.isfinite(t_pq_raw), t_pq_raw, t_qt_e)   # fold undefined post_q into QT

PHASES = ["star-forming", "quenching", "post-quench", "quenched"]
PH_COL = ["#4c9f70", "#e1a730", "#7b5cc0", "#9aa0a6"]
egrid  = np.linspace(np.nanmin(t_start_e), COSMO.age(0).value, 220)

def _phase_frac(mask):
    """(4, len(egrid)) FRACTION per phase among galaxies in `mask` that have formed by t.
       Each column sums to 1 where >=1 galaxy has formed, else 0."""
    ts, sf, qt, pq = t_start_e[mask], t_sft_e[mask], t_qt_e[mask], t_pq_e[mask]
    out = np.zeros((4, len(egrid)))
    for k, t in enumerate(egrid):
        formed = t >= ts
        nf = int(formed.sum())
        if nf == 0:
            continue
        out[0, k] = np.sum(formed & (t < sf))
        out[1, k] = np.sum(formed & (t >= sf) & (t < qt))
        out[2, k] = np.sum(formed & (t >= qt) & (t < pq))
        out[3, k] = np.sum(formed & (t >= pq))
        out[:, k] /= nf
    return out

def _phase_panel(ax, mask, title, top_axis=False, ylab=False):
    ax.stackplot(egrid, _phase_frac(mask), colors=PH_COL, labels=PHASES, alpha=0.9)
    ax.set_xlim(egrid[0], egrid[-1]); ax.set_ylim(0, 1); ax.set_title(title, fontsize=9)
    ax.set_xlabel("cosmic time t [Gyr]")
    if ylab:
        ax.set_ylabel("fraction of formed galaxies in phase")
    if top_axis:
        sx = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t)); sx.set_xlabel("redshift z", fontsize=8)

# (a) faceted by STELLAR-mass bin
fig, axes = plt.subplots(1, NBINS, figsize=(4.4 * NBINS, 4.2), squeeze=False, sharey=True)
for bi in range(NBINS):
    _phase_panel(axes[0, bi], mbin == bi, f"logM* {MASS_BIN_LABELS[bi]} (n={(mbin == bi).sum()})",
                 top_axis=True, ylab=(bi == 0))
axes[0, -1].legend(loc="lower left", fontsize=8, framealpha=0.9)
fig.suptitle(f"Phase occupancy (fraction) over cosmic time, by stellar-mass bin — sample={len(records)}", y=1.04)
plt.tight_layout(); plt.savefig(figpath( "phase_occupancy_by_stellarmass.png"), dpi=130, bbox_inches="tight"); plt.show()

# (b) faceted by HALO-mass bin (canonical §7·setup bins, Part II)
HMASS_LABELS_E, hmbin_e = HMASS_BIN_LABELS, hmbin
print("§3e halo-mass bin populations:",
      {HMASS_LABELS_E[b]: int((hmbin_e == b).sum()) for b in range(len(HMASS_LABELS_E))},
      " unassigned(<11.5):", int((hmbin_e == -1).sum()))
NHB = len(HMASS_LABELS_E)
fig, axes = plt.subplots(1, NHB, figsize=(4.4 * NHB, 4.2), squeeze=False, sharey=True)
for bi in range(NHB):
    _phase_panel(axes[0, bi], hmbin_e == bi, f"logMh {HMASS_LABELS_E[bi]} (n={(hmbin_e == bi).sum()})",
                 top_axis=True, ylab=(bi == 0))
axes[0, -1].legend(loc="lower left", fontsize=8, framealpha=0.9)
fig.suptitle(f"Phase occupancy (fraction) over cosmic time, by halo-mass bin — sample={len(records)}", y=1.04)
plt.tight_layout(); plt.savefig(figpath( "phase_occupancy_by_halomass.png"), dpi=130, bbox_inches="tight"); plt.show()

# Part 3 — Cached products & per-stage diagnostics

Load (or build, gated) the heavy anchor-dependent products once — particle dust/HI/H₂/star profiles, BH history, satellite catalogue, CGM temperature, local-overdensity history — and assemble the per-stage catalogue diagnostics `DIAGS`. Every domain part below only reads these.

## II-b.1 (§5) · Particle-based dust/HI/H$_2$ radial profiles — the reduced product

The catalogs carry **no** dust half-mass radius, so dust extension must come from the
**particle-level radial distribution**. Writing full per-galaxy particle files for the
whole sample × all critical points is far too heavy, so instead we build a **compact
reduced product**:

- each needed snapshot is opened **once**;
- only each galaxy's own gas/star particles are read (via CAESAR `glist`/`slist`) — not the
  whole snapshot, not all fields;
- the face-on Σ-profiles and effective radii ($R_{50}, R_{90}$) of dust / HI / H₂ / stars
  are computed on the fly and **only those small arrays are stored** in one HDF5.

No particle files are written. The product loads instantly and covers **every galaxy at
every critical point**, so the dust-extension comparisons below run on the *full* sample
(fast vs slow, by mass bin) — the §4c dust-radius plots we could not get from the catalog.

In [ ]:
# configuration for the reduced dust-profile product
PROF_STAGES   = STAGES                      # profile at ALL critical points
RMAX_KPC      = 50.0                        # radial extent of the profiles [physical kpc]
NBINS_R       = 25                          # radial bins
STORE_SIGMA   = True                        # also store Sigma(R) curves (for example plots)
DUSTPROF_PATH = os.path.join(SFHDIR, "dust_profiles_allcrit.hdf5")
BUILD_DUST_PROFILES = False                # HEAVY (hours): built on the cluster via the SLURM job
                                           # (run §5a·plan, submit build_profiles_job.py, then §5b loads).
                                           # Set True only to build in-kernel for a tiny test sample.

edges_R = np.linspace(0.0, RMAX_KPC, NBINS_R + 1)
rmid_R  = 0.5 * (edges_R[:-1] + edges_R[1:])
area_R  = np.pi * (edges_R[1:] ** 2 - edges_R[:-1] ** 2)
print("dust-profile product:", DUSTPROF_PATH, "| build =", BUILD_DUST_PROFILES)

In [ ]:
# 5·helpers — unit conversions + a single-galaxy face-on profile from raw particle arrays.
def header_units(f):
    h = f["Header"].attrs if "Header" in f else {}
    return float(h.get("Time", 1.0)), float(h.get("HubbleParam", 0.68))   # (scale factor a, h)

def _to_kpc(x, a, hub):   return x * a / hub        # comoving kpc/h -> physical kpc
def _to_msun(m, hub):     return m * 1e10 / hub     # 1e10 Msun/h -> Msun

def _components(gf, gmass_code, hub):
    # gf: dict of raw arrays already sliced to the galaxy's gas particles
    m_gas = _to_msun(gmass_code, hub)
    m_dust = _to_msun(gf["dust"], hub) if gf.get("dust") is not None else np.full_like(m_gas, np.nan)
    if gf.get("Z") is not None and gf["Z"].ndim == 2 and gf["Z"].shape[1] >= 2:
        XH = np.clip(1.0 - gf["Z"][:, 0] - gf["Z"][:, 1], 0.0, 1.0)
    else:
        XH = np.full_like(m_gas, 0.76)
    m_H = m_gas * XH
    fneut = gf["fneut"] if gf.get("fneut") is not None else np.full_like(m_gas, np.nan)
    fmol  = gf.get("fmol")
    if fmol is not None:
        m_H2 = m_H * fneut * fmol; m_HI = m_H * fneut * (1.0 - fmol)
    else:
        m_H2 = np.full_like(m_gas, np.nan); m_HI = m_H * fneut
    return m_dust, m_HI, m_H2

def _profile_one(gpos, gdust, gHI, gH2, spos, smass):
    # face-on cylindrical radius in the stellar principal frame; sigmas + effective radii
    if len(spos) >= 10 and np.sum(smass) > 0:
        center = shrink_center(spos, masses=smass)
        _, _, evecs, _ = principal_axes(spos - center, masses=smass)
    elif len(gpos):
        center = np.median(gpos, axis=0); evecs = np.eye(3)
    else:
        return None
    R  = np.sqrt(np.sum(rotate_to_frame(gpos, center, evecs)[:, :2] ** 2, axis=1)) if len(gpos) else np.array([])
    Rs = np.sqrt(np.sum(rotate_to_frame(spos, center, evecs)[:, :2] ** 2, axis=1)) if len(spos) else np.array([])
    def _sig(Rarr, mass):
        if mass is None or len(Rarr) == 0 or not np.any(np.isfinite(mass)):
            return np.full(NBINS_R, np.nan)
        s, _ = np.histogram(Rarr, bins=edges_R, weights=np.nan_to_num(mass)); return s / area_R
    def _renc(Rarr, mass, frac):
        if mass is None or len(Rarr) == 0:
            return np.nan
        m = np.nan_to_num(mass); o = np.argsort(Rarr); c = np.cumsum(m[o])
        if c[-1] <= 0:
            return np.nan
        return float(Rarr[o][np.searchsorted(c, frac * c[-1])])
    comps = {"dust": (R, gdust), "HI": (R, gHI), "H2": (R, gH2), "star": (Rs, smass)}
    out = {f"sigma_{c}": _sig(Ra, m) for c, (Ra, m) in comps.items()}
    for c, (Ra, m) in comps.items():               # exact percentile radii (no binning)
        out[f"R20_{c}"] = _renc(Ra, m, 0.2)
        out[f"R50_{c}"] = _renc(Ra, m, 0.5)
        out[f"R80_{c}"] = _renc(Ra, m, 0.8)
    out["R90_dust"] = _renc(R, gdust, 0.9)
    return out

In [ ]:
# 5a·plan — dump the extraction plan for the SLURM job (records + _PROG_INDEX -> flat table).
# Run this instead of (or before) §5a when building on the cluster: it writes the inputs the
# array workers need, then submit build_profiles_job.py via submit_profiles.sh.
PLAN_PATH = os.path.join(SFHDIR, "dust_profile_plan.hdf5")
_ri, _si, _gx, _snap = [], [], [], []
for ri, r in enumerate(records):
    for si, st in enumerate(PROF_STAGES):
        row = r[f"row_{st}"]
        if row < 0:
            continue
        gx = _PROG_INDEX[row, r["col"]]
        if not np.isfinite(gx):
            continue
        _ri.append(ri); _si.append(si); _gx.append(int(gx)); _snap.append(int(snaps_arr[row]))

with h5py.File(PLAN_PATH, "w") as out:
    out.attrs["sim_name"]    = SIM_NAME
    out.attrs["rmax_kpc"]    = RMAX_KPC
    out.attrs["nbins_r"]     = NBINS_R
    out.attrs["store_sigma"] = int(STORE_SIGMA)
    out.attrs["stages"]      = ",".join(PROF_STAGES)
    out.attrs["nrec"]        = len(records)
    out.attrs["nst"]         = len(PROF_STAGES)
    out.create_dataset("entry_ri",   data=np.array(_ri, np.int32))
    out.create_dataset("entry_si",   data=np.array(_si, np.int32))
    out.create_dataset("entry_gx",   data=np.array(_gx, np.int64))
    out.create_dataset("entry_snap", data=np.array(_snap, np.int32))
    out.create_dataset("gid",        data=np.array([r["gid"] for r in records], np.int64))
print(f"wrote plan: {len(_ri)} (galaxy,stage) entries over {len(set(_snap))} snapshots -> {PLAN_PATH}")
print("submit on the cluster — ONE unified job builds ISM dust + CGM profiles + CGM temperature")
print("in a single snapshot pass (replaces submit_profiles / submit_profiles /")
print("submit_profiles):  mkdir -p logs && \\")
print("  jid=$(sbatch --parsable submit_profiles.sh) && \\")
print("  sbatch --dependency=afterok:$jid submit_profiles_merge.sh")
print("-> dust_profiles / cgm_profiles / cgm_temperature _allcrit.hdf5; reload in §5b / §7p / §7g")
print("   (BUILD_DUST_PROFILES stays False; the notebook only loads).")

In [ ]:
# 5a. Build the reduced dust-profile product. Work is grouped by snapshot, so each
# snapshot is opened ONCE; per galaxy we read only its glist/slist particle slices.
def _detect_gas_fields(f):
    have = set(f["PartType0"].keys())
    dust = next((k for k in ("Dust_Masses", "DustMasses") if k in have), None)
    fmol = next((k for k in ("FractionH2", "fH2", "GrackleH2", "f_H2") if k in have), None)
    Zk   = "Metallicity" if "Metallicity" in have else None
    fnk  = "NeutralHydrogenAbundance" if "NeutralHydrogenAbundance" in have else None
    return dust, fmol, Zk, fnk

def build_dust_profiles():
    nrec, nst = len(records), len(PROF_STAGES)
    R20  = {c: np.full((nrec, nst), np.nan) for c in ("dust", "HI", "H2", "star")}
    R50  = {c: np.full((nrec, nst), np.nan) for c in ("dust", "HI", "H2", "star")}
    R80  = {c: np.full((nrec, nst), np.nan) for c in ("dust", "HI", "H2", "star")}
    R90d = np.full((nrec, nst), np.nan)
    SIG  = {c: np.full((nrec, nst, NBINS_R), np.nan) for c in ("dust", "HI", "H2", "star")} if STORE_SIGMA else None

    need = {}                                   # snapshot -> list of (rec_idx, stage_idx, gidx)
    for ri, r in enumerate(records):
        for si, st in enumerate(PROF_STAGES):
            row = r[f"row_{st}"]
            if row < 0:
                continue
            gx = _PROG_INDEX[row, r["col"]]
            if not np.isfinite(gx):
                continue
            need.setdefault(int(snaps_arr[row]), []).append((ri, si, int(gx)))

    snaps_todo = sorted(need, reverse=True)
    print(f"profiling {sum(len(v) for v in need.values())} (galaxy,stage) entries over {len(snaps_todo)} snapshots")
    for k, snap in enumerate(snaps_todo):
        if snap in CORRUPT_SNAPS:
            continue
        try:
            cs = sim.load_catalog(snap=snap)
            with h5py.File(sim.get_snapshot_file(snap), "r") as f:
                a, hub = header_units(f)
                dk, fk, Zk, fnk = _detect_gas_fields(f)
                g = f["PartType0"]; s = f["PartType4"] if "PartType4" in f else None
                for (ri, si, gx) in need[snap]:
                    gal = cs.galaxies[gx]
                    gl = np.unique(np.asarray(gal.glist, dtype=np.int64))
                    sl = np.unique(np.asarray(getattr(gal, "slist", []), dtype=np.int64))
                    if len(gl) == 0:
                        continue
                    gpos = _to_kpc(g["Coordinates"][gl], a, hub)
                    gf = {"dust":  g[dk][gl]  if dk  else None,
                          "Z":     g[Zk][gl]  if Zk  else None,
                          "fneut": g[fnk][gl] if fnk else None,
                          "fmol":  g[fk][gl]  if fk  else None}
                    m_dust, m_HI, m_H2 = _components(gf, g["Masses"][gl], hub)
                    if s is not None and len(sl):
                        spos = _to_kpc(s["Coordinates"][sl], a, hub); smass = _to_msun(s["Masses"][sl], hub)
                    else:
                        spos, smass = np.empty((0, 3)), np.empty(0)
                    prof = _profile_one(gpos, m_dust, m_HI, m_H2, spos, smass)
                    if prof is None:
                        continue
                    for c in ("dust", "HI", "H2", "star"):
                        R20[c][ri, si] = prof[f"R20_{c}"]; R50[c][ri, si] = prof[f"R50_{c}"]; R80[c][ri, si] = prof[f"R80_{c}"]
                    R90d[ri, si] = prof["R90_dust"]
                    if STORE_SIGMA:
                        for c in ("dust", "HI", "H2", "star"):
                            SIG[c][ri, si] = prof[f"sigma_{c}"]
            del cs; gc.collect()
        except OSError:
            print(f"  [skip] corrupt snapshot {snap}"); CORRUPT_SNAPS.add(snap); continue
        if (k + 1) % 10 == 0 or k == len(snaps_todo) - 1:
            print(f"  processed {k+1}/{len(snaps_todo)} snapshots")

    with h5py.File(DUSTPROF_PATH, "w") as out:
        out.create_dataset("rmid", data=rmid_R)
        out.create_dataset("gid", data=np.array([r["gid"] for r in records]))
        out.attrs["stages"] = ",".join(PROF_STAGES)
        for c in ("dust", "HI", "H2", "star"):
            out.create_dataset(f"R20_{c}", data=R20[c])
            out.create_dataset(f"R50_{c}", data=R50[c])
            out.create_dataset(f"R80_{c}", data=R80[c])
        out.create_dataset("R90_dust", data=R90d)
        if STORE_SIGMA:
            for c in ("dust", "HI", "H2", "star"):
                out.create_dataset(f"sigma_{c}", data=SIG[c], compression="gzip")
    print("wrote", DUSTPROF_PATH)

if BUILD_DUST_PROFILES:
    build_dust_profiles()
else:
    print("BUILD_DUST_PROFILES=False -> expecting cached", DUSTPROF_PATH)

In [ ]:
# Shared guard: cached record-aligned products can be STALE when the sample changes size
# (e.g. the history is rebuilt over a different selection). Remap a cached product onto the
# current `records` by galaxy id (rows/cols absent from the cache -> NaN), so a stale cache
# degrades to partial coverage with a warning instead of a broadcasting/index error. Rebuild
# the product (BUILD_*=True) to cover the full sample.
def _align_records_by_gid(arrays, cache_gid, cur_gid, axis=0, name=""):
    import numpy as _np
    cache_gid = _np.asarray(cache_gid).astype(_np.int64)
    cur_gid   = _np.asarray(cur_gid).astype(_np.int64)
    if len(cache_gid) == len(cur_gid) and _np.array_equal(cache_gid, cur_gid):
        return arrays, len(cur_gid)
    pos = {int(g): j for j, g in enumerate(cache_gid)}
    sel = _np.array([pos.get(int(g), -1) for g in cur_gid])
    found = sel >= 0
    out = {}
    for k, a in arrays.items():
        a = _np.asarray(a)
        if a.ndim <= axis or a.shape[axis] != len(cache_gid):   # not record-aligned -> keep
            out[k] = a; continue
        nshape = list(a.shape); nshape[axis] = len(cur_gid)
        b = _np.full(nshape, _np.nan, dtype=float)
        idx = _np.where(found)[0]
        sl = [slice(None)] * b.ndim; sl[axis] = idx
        b[tuple(sl)] = _np.take(a, sel[found], axis=axis)
        out[k] = b
    print(f"[warn] {name}: cached product holds {len(cache_gid)} galaxies but the sample has "
          f"{len(cur_gid)} ({int(found.sum())} matched, {int((~found).sum())} -> NaN). "
          f"Rebuild it to cover the full sample.")
    return out, int(found.sum())

# 5b. load the reduced product (aligned to `records`)
if os.path.exists(DUSTPROF_PATH):
    with h5py.File(DUSTPROF_PATH, "r") as f:
        DP_STAGES = list(f.attrs["stages"].split(","))
        DP = {k: f[k][:] for k in f.keys()}
    _cur_gid = np.array([r["gid"] for r in records])
    if "gid" in DP:
        DP, _ = _align_records_by_gid(DP, DP["gid"], _cur_gid, name="dust profiles (§5)")
        DP["gid"] = _cur_gid
    DP_RMID = DP["rmid"]
    DP_COMPS = ("dust", "HI", "H2", "star")
    with np.errstate(all="ignore"):
        DP_RATIO = np.where(DP["R50_star"] > 0, DP["R50_dust"] / DP["R50_star"], np.nan)
        # size-ratio (concentration) diagnostics per component. Populated only when the product
        # carries R20/R80 (rebuild §5a); older products -> NaN so downstream plots no-op cleanly.
        DP_HAS_CONC = all(f"R20_{c}" in DP and f"R80_{c}" in DP for c in DP_COMPS)
        DP_C2050, DP_C2080 = {}, {}                 # R20/R50 (inner) and R20/R80 (overall); both <1
        for c in DP_COMPS:
            if DP_HAS_CONC:
                DP_C2050[c] = np.where(DP[f"R50_{c}"] > 0, DP[f"R20_{c}"] / DP[f"R50_{c}"], np.nan)
                DP_C2080[c] = np.where(DP[f"R80_{c}"] > 0, DP[f"R20_{c}"] / DP[f"R80_{c}"], np.nan)
            else:
                DP_C2050[c] = np.full_like(DP["R50_dust"], np.nan)
                DP_C2080[c] = np.full_like(DP["R50_dust"], np.nan)
    DP_OK = True
    n_ok = int(np.sum(np.isfinite(DP["R50_dust"])))
    print(f"loaded {DUSTPROF_PATH}: stages={DP_STAGES}; {n_ok} finite dust R50 entries; "
          f"concentration={'present' if DP_HAS_CONC else 'absent (rebuild §5a)'}")
else:
    DP, DP_OK = None, False
    print("no dust-profile product yet -> set BUILD_DUST_PROFILES=True and run §5a.")

In [ ]:
# load BH history and derive AGN-feeding metrics per galaxy (aligned to `records`)

with h5py.File(BH_HIST_PATH, "r") as f:
    BH = {k: f[k][:] for k in f.keys()}

# Guard against a STALE cached BH history whose snapshot axis is shorter/longer than the
# current property history (e.g. P rebuilt over a different snapshot range). BH rows are
# snaps_arr-ordered from the shared START_SNAP, so align by NaN-padding / truncating the
# high-z end; covered (low-z) rows stay correctly aligned. Proper refresh: BUILD_BH=True.
_nsnap = len(snaps_arr)
_bh_rows0 = next(iter(BH.values())).shape[0]
if _bh_rows0 != _nsnap:
    print(f"[warn] cached BH history has {_bh_rows0} snapshot rows but the property history "
          f"has {_nsnap}; aligning (NaN-pad/truncate the high-z end). Rebuild with "
          f"BUILD_BH=True to refresh for full high-z coverage.")
    for k in list(BH):
        a = BH[k]
        if a.shape[0] < _nsnap:
            a = np.vstack([a, np.full((_nsnap - a.shape[0], a.shape[1]), np.nan)])
        else:
            a = a[:_nsnap]
        BH[k] = a
if int(cols.max()) >= next(iter(BH.values())).shape[1]:
    print(f"[warn] BH history galaxy axis ({next(iter(BH.values())).shape[1]}) is smaller than "
          f"the sample needs (max col {int(cols.max())}) -> rebuild BH with BUILD_BH=True.")

def bh_stage(arr, stage):
    rows = np.array([r[f"row_{stage}"] for r in records])
    out = np.full(len(records), np.nan)
    for i, (row, col) in enumerate(zip(rows, cols)):
        if row >= 0:
            out[i] = arr[row, col]
    return out

def _sft_qt_rows(r):
    rs, rq = r["row_sft"], r["row_qt"]
    if rs < 0 or rq < 0:
        return np.array([], dtype=int)
    lo, hi = sorted((rs, rq))
    return np.arange(lo, hi + 1)

c2_erg_per_Msun = (const.c ** 2 * u.Msun).to("erg").value     # E = M c^2 for 1 Msun

dMbh, Eagn, jetfrac, mdot_qt = (np.full(len(records), np.nan) for _ in range(4))
for i, r in enumerate(records):
    col = r["col"]
    mdot_qt[i] = BH["bh_mdot"][r["row_qt"], col] if r["row_qt"] >= 0 else np.nan
    idx = _sft_qt_rows(r)
    if len(idx) < 2:
        continue
    t = t_cosmic_yr[idx]; o = np.argsort(t); idx = idx[o]; t = t[o]
    md = BH["bh_mdot"][idx, col]; mb = BH["bh_mass"][idx, col]; fe = BH["bh_fedd"][idx, col]
    good = np.isfinite(md) & np.isfinite(t)
    if good.sum() >= 2:
        M_acc = np.trapz(md[good], t[good])               # Msun (Msun/yr * yr)
        Eagn[i] = EPS_R * M_acc * c2_erg_per_Msun          # erg
    if np.isfinite(mb).sum() >= 2:
        mbv = mb[np.isfinite(mb)]
        dMbh[i] = mbv[-1] - mbv[0]
    jet = np.isfinite(mb) & np.isfinite(fe) & (mb > 10 ** 7.5) & (fe < 0.2)
    valid_state = np.isfinite(mb) & np.isfinite(fe)
    jetfrac[i] = jet.sum() / valid_state.sum() if valid_state.sum() > 0 else np.nan

print(f"galaxies with E_AGN(SFT->QT): {np.isfinite(Eagn).sum()} / {len(records)}")

### II-b.2 (§7a) · Satellites in the host halo — counts and their gas/dust

How many galaxies share each tracked QG's halo, and how much gas/dust those companions
carry. This is the satellite ISM that inflates the §7 `M_CGM` proxy (halo gas counts
satellite ISM as "CGM"). Catalog-level, grouped by snapshot so each CAESAR catalog opens
once; **needs `_PROG_INDEX` from §5·pre**. The central is taken as the most massive
(stellar) galaxy in the halo; satellite sums exclude it. A cleaner CGM correction is
`M_CGM_clean = M_halo_gas − Mgal_gas_halo` (subtract *all* member-galaxy ISM).

In [ ]:
# 7a. Build the per-halo satellite catalogue (catalog-only; light). One CAESAR file per snap.
assert "_PROG_INDEX" in globals(), "run §5·pre first to build _PROG_INDEX"

SAT_PATH  = os.path.join(SFHDIR, "satellites_allcrit.hdf5")
BUILD_SAT = True   # rebuild for the current sample (set back to `not os.path.exists(SAT_PATH)` once cached)

SAT_FIELDS = {
    "star": "galaxy_data/dicts/masses.stellar",
    "gas":  "galaxy_data/dicts/masses.gas",
    "dust": "galaxy_data/dicts/masses.dust",
    "HI":   "galaxy_data/dicts/masses.HI",
    "H2":   "galaxy_data/dicts/masses.H2",
}
SAT_KEYS = ["Nmemb", "Nsat", "is_central", "Msat_gas", "Msat_dust",
            "Msat_HI", "Msat_H2", "Mgal_gas_halo"]

def build_satellites():
    nrec, nst = len(records), len(STAGES)
    S = {k: np.full((nrec, nst), np.nan) for k in SAT_KEYS}
    need = {}
    for ri, r in enumerate(records):
        for si, st in enumerate(STAGES):
            row = r[f"row_{st}"]
            if row < 0:
                continue
            gx = _PROG_INDEX[row, r["col"]]
            if not np.isfinite(gx):
                continue
            need.setdefault(int(snaps_arr[row]), []).append((ri, si, int(gx)))
    print(f"satellites: {sum(len(v) for v in need.values())} (galaxy,stage) over {len(need)} snapshots")

    for snap in sorted(need, reverse=True):
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                parent = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
                arr = {c: (np.asarray(f[p][:], dtype=np.float64) if p in f else None)
                       for c, p in SAT_FIELDS.items()}
        except (OSError, KeyError):
            print(f"  [skip] snapshot {snap}"); CORRUPT_SNAPS.add(snap); continue

        valid = parent >= 0
        if not np.any(valid):
            continue
        nhalo = int(parent[valid].max()) + 1
        cnt = np.bincount(parent[valid], minlength=nhalo)
        sums = {c: (np.bincount(parent[valid], weights=arr[c][valid], minlength=nhalo)
                    if arr[c] is not None else None) for c in SAT_FIELDS}
        # central = most massive stellar (fallback gas) galaxy per halo: last write in
        # ascending-mass order wins.
        key = "star" if arr["star"] is not None else "gas"
        gv = np.where(valid)[0]
        order = gv[np.argsort(arr[key][gv], kind="stable")]
        central = np.full(nhalo, -1, dtype=np.int64)
        central[parent[order]] = order

        for (ri, si, gx) in need[snap]:
            h = parent[gx]
            if h < 0:
                continue
            c = central[h]
            S["Nmemb"][ri, si]      = cnt[h]
            S["Nsat"][ri, si]       = cnt[h] - 1
            S["is_central"][ri, si] = float(gx == c)
            if sums["gas"] is not None:
                S["Msat_gas"][ri, si]      = max(sums["gas"][h] - arr["gas"][c], 0.0)
                S["Mgal_gas_halo"][ri, si] = sums["gas"][h]
            if sums["dust"] is not None:
                S["Msat_dust"][ri, si] = max(sums["dust"][h] - arr["dust"][c], 0.0)
            if sums["HI"] is not None:
                S["Msat_HI"][ri, si]   = max(sums["HI"][h]   - arr["HI"][c],   0.0)
            if sums["H2"] is not None:
                S["Msat_H2"][ri, si]   = max(sums["H2"][h]   - arr["H2"][c],   0.0)

    with h5py.File(SAT_PATH, "w") as out:
        out.attrs["stages"] = ",".join(STAGES)
        out.create_dataset("gid", data=np.array([r["gid"] for r in records]))
        for k in SAT_KEYS:
            out.create_dataset(k, data=S[k])
    print("wrote", SAT_PATH)
    return S

if BUILD_SAT:
    SAT = build_satellites()
else:
    with h5py.File(SAT_PATH, "r") as f:
        SAT = {k: f[k][:] for k in SAT_KEYS}
        _sat_gid = f["gid"][:] if "gid" in f else None
    print("loaded", SAT_PATH)
    if _sat_gid is not None:
        SAT, _ = _align_records_by_gid(SAT, _sat_gid, np.array([r["gid"] for r in records]),
                                       name="satellites (§7a)")

# tidy diagnostics dict for the generalized plotters
_sij = {st: i for i, st in enumerate(STAGES)}
DIAGS_SAT = {}
for st in STAGES:
    j = _sij[st]
    DIAGS_SAT[st] = {
        "Nsat":          SAT["Nsat"][:, j],
        "Msat_gas":      SAT["Msat_gas"][:, j],
        "Msat_dust":     SAT["Msat_dust"][:, j],
        "Msat_cold":     SAT["Msat_HI"][:, j] + SAT["Msat_H2"][:, j],
        "Msat_dust/gas": np.where(SAT["Msat_gas"][:, j] > 0,
                                  SAT["Msat_dust"][:, j] / SAT["Msat_gas"][:, j], np.nan),
    }

# descriptive summary at z=0
jz = _sij["z0"]; nsat0 = SAT["Nsat"][:, jz]; cen0 = SAT["is_central"][:, jz]
print(f"\nat z=0: tracked QG is the halo central for "
      f"{np.nansum(cen0):.0f}/{np.sum(np.isfinite(cen0)):.0f}; "
      f"median N_sat={np.nanmedian(nsat0):.1f}, max N_sat={np.nanmax(nsat0):.0f}")
for _b in range(NHBINS):
    m = (hmbin == _b) & np.isfinite(nsat0)
    if m.sum():
        print(f"   logMhalo {HMASS_BIN_LABELS[_b]:9s}: median N_sat={np.nanmedian(nsat0[m]):.1f}, "
              f"median Msat_gas={np.nanmedian(SAT['Msat_gas'][m, jz]):.2e} Msun (n={m.sum()})")

### II-b.3 (§7g) · CGM temperature — particle-level cross-check (optional)

Optional higher-fidelity cross-check of the catalog `temperatures.mass_weighted_cgm`
already plotted in §7b/§7d. Mass-weighted temperature of the **non-star-forming halo gas** (the diffuse CGM, from
`halo.glist` with `SFR==0`) at each critical point, faceted by halo mass, fast vs slow.
Heavy particle read — produced by the SLURM job, **not** on the fly:

```
mkdir -p logs
jid=$(sbatch --parsable submit_profiles.sh)
sbatch --dependency=afterok:$jid submit_profiles_merge.sh
```

It reuses the dust-profile plan (`dust_profile_plan.hdf5`), so no new plan is needed — just
make sure §5a·plan has been run. `T_mw` is the mass-weighted geometric-mean temperature;
`f_hot/f_warm/f_cold` split the CGM mass at 10^5 / 10^4 K.

In [ ]:
# 7g. Load the CGM-temperature product and build diagnostics aligned to `records`.
CGMT_PATH = os.path.join(SFHDIR, "cgm_temperature_allcrit.hdf5")
if os.path.exists(CGMT_PATH):
    with h5py.File(CGMT_PATH, "r") as f:
        CT_STAGES = list(f.attrs["stages"].split(","))
        CT = {k: f[k][:] for k in f.keys()}
    if "gid" in CT:
        _cur_gid = np.array([r["gid"] for r in records])
        CT, _ = _align_records_by_gid(CT, CT["gid"], _cur_gid, name="CGM temperature (§7g)")
        CT["gid"] = _cur_gid
    _nrec_ct = CT["T_mw"].shape[0]
    _ct_keys = ("T_CGM", "T_CGM_med", "M_CGM_T", "fhot", "fwarm", "fcold")
    DIAGS_CGMT = {}
    for st in STAGES:
        if st not in CT_STAGES:                 # stage absent from this product build -> NaN
            DIAGS_CGMT[st] = {k: np.full(_nrec_ct, np.nan) for k in _ct_keys}
            continue
        j = CT_STAGES.index(st)
        DIAGS_CGMT[st] = {
            "T_CGM":     CT["T_mw"][:, j],      # mass-weighted geometric-mean temperature [K]
            "T_CGM_med": CT["T_med"][:, j],     # mass-weighted median temperature [K]
            "M_CGM_T":   CT["M_cgm"][:, j],     # non-SF halo gas mass [Msun]
            "fhot":      CT["f_hot"][:, j],     # mass fraction T > 1e5 K
            "fwarm":     CT["f_warm"][:, j],    # mass fraction 1e4-1e5 K
            "fcold":     CT["f_cold"][:, j],    # mass fraction T < 1e4 K
        }
    CGMT_OK = True
    n_ok = int(np.sum(np.isfinite(CT["T_mw"])))
    print(f"loaded {CGMT_PATH}: stages={CT_STAGES}; {n_ok} finite T_CGM entries")
else:
    CGMT_OK = False
    print("no CGM-temperature product yet -> run submit_profiles.sh + merge "
          "(reuses dust_profile_plan.hdf5), then re-run this cell.")

In [ ]:
# 8c·pre — full local-overdensity HISTORY per galaxy (n_snap × n_records), so the ENVIRONMENT
#   can be timed exactly like AGN ignition. Generalizes §7N from the 7 stage-rows to every
#   snapshot. Stored as (n_snap, n_records) to reuse the same time-reordering as BH/P arrays.
#   Cached; delete env_history_passive_z0.hdf5 (or set BUILD_ENV_HISTORY=True) to rebuild.
ENV_HIST_PATH = os.path.join(SFHDIR, "env_history_passive_z0.hdf5")
ENV_SPHERE_R  = globals().get("SPHERE_R", 2.0)          # Mpc/h comoving (matches §7N default)
ENV_MSTAR_MIN = globals().get("NEIGH_MSTAR_MIN", 1e9)   # tracer stellar-mass floor [Msun]
BUILD_ENV_HISTORY = not os.path.exists(ENV_HIST_PATH)
if not BUILD_ENV_HISTORY:
    with h5py.File(ENV_HIST_PATH, "r") as _f:
        _ehs = _f["overdens_hist"].shape
    if _ehs != (n_snap, len(records)):    # stale (sample/snap range changed); no gid -> must rebuild
        print(f"[warn] env history {_ehs} is stale vs (n_snap={n_snap}, records={len(records)}); "
              f"rebuilding (no gid stored, cannot remap by id).")
        BUILD_ENV_HISTORY = True

if BUILD_ENV_HISTORY:
    assert "_PROG_INDEX" in globals(), "run §5·pre first (builds _PROG_INDEX)"
    overdens_hist = np.full((n_snap, len(records)), np.nan)   # [snap-row, record]
    _Lh = globals().get("_L", None)
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                gpos_raw = np.asarray(f["galaxy_data/pos"][:], float)
                gms      = np.asarray(f["galaxy_data/dicts/masses.stellar"][:], float)
        except (OSError, KeyError):
            CORRUPT_SNAPS.add(snap); continue
        if _Lh is None:
            _Lh = float(BOX) if np.nanmax(gpos_raw) < 1000 else float(BOX) * 1000.0
        R = ENV_SPHERE_R if _Lh <= 1000 else ENV_SPHERE_R * 1000.0
        gpos = np.clip(np.mod(gpos_raw, _Lh), 0.0, _Lh * (1 - 1e-12))
        tracer = gms >= ENV_MSTAR_MIN
        if tracer.sum() < 5:
            continue
        tree = cKDTree(gpos[tracer], boxsize=_Lh)
        nmean = tracer.sum() / _Lh ** 3; Vsph = 4.0 / 3.0 * np.pi * R ** 3
        for rci, col in enumerate(cols):
            gx = _PROG_INDEX[ri, col]
            if not np.isfinite(gx):
                continue
            gx = int(gx); p = gpos[gx]
            if not np.all(np.isfinite(p)):
                continue
            cnt = len(tree.query_ball_point(p, R)) - (1 if tracer[gx] else 0)
            overdens_hist[ri, rci] = cnt / (nmean * Vsph)
    with h5py.File(ENV_HIST_PATH, "w") as f:
        f.create_dataset("overdens_hist", data=overdens_hist)
    print("env history written:", ENV_HIST_PATH, overdens_hist.shape)
else:
    with h5py.File(ENV_HIST_PATH, "r") as f:
        overdens_hist = f["overdens_hist"][:]
    print("env history loaded:", ENV_HIST_PATH, overdens_hist.shape)


## IV.6 (§4) · Catalogue distributions at the critical points

Using CAESAR integrated quantities only. At each evolutionary anchor we read each
galaxy's $M_{\rm dust}, M_{H_2}, M_{HI}, M_\star, M_{\rm gas}$ and both half-mass
radii from the history row that the anchor maps to. We then compare the
**distributions** between **fast** and **slow** quenchers, with the **SFT** anchor
playing the role of the star-forming reference state.

Key catalogue-level diagnostics of the hypothesis:
- $M_{\rm dust}/M_\star$, $M_{H_2}/M_\star$, $M_{HI}/M_\star$ — how much reservoir survives;
- $R_{\rm gas,1/2}/R_{\star,1/2}$ — is the cold gas **extended** vs centrally concentrated;
- dust-to-gas and HI/H₂ — phase of the surviving ISM.

In [ ]:
def gather_stage(prop, stage):
    '''Value of `prop` for every record at its `stage` anchor row (NaN if missing).'''
    rows = np.array([r[f"row_{stage}"] for r in records])
    out = np.full(len(records), np.nan)
    arr = P.get(prop)
    if arr is None:                       # property absent from cached history -> all-NaN
        return out
    for i, (row, col) in enumerate(zip(rows, cols)):
        if row >= 0:
            out[i] = arr[row, col]
    return out

def ratio_stage(num, den, stage):
    a = gather_stage(num, stage); b = gather_stage(den, stage)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(b > 0, a / b, np.nan)

# build a tidy dict: diagnostics[stage][quantity] = array over galaxies
DIAGS = {}
for st in STAGES:
    DIAGS[st] = {
        "Mdust":   gather_stage("masses.dust", st),
        "MH2":     gather_stage("masses.H2", st),
        "MHI":     gather_stage("masses.HI", st),
        "Mstar":   gather_stage("masses.stellar", st),
        "Mgas":    gather_stage("masses.gas", st),
        "sSFR":    ratio_stage("sfr", "masses.stellar", st),  # specific SFR = SFR / M*
        "dust/M*": ratio_stage("masses.dust", "masses.stellar", st),
        "H2/M*":   ratio_stage("masses.H2", "masses.stellar", st),
        "HI/M*":   ratio_stage("masses.HI", "masses.stellar", st),
        "dust/gas": ratio_stage("masses.dust", "masses.gas", st),
        "HI/H2":   ratio_stage("masses.HI", "masses.H2", st),
        "Rgas/Rstar": ratio_stage("radii.gas_half_mass", "radii.stellar_half_mass", st),
        "Rgas":    gather_stage("radii.gas_half_mass", st),
        # catalog concentration (size ratios), available at every snapshot for stars & total gas.
        # r20/r80 -> overall concentration (<1, smaller = more concentrated); r20/r50 -> inner.
        "C20/80*":   ratio_stage("radii.stellar_r20", "radii.stellar_r80", st),
        "C20/50*":   ratio_stage("radii.stellar_r20", "radii.stellar_half_mass", st),
        "C20/80gas": ratio_stage("radii.gas_r20", "radii.gas_r80", st),
        "C20/50gas": ratio_stage("radii.gas_r20", "radii.gas_half_mass", st),
    }
print("diagnostics built for stages:", list(DIAGS.keys()))
# NB: the catalog has NO dust half-mass radius, so dust extension is computed from the
# particle reduced product (DP) in §5 — see §5c (stage tracks) and §5d (quenching clock).

# Part 4 — Stellar properties

Stellar (and total-gas) structure: concentration / size ratios across the evolutionary stages. Broader stellar mass / age / size / metallicity distributions appear in the population-character grid (Part 9) and the observable grids (Part 11).

### Stellar & gas concentration (size ratios)

Catalog concentration across the evolutionary stages: $R_{20}/R_{80}$ (overall) and
$R_{20}/R_{50}$ (inner) for stars and total gas — smaller means more centrally concentrated.
Fast vs slow, faceted by stellar- and halo-mass bin. Uses the catalog `radii.*_r20/half_mass/r80`
already in the history (available at every snapshot).

In [ ]:
# stellar/gas concentration across stages. Needs §4 (DIAGS), §7·setup (violin_stage_g,
#   median_track_g, bins).
for q, lab in [("C20/80*",   r"$R_{20}/R_{80}$ (stars)"),
               ("C20/50*",   r"$R_{20}/R_{50}$ (stars)"),
               ("C20/80gas", r"$R_{20}/R_{80}$ (gas)")]:
    _safe = q.replace("/", "_").replace("*", "star")
    violin_stage_g(DIAGS, q, mbin, MASS_BIN_LABELS, binname="logM*", logy=False, ylabel=lab,
                   fname=f"conc_{_safe}_Mstar.png")
    median_track_g(DIAGS, q, hmbin, HMASS_BIN_LABELS, binname="logMhalo", logy=False,
                   fname=f"conc_{_safe}_track_Mhalo.png")

# Part 5 — ISM: reservoirs & dust survival

The cold-ISM *content*: dust / H₂ / HI reservoir distributions and stage tracks, then the headline result — how much dust survives quenching (fast-fraction, dust clock, threshold-free ledger, P(dusty|quiescent) and τ_dust). The ISM *spatial distribution* is in Part 6.

In [ ]:
# 4a. Distributions at each stage, fast vs slow — one column per mass bin
# (generalized violin_stage_g from §7·setup, faceted by the z=0 stellar-mass bins)
for q in ["dust/M*", "H2/M*", "HI/M*"]:
    violin_stage_g(DIAGS, q, mbin, MASS_BIN_LABELS, binname="logM*",
                   logy=True, fname=f"dist_{q.replace('/','_')}.png")


In [ ]:
# 4b. Median stage evolution (fast vs slow) for the reservoir masses — one column per mass bin
# (thin wrapper over the generalized median_track_g from §7·setup; §4c/§4d reuse it)
def median_track(qty, logy=True):
    median_track_g(DIAGS, qty, mbin, MASS_BIN_LABELS, binname="logM*", logy=logy,
                   fname=f"track_{qty.replace('/','_')}.png")

for q in ["Mstar", "Mdust", "MH2", "MHI", "sSFR"]:
    median_track(q)


In [ ]:
# 4c. THE catalogue-level extension test: is the cold *gas* extended (R_gas/R_star)?
# (The catalog has no dust radius; the dust-extension analogue is §5c/§5d from particles.)
median_track("Rgas/Rstar", logy=False)
median_track("Rgas", logy=False)
# 4d. surviving-ISM phase
median_track("dust/gas", logy=True)
median_track("HI/H2", logy=True)

### IV.1 (§3f) · Who dominates when? Fast fraction of the quenched & still-dusty population

The literal question — at each cosmic time, are the quenched (and still-**dusty**) galaxies mostly
fast or slow — but **faceted by halo mass**, because fast/slow is heavily confounded with mass
(fast quenchers are almost all low-M\*). Reading at fixed halo mass separates the two scenarios:

- **timing (scenario 1)**: within a halo bin, the fast fraction *declines with cosmic time* (fast
  dominate the early/high-z quenched & dusty population, slow dominate at low z).
- **environment (scenario 2)**: the split is set by which halo bin you are in, with little time-trend.

Solid = all quenched, dashed = still-dusty (`M_dust/M_dust,peak > RETAIN_THR` $= 10^{-1.5}\approx 3\%$), band = Wilson 68%.
Shown for both the **first** and **last** quench definition (top/bottom rows).

In [ ]:
# ── §3f. Who dominates when? Fast fraction of the quenched (and still-dusty) population ──
#   The literal question: at each cosmic time, are the quenched / still-dusty galaxies mostly
#   FAST or SLOW? fast/slow is strongly confounded with mass (fast are almost all low-M*), so we
#   FACET BY HALO MASS — a time-trend that survives at fixed halo mass is a real timing effect
#   (scenario 1); a split that is purely set by halo mass with no time-trend is environment
#   (scenario 2). Compared for the FIRST vs LAST quench definition (top vs bottom row).

# --- shared quantities for §3f–§3g (built once, reused below) ---
_o      = np.argsort(t_cosmic_yr)
tg      = t_cosmic_yr[_o] / 1e9                       # Gyr, ascending
zg_asc  = redshift[_o]
QT = {"first": np.array([r["t_qt_first"] for r in records]) / 1e9,
      "last":  np.array([r["t_qt"]       for r in records]) / 1e9}

# dust/M* history, log dust/M*, and retained-dust fraction (M_dust / per-galaxy peak), ascending time
_mstar = P["masses.stellar"][:, cols][_o, :]
_mdust = P["masses.dust"][:, cols][_o, :]
with np.errstate(divide="ignore", invalid="ignore"):
    dmh   = np.where(_mstar > 0, _mdust / _mstar, np.nan)         # (n_snap, n_rec) dust/M*
    ldmh  = np.where(dmh > 0, np.log10(dmh), np.nan)             # log10 dust/M*
    _dpeak = np.nanmax(np.where(_mdust > 0, _mdust, np.nan), axis=0)         # per-rec peak dust mass
    retain = np.where(_dpeak[None, :] > 0, _mdust / _dpeak[None, :], np.nan)  # (n_snap, n_rec)
RETAIN_THR = 10**(-1.5)                                          # 'still dusty' = holds >~3% (10**-1.5) of peak dust
dustyR = np.isfinite(retain) & (retain > RETAIN_THR)     # (n_snap, n_rec) time-varying dusty

# halo-mass bins: canonical §7·setup definition (Part II), aliased for §3f-§3i
HMASS_EDGES, HMASS_LABELS, NHB = HMASS_BIN_EDGES, HMASS_BIN_LABELS, NHBINS
print("§3f halo-mass bins:", ", ".join(f"{HMASS_LABELS[b]}={int((hmbin==b).sum())}" for b in range(NHB)),
      f"| fast={int(is_fast.sum())} slow={int(is_slow.sum())}")

# _wilson (binomial CI): canonical definition in §7·setup (Part II)
def _fastfrac_line(ax, qt_gyr, pop_extra, bm, ls, lab):
    """Fast fraction vs cosmic time among (quenched & fast|slow & bm [& pop_extra]) + Wilson band."""
    fs = bm & (is_fast | is_slow)
    sel = (tg[:, None] >= qt_gyr[None, :]) & fs[None, :]          # quenched & in fast/slow & bin
    if pop_extra is not None:
        sel = sel & pop_extra
    nfast = (sel & is_fast[None, :]).sum(1); ntot = sel.sum(1)
    p = np.full(len(tg), np.nan); lo = p.copy(); hi = p.copy()
    for s in range(len(tg)):
        if ntot[s] >= 5:
            p[s], lo[s], hi[s] = _wilson(int(nfast[s]), int(ntot[s]))
    ax.plot(tg, p, ls, color=COL_A, lw=1.8, label=lab)
    ax.fill_between(tg, lo, hi, color=COL_A, alpha=0.12)

fig, axes = plt.subplots(2, NHB, figsize=(4.3 * NHB, 7.6), squeeze=False, sharex=True, sharey=True)
for ri, mode in enumerate(["first", "last"]):
    for bi in range(NHB):
        ax = axes[ri, bi]; bm = hmbin == bi
        _fastfrac_line(ax, QT[mode], None,   bm, "-",  "all quenched")
        _fastfrac_line(ax, QT[mode], dustyR, bm, "--", "still-dusty")
        ax.axhline(0.5, color="0.7", lw=0.8, ls=":"); ax.set_ylim(0, 1)
        if ri == 0:
            ax.set_title(f"logMh {HMASS_LABELS[bi]} (n={int(bm.sum())})", fontsize=9)
            sx = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t)); sx.set_xlabel("z", fontsize=8)
        if ri == 1:
            ax.set_xlabel("cosmic time t [Gyr]")
        if bi == 0:
            ax.set_ylabel(f"QT={mode}\nfast fraction", fontsize=9)
    axes[ri, -1].legend(fontsize=7, loc="upper right")
fig.suptitle(f"Who dominates the quenched population? Fast fraction vs cosmic time, by halo mass "
             f"(>0.5 = {LBL_A}-dominated; dashed = still-dusty; band = Wilson 68%)", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "crossover_fastfrac_by_halomass.png"), dpi=130, bbox_inches="tight"); plt.show()

### IV.2 (§3g) · Dust timescale — dust/M\* and retained dust, fast vs slow

The heart of scenario 1. Median dust/M\* (top) and retained-dust fraction `M_dust/M_dust,peak`
(bottom), fast vs slow, on three time axes: **absolute cosmic time** (left) and the **quench clock**
`t − t_QT` for the **first** and **last** quench (middle/right). The quench clock isolates dust
*persistence after quenching* from the quench *epoch*: if slow quenchers keep their dust longer,
their retained-dust curve stays above `RETAIN_THR` ($10^{-1.5}\approx 3\%$ of peak) to larger `t − t_QT`. The printed table gives the
median **dust-survival time** (how long retained dust stays above `RETAIN_THR` after QT) for each class.

In [ ]:
# ── §3g. Dust timescale: how dust/M* and retained dust evolve, fast vs slow ──
#   (a) vs absolute cosmic time; (b,c) vs the quench clock (t - t_QT) for FIRST & LAST quench.
#   The clock isolates dust PERSISTENCE after quenching from the quench EPOCH itself — the core
#   of scenario 1 (fast: dust peaks early & drops fast; slow: dusty later & for longer).
#   Needs §3f (tg, QT, ldmh, retain, RETAIN_THR).
CLK = np.linspace(-6, 8, 71)                              # quench-clock grid [Gyr]

def _band_abs(vals2d, gmask):
    sub = vals2d[:, gmask]
    return np.nanmedian(sub, 1), np.nanpercentile(sub, 25, 1), np.nanpercentile(sub, 75, 1)

def _band_clock(vals2d, gmask, qt_gyr):
    idx = np.where(gmask)[0]
    M = np.full((len(idx), len(CLK)), np.nan)
    for j, i in enumerate(idx):
        y = vals2d[:, i]; ok = np.isfinite(y)
        if ok.sum() >= 3:
            M[j] = np.interp(CLK, (tg - qt_gyr[i])[ok], y[ok], left=np.nan, right=np.nan)
    return np.nanmedian(M, 0), np.nanpercentile(M, 25, 0), np.nanpercentile(M, 75, 0)

def _draw(ax, x, getband, title, xlabel, ylabel=None, mark0=False, ztop=False):
    for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
        m, q1, q3 = getband(msk)
        ax.plot(x, m, color=c, lw=2, label=lab); ax.fill_between(x, q1, q3, color=c, alpha=0.18)
    if mark0:
        ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title(title, fontsize=9); ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if ztop:
        sx = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t)); sx.set_xlabel("z", fontsize=8)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# col 0 = absolute cosmic time
_draw(axes[0, 0], tg, lambda m: _band_abs(ldmh, m),   "log dust/M* vs cosmic time", "t [Gyr]", "log dust/M*", ztop=True)
_draw(axes[1, 0], tg, lambda m: _band_abs(retain, m), "retained dust vs cosmic time", "t [Gyr]", "M_dust / M_dust,peak", ztop=True)
# cols 1,2 = quench clock for first / last
for ci, mode in enumerate(["first", "last"], start=1):
    _draw(axes[0, ci], CLK, lambda m, q=QT[mode]: _band_clock(ldmh, m, q),
          f"log dust/M* vs (t - QT_{mode})", f"t - QT_{mode} [Gyr]", mark0=True)
    _draw(axes[1, ci], CLK, lambda m, q=QT[mode]: _band_clock(retain, m, q),
          f"retained dust vs (t - QT_{mode})", f"t - QT_{mode} [Gyr]", mark0=True)
    axes[1, ci].axhline(RETAIN_THR, color="0.6", lw=0.8, ls=":")
fig.suptitle(f"Dust evolution, {LBL_A} vs {LBL_B} — absolute time vs quench clock (band = 25-75%)", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "dust_timescale_fast_slow.png"), dpi=130, bbox_inches="tight"); plt.show()

# headline numbers: median dust-survival time = how long retained dust stays > RETAIN_THR after QT
print(f"median dust-survival after quenching (retained > {RETAIN_THR:g}), Gyr:")
for mode in ["first", "last"]:
    for msk, lab in [(is_fast, LBL_A), (is_slow, LBL_B)]:
        m, _, _ = _band_clock(retain, msk, QT[mode])
        above = CLK[(CLK >= 0) & (m > RETAIN_THR)]
        surv = (above.max() if above.size else 0.0)
        print(f"  QT={mode:5s}  {lab:5s}: {surv:4.1f}")

### IV.3 (§3h) · SF-peak premise — do fast quenchers peak earlier and dustier?

Tests the premise behind scenario 1 directly: **when** the SF peak happens (`t_sf_peak`, top) and
**how dusty** the galaxy is at that peak (`log dust/M*` at `sf_peak`, bottom), fast vs slow, faceted
by halo mass so "earlier" isn't just standing in for environment. If fast quenchers peak earlier and
dustier *at fixed halo mass*, scenario 1 holds; if the peak timing is the same across halo bins and
only the halo bin differs, scenario 2 dominates.

In [ ]:
# ── §3h. SF-peak premise: do fast quenchers peak earlier and dustier? ──
#   Tests the premise behind scenario 1: WHEN (t_sf_peak) and HOW DUSTY (dust/M* at sf_peak)
#   the SF peak is, fast vs slow, faceted by halo mass (so 'earlier' isn't just an environment proxy).
#   Needs §3f (hmbin, HMASS_LABELS, NHB).
row_sfpk = np.array([r["row_sf_peak"] for r in records])
tsfpk    = np.array([r["t_sf_peak"] for r in records]) / 1e9

with np.errstate(divide="ignore", invalid="ignore"):
    _dmfull = np.where(P["masses.stellar"] > 0,
                       P["masses.dust"] / np.where(P["masses.stellar"] > 0, P["masses.stellar"], np.nan), np.nan)
_dpk = np.full(len(records), np.nan)
for i, (rw, cl) in enumerate(zip(row_sfpk, cols)):
    if rw >= 0:
        _dpk[i] = _dmfull[rw, cl]
dustatpk = np.where(_dpk > 0, np.log10(np.where(_dpk > 0, _dpk, np.nan)), np.nan)

_tbins = np.linspace(np.nanpercentile(tsfpk, 2), np.nanpercentile(tsfpk, 98), 16)
_dbins = np.linspace(np.nanpercentile(dustatpk, 2), np.nanpercentile(dustatpk, 98), 16)
fig, axes = plt.subplots(2, NHB, figsize=(4.3 * NHB, 7.2), squeeze=False)
for bi in range(NHB):
    bm = hmbin == bi
    ax = axes[0, bi]                                    # top: WHEN the SF peak happens
    for msk, c, lab in [(is_fast & bm, COL_A, LBL_A), (is_slow & bm, COL_B, LBL_B)]:
        v = tsfpk[msk]; v = v[np.isfinite(v)]
        if len(v):
            ax.hist(v, bins=_tbins, color=c, alpha=0.5, density=True, label=f"{lab} ({len(v)})")
            ax.axvline(np.median(v), color=c, ls="--", lw=1.2)
    ax.set_title(f"logMh {HMASS_LABELS[bi]} (n={int(bm.sum())})", fontsize=9); ax.legend(fontsize=6)
    sx = ax.secondary_xaxis("top", functions=(_t_to_z, _z_to_t)); sx.set_xlabel("z", fontsize=8)
    if bi == 0:
        ax.set_ylabel("t_sf_peak [Gyr]\npdf", fontsize=9)
    ax = axes[1, bi]                                    # bottom: HOW DUSTY at the SF peak
    for msk, c, lab in [(is_fast & bm, COL_A, LBL_A), (is_slow & bm, COL_B, LBL_B)]:
        v = dustatpk[msk]; v = v[np.isfinite(v)]
        if len(v):
            ax.hist(v, bins=_dbins, color=c, alpha=0.5, density=True, label=lab)
            ax.axvline(np.median(v), color=c, ls="--", lw=1.2)
    ax.set_xlabel("log dust/M* at SF peak"); ax.legend(fontsize=6)
    if bi == 0:
        ax.set_ylabel("log dust/M* @ peak\npdf", fontsize=9)
fig.suptitle(f"SF-peak premise — when (top) & how dusty (bottom) at the SF peak, {LBL_A} vs {LBL_B}, by halo mass", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "sfpeak_premise_by_halomass.png"), dpi=130, bbox_inches="tight"); plt.show()

print(f"{'halo bin':10s}{'t_sfpk '+LBL_A:>11s}{'t_sfpk '+LBL_B:>11s}{'dust@pk '+LBL_A:>13s}{'dust@pk '+LBL_B:>13s}")
for bi in range(NHB):
    bm = hmbin == bi
    print(f"{HMASS_LABELS[bi]:10s}{np.nanmedian(tsfpk[is_fast&bm]):11.2f}{np.nanmedian(tsfpk[is_slow&bm]):11.2f}"
          f"{np.nanmedian(dustatpk[is_fast&bm]):13.2f}{np.nanmedian(dustatpk[is_slow&bm]):13.2f}")

## IV.4 (§3j) · How much dust actually survives quenching? (threshold-free)

You don't need to pick a "dusty" cut to answer this — measure the **dust budget** directly. For every
galaxy we read `M_dust` at each evolutionary anchor and form **survival fractions**:

- **ledger / staircase** — median `M_dust/M_dust,peak` (and `dust/M*`) across `sf_peak → sft → qt →
  post_quench → z0`, fast vs slow, by halo mass, with 25–75% scatter: what fraction of the SF-built
  dust is left at each step.
- **transition fractions** — `D_QT/D_SFT` (survives the quench itself) and `D_z0/D_QT` (survives
  afterwards), as full distributions. If these are broad and *not* bimodal, that is exactly why a
  single "dusty" threshold is arbitrary — the continuous fraction is the better quantity.
- **what controls it** — a per-galaxy post-quench dust e-folding time `τ_decay`, and a standardized
  regression of the survival fraction on `logM*, logM_halo, log τ_q, Δt_build, log D_peak`, ranking
  which parameters set how much dust survives.

In [ ]:
# ── §3j·a. How much dust survives quenching? (threshold-free ledger + transition fractions) ──
#   Needs §3 (records, cols, P), §0 (mass bins, COL_A/B, LBL_A/B).
ANCH = ["sf_peak", "sft", "qt", "post_quench", "z0"]
_md_full = P["masses.dust"]; _ms_full = P["masses.stellar"]
_dpeak_rec = np.nanmax(np.where(_md_full[:, cols] > 0, _md_full[:, cols], np.nan), axis=0)  # per-rec peak dust

def _at_anchor(arr, stage):
    out = np.full(len(records), np.nan)
    for i, r in enumerate(records):
        rw = r[f"row_{stage}"]
        if rw >= 0:
            out[i] = arr[rw, cols[i]]
    return out

_Dm = {st: _at_anchor(_md_full, st) for st in ANCH}          # M_dust at each anchor
_Ms = {st: _at_anchor(_ms_full, st) for st in ANCH}          # M_star at each anchor
Dret = {st: _Dm[st] / _dpeak_rec for st in ANCH}             # M_dust / peak  (dust budget remaining)
with np.errstate(divide="ignore", invalid="ignore"):
    DsM = {st: np.log10(np.where((_Dm[st] > 0) & (_Ms[st] > 0), _Dm[st] / _Ms[st], np.nan)) for st in ANCH}

# halo-mass bins: canonical §7·setup definition (Part II); _lmh kept for §3j·b's regression
HLAB, hb = HMASS_BIN_LABELS, hmbin
_lmh = logmh_rec

# --- staircase: median D/D_peak and log dust/M* across anchors, fast vs slow, by halo mass ---
xi = np.arange(len(ANCH))
fig, axes = plt.subplots(2, len(HLAB), figsize=(4.3 * len(HLAB), 7.4), squeeze=False, sharex=True)
for bi in range(len(HLAB)):
    bm = hb == bi
    for row, D, ylab in [(0, Dret, "M_dust / M_dust,peak"), (1, DsM, "log dust/M*")]:
        ax = axes[row, bi]
        for msk, c, lab in [(is_fast & bm, COL_A, LBL_A), (is_slow & bm, COL_B, LBL_B)]:
            med = [np.nanmedian(D[st][msk]) for st in ANCH]
            q1  = [np.nanpercentile(D[st][msk], 25) for st in ANCH]
            q3  = [np.nanpercentile(D[st][msk], 75) for st in ANCH]
            ax.plot(xi, med, "o-", color=c, lw=2, label=lab); ax.fill_between(xi, q1, q3, color=c, alpha=0.18)
        ax.set_xticks(xi); ax.set_xticklabels(ANCH, rotation=30, fontsize=8)
        if row == 0:
            ax.set_title(f"logMh {HLAB[bi]} (n={int(bm.sum())})", fontsize=9); ax.set_ylim(0, 1)
        if bi == 0:
            ax.set_ylabel(ylab, fontsize=9)
    axes[0, bi].legend(fontsize=7)
fig.suptitle("Dust survival ledger: fraction of peak dust (top) & dust/M* (bottom) across the quenching sequence", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "dust_survival_staircase.png"), dpi=130, bbox_inches="tight"); plt.show()

# --- transition fractions (threshold-free): survives SFT->QT and QT->z0 ---
with np.errstate(divide="ignore", invalid="ignore"):
    s_quench = np.log10(np.where(Dret["sft"] > 0, Dret["qt"] / Dret["sft"], np.nan))   # log10 D_qt/D_sft (survives SFT->QT)
    s_post   = np.log10(np.where(Dret["qt"]  > 0, Dret["z0"] / Dret["qt"], np.nan))    # log10 D_z0/D_qt  (survives QT->z0)
print("median dust surviving the transition (dex; 0 = no loss, -1 = 90% lost):")
print(f"{'logMh':10s}{'SFT->QT '+LBL_A:>14s}{'SFT->QT '+LBL_B:>14s}{'QT->z0 '+LBL_A:>14s}{'QT->z0 '+LBL_B:>14s}")
for bi in range(len(HLAB)):
    bm = hb == bi
    print(f"{HLAB[bi]:10s}{np.nanmedian(s_quench[is_fast&bm]):14.2f}{np.nanmedian(s_quench[is_slow&bm]):14.2f}"
          f"{np.nanmedian(s_post[is_fast&bm]):14.2f}{np.nanmedian(s_post[is_slow&bm]):14.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, val, ttl in [(axes[0], s_quench, "log10 D_qt / D_sft  (survives SFT->QT)"),
                     (axes[1], s_post,   "log10 D_z0 / D_qt  (survives QT->z0)")]:
    for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
        v = val[msk]; v = v[np.isfinite(v)]
        if len(v):
            ax.hist(v, bins=30, color=c, alpha=0.5, density=True, label=f"{lab} (med {np.median(v):.2f})")
            ax.axvline(np.median(v), color=c, ls="--")
    ax.axvline(0, color="k", lw=0.8); ax.set_xlabel(ttl); ax.legend(fontsize=8)
axes[0].set_ylabel("pdf")
fig.suptitle("How much dust survives the quenching transition — full distributions (is there a natural threshold?)", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "dust_survival_fractions.png"), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ── §3j·b. What controls dust survival? per-galaxy decay timescale + standardized regression ──
#   Needs §3j·a (s_post, _dpeak_rec, _lmh, hb). Threshold-free: fit log10 M_dust vs (t - t_QT) after
#   quenching -> e-folding time; then regress the survival fraction on parameters to rank drivers.
_o2 = np.argsort(t_cosmic_yr); tg2 = t_cosmic_yr[_o2] / 1e9
_md_s = _md_full[:, cols][_o2, :]                      # (n_snap, n_rec) dust history, ascending
tqt_last = np.array([r["t_qt"] for r in records]) / 1e9

tau_decay = np.full(len(records), np.nan)              # e-folding time of dust after QT [Gyr]
for i in range(len(records)):
    dt = tg2 - tqt_last[i]; yv = _md_s[:, i]
    ok = (dt >= 0) & (dt <= 6) & np.isfinite(yv) & (yv > 0)
    if ok.sum() >= 4:
        sl = np.polyfit(dt[ok], np.log10(yv[ok]), 1)[0]    # dex per Gyr
        if sl < 0:
            tau_decay[i] = -0.4343 / sl                     # e-folding time [Gyr]
print(f"post-quench dust e-folding tau_decay [Gyr]: median {LBL_A}={np.nanmedian(tau_decay[is_fast]):.2f}, "
      f"{LBL_B}={np.nanmedian(tau_decay[is_slow]):.2f}  (defined for {int(np.isfinite(tau_decay).sum())}/{len(records)})")

# regression: what sets the post-quench dust survival (target = s_post = log10 D_z0/D_qt)?
_tauq = np.array([r["tau_q"] for r in records], float)
preds = {
    "logM*":     np.log10(np.where(_ms_full[0, cols] > 0, _ms_full[0, cols], np.nan)),
    "logMh":     _lmh,
    "log tau_q": np.log10(np.where(_tauq > 0, _tauq, np.nan)),
    "dt_build":  (np.array([r["t_qt_first"] for r in records]) - np.array([r["t_sf_peak"] for r in records])) / 1e9,
    "logDpeak":  np.log10(np.where(_dpeak_rec > 0, _dpeak_rec, np.nan)),
}
pnames = list(preds); Pm = np.column_stack([preds[k] for k in pnames]); Y = s_post.copy()
good = np.isfinite(Y) & np.all(np.isfinite(Pm), axis=1)
Pz = (Pm[good] - Pm[good].mean(0)) / Pm[good].std(0); Yg = Y[good]
try:
    import statsmodels.api as sm
    res = sm.OLS(Yg, sm.add_constant(Pz)).fit()
    coefs, pvs, r2 = res.params[1:], res.pvalues[1:], res.rsquared
except Exception:
    beta = np.linalg.lstsq(np.column_stack([np.ones(len(Yg)), Pz]), Yg, rcond=None)[0]
    coefs, pvs, r2 = beta[1:], np.full(len(pnames), np.nan), np.nan
print(f"\nwhat controls post-quench dust survival (log D_z0/D_qt) — standardized OLS (R2={r2:.2f}, N={int(good.sum())}):")
print("  beta per +1 SD (more negative = stronger dust LOSS):")
for nm, b, p in sorted(zip(pnames, coefs, pvs), key=lambda t: -abs(t[1])):
    print(f"  {nm:10s} beta={b:+.3f}" + (f"  p={p:.1e}" if np.isfinite(p) else ""))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
    v = tau_decay[msk]; v = v[np.isfinite(v) & (v < 20)]
    if len(v):
        ax[0].hist(v, bins=30, color=c, alpha=0.5, density=True, label=f"{lab} (med {np.median(v):.1f})")
        ax[0].axvline(np.median(v), color=c, ls="--")
ax[0].set_xlabel(r"post-quench dust e-folding $\tau_{\rm decay}$ [Gyr]"); ax[0].set_ylabel("pdf"); ax[0].legend(fontsize=8)
order = np.argsort(np.abs(coefs))
ax[1].barh(np.array(pnames)[order], np.array(coefs)[order], color="C0")
ax[1].axvline(0, color="k", lw=0.8); ax[1].set_xlabel(r"std. OLS $\beta$ on log $D_{z0}/D_{QT}$")
ax[1].set_title("drivers of dust survival")
fig.suptitle("Post-quench dust decay timescale & what controls how much dust survives", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "dust_survival_drivers.png"), dpi=130, bbox_inches="tight"); plt.show()

## IV.5 (§3i) · What is a dusty *quiescent* galaxy? — connecting quench timescale, dust budget & cosmic time

The synthesis. We build one row per **galaxy × snapshot where the galaxy is passive** (the §0
criterion `sSFR < 0.2/t_H`), and model whether it is **also dusty** (`M_dust/M_dust,peak > RETAIN_THR` $= 10^{-1.5}\approx 3\%$)
from: the snapshot epoch `t_cos` (cosmic time, Gyr), **time since quench** `Δt_q = t − t_QT` (the master clock), the **build
timescale** `Δt_build = t_QT − t_sf_peak`, the **dust budget** `D_qt = log M_dust` at quench,
`logM*`, `logM_halo` (both *at the observed snapshot*), and **quench speed** (`log τ_q`, fast/slow).

Three views, increasing in rigor:
1. **Empirical maps** — `P(dusty | quiescent)` vs cosmic time × {halo mass, stellar mass, Δt_build}, faceted fast/slow. Reads off "which type is dusty-and-quiescent at epoch t".
2. **Logistic regression** — odds ratios rank which property drives residual dust *with the others held fixed* (essential, given the speed/mass confounds), plus `P(dusty|t)` partial-dependence.
3. **Dust-survival timescale** `τ_dust` — the `Δt_q` where `P(dusty)` falls to half its initial post-quench value (the absolute 0.5 level is unreachable at the ~5% overall dusty fraction), per halo/stellar/speed cell: the mechanistic number that, times the quench rate at z, sets the dusty-quiescent abundance.

In [ ]:
# ── §3i·a. Build the quiescent-snapshot table + empirical P(dusty|quiescent) maps ──
#   Needs §3f (tg, zg_asc, QT, _o, _mstar, _mdust, retain, dustyR, hmbin, HMASS_EDGES/LABELS).
import pandas as pd

n_snap = len(tg); n_rec = len(records)
_sfr_s = P["sfr"][:, cols][_o, :]
with np.errstate(divide="ignore", invalid="ignore"):
    ssfr_s  = np.where(_mstar > 0, _sfr_s / _mstar, np.nan)
    logM_s  = np.log10(np.where(_mstar > 0, _mstar, np.nan))
    _mh_s   = P["masses.total"][:, cols][_o, :]
    logMh_s = np.log10(np.where(_mh_s > 0, _mh_s, np.nan))
tH_s = tg * 1e9                                              # t_H(z_s) = cosmic age [yr]
quiescent = np.isfinite(ssfr_s) & (ssfr_s < PASSIVE_FACTOR / tH_s[:, None])   # (n_snap, n_rec)

# per-galaxy scalars (quench reference = FIRST / main quench)
t_qref   = QT["first"]                                       # Gyr
t_sfpk_r = np.array([r["t_sf_peak"] for r in records]) / 1e9
dt_build = t_qref - t_sfpk_r                                 # sf_peak -> qt timescale [Gyr]
tau_q_r  = np.array([r["tau_q"] for r in records])           # yr (SFT->QT)
ltau_r   = np.log10(np.where(tau_q_r > 0, tau_q_r, np.nan))
_sref    = np.array([int(np.argmin(np.abs(tg - tq))) for tq in t_qref])   # snapshot nearest t_qref
_dbud    = _mdust[_sref, np.arange(n_rec)]
D_qt_r   = np.log10(np.where(_dbud > 0, _dbud, np.nan))      # log M_dust at quench (dust budget)

SS, II = np.where(quiescent)                                 # snapshot idx, record idx
tab = pd.DataFrame(dict(
    t_cos    = tg[SS],            # snapshot epoch as COSMIC TIME [Gyr] (use zg_asc[SS] for redshift)
    dt_q     = tg[SS] - t_qref[II],
    dt_build = dt_build[II],
    D_qt     = D_qt_r[II],
    logM     = logM_s[SS, II],
    logMh    = logMh_s[SS, II],
    ltau     = ltau_r[II],
    fast     = is_fast[II].astype(float),
    dusty    = dustyR[SS, II].astype(float),
)).replace([np.inf, -np.inf], np.nan).dropna()
print(f"§3i quiescent-snapshot table: {len(tab)} rows / {n_rec} galaxies; "
      f"overall dusty fraction = {tab['dusty'].mean():.3f}")

# ---- empirical 2D maps: P(dusty|quiescent) vs redshift x property, faceted fast/slow ----
_tb = np.linspace(0, np.nanpercentile(tab["t_cos"], 98), 13)
def _pmap(ax, sub, ycol, yb, title, ylabel, nmin=15):
    H, _, _ = np.histogram2d(sub["t_cos"], sub[ycol], bins=[_tb, yb])
    S, _, _ = np.histogram2d(sub["t_cos"], sub[ycol], bins=[_tb, yb], weights=sub["dusty"])
    with np.errstate(invalid="ignore"):
        Pr = np.where(H >= nmin, S / H, np.nan)
    pcm = ax.pcolormesh(_tb, yb, Pr.T, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9); ax.set_xlabel("cosmic time t [Gyr]"); ax.set_ylabel(ylabel)
    return pcm

PMAP_PROPS = [("logMh",   np.linspace(11.5, 14.0, 14), r"log $M_{\rm halo}$"),
         ("logM",    np.linspace(9.5, 11.5, 14),  r"log $M_*$"),
         ("dt_build", np.linspace(0, np.nanpercentile(tab["dt_build"], 98), 14), r"$\Delta t_{\rm build}$ [Gyr]")]
fig, axes = plt.subplots(len(PMAP_PROPS), 2, figsize=(11, 4 * len(PMAP_PROPS)), squeeze=False, constrained_layout=True)
for pi, (ycol, yb, ylab) in enumerate(PMAP_PROPS):
    pcm = None
    for ci, (msk, lab) in enumerate([(tab["fast"] == 1, LBL_A), (tab["fast"] == 0, LBL_B)]):
        pcm = _pmap(axes[pi, ci], tab[msk], ycol, yb, lab, ylab)
    fig.colorbar(pcm, ax=axes[pi, :].tolist(), label="P(dusty | quiescent)", shrink=0.8)
fig.suptitle("P(dusty | quiescent) vs cosmic time and galaxy property (left = fast, right = slow)")
plt.savefig(figpath( "pdusty_maps.png"), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ── §3i·b. Logistic regression: which property drives residual dust (confounds controlled)? ──
#   Needs §3i·a (tab). Standardized continuous predictors -> odds ratio per +1 SD.
#   Fits TWO models — full (with t_cos) and reduced (t_cos dropped, so the quench clock dt_q carries the
#   cosmic-time information) — and compares their odds ratios side by side.
try:
    import statsmodels.api as sm
    _HAVE_SM = True
except Exception:
    from sklearn.linear_model import LogisticRegression
    _HAVE_SM = False

FEATS = ["t_cos", "dt_q", "dt_build", "D_qt", "logM", "logMh", "ltau"]   # continuous (+ binary 'fast')
Xc = tab[FEATS].to_numpy(float)
mu = Xc.mean(0); sd = Xc.std(0); sd[sd == 0] = 1.0
Xs = (Xc - mu) / sd
fastcol = tab["fast"].to_numpy(float)
X  = np.column_stack([Xs, fastcol])
y  = tab["dusty"].to_numpy(float)
names = FEATS + ["fast"]

# ── multicollinearity diagnostics: predictor correlation matrix + VIF ──
#   VIF_j = (R^-1)_jj from the predictor correlation matrix R. VIF>5 (esp. >10) means feature j is
#   largely explained by the others, so its odds ratio below is unstable -> trust the maps (§3i·a)
#   over the individual coefficient, or drop/merge the collinear feature and refit.
Cmat = np.corrcoef(X, rowvar=False)
try:
    vif = np.diag(np.linalg.inv(Cmat))
except np.linalg.LinAlgError:
    vif = np.diag(np.linalg.pinv(Cmat)); print("  (near-singular correlation matrix -> pinv used)")
print("predictor collinearity (VIF; >5 concerning, >10 severe):")
for nm, v in sorted(zip(names, vif), key=lambda t: -t[1]):
    flag = "  <-- severe" if v > 10 else ("  <- high" if v > 5 else "")
    print(f"  {nm:9s}  VIF={v:6.2f}{flag}")

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(Cmat, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{Cmat[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(Cmat[i, j]) > 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=ax, label="Pearson r", shrink=0.85)
ax.set_title("Predictor correlation matrix (standardized features)")
plt.tight_layout(); plt.savefig(figpath( "pdusty_logit_vif_corr.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── fit a logistic model on a chosen subset of the continuous features (+ binary 'fast') ──
def _fit(feats):
    idx = [FEATS.index(f) for f in feats]
    Xsub = np.column_stack([Xs[:, idx], fastcol]); nm = feats + ["fast"]
    if _HAVE_SM:
        r = sm.Logit(y, sm.add_constant(Xsub)).fit(disp=False)
        params, pvals, r2 = r.params[1:], r.pvalues[1:], r.prsquared
        pred = lambda vec, rr=r: float(rr.predict(np.concatenate([[1.0], vec]))[0])
    else:
        clf = LogisticRegression(max_iter=1000).fit(Xsub, y)
        params, pvals, r2 = clf.coef_[0], np.full(len(nm), np.nan), clf.score(Xsub, y)
        pred = lambda vec, cc=clf: float(cc.predict_proba(vec[None, :])[0, 1])
    return dict(names=nm, idx=idx, params=params, pvals=pvals, r2=r2, predict=pred)

FULL = _fit(FEATS)
REDU = _fit([f for f in FEATS if f != "t_cos"])
_score = "pseudo-R2" if _HAVE_SM else "accuracy"
print(f"\nlogistic P(dusty|quiescent)  ({_score}: full={FULL['r2']:.3f}, reduced(no t_cos)={REDU['r2']:.3f}, N={len(y)})")
print("  odds ratio per +1 SD (per unit for 'fast'); >1 raises dusty-quiescent probability")
_of = dict(zip(FULL["names"], zip(FULL["params"], FULL["pvals"])))
_or = dict(zip(REDU["names"], zip(REDU["params"], REDU["pvals"])))
print(f"  {'feature':9s}{'OR_full':>10s}{'OR_noT':>10s}{'  p_full':>10s}{'  p_noT':>10s}")
for nm in names:
    bf = _of.get(nm); br = _or.get(nm)
    sf = f"{np.exp(bf[0]):10.2f}" if bf else f"{'--':>10s}"
    sr = f"{np.exp(br[0]):10.2f}" if br else f"{'--':>10s}"
    pf = f"{bf[1]:10.1e}" if (bf and np.isfinite(bf[1])) else f"{'':>10s}"
    pr = f"{br[1]:10.1e}" if (br and np.isfinite(br[1])) else f"{'':>10s}"
    print(f"  {nm:9s}{sf}{sr}{pf}{pr}")

# ── partial dependence, fast vs slow: full model vs epoch t_cos (left), reduced model vs quench clock (right) ──
def _pd(fit, varname, grid, fastval):
    medw = (np.median(Xc, 0) - mu) / sd                     # full-length standardized medians
    iv = FEATS.index(varname); out = []
    for g in grid:
        fv = medw.copy(); fv[iv] = (g - mu[iv]) / sd[iv]
        out.append(fit["predict"](np.concatenate([fv[fit["idx"]], [fastval]])))
    return out

tgrid = np.linspace(np.nanpercentile(tab["t_cos"], 2), np.nanpercentile(tab["t_cos"], 98), 40)
dg = np.linspace(np.nanpercentile(tab["dt_q"], 2), np.nanpercentile(tab["dt_q"], 98), 40)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for fastval, c, lab in [(1.0, COL_A, LBL_A), (0.0, COL_B, LBL_B)]:
    axes[0].plot(tgrid, _pd(FULL, "t_cos", tgrid, fastval), color=c, lw=2, label=lab)
    axes[1].plot(dg, _pd(REDU, "dt_q", dg, fastval), color=c, lw=2, label=lab)
axes[0].set_xlabel("cosmic time t [Gyr]"); axes[0].set_title("FULL model — P(dusty) vs epoch (others at median)")
axes[1].set_xlabel("time since quench  t - t_QT [Gyr]"); axes[1].set_title("REDUCED model (no z) — P(dusty) vs quench clock")
for a in axes:
    a.set_ylabel("P(dusty | quiescent)"); a.set_ylim(0, 1); a.legend()
plt.tight_layout(); plt.savefig(figpath( "pdusty_logit_partial.png"), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ── §3i·c. Dust-survival timescale tau_dust = time-since-quench where P(dusty) = 0.5 ──
#   Needs §3i·a (tab), §3f (HMASS_EDGES/LABELS), §0 (MASS_BIN_EDGES/LABELS).
DTB = np.linspace(0, 8, 33)                                  # dt_q bins [Gyr]
_xc = 0.5 * (DTB[1:] + DTB[:-1])

def _tau50(sub, nmin=10):
    """time-since-quench (Gyr) where the dusty fraction falls to HALF its post-quench value.
       Absolute P=0.5 is unreachable here (overall dusty frac ~5%), so the threshold is
       RELATIVE: half of P(dusty) in the first populated dt_q bin (linear interp)."""
    dq = sub["dt_q"].to_numpy(); du = sub["dusty"].to_numpy()
    idx = np.digitize(dq, DTB)
    p = np.array([du[idx == k].mean() if np.sum(idx == k) >= nmin else np.nan for k in range(1, len(DTB))])
    fin = np.where(np.isfinite(p))[0]
    if fin.size == 0 or not (p[fin[0]] > 0):
        return np.nan                                       # no post-quench dust -> undefined
    half = 0.5 * p[fin[0]]                                   # half the post-quench dusty fraction
    below = np.where(np.isfinite(p) & (p < half))[0]
    below = below[below > fin[0]]                            # only AFTER the first populated bin
    if below.size == 0:
        return _xc[fin[-1]]                                  # still > half at the end of the window
    k = below[0]
    if k == 0 or not np.isfinite(p[k - 1]) or p[k - 1] == p[k]:
        return _xc[k]
    return _xc[k - 1] + (half - p[k - 1]) * (_xc[k] - _xc[k - 1]) / (p[k] - p[k - 1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (edges, labels, name, col) in zip(
        axes, [(HMASS_EDGES, HMASS_LABELS, r"halo mass", "logMh"),
               (MASS_BIN_EDGES, MASS_BIN_LABELS, r"stellar mass", "logM")]):
    tf, ts = [], []
    print(f"tau_dust [Gyr] (P=half-peak) by {name}:  {'bin':12s}{LBL_A:>7s}{LBL_B:>7s}")
    for b in range(len(labels)):
        sb = tab[(tab[col] >= edges[b]) & (tab[col] < edges[b + 1])]
        a, s = _tau50(sb[sb.fast == 1]), _tau50(sb[sb.fast == 0])
        tf.append(a); ts.append(s)
        print(f"{'':38s}{labels[b]:12s}{a:7.2f}{s:7.2f}")
    x = np.arange(len(labels))
    ax.bar(x - 0.2, np.nan_to_num(tf), 0.4, color=COL_A, label=LBL_A)
    ax.bar(x + 0.2, np.nan_to_num(ts), 0.4, color=COL_B, label=LBL_B)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
    ax.set_title(f"$\\tau_{{\\rm dust}}$ by {name}"); ax.set_ylabel(r"$\tau_{\rm dust}$ [Gyr] (P=$\frac{1}{2}$ post-quench)")
    ax.legend(fontsize=8)
fig.suptitle("Dust-survival timescale after quenching — how long a quenched galaxy stays dusty", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "tau_dust_by_mass_speed.png"), dpi=130, bbox_inches="tight"); plt.show()

# Part 6 — CGM

The circumgalactic reservoir at each critical point: M_CGM (raw and satellite-corrected), cold fraction, satellite ISM contribution, CGM temperature, and the radial surface-density profiles of gas & dust.

## V.3 (§7) · CGM reservoir & satellites at the critical points — fast vs slow, binned by halo mass

Parallel to §4 (which facets by z=0 stellar mass): every comparison here is faceted by z=0
**halo** mass (parent-halo total, `masses.total`), split fast vs slow.

Catalog-level CGM proxies at each critical-point anchor:
- **raw**   `M_CGM     = M_halo_gas − M_central_ISM`  (counts satellite ISM as CGM)
- **clean** `M_CGM_sub = M_halo_gas − Σ member-galaxy ISM`  (satellite-corrected, uses §7a)

Order: §7a builds the per-halo satellite catalogue (**needs `_PROG_INDEX` from §5·pre**), then
§7b–7d use it for both the raw and satellite-corrected CGM. Requires the halo fields added to
§1b `PROPS["halo_data"]` — rerun `BUILD_HISTORY` once if keys are reported missing.

In [ ]:
# 7b. CGM diagnostics — catalog-level reservoir at each stage anchor (fast vs slow).
#   raw   M_CGM     = M_halo_gas - M_central_ISM          (counts satellite ISM as CGM)
#   clean M_CGM_sub = M_halo_gas - sum(all member ISM)    (satellite-corrected; uses §7a SAT)
HALO_KEYS = {
    "Mgas":  "halo_data/dicts/masses.gas",
    "Mstar": "halo_data/dicts/masses.stellar",
    "MHI":   "halo_data/dicts/masses.HI",
    "MH2":   "halo_data/dicts/masses.H2",
    "Mdust": "halo_data/dicts/masses.dust",
    "Z":     "halo_data/dicts/metallicities.mass_weighted",
    "Tcgm":  "halo_data/dicts/temperatures.mass_weighted_cgm",
    "Tvir":  "halo_data/dicts/virial_quantities.temperature",
    "Mtot":  "masses.total",
}
_missing = [v for v in HALO_KEYS.values() if v not in P]
if _missing:
    print("MISSING halo history keys -> set BUILD_HISTORY=True and rerun §1b with these in"
          " PROPS['halo_data']:")
    for m in _missing:
        print("   ", m)
else:
    print("all halo CGM keys present in the history.")

_sij = {st: i for i, st in enumerate(STAGES)}
_have_sat = ("SAT" in globals()) and ("Mgal_gas_halo" in SAT)
if not _have_sat:
    print("note: §7a satellite product not found -> M_CGM_sub will be NaN (run §7a first).")

# BH accretion / jet mode (from the §4b BH history). Jet-mode in SIMBA = the central BH is
# massive AND in the low-Eddington kinetic branch -> the dominant CGM-heating channel.
JET_LOGMBH, JET_FEDD = 7.5, 0.2      # log10(M_BH/Msun) > 7.5  AND  f_Edd < 0.2
_have_bh = ("BH" in globals()) and ("bh_stage" in globals())
if not _have_bh:
    print("note: §4b BH history not found -> BH/jet diagnostics will be NaN (run §4b first).")

def _hstage(short, stage):
    """Halo property (short alias) at the stage anchor row; NaN if key absent/missing."""
    key = HALO_KEYS[short]
    out = np.full(len(records), np.nan)
    if key not in P:
        return out
    arr = P[key]
    rows = np.array([r[f"row_{stage}"] for r in records])
    for i, (row, col) in enumerate(zip(rows, cols)):
        if row >= 0:
            out[i] = arr[row, col]
    return out

DIAGS_CGM = {}
for st in STAGES:
    Mhg  = _hstage("Mgas", st)
    Mhhi = _hstage("MHI", st); Mhh2 = _hstage("MH2", st)
    Mhtot = _hstage("Mtot", st)
    gg  = DIAGS[st]["Mgas"]                        # central-galaxy ISM gas (from §4 DIAGS)
    ghi = DIAGS[st]["MHI"]; gh2 = DIAGS[st]["MH2"]
    M_cgm = np.clip(Mhg - gg, 0, None)            # raw: halo gas - central ISM
    cold_h = Mhhi + Mhh2
    cold_g = np.nan_to_num(ghi) + np.nan_to_num(gh2)
    M_cgm_cold = np.clip(cold_h - cold_g, 0, None)
    # satellite-corrected: subtract ALL member-galaxy ISM gas
    if _have_sat:
        Mgal_halo = SAT["Mgal_gas_halo"][:, _sij[st]]
        M_cgm_sub = np.clip(Mhg - Mgal_halo, 0, None)
    else:
        M_cgm_sub = np.full(len(records), np.nan)
    # BH accretion at this stage anchor + jet-mode indicator (NaN where BH state is missing)
    if _have_bh:
        bh_md = bh_stage(BH["bh_mdot"], st)
        bh_fe = bh_stage(BH["bh_fedd"], st)
        bh_mb = bh_stage(BH["bh_mass"], st)
        _bh_ok = np.isfinite(bh_mb) & np.isfinite(bh_fe)
        bh_jet = np.where(_bh_ok, ((bh_mb > 10 ** JET_LOGMBH) & (bh_fe < JET_FEDD)).astype(float), np.nan)
    else:
        bh_md = bh_fe = bh_mb = bh_jet = np.full(len(records), np.nan)
    Mstar = DIAGS[st]["Mstar"]
    DIAGS_CGM[st] = {
        "bh_mdot":  bh_md,                                               # central BH accretion rate [Msun/yr]
        "f_Edd":    bh_fe,                                               # Eddington ratio
        "M_BH":     bh_mb,                                               # BH mass [Msun]
        "jet_mode": bh_jet,                                              # 1 if in jet mode, else 0 (NaN if no BH)
        "M*": Mstar,
        "sSFR": np.where(Mstar > 0, gather_stage("sfr", st) / Mstar, np.nan),  # central-galaxy sSFR
        "M_CGM":         M_cgm,                                          # raw CGM gas
        "M_CGM_sub":     M_cgm_sub,                                      # satellite-corrected CGM gas
        "M_CGM/M*":      np.where(Mstar > 0, M_cgm / Mstar, np.nan),
        "M_CGM_sub/M*":  np.where(Mstar > 0, M_cgm_sub / Mstar, np.nan),
        "fCGM_halo":     np.where(Mhtot > 0, M_cgm / Mhtot, np.nan),     # raw CGM / halo mass
        "fCGM_sub_halo": np.where(Mhtot > 0, M_cgm_sub / Mhtot, np.nan), # clean CGM / halo mass
        "fsat_ISM":      np.where(M_cgm > 0, (M_cgm - M_cgm_sub) / M_cgm, np.nan),  # satellite share of raw proxy
        "fCGM_gas":      np.where(Mhg > 0, M_cgm / Mhg, np.nan),
        "fbaryon_gas":   np.where(Mhtot > 0, Mhg / Mhtot, np.nan),
        "M_CGM_cold":    M_cgm_cold,
        "fcold_CGM":     np.where(M_cgm > 0, M_cgm_cold / M_cgm, np.nan),
        "Z_halo":        _hstage("Z", st),
        "T_CGM":         _hstage("Tcgm", st),   # CAESAR mass-weighted CGM temperature [K]
        "T_vir":         _hstage("Tvir", st),   # halo virial temperature [K]
    }
print("CGM diagnostics built for stages:", list(DIAGS_CGM.keys()))
if _have_bh:
    print(f"BH/jet diagnostics attached (jet = logM_BH>{JET_LOGMBH} & f_Edd<{JET_FEDD}).")


In [ ]:
# 7c. CGM reservoir distributions, fast vs slow — raw vs satellite-corrected, by halo-mass bin
for q, logy in [("M_CGM", True), ("M_CGM_sub", True), ("fsat_ISM", False), ("fcold_CGM", False)]:
    violin_stage_g(DIAGS_CGM, q, hmbin, HMASS_BIN_LABELS, logy=logy,
                   fname=f"cgm_dist_{q.replace('/','_')}.png")

In [ ]:
# 7d. Median CGM stage evolution, fast vs slow — raw and satellite-corrected, by halo-mass bin
for q, logy in [("M_CGM", True), ("M_CGM_sub", True),
                ("M_CGM/M*", True), ("M_CGM_sub/M*", True),
                ("fCGM_halo", False), ("fCGM_sub_halo", False),
                ("fsat_ISM", False), ("fbaryon_gas", False),
                ("fcold_CGM", False), ("Z_halo", True),
                ("T_CGM", True), ("T_vir", True),
               ("M*", True), ("sSFR", True)]:
    median_track_g(DIAGS_CGM, q, hmbin, HMASS_BIN_LABELS, logy=logy,
                   fname=f"cgm_track_{q.replace('/','_')}.png")

In [ ]:
# 7e. §4b median stage evolution (fast vs slow), but faceted by HALO-mass bin instead of
# z=0 stellar mass — same galaxy quantities as §4b; shows which halos these QGs sit in.
for q in ["Mdust", "MH2", "MHI"]:
    median_track_g(DIAGS, q, hmbin, HMASS_BIN_LABELS, logy=True,
                   fname=f"halobin_track_{q.replace('/','_')}.png")

In [ ]:
# 7f. Satellites across stages, fast vs slow — one column per halo-mass bin.
median_track_g(DIAGS_SAT, "Nsat", hmbin, HMASS_BIN_LABELS, logy=False, fname="sat_track_Nsat.png")
for q in ["Msat_gas", "Msat_dust", "Msat_cold"]:
    median_track_g(DIAGS_SAT, q, hmbin, HMASS_BIN_LABELS, logy=True, fname=f"sat_track_{q}.png")

In [ ]:
# 7h. CGM temperature across stages, fast vs slow — one column per halo-mass bin.
if CGMT_OK:
    median_track_g(DIAGS_CGMT, "T_CGM", hmbin, HMASS_BIN_LABELS, logy=True, fname="cgmT_track_T.png")
    for q in ["fhot", "fwarm", "fcold"]:
        median_track_g(DIAGS_CGMT, q, hmbin, HMASS_BIN_LABELS, logy=False, fname=f"cgmT_track_{q}.png")
    violin_stage_g(DIAGS_CGMT, "T_CGM", hmbin, HMASS_BIN_LABELS, logy=True, fname="cgmT_dist_T.png")
else:
    print("CGMT_OK is False -> build/merge the CGM-temperature product first (see §7g).")

### V.4 (§7p) · CGM radial surface-density profiles — dust & gas across stages

The CGM analogue of the ISM §5e panels: face-on **projected** $\Sigma(R)$ of the parent halo's
**diffuse (non-star-forming) gas and dust**, computed in the central galaxy's stellar principal
frame out to ~300 kpc. Built by `build_profiles_job.py` + `merge_profiles.py`, which
**reuse the existing `dust_profile_plan.hdf5`** (same galaxies/stages/snapshots) — no new plan:

```
mkdir -p logs && jid=$(sbatch --parsable submit_profiles.sh) && \
sbatch --dependency=afterok:$jid submit_profiles_merge.sh
```

Radial extent/binning live in `submit_profiles.sh` (`CGM_RMAX_KPC=300`, `CGM_NBINS_R=30`). The
cell shows **(A)** the sample-median $\Sigma(R)$ per stage for gas/dust/HI/H$_2$, and **(B)** the
example-galaxy profiles per class per mass bin (the §5e view), for gas and dust.

In [ ]:
# 7p · CGM radial surface-density profiles of gas & dust across stages (projected Σ, mirror §5e).
#   Product: cgm_profiles_allcrit.hdf5 from build_profiles_job.py + merge_profiles.py
#   (reuses dust_profile_plan.hdf5). Schema: rmid (nbins); sigma_{gas,dust,HI,H2} (nrec,nst,nbins);
#   R50_gas/R50_dust/M_cgm/Ngas (nrec,nst); gid; 'stages' attr.
CGMPROF_PATH = os.path.join(SFHDIR, "cgm_profiles_allcrit.hdf5")
if os.path.exists(CGMPROF_PATH):
    with h5py.File(CGMPROF_PATH, "r") as f:
        CGP_STAGES = list(f.attrs["stages"].split(","))
        CGP = {k: f[k][:] for k in f.keys()}
    CGP_RMID = CGP["rmid"]
    CGP_OK = True
    n_ok = int(np.sum(np.isfinite(CGP["R50_gas"])))
    print(f"loaded {CGMPROF_PATH}: stages={CGP_STAGES}; rmax={CGP_RMID[-1]:.0f} kpc, "
          f"{len(CGP_RMID)} bins; {n_ok} finite CGM R50_gas entries")
else:
    CGP, CGP_OK = None, False
    print("no CGM-profile product yet. It reuses the existing dust_profile_plan.hdf5 — on the cluster run:")
    print("  mkdir -p logs && jid=$(sbatch --parsable submit_profiles.sh) && \\")
    print("  sbatch --dependency=afterok:$jid submit_profiles_merge.sh")
    print("then re-run this cell (no new plan needed).")

# (A) sample-median Σ(R) per stage — gas, dust (and HI, H2): the population radial distribution
if CGP_OK:
    comps = [(c, l) for c, l in [("gas", r"\Sigma_{\rm gas}"), ("dust", r"\Sigma_{\rm dust}"),
                                 ("HI", r"\Sigma_{\rm HI}"), ("H2", r"\Sigma_{\rm H_2}")]
             if f"sigma_{c}" in CGP]
    fig, axs = plt.subplots(1, len(comps), figsize=(4.2 * len(comps), 4.0), sharex=True)
    axs = np.atleast_1d(axs)
    for ai, (c, lab) in enumerate(comps):
        sig = CGP[f"sigma_{c}"]                                   # (nrec, nst, nbins)
        for si, st in enumerate(CGP_STAGES):
            with np.errstate(all="ignore"):
                prof = np.where(sig[:, si, :] > 0, sig[:, si, :], np.nan)
                med = np.nanmedian(prof, axis=0)
            if np.all(~np.isfinite(med)):
                continue
            axs[ai].plot(CGP_RMID, med, label=st)
        axs[ai].set_yscale("log"); axs[ai].set_xlabel("R [kpc]")
        axs[ai].set_title(f"CGM ${lab}$"); axs[ai].legend(fontsize=7)
    axs[0].set_ylabel(r"median $\Sigma$ [$M_\odot$ kpc$^{-2}$]")
    fig.suptitle("CGM radial surface-density profiles across stages (sample median)", y=1.02)
    plt.tight_layout()
    plt.savefig(figpath( "cgm_sigma_profiles_median.png"), dpi=130, bbox_inches="tight")
    plt.show()

# (B) example galaxies (most finite CGM stages) per class per mass bin — direct §5e analogue
if CGP_OK:
    for bi in range(NBINS):
        binmask = mbin == bi
        for cmask, name in [(is_fast & binmask, LBL_A), (is_slow & binmask, LBL_B)]:
            ridx = np.where(cmask)[0]
            if len(ridx) == 0:
                continue
            best = max(ridx, key=lambda ri: int(np.sum(np.isfinite(CGP["R50_gas"][ri]))))
            if np.sum(np.isfinite(CGP["R50_gas"][best])) == 0:
                continue
            fig, axx = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
            for ai, (c, lab) in enumerate([("gas", r"\Sigma_{\rm gas}"), ("dust", r"\Sigma_{\rm dust}")]):
                for si, st in enumerate(CGP_STAGES):
                    sig = CGP[f"sigma_{c}"][best, si]
                    if np.all(~np.isfinite(sig)) or np.nansum(sig) <= 0:
                        continue
                    axx[ai].plot(CGP_RMID, sig, label=st)
                axx[ai].set_yscale("log"); axx[ai].set_xlabel("R [kpc]")
                axx[ai].set_title(f"{name} {MASS_BIN_LABELS[bi]} — CGM ${lab}$"); axx[ai].legend(fontsize=7)
            axx[0].set_ylabel(r"$\Sigma$ [$M_\odot$ kpc$^{-2}$]")
            fig.suptitle(f"example galaxy (gid {int(CGP['gid'][best])}) CGM profiles", y=1.03)
            _slug = "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")
            plt.tight_layout()
            plt.savefig(figpath( f"cgm_sigma_example_bin{bi}_{_slug}.png"),
                        dpi=130, bbox_inches="tight")
            plt.show()

# Part 7 — Kinematics, morphology & ISM distribution

Where the ISM sits and how concentrated it is: dust / HI / H₂ radial extension across stages and on the quenching clock, example face-on Σ(R) profiles, and the r20/r80, r20/r50 concentration of dust / HI / H₂ / stars from the particle product. (Disc-stability & spin-misalignment kinematics are *event-clocked* and live in Part 9, where the event engine is built.)

In [ ]:
# 5c. Dust extension across stages (WHOLE sample): median R50_dust and R50_dust/R50_star,
# fast vs slow, one column per mass bin.
def _dp_stage(arr2d, st):
    if st not in DP_STAGES:                       # stage absent from this product build -> NaN
        return np.full(arr2d.shape[0], np.nan)
    return arr2d[:, DP_STAGES.index(st)]

def dust_track(arr2d, ylabel, logy, fname):
    xs = np.arange(len(STAGES))
    fig, axs = plt.subplots(1, NBINS, figsize=(5.0 * NBINS, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(NBINS):
        ax = axs[bi]; binmask = mbin == bi
        for cmask, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            med, lo, hi = [], [], []
            for st in STAGES:
                v = _dp_stage(arr2d, st)[cmask]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    med.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
                vv = np.log10(v) if logy else v
                med.append(np.median(vv)); lo.append(np.percentile(vv, 25)); hi.append(np.percentile(vv, 75))
            ax.plot(xs, med, "-o", color=c, label=lab); ax.fill_between(xs, lo, hi, color=c, alpha=0.15)
        if not logy:
            ax.axhline(1, ls=":", c="0.6", lw=0.8)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(ylabel)
    plt.tight_layout(); plt.savefig(figpath( fname), dpi=130, bbox_inches="tight"); plt.show()

if DP_OK:
    dust_track(DP["R50_dust"], r"median $\log_{10} R_{50}^{\rm dust}$ [kpc]", True, "dustR50_track.png")
    dust_track(DP_RATIO,       r"median $R_{50}^{\rm dust}/R_{50}^{\star}$", False, "dustR50_ratio_track.png")

In [ ]:
# 5d. Dust extension AROUND QUENCHING: each galaxy's per-stage R50 placed on the quenching
# clock (QT-aligned and phase-normalized), binned-median fast vs slow, per mass bin.
def _clock_points(arr2d, ridx, mode, logy):
    xs, ys = [], []
    for ri in ridx:
        r = records[ri]
        for si, st in enumerate(DP_STAGES):
            v = arr2d[ri, si]
            if not np.isfinite(v) or (logy and v <= 0):
                continue
            tt = r[f"t_{st}"]
            if not np.isfinite(tt):
                continue
            if mode == "qt":
                xs.append((tt - r["t_qt"]) / 1e9)
            else:
                den = r["t_qt"] - r["t_sft"]
                if den <= 0:
                    continue
                xs.append((tt - r["t_sft"]) / den)
            ys.append(np.log10(v) if logy else v)
    return np.asarray(xs), np.asarray(ys)

def _binned_median(x, y, xedges):
    mid = 0.5 * (xedges[:-1] + xedges[1:]); med = np.full(len(mid), np.nan); lo = med.copy(); hi = med.copy()
    idx = np.digitize(x, xedges) - 1
    for b in range(len(mid)):
        yy = y[idx == b]
        if len(yy) >= 3:
            med[b] = np.median(yy); lo[b] = np.percentile(yy, 25); hi[b] = np.percentile(yy, 75)
    return mid, med, lo, hi

_cg = {"qt": np.linspace(-2.0, 2.0, 13), "phase": np.linspace(0.0, 1.5, 10)}
_cl = {"qt": r"$t-t_{\rm QT}$ [Gyr]", "phase": r"$(t-t_{\rm SFT})/(t_{\rm QT}-t_{\rm SFT})$"}

if DP_OK:
    for arr2d, logy, ylab, fname in [
            (DP["R50_dust"], True,  r"$\log_{10} R_{50}^{\rm dust}$ [kpc]", "dustR50_clock.png"),
            (DP_RATIO,       False, r"$R_{50}^{\rm dust}/R_{50}^{\star}$", "dustR50_ratio_clock.png")]:
        fig, axes = plt.subplots(NBINS, 2, figsize=(11, 3.6 * NBINS), squeeze=False)
        for bi in range(NBINS):
            binmask = mbin == bi
            for mj, mode in enumerate(["qt", "phase"]):
                ax = axes[bi, mj]
                for cmask, c, name in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
                    x, y = _clock_points(arr2d, np.where(cmask)[0], mode, logy)
                    if len(x) < 3:
                        continue
                    mid, med, lo, hi = _binned_median(x, y, _cg[mode])
                    ax.plot(mid, med, "-o", color=c, lw=2, ms=4, label=f"{name} (N={len(x)})")
                    ax.fill_between(mid, lo, hi, color=c, alpha=0.15)
                ax.axvline(0, ls=":", c="k", lw=1)
                if mode == "phase":
                    ax.axvline(1, ls="--", c="0.5", lw=1)
                if not logy:
                    ax.axhline(1, ls=":", c="0.6", lw=0.8)
                if bi == NBINS - 1:
                    ax.set_xlabel(_cl[mode])
                if mj == 0:
                    ax.set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\n{ylab}", fontsize=9)
                if bi == 0:
                    ax.set_title("QT-aligned" if mode == "qt" else "phase-norm")
                ax.legend(fontsize=7)
        fig.suptitle(f"{ylab} around quenching (particle dust) — {SPLIT_DESC}, by mass bin", y=1.005)
        plt.tight_layout(); plt.savefig(figpath( fname), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 5d-gas. HI / H2 extension AROUND QUENCHING — same QT-clock as §5d, for the cold-gas phases.
# Reuses §5d's _clock_points / _binned_median / _cg / _cl. R50_HI, R50_H2 come from the §5
# reduced product (DP); ratios are taken against the stellar R50.
assert DP_OK and "_clock_points" in globals(), "run §5b and §5d first"
_missing = [k for k in ("R50_HI", "R50_H2", "R50_star") if k not in DP]
if _missing:
    print("gas R50 not in product:", _missing, "-> rebuild §5a with HI/H2 profiles")
else:
    with np.errstate(all="ignore"):
        DP_RATIO_HI = np.where(DP["R50_star"] > 0, DP["R50_HI"] / DP["R50_star"], np.nan)
        DP_RATIO_H2 = np.where(DP["R50_star"] > 0, DP["R50_H2"] / DP["R50_star"], np.nan)
    GAS_SPECIES = [
        ("HI", DP["R50_HI"], DP_RATIO_HI, r"R_{50}^{\rm HI}",  "HI"),
        ("H2", DP["R50_H2"], DP_RATIO_H2, r"R_{50}^{\rm H_2}", "H2"),
    ]
    for sp, r50, ratio, sym, tag in GAS_SPECIES:
        for arr2d, logy, ylab, fname in [
                (r50,   True,  rf"$\log_{{10}} {sym}$ [kpc]", f"{tag}R50_clock.png"),
                (ratio, False, rf"${sym}/R_{{50}}^{{\star}}$", f"{tag}R50_ratio_clock.png")]:
            fig, axes = plt.subplots(NBINS, 2, figsize=(11, 3.6 * NBINS), squeeze=False)
            for bi in range(NBINS):
                binmask = mbin == bi
                for mj, mode in enumerate(["qt", "phase"]):
                    ax = axes[bi, mj]
                    for cmask, c, name in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
                        x, y = _clock_points(arr2d, np.where(cmask)[0], mode, logy)
                        if len(x) < 3:
                            continue
                        mid, med, lo, hi = _binned_median(x, y, _cg[mode])
                        ax.plot(mid, med, "-o", color=c, lw=2, ms=4, label=f"{name} (N={len(x)})")
                        ax.fill_between(mid, lo, hi, color=c, alpha=0.15)
                    ax.axvline(0, ls=":", c="k", lw=1)
                    if mode == "phase":
                        ax.axvline(1, ls="--", c="0.5", lw=1)
                    if not logy:
                        ax.axhline(1, ls=":", c="0.6", lw=0.8)
                    if bi == NBINS - 1:
                        ax.set_xlabel(_cl[mode])
                    if mj == 0:
                        ax.set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\n{ylab}", fontsize=9)
                    if bi == 0:
                        ax.set_title("QT-aligned" if mode == "qt" else "phase-norm")
                    ax.legend(fontsize=7)
            fig.suptitle(f"{ylab} around quenching (particle {sp}) — {SPLIT_DESC}, by mass bin", y=1.005)
            plt.tight_layout(); plt.savefig(figpath( fname), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 5e. Example face-on Sigma profiles (dust / HI / H2) across stages — one galaxy per
# class per mass bin (the one with the most finite dust stages).
if DP_OK and STORE_SIGMA and "sigma_dust" in DP:
    for bi in range(NBINS):
        binmask = mbin == bi
        for cmask, name in [(is_fast & binmask, LBL_A), (is_slow & binmask, LBL_B)]:
            ridx = np.where(cmask)[0]
            if len(ridx) == 0:
                continue
            best = max(ridx, key=lambda ri: np.sum(np.isfinite(DP["R50_dust"][ri])))
            if np.sum(np.isfinite(DP["R50_dust"][best])) == 0:
                continue
            fig, axx = plt.subplots(1, 3, figsize=(14, 3.6), sharex=True)
            for si, st in enumerate(DP_STAGES):
                for ai, comp in enumerate(["dust", "HI", "H2"]):
                    sig = DP[f"sigma_{comp}"][best, si]
                    if np.all(~np.isfinite(sig)) or np.nansum(sig) <= 0:
                        continue
                    axx[ai].plot(DP_RMID, sig, label=st)
            for ai, comp in enumerate([r"\Sigma_{\rm dust}", r"\Sigma_{\rm HI}", r"\Sigma_{\rm H_2}"]):
                axx[ai].set_yscale("log"); axx[ai].set_xlabel("R [kpc]")
                axx[ai].set_title(f"{name} {MASS_BIN_LABELS[bi]} — ${comp}$"); axx[ai].legend(fontsize=7)
            axx[0].set_ylabel(r"$\Sigma$ [$M_\odot$ kpc$^{-2}$]")
            plt.tight_layout(); plt.show()
else:
    print("no Sigma curves to show (need DP_OK and STORE_SIGMA=True).")

### ISM spatial distribution (concentration of dust / HI / H$_2$ / stars)

The particle-profile analogue of the catalog sizes: $R_{20}/R_{80}$ and $R_{20}/R_{50}$ for
**dust, HI, H$_2$ and stars** from the §5 reduced product (exact percentile radii), across the
critical stages and on the quenching clock. Needs the rebuilt product (`DP_HAS_CONC`).

In [ ]:
# ISM concentration from the particle profiles. Needs §5b (DP, DP_C2050/DP_C2080, DP_HAS_CONC),
#   §5c (dust_track, figpath), §5d (_clock_points, _binned_median, _cg, _cl).
if DP_OK and DP_HAS_CONC:
    for comp, sym in [("dust", "dust"), ("H2", r"H_2"), ("HI", "HI"), ("star", r"\star")]:
        dust_track(DP_C2080[comp], rf"$R_{{20}}/R_{{80}}^{{\rm {sym}}}$", False, f"conc2080_{comp}_track.png")
        dust_track(DP_C2050[comp], rf"$R_{{20}}/R_{{50}}^{{\rm {sym}}}$", False, f"conc2050_{comp}_track.png")
    for comp, sym in [("dust", "dust"), ("H2", r"H_2")]:    # quenching-clock view (reuse §5d machinery)
        fig, axes = plt.subplots(NBINS, 2, figsize=(11, 3.6 * NBINS), squeeze=False)
        for bi in range(NBINS):
            binmask = mbin == bi
            for mj, mode in enumerate(["qt", "phase"]):
                ax = axes[bi, mj]
                for cmask, c, name in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
                    x, y = _clock_points(DP_C2080[comp], np.where(cmask)[0], mode, False)
                    if len(x) < 3:
                        continue
                    mid, med, lo, hi = _binned_median(x, y, _cg[mode])
                    ax.plot(mid, med, "-o", color=c, label=name); ax.fill_between(mid, lo, hi, color=c, alpha=.15)
                ax.set_xlabel(_cl[mode])
                ax.set_title(rf"logM* {MASS_BIN_LABELS[bi]} — $R_{{20}}/R_{{80}}^{{\rm {sym}}}$")
                if bi == 0 and mj == 0:
                    ax.legend(fontsize=8)
        plt.tight_layout(); plt.savefig(figpath(f"conc2080_{comp}_clock.png"), dpi=130, bbox_inches="tight"); plt.show()
else:
    print("ISM concentration needs DP_OK and the rebuilt product (DP_HAS_CONC) -> rebuild §5a "
          "with BUILD_DUST_PROFILES=True.")

# Part 8 — Environment

Central vs satellite at each critical point, the cluster-infall-vs-quenching test, and the local environment density — the environmental axis feeding the causal engine in Part 9.

### V.5 (§7i) · Central or satellite at each critical point — fast vs slow

For every tracked z=0 quiescent galaxy we ask, **at each stage anchor**, whether it is the
**central** of its parent halo (`is_central = 1`, the most-massive halo member from §7a) or a
**satellite** (`is_central = 0`). The curves below show the *central fraction* per stage,
fast vs slow, faceted by z=0 halo mass — the satellite fraction is simply its complement.
Error bars are binomial ($\sqrt{p(1-p)/N}$); the dotted line marks the 50/50 split.

In [ ]:
# 7i. Central fraction at each critical point, fast vs slow, by halo-mass bin.
#     Reuses §7a SAT["is_central"] (1 = halo central, 0 = satellite). Also defines the
#     generalized fraction plotter frac_stage_g, reused by §7j below.
def frac_stage_g(diags, qty, binarr, binlabels, binname="logMhalo",
                 ylabel=None, fname=None, ymax=1.05):
    """Fraction (mean of a 0/1 indicator) satisfying `qty` at each stage, fast vs slow,
    faceted by `binarr`. Binomial error bars; NaNs dropped per stage; bins with <3 shown blank."""
    xs = np.arange(len(STAGES)); nbn = len(binlabels)
    fig, axs = plt.subplots(1, nbn, figsize=(5.0 * nbn, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(nbn):
        ax = axs[bi]; binmask = binarr == bi
        for msk, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            frac, err = [], []
            for st in STAGES:
                v = diags[st][qty][msk]
                v = v[np.isfinite(v)]
                if len(v) < 3:
                    frac.append(np.nan); err.append(0.0); continue
                p = float(np.mean(v)); n = len(v)
                frac.append(p); err.append(np.sqrt(max(p * (1 - p), 0) / n))
            ax.errorbar(xs, frac, yerr=err, fmt="-o", color=c, label=lab, capsize=3)
        ax.axhline(0.5, ls=":", c="grey", lw=0.8)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20); ax.set_ylim(-0.03, ymax)
        ax.set_title(f"{binname} {binlabels[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(ylabel or f"fraction {qty}")
    fig.suptitle(f"{qty}: fraction across stages  ({LBL_A} vs {LBL_B})", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath( fname), dpi=130, bbox_inches="tight")
    plt.show()

# central/satellite indicator per stage, drawn from the §7a satellite catalogue
DIAGS_CEN = {st: {"is_central": SAT["is_central"][:, _sij[st]],
                  "is_sat":     1.0 - SAT["is_central"][:, _sij[st]]} for st in STAGES}

frac_stage_g(DIAGS_CEN, "is_central", hmbin, HMASS_BIN_LABELS,
             ylabel="central fraction  (1=central, 0=satellite)",
             fname="cgm_central_fraction.png")

# quick text summary: central fraction at each stage, fast vs slow (all halo masses)
print("central fraction per stage (fast / slow):")
for st in STAGES:
    cv = SAT["is_central"][:, _sij[st]]
    ff = cv[is_fast & np.isfinite(cv)]; ss = cv[(is_slow) & np.isfinite(cv)]
    sf = f"{ff.mean():.2f} (n={ff.size})" if ff.size else "  -  "
    sl = f"{ss.mean():.2f} (n={ss.size})" if ss.size else "  -  "
    print(f"  {st:12s}: {LBL_A} {sf:>14s}   {LBL_B} {sl:>14s}")


### V.6 (§7L) · Did quenching start on cluster infall? — fast vs slow

Environmental test competing with the internal-AGN picture (§4b/§7j). For every tracked z=0
quiescent galaxy we reconstruct, at **every** snapshot on its progenitor track, whether it is
the **central** of its parent halo or a **satellite**, and the parent-halo total mass (from
`P["masses.total"]`). The **cluster-entry (infall)** time is the first time — going *forward*
in cosmic time — the galaxy becomes a *satellite* inside a halo above `CLUSTER_LOGMH` and
**stays** one (`SAT_PERSIST` consecutive snaps, anti-flicker). We then compare that infall time
to the quench-start (`SFT`) and quench-end (`QT`) anchors.

If fast quenchers are environmentally quenched we expect, relative to slow quenchers:
(i) a **higher infall fraction**, (ii) infall **preceding** quench start (`t_infall ≤ t_SFT`),
and (iii) a small, **positive** offset `t_SFT − t_infall` (quenching switches on at/just after
entry). Slow quenchers should instead quench while still centrals (internal/secular).

> **Box caveat.** `m100n1024` hosts very few true clusters (>10¹⁴ M☉) — the §7 halo bins top out
> at ">=13". `CLUSTER_LOGMH` defaults to **13.0** (groups+clusters); treat 10¹⁴ as a
> statistics-limited sensitivity check. Needs `_PROG_INDEX` from §5·pre.

In [ ]:
# 7L. Cluster-infall vs quenching test (fast vs slow). Needs _PROG_INDEX (§5·pre),
#     records / cols / is_fast (§3), P["masses.total"], snaps_arr, t_cosmic_yr.

CLUSTER_LOGMH = 13.0     # log10(Mhalo/Msun) for "cluster/group" entry (box has few >14)
SAT_PERSIST   = 2        # require satellite for >= this many consecutive (finite) snaps
SYNC_TOL_GYR  = 0.5      # |t_SFT - t_infall| < this  => "quenching starts on entry"

nrec, nsnap = len(records), len(snaps_arr)
cen_full = np.full((nrec, nsnap), np.nan)                 # 1=central, 0=satellite, per row
nsat_full = np.full((nrec, nsnap), np.nan)               # N_sat (halo members - 1), per row, full history
# parent-halo mass history is already in the catalogue-validated Msun units of §7
mh_full  = np.log10(np.where(P["masses.total"][:, cols].T > 0,
                             P["masses.total"][:, cols].T, np.nan))   # (nrec, nsnap)

# --- one catalog read per snapshot: is the tracked galaxy the central of its halo? ---
need = {}
for ri, r in enumerate(records):
    for row in range(nsnap):
        gx = _PROG_INDEX[row, r["col"]]
        if np.isfinite(gx):
            need.setdefault(int(snaps_arr[row]), []).append((ri, row, int(gx)))

for snap in sorted(need, reverse=True):
    if snap in CORRUPT_SNAPS:
        continue
    try:
        with h5py.File(sim.get_caesar_file(snap), "r") as f:
            parent = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
            gstar  = np.asarray(f["galaxy_data/dicts/masses.stellar"][:], dtype=np.float64)
    except (OSError, KeyError):
        print(f"  [skip] snapshot {snap}"); CORRUPT_SNAPS.add(snap); continue
    valid = parent >= 0
    if not np.any(valid):
        continue
    nhalo = int(parent[valid].max()) + 1
    cnt = np.bincount(parent[valid], minlength=nhalo)        # halo membership count -> N_sat
    gv = np.where(valid)[0]
    order = gv[np.argsort(gstar[gv], kind="stable")]     # most-massive-stellar wins (== §7a)
    central = np.full(nhalo, -1, dtype=np.int64)
    central[parent[order]] = order
    for (ri, row, gx) in need[snap]:
        h = parent[gx]
        if h >= 0:
            cen_full[ri, row] = 1.0 if central[h] == gx else 0.0
            nsat_full[ri, row] = cnt[h] - 1

# --- infall time: first persistent (satellite & massive-halo) run, forward in cosmic time ---
t_infall = np.full(nrec, np.nan)
for ri in range(nrec):
    cond = (cen_full[ri] == 0) & (mh_full[ri] >= CLUSTER_LOGMH)
    fin  = np.isfinite(cen_full[ri]) & np.isfinite(mh_full[ri])
    idx  = np.where(fin)[0]
    idx  = idx[np.argsort(t_cosmic_yr[idx])]             # finite rows, forward in time
    run = 0
    for k, j in enumerate(idx):
        run = run + 1 if cond[j] else 0
        if run >= SAT_PERSIST:
            t_infall[ri] = t_cosmic_yr[idx[k - SAT_PERSIST + 1]]
            break

# --- align to quench anchors and build the indicators ---
t_sft = np.array([r["t_sft"] for r in records])
t_qt  = np.array([r["t_qt"]  for r in records])
has_infall       = np.isfinite(t_infall)
infall_before_qt = has_infall & (t_infall <= t_qt)             # entered before quenching finished
order_ok         = has_infall & (t_infall <= t_sft)            # infall precedes quench start
sync             = has_infall & (np.abs(t_sft - t_infall) < SYNC_TOL_GYR * 1e9)
dt_sft = (t_sft - t_infall) / 1e9                             # Gyr; >0 => quench starts after entry

def _fisher(ind):
    a = int(np.sum(ind & is_fast));  b = int(np.sum(~ind & is_fast))
    c = int(np.sum(ind & is_slow)); d = int(np.sum(~ind & is_slow))
    _, p = fisher_exact([[a, b], [c, d]])
    return (a / (a + b) if a + b else np.nan), (c / (c + d) if c + d else np.nan), p

print(f"cluster-entry test  (CLUSTER_LOGMH={CLUSTER_LOGMH}, persist={SAT_PERSIST} snaps)")
print(f"  {'indicator':24s}  {LBL_A[:10]:>10s}  {LBL_B[:10]:>10s}   p(Fisher)")
for name, ind in [("ever infell", has_infall),
                  ("infall before QT", infall_before_qt),
                  ("infall before SFT", order_ok),
                  (f"sync (<{SYNC_TOL_GYR} Gyr)", sync)]:
    ff, ss, p = _fisher(ind)
    print(f"  {name:24s}  {ff:10.2f}  {ss:10.2f}   {p:8.3g}")
df, ds = dt_sft[is_fast & np.isfinite(dt_sft)], dt_sft[is_slow & np.isfinite(dt_sft)]
if len(df) > 2 and len(ds) > 2:
    print(f"  median (t_SFT - t_infall):  {LBL_A} {np.median(df):+.2f} Gyr   {LBL_B} {np.median(ds):+.2f} Gyr"
          f"   KS p={ks_2samp(df, ds).pvalue:.3g}")

# --- figures ---
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
# (1) t_infall vs t_SFT — points above the 1:1 line quench AFTER entering
for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
    m = msk & has_infall
    ax[0].scatter(t_infall[m] / 1e9, t_sft[m] / 1e9, s=22, c=c, alpha=0.6, label=f"{lab} ({m.sum()})")
lims = [0, np.nanmax(t_cosmic_yr) / 1e9]
ax[0].plot(lims, lims, "k--", lw=1); ax[0].set_xlim(lims); ax[0].set_ylim(lims)
ax[0].set_xlabel("t_infall [Gyr]"); ax[0].set_ylabel("t_SFT (quench start) [Gyr]")
ax[0].set_title("entry vs quench start  (above line = quench after entry)"); ax[0].legend(fontsize=8)
# (2) offset distribution
bins = np.linspace(-6, 6, 25)
for v, c, lab in [(df, COL_A, LBL_A), (ds, COL_B, LBL_B)]:
    if len(v):
        ax[1].hist(v, bins=bins, color=c, alpha=0.5, density=True, label=f"{lab} (med {np.median(v):+.1f})")
ax[1].axvline(0, c="k", lw=1); ax[1].axvspan(0, SYNC_TOL_GYR, color="grey", alpha=0.2)
ax[1].set_xlabel("t_SFT - t_infall [Gyr]"); ax[1].set_ylabel("pdf")
ax[1].set_title("offset: quench start relative to entry"); ax[1].legend(fontsize=8)
# (3) indicator fractions
names = ["ever\ninfell", "infall\n<QT", "infall\n<SFT", f"sync\n<{SYNC_TOL_GYR}Gyr"]
inds  = [has_infall, infall_before_qt, order_ok, sync]
x = np.arange(len(inds))
ax[2].bar(x - 0.18, [np.mean(i[is_fast]) for i in inds], 0.36, color=COL_A, label=LBL_A)
ax[2].bar(x + 0.18, [np.mean(i[is_slow]) for i in inds], 0.36, color=COL_B, label=LBL_B)
ax[2].set_xticks(x); ax[2].set_xticklabels(names); ax[2].set_ylim(0, 1.05)
ax[2].set_ylabel("fraction"); ax[2].set_title("environmental indicators"); ax[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(figpath( "cluster_infall_quenching.png"), dpi=130, bbox_inches="tight")
plt.show()

### V.7 (§7N) · Local environment density at each critical point — fast vs slow

A **quantitative** replacement for the §7M contour eyeball. For every tracked galaxy, at each
critical point, we reduce its environment to a per-galaxy scalar **local overdensity** `1+δ`,
measured against the *full* galaxy population of that snapshot (tracers above `NEIGH_MSTAR_MIN`):

- **counts-in-sphere**: `(1+δ) = N(<R) / (n̄·V)`, `R = SPHERE_R` (comoving);
- **Nth-nearest-neighbour**: `Σ_N = N / (4/3·π·d_N³)`, reported as `Σ_N/n̄`.

Both are **normalized to the mean density at that snapshot**, so they are directly comparable
*across* critical points despite the different redshifts — the thing the raw contour map cannot do.
Neighbours are found with a **periodic** `cKDTree` (correct at the box edges). We then compare the
fast vs slow distributions at each stage with a Mann–Whitney test. If fast quenchers quench in
denser environments, their `1+δ` should sit significantly above the slow population, most sharply
at `QT`. Needs `_PROG_INDEX` (§5·pre); host-halo mass (`P["masses.total"]`) is the complementary
single-number environment proxy already plotted in §7.

In [ ]:
# 7N. Local environment density at each critical point — fast vs slow.

NEIGH_K         = 5        # Nth nearest neighbour for Sigma_N
SPHERE_R        = 2.0      # counts-in-sphere radius [Mpc/h comoving] (auto-scaled if pos in kpc/h)
NEIGH_MSTAR_MIN = 1e9      # tracer-population stellar-mass floor [Msun]

# group tracked-galaxy lookups by snapshot (same pattern as §7L)
need = {}
for ri, r in enumerate(records):
    for si, st in enumerate(STAGES):
        row = r[f"row_{st}"]
        if row < 0:
            continue
        gx = _PROG_INDEX[row, r["col"]]
        if np.isfinite(gx):
            need.setdefault(int(snaps_arr[row]), []).append((ri, si, int(gx)))

_sij = {st: i for i, st in enumerate(STAGES)}
nrec = len(records)
overdens = np.full((nrec, len(STAGES)), np.nan)   # counts-in-sphere overdensity 1+delta
sigmaN   = np.full((nrec, len(STAGES)), np.nan)   # Nth-NN density / mean (overdensity)

_L = None
for snap in sorted(need, reverse=True):
    if snap in CORRUPT_SNAPS:
        continue
    try:
        with h5py.File(sim.get_caesar_file(snap), "r") as f:
            gpos_raw = np.asarray(f["galaxy_data/pos"][:], dtype=np.float64)
            gms      = np.asarray(f["galaxy_data/dicts/masses.stellar"][:], dtype=np.float64)
    except (OSError, KeyError):
        print(f"  [skip] snapshot {snap}"); CORRUPT_SNAPS.add(snap); continue
    if _L is None:
        _L = float(BOX) if np.nanmax(gpos_raw) < 1000 else float(BOX) * 1000.0
    R = SPHERE_R if _L <= 1000 else SPHERE_R * 1000.0
    gpos = np.clip(np.mod(gpos_raw, _L), 0.0, _L * (1 - 1e-12))   # wrap into [0, L) for periodic tree
    tracer = gms >= NEIGH_MSTAR_MIN
    if tracer.sum() < NEIGH_K + 2:
        continue
    tree  = cKDTree(gpos[tracer], boxsize=_L)
    nmean = tracer.sum() / _L**3
    Vsph  = 4.0 / 3.0 * np.pi * R**3
    for (ri, si, gx) in need[snap]:
        p = gpos[gx]
        if not np.all(np.isfinite(p)):
            continue
        cnt = len(tree.query_ball_point(p, R)) - (1 if tracer[gx] else 0)   # exclude self
        overdens[ri, si] = cnt / (nmean * Vsph)
        dd, _ = tree.query(p, k=NEIGH_K + 1)
        dd = dd[dd > 0][:NEIGH_K]                                            # drop self at d=0
        if len(dd) == NEIGH_K and dd[-1] > 0:
            sigmaN[ri, si] = (NEIGH_K / (4.0 / 3.0 * np.pi * dd[-1]**3)) / nmean

# ---- stats: Mann-Whitney fast vs slow at each stage ----
print(f"local overdensity 1+δ  (sphere R={SPHERE_R} Mpc/h, tracers logM*>={np.log10(NEIGH_MSTAR_MIN):.1f})")
print(f"  {'stage':12s}  {'fast med':>9s}  {'slow med':>9s}   MW-p")
for st in STAGES:
    v = overdens[:, _sij[st]]
    ff = v[is_fast & np.isfinite(v)]; ss = v[(is_slow) & np.isfinite(v)]
    if len(ff) > 2 and len(ss) > 2:
        p = mannwhitneyu(ff, ss, alternative="two-sided").pvalue
        print(f"  {st:12s}  {np.median(ff):9.2f}  {np.median(ss):9.2f}   {p:7.3g}")

# ---- plots: median overdensity track + violins, fast vs slow (all, then by halo-mass bin) ----
DIAGS_ENV = {st: {"overdensity": overdens[:, _sij[st]], "sigmaN": sigmaN[:, _sij[st]]} for st in STAGES}
_allbin = np.zeros(nrec, dtype=int)
median_track_g(DIAGS_ENV, "overdensity", _allbin, ["all galaxies"], binname="", logy=True,
               fname="env_overdensity_track.png")
violin_stage_g(DIAGS_ENV, "overdensity", _allbin, ["all galaxies"], binname="", logy=True,
               ylabel="log10 (1+δ)  [sphere]", fname="env_overdensity_violin.png")
median_track_g(DIAGS_ENV, "overdensity", hmbin, HMASS_BIN_LABELS, logy=True,
               fname="env_overdensity_track_byhalo.png")

# Part 9 — What is the primary axis? fast/slow vs halo mass vs AGN mode

The drivers apparatus: AGN-feeding cross-check and jet-mode incidence, the M*–M_halo degeneracy breaking (SHMR residual, matched twins), the event-aligned causal engine (env → AGN → quench) and its consumers (dust split, DTM, disc-stability kinematics, the working-scenario tests, population character, post-QT AGN regime), and finally a direct η² test of which axis — fast/slow, halo mass, or jet mode — organizes the population.

## V.1 (§4b) · AGN-feeding cross-check (BH accretion / AGN energy)

The dust-survival hypothesis hinges on a *negative*: the gas **did not** feed the
AGN, so there was no central energy injection to destroy the grains. We test that
link directly with the black hole's own history. CAESAR exposes galaxy-level BH mass
(`masses.bh`), accretion rate (`bhmdot`) and Eddington ratio (`bh_fedd`); SIMBA's
**jet-mode** (the channel that actually heats/ejects gas) triggers at
$M_{\rm BH} > 10^{7.5}\,M_\odot$ **and** $f_{\rm Edd} < 0.2$.

For each galaxy we read these along the progenitor chain (same index machinery as the
property history) and, over the **SFT→QT** window, compute:
- the BH mass *grown* $\Delta M_{\rm BH}$ and the integrated feedback energy
  $E_{\rm AGN}=\epsilon_r\,c^2\!\int \dot M_{\rm BH}\,dt$ (with $\epsilon_r=0.1$);
- the **fraction of time spent in jet mode**.

**Prediction:** dust-rich / slow quenchers should show **little AGN feeding** around
quenching (low $E_{\rm AGN}$, little jet-mode time), and residual dust at z=0 should
**anti-correlate** with $E_{\rm AGN}$ / jet-mode time.

> *Units caveat:* `bhmdot` is taken as $M_\odot\,{\rm yr}^{-1}$ and `masses.bh` as
> $M_\odot$ (CAESAR convention). Confirm against your build; the §0.1 probe lists the
> available keys and `BH_CANDIDATES` below tries the common name variants.

In [ ]:
# 4b-i. median BH accretion rate & Eddington ratio across stages — one column per mass bin
def bh_median_track(arr, label, logy=True):
    xs = np.arange(len(STAGES))
    fig, axs = plt.subplots(1, NBINS, figsize=(5.0 * NBINS, 4.5), sharey=True)
    axs = np.atleast_1d(axs)
    for bi in range(NBINS):
        ax = axs[bi]; binmask = mbin == bi
        for msk, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            med, lo, hi = [], [], []
            for st in STAGES:
                v = bh_stage(arr, st)[msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    med.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
                vv = np.log10(v) if logy else v
                med.append(np.median(vv)); lo.append(np.percentile(vv, 25)); hi.append(np.percentile(vv, 75))
            ax.plot(xs, med, "-o", color=c, label=lab); ax.fill_between(xs, lo, hi, color=c, alpha=0.15)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(("median log10 " if logy else "median ") + label)
    fig.suptitle(f"stage evolution of {label}", y=1.02)
    plt.tight_layout(); plt.savefig(figpath( f"bh_{label}.png".replace(" ", "_")), dpi=130, bbox_inches="tight"); plt.show()

bh_median_track(BH["bh_mdot"], "BH Mdot", logy=True)
bh_median_track(BH["bh_fedd"], "f_Edd", logy=True)

In [ ]:
# 4b-ii. THE cross-check: residual dust at z=0 vs AGN feeding during SFT->QT
# rows = mass bin, cols = the three AGN-feeding metrics; Spearman printed per bin.
dust_z0 = DIAGS["z0"]["dust/M*"]          # aligned to `records`
metrics = [("E_AGN", Eagn, r"$\log_{10} E_{\rm AGN}$(SFT$\to$QT) [erg]", True),
           ("dMbh",  dMbh, r"$\log_{10}\,\Delta M_{\rm BH}$(SFT$\to$QT) [$M_\odot$]", True),
           ("jetfrac", jetfrac, "jet-mode time fraction (SFT$\to$QT)", False)]
fig, axes = plt.subplots(NBINS, 3, figsize=(15, 4.2 * NBINS), squeeze=False)
for bi in range(NBINS):
    binmask = mbin == bi
    for mi, (name, x, xlab, logx) in enumerate(metrics):
        ax = axes[bi, mi]
        for cmask, c, lab in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            xv = np.log10(x[cmask]) if logx else x[cmask]
            ax.scatter(xv, np.log10(dust_z0[cmask]), s=18, c=c, label=lab, alpha=0.8)
        if bi == NBINS - 1:
            ax.set_xlabel(xlab)
        ax.set_ylabel(r"$\log_{10}(M_{\rm dust}/M_\star)_{z=0}$"); ax.legend(fontsize=7)
        m = binmask & np.isfinite(x) & np.isfinite(dust_z0) & (dust_z0 > 0)
        if m.sum() > 5:
            rho, p = spearmanr(x[m], dust_z0[m])
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}: rho={rho:+.2f} (p={p:.2g}, N={m.sum()})", fontsize=8)
            print(f"[{MASS_BIN_LABELS[bi]:9s}] Spearman(dust z=0, {name:7s}): rho={rho:+.2f} p={p:.3g} N={m.sum()}")
        else:
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}: N<6", fontsize=8)
fig.suptitle("residual dust vs AGN feeding, by mass bin  (hypothesis: anti-correlation)", y=1.005)
plt.tight_layout(); plt.savefig(figpath( "agn_feeding_vs_residual_dust.png"), dpi=130, bbox_inches="tight"); plt.show()

### V.2 (§4b-iv) · Quenching-clock view — does a difference appear *around* quenching?

If the full history (4b-iii) is degenerate, a difference may still hide in the brief
window when quenching actually happens. We re-stack every galaxy's BH history on its
**own quenching clock**, two ways:

- **QT-aligned:** $x = t - t_{QT}$ [Gyr] — absolute lookback around the quenching event;
- **phase-normalized:** $\phi = (t-t_{SFT})/(t_{QT}-t_{SFT})$, so $\phi{=}0$ is SFT, $\phi{=}1$
  is QT — this removes the $\tau_q$ duration difference and compares fast vs slow *by
  phase*, isolating any contrast in how hard the BH is fed approaching quenching.

KS tests on the **peak** $\dot M_{\rm BH}$ and $f_{\rm Edd}$ *within* each galaxy's
SFT→QT window quantify it.

In [ ]:
# stack a BH quantity on the quenching clock and return median + 25/75 band
def _stack(recs_sub, key, xgrid, mode):
    rows_acc = []
    for r in recs_sub:
        y = BH[key][:, r["col"]].astype(float)
        good = np.isfinite(y) & (y > 0) & np.isfinite(t_cosmic_yr)
        if good.sum() < 3:
            continue
        ts = t_cosmic_yr[good]; ys = np.log10(y[good])
        o = np.argsort(ts); ts, ys = ts[o], ys[o]
        if mode == "qt":
            xx = (ts - r["t_qt"]) / 1e9
        else:
            den = r["t_qt"] - r["t_sft"]
            if den <= 0:
                continue
            xx = (ts - r["t_sft"]) / den
        rows_acc.append(np.interp(xgrid, xx, ys, left=np.nan, right=np.nan))
    if not rows_acc:
        return None
    M = np.array(rows_acc)
    with np.errstate(all="ignore"):
        return np.nanmedian(M, axis=0), np.nanpercentile(M, 25, axis=0), np.nanpercentile(M, 75, axis=0)

grids   = {"qt": np.linspace(-2.0, 2.0, 41), "phase": np.linspace(0.0, 1.5, 31)}
xlabels = {"qt": r"$t - t_{\rm QT}$ [Gyr]",
           "phase": r"phase $(t-t_{\rm SFT})/(t_{\rm QT}-t_{\rm SFT})$"}
qkeys = [("bh_mdot", r"$\log\,\dot M_{\rm BH}$"), ("bh_fedd", r"$\log f_{\rm Edd}$")]

def _window_peak(key, sel):
    fa, sa = [], []
    for i, (r, isf, iss) in enumerate(zip(records, is_fast, is_slow)):
        if not sel[i] or not (isf or iss):
            continue
        idx = _sft_qt_rows(r)
        if len(idx) < 2:
            continue
        v = BH[key][idx, r["col"]]; v = v[np.isfinite(v) & (v > 0)]
        if len(v) == 0:
            continue
        (fa if isf else sa).append(np.max(v))
    return np.array(fa), np.array(sa)

# one quenching-clock figure (2 qty x 2 clock modes) per mass bin
for bi in range(NBINS):
    binmask = mbin == bi
    fast_recs = [r for r, f, b in zip(records, is_fast, binmask) if (f and b)]
    slow_recs = [r for r, s, b in zip(records, is_slow, binmask) if (s and b)]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
    for ci, (key, lab) in enumerate(qkeys):
        for ri, mode in enumerate(["qt", "phase"]):
            ax = axes[ri, ci]; xg = grids[mode]
            for recs, c, name in [(fast_recs, COL_A, LBL_A), (slow_recs, COL_B, LBL_B)]:
                res = _stack(recs, key, xg, mode)
                if res is None:
                    continue
                med, lo, hi = res
                ax.plot(xg, med, "-", color=c, lw=2, label=f"{name} (n={len(recs)})")
                ax.fill_between(xg, lo, hi, color=c, alpha=0.15)
            ax.axvline(0, ls=":", c="k", lw=1)
            if mode == "phase":
                ax.axvline(1, ls="--", c="0.5", lw=1)
            ax.set_xlabel(xlabels[mode]); ax.set_ylabel(lab); ax.legend(fontsize=7)
            ax.set_title(f"{lab} — {'QT-aligned' if mode == 'qt' else 'phase-norm'}")
    fig.suptitle(f"Quenching clock — logM* {MASS_BIN_LABELS[bi]}", y=1.01)
    plt.tight_layout(); plt.savefig(figpath( f"bh_quenching_clock_bin{bi}.png"), dpi=130, bbox_inches="tight"); plt.show()

    print(f"--- logM* {MASS_BIN_LABELS[bi]} ---")
    for key, lab in [("bh_mdot", "peak Mdot (SFT->QT)"), ("bh_fedd", "peak f_Edd (SFT->QT)")]:
        fa, sa = _window_peak(key, binmask)
        if len(fa) > 5 and len(sa) > 5:
            ks = ks_2samp(np.log10(fa), np.log10(sa))
            print(f"  KS {lab:22s}: D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
                  f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")

### V.8 (§7j) · Black-hole accretion & jet-mode incidence at the critical points

AGN **jet mode** is SIMBA's dominant CGM-heating channel, so it belongs next to the CGM
reservoir. Using the §4b BH history sampled at each stage anchor we track the BH accretion
rate $\dot M_{\rm BH}$, the Eddington ratio $f_{\rm Edd}$ and the BH mass $M_{\rm BH}$, and the
**jet-mode fraction** — galaxies with $\log_{10}(M_{\rm BH}/M_\odot) > 7.5$ **and**
$f_{\rm Edd} < 0.2$ — fast vs slow, faceted by halo mass (same facets as the CGM panels).

In [ ]:
# 7j. BH accretion / jet mode at the critical points (fast vs slow), by halo-mass bin.
#     Pulls bh_mdot / f_Edd / M_BH / jet_mode straight from the §7b DIAGS_CGM dict.
assert _have_bh, "BH history missing -> run §4b (cells 24-25) before this cell."
for q, logy in [("bh_mdot", True), ("f_Edd", True), ("M_BH", True)]:
    median_track_g(DIAGS_CGM, q, hmbin, HMASS_BIN_LABELS, logy=logy,
                   fname=f"cgm_bh_{q}.png")

# jet-mode incidence: fraction of galaxies in jet mode at each stage (fast vs slow)
frac_stage_g(DIAGS_CGM, "jet_mode", hmbin, HMASS_BIN_LABELS,
             ylabel=f"jet-mode fraction  (logM_BH>{JET_LOGMBH}, f_Edd<{JET_FEDD})",
             fname="cgm_jetmode_fraction.png")

print("jet-mode fraction per stage (fast / slow):")
for st in STAGES:
    jv = DIAGS_CGM[st]["jet_mode"]
    ff = jv[is_fast & np.isfinite(jv)]; ss = jv[(is_slow) & np.isfinite(jv)]
    sf = f"{ff.mean():.2f} (n={ff.size})" if ff.size else "  -  "
    sl = f"{ss.mean():.2f} (n={ss.size})" if ss.size else "  -  "
    print(f"  {st:12s}: {LBL_A} {sf:>14s}   {LBL_B} {sl:>14s}")


## V.9 (§8) · Breaking the M\*–M_halo degeneracy: SHMR residual, matched twins, and event-timing

The §7k grids showed no fast/slow contrast because a grid of per-bin medians is the *least*
sensitive readout, and because the real signal is not "different value at a fixed cell" but a
**locus shift**: at fixed M\* fast quenchers feed the BH harder, at fixed M_halo they feed it
less — the classic signature of fast quenchers being **under-massive in halo for their stars**.
This section attacks that directly, in three escalating ways:

- **§8a — one orthogonalized axis.** Collapse the 2-D degeneracy onto the offset from the
  stellar-to-halo-mass relation, `ΔlogM_halo = logM_halo − ⟨logM_halo|logM*⟩`, and plot AGN
  feeding and environment against it.
- **§8b — matched twins + partial correlation.** Compare each fast quencher to its nearest
  slow twin at **fixed both masses**, and measure whether AGN/environment still predict "fast"
  after both masses are regressed out.
- **§8c — the causal test.** Re-zero every track on its candidate trigger (AGN ignition,
  environment entry) and test the ordering `env → AGN → quench`.

All three reuse existing products: `records`, `is_fast`, `cols`, `P`, `BH`, `DIAGS`,
`gather_stage`, and (for environment) `overdens`/`_sij` from §7N and `_PROG_INDEX` from §5·pre.


In [ ]:
# 8·setup — orthogonalize the two masses. The degeneracy you're fighting is that fast & slow
# differ in OPPOSITE directions at fixed M* vs fixed M_halo. Collapse it to ONE axis:
#   ΔlogM_halo = logM_halo − <logM_halo | logM*>   (offset from the stellar-to-halo-mass ridge)
# Positive = over-massive halo for its stars; negative = under-massive halo.
# Ridge = RUNNING MEDIAN of logM_halo in logM* bins (robust to outliers, unlike a polyfit),
# fit SEPARATELY in quantile bins of the anchor redshift: the SHMR evolves, and the 'sft'/'qt'
# anchors mix epochs, so a single global fit would alias quench epoch into the residual.

SHMR_NZBINS = 3      # anchor-redshift bins (collapses to 1 when the z spread is tiny, e.g. z0)
SHMR_NMBINS = 7      # logM* bins for the running-median ridge

def _log(a):
    a = np.asarray(a, float)
    return np.log10(np.where(a > 0, a, np.nan))

def _shmr_residual(logMstar, logMhalo, z_anchor, nzbins=SHMR_NZBINS, nmbins=SHMR_NMBINS, nmin=10):
    """ΔlogM_halo about the running-median logM_halo|logM* ridge, per anchor-redshift bin.
       Returns (residuals, ridge) with ridge[k] = (z_lo, z_hi, mid_logM*, med_logMh)."""
    logMstar = np.asarray(logMstar, float); logMhalo = np.asarray(logMhalo, float)
    res, ridge = np.full(logMstar.shape, np.nan), {}
    g = np.isfinite(logMstar) & np.isfinite(logMhalo) & np.isfinite(z_anchor)
    if g.sum() < 2 * nmin:
        return res, ridge
    if np.nanmax(z_anchor[g]) - np.nanmin(z_anchor[g]) < 0.05:
        nzbins = 1                                       # all anchors at ~one epoch (e.g. z0)
    zed = np.nanquantile(z_anchor[g], np.linspace(0, 1, nzbins + 1)); zed[-1] += 1e-9
    for k in range(nzbins):
        inz = g & (z_anchor >= zed[k]) & (z_anchor < zed[k + 1])
        if inz.sum() < nmin:
            continue
        med = np.nanquantile(logMstar[inz], np.linspace(0, 1, nmbins + 1))
        mids, meds = [], []
        for b in range(nmbins):
            inb = inz & (logMstar >= med[b]) & (logMstar <= med[b + 1])
            if inb.sum() >= 5:
                mids.append(np.median(logMstar[inb])); meds.append(np.median(logMhalo[inb]))
        if len(mids) < 3:
            continue
        ridge[k] = (float(zed[k]), float(zed[k + 1]), np.asarray(mids), np.asarray(meds))
        res[inz] = logMhalo[inz] - np.interp(logMstar[inz], mids, meds)
    return res, ridge

logMs, logMh, dMh, shmr_ridge = {}, {}, {}, {}
for st in ("sft", "qt", "z0"):
    logMs[st] = _log(gather_stage("masses.stellar", st))
    logMh[st] = _log(gather_stage("masses.total", st))
    _zanc = np.array([redshift[r[f"row_{st}"]] if r[f"row_{st}"] >= 0 else np.nan for r in records])
    dMh[st], shmr_ridge[st] = _shmr_residual(logMs[st], logMh[st], _zanc)

HAS_ENV = ("overdens" in globals()) and ("_sij" in globals())
print("§8 setup: SHMR residual (running median, z-binned) built at", list(dMh),
      "| environment (overdens) available:", HAS_ENV)
for st in ("sft", "qt", "z0"):
    f = dMh[st][is_fast]; s = dMh[st][is_slow]
    f = f[np.isfinite(f)]; s = s[np.isfinite(s)]
    if len(f) > 2 and len(s) > 2:
        p = mannwhitneyu(f, s, alternative="two-sided").pvalue
        print(f"  {st:4s}:  ΔlogM_halo  fast med={np.median(f):+.3f}  slow med={np.median(s):+.3f}  MW-p={p:.2g}")


### V.10 (§8a) · The degeneracy collapsed onto one axis — offset from the SHMR

Four panels at the chosen anchor (`ANCHOR`, default `qt` = quench complete):
1. the (M\*, M_halo) plane with the fitted SHMR ridge — *where* fast vs slow sit;
2. the distribution of `ΔlogM_halo` (under/over-massive halo) for fast vs slow, with a
   Mann–Whitney p-value;
3. AGN feeding (`log Ṁ_BH`) vs `ΔlogM_halo`, with running medians;
4. local environment (`log(1+δ)`) vs `ΔlogM_halo`.

**Prediction:** fast quenchers cluster at `ΔlogM_halo < 0` (under-massive halos), and within
that branch carry the strongest BH feeding and the densest environment — i.e. the contrast the
2-D grid washed out becomes a clean 1-D trend.


In [ ]:
# 8a. Degeneracy on ONE axis: offset from the SHMR ridge, vs AGN feeding and environment.
ANCHOR = "qt"          # stage to read masses / BHAR / environment ("qt" = quench complete)

x = logMs[ANCHOR]; y = logMh[ANCHOR]; d = dMh[ANCHOR]
bhar = _log(mdot_qt) if ANCHOR == "qt" else _log(bh_stage(BH["bh_mdot"], ANCHOR))
env  = overdens[:, _sij[ANCHOR]] if HAS_ENV else np.full(len(records), np.nan)

def _runmed(xx, yy, nb=6):
    m = np.isfinite(xx) & np.isfinite(yy); xx, yy = xx[m], yy[m]
    if m.sum() < nb: return None
    e = np.nanpercentile(xx, np.linspace(0, 100, nb + 1)); cx = 0.5 * (e[:-1] + e[1:])
    cy = [np.median(yy[(xx >= e[i]) & (xx < e[i + 1])]) if ((xx >= e[i]) & (xx < e[i + 1])).any()
          else np.nan for i in range(nb)]
    return cx, np.array(cy)

fig, ax = plt.subplots(2, 2, figsize=(12, 10))
# (1) M*-Mhalo plane with the z-binned running-median SHMR ridges from §8·setup
for _zlo, _zhi, _mids, _meds in sorted(shmr_ridge[ANCHOR].values(), key=lambda t: t[0]):
    ax[0, 0].plot(_mids, _meds, "--", lw=1.6, label=f"ridge z=[{_zlo:.2f},{_zhi:.2f})")
ax[0, 0].scatter(x[is_fast], y[is_fast], s=16, c=COL_A, alpha=.7, label=LBL_A)
ax[0, 0].scatter(x[is_slow], y[is_slow], s=16, c=COL_B, alpha=.7, label=LBL_B)
ax[0, 0].set_xlabel(r"$\log_{10}M_*$"); ax[0, 0].set_ylabel(r"$\log_{10}M_{\rm halo}$")
ax[0, 0].set_title(f"M*-M_halo at {ANCHOR}"); ax[0, 0].legend(fontsize=8)
# (2) residual distributions
for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
    v = d[msk]; v = v[np.isfinite(v)]
    if len(v): ax[0, 1].hist(v, bins=18, color=c, alpha=.5, density=True, label=f"{lab} (med {np.median(v):+.2f})")
ax[0, 1].axvline(0, ls=":", c="k")
ax[0, 1].set_xlabel(r"$\Delta\log M_{\rm halo}$ at fixed $M_*$ (offset from SHMR)"); ax[0, 1].set_ylabel("pdf")
ff = d[is_fast]; ss = d[is_slow]; ff = ff[np.isfinite(ff)]; ss = ss[np.isfinite(ss)]
if len(ff) > 2 and len(ss) > 2:
    ax[0, 1].set_title(f"under/over-massive halo  (MW-p={mannwhitneyu(ff, ss).pvalue:.2g})")
ax[0, 1].legend(fontsize=8)
# (3) residual vs BH feeding
for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
    ax[1, 0].scatter(d[msk], bhar[msk], s=16, c=c, alpha=.6, label=lab)
    rm = _runmed(d[msk], bhar[msk])
    if rm: ax[1, 0].plot(*rm, "-o", color=c, lw=2)
ax[1, 0].axvline(0, ls=":", c="k"); ax[1, 0].set_xlabel(r"$\Delta\log M_{\rm halo}$")
ax[1, 0].set_ylabel(r"$\log_{10}\dot M_{\rm BH}$ at " + ANCHOR); ax[1, 0].set_title("AGN feeding vs SHMR offset")
ax[1, 0].legend(fontsize=8)
# (4) residual vs environment
if HAS_ENV:
    le = _log(env)
    for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
        ax[1, 1].scatter(d[msk], le[msk], s=16, c=c, alpha=.6, label=lab)
        rm = _runmed(d[msk], le[msk])
        if rm: ax[1, 1].plot(*rm, "-o", color=c, lw=2)
    ax[1, 1].axvline(0, ls=":", c="k"); ax[1, 1].set_xlabel(r"$\Delta\log M_{\rm halo}$")
    ax[1, 1].set_ylabel(r"$\log_{10}(1+\delta)$ at " + ANCHOR); ax[1, 1].set_title("environment vs SHMR offset")
    ax[1, 1].legend(fontsize=8)
else:
    ax[1, 1].text(.5, .5, "run §7N first\nto populate `overdens`", ha="center", va="center"); ax[1, 1].axis("off")
fig.suptitle(f"§8a · degeneracy-breaking axis: offset from the SHMR at {ANCHOR}", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "shmr_residual_axis.png"), dpi=130, bbox_inches="tight"); plt.show()


### V.11 (§8b) · Matched twins at fixed (M\*, M_halo) + partial correlation

The rigorous "all else equal" comparison. For every fast quencher we find its nearest slow
quencher in the **standardized** (logM\*, logM_halo) plane (`cKDTree`, twin distance capped at
`TOL_STD`), then compare the quantity of interest **pairwise**; a one-sample Wilcoxon test on
the paired `fast − slow` differences asks whether they differ at *truly* fixed both masses — far
more sensitive than median-in-bin colour.

Then the compact statistic: the **partial correlation** of `P(fast)` with each candidate driver
(`Ṁ_BH`, `E_AGN`, `1+δ`) after regressing out **both** masses. If a driver keeps a significant
partial correlation, it carries fast/slow information that neither mass explains — your
degeneracy-broken evidence.


In [ ]:
# 8b. Matched twins at FIXED (M*, M_halo) + partial correlation of P(fast) on the drivers.
ANCHOR_B = "qt"
TOL_STD  = 0.6          # max twin distance in std-units of the 2-D mass plane (None -> nearest always)

xb = logMs[ANCHOR_B]; yb = logMh[ANCHOR_B]
QTYS = {
    r"$\log\,\dot M_{\rm BH}$":     _log(mdot_qt) if ANCHOR_B == "qt" else _log(bh_stage(BH["bh_mdot"], ANCHOR_B)),
    r"$\log\,E_{\rm AGN}$":         _log(Eagn),
    r"$\log\,M_{\rm dust}/M_*$":    _log(DIAGS[ANCHOR_B]["dust/M*"]),
    r"$\log\,M_{\rm H_2}/M_*$":     _log(DIAGS[ANCHOR_B]["H2/M*"]),
    r"$R_{\rm gas}/R_*$":           DIAGS[ANCHOR_B]["Rgas/Rstar"],
}
if HAS_ENV:
    QTYS[r"$\log\,(1+\delta)$"] = _log(overdens[:, _sij[ANCHOR_B]])

# standardize the mass plane, match each fast -> nearest slow
mg = np.isfinite(xb) & np.isfinite(yb)
mu = np.array([np.nanmean(xb[mg]), np.nanmean(yb[mg])]); sd = np.array([np.nanstd(xb[mg]), np.nanstd(yb[mg])])
Z = np.column_stack(((xb - mu[0]) / sd[0], (yb - mu[1]) / sd[1]))
fast_idx = np.where(is_fast & mg)[0]; slow_idx = np.where((is_slow) & mg)[0]
dist, j = cKDTree(Z[slow_idx]).query(Z[fast_idx], k=1)
keep = np.isfinite(dist) & (dist <= (TOL_STD if TOL_STD else np.inf))
pairs = list(zip(fast_idx[keep], slow_idx[j[keep]]))
print(f"matched {len(pairs)} fast<->slow twins at {ANCHOR_B} (TOL_STD={TOL_STD}, of {len(fast_idx)} fast)")
if pairs:
    dm = np.array([np.hypot(xb[a] - xb[b], yb[a] - yb[b]) for a, b in pairs])
    print(f"  median twin mass mismatch (combined ΔlogM*,ΔlogMh): {np.nanmedian(dm):.3f} dex")

nq = len(QTYS)
fig, axs = plt.subplots(1, nq, figsize=(3.3 * nq, 3.8)); axs = np.atleast_1d(axs)
for ax, (lab, q) in zip(axs, QTYS.items()):
    diff = np.array([q[a] - q[b] for a, b in pairs]) if pairs else np.array([])
    diff = diff[np.isfinite(diff)]
    if len(diff) < 3:
        ax.text(.5, .5, "n<3", ha="center"); ax.set_title(lab, fontsize=8); continue
    ax.hist(diff, bins=15, color="0.6", edgecolor="k")
    ax.axvline(0, ls=":", c="k"); ax.axvline(np.median(diff), c="C3", lw=2)
    try:
        p = wilcoxon(diff).pvalue
    except ValueError:
        p = np.nan
    ax.set_title(f"{lab}\n" + r"$\Delta$med=" + f"{np.median(diff):+.2f}  p={p:.2g}", fontsize=8)
    ax.set_xlabel("fast - slow (twin)")
fig.suptitle(f"§8b · paired {LBL_A}-{LBL_B} at fixed (M*, M_halo), anchor={ANCHOR_B}", y=1.05)
plt.tight_layout(); plt.savefig(figpath( "matched_twins_diff.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── partial correlation: does each driver predict 'fast' AFTER both masses are removed? ──
def _partial_corr(target, driver, controls):
    M = np.column_stack([np.ones_like(target, dtype=float)] + [np.asarray(c, float) for c in controls])
    g = np.isfinite(target) & np.isfinite(driver) & np.all(np.isfinite(M), axis=1)
    if g.sum() < 8:
        return np.nan, g.sum()
    def _resid(v):
        b, *_ = np.linalg.lstsq(M[g], v[g].astype(float), rcond=None)
        return v[g].astype(float) - M[g] @ b
    rt, rd = _resid(target), _resid(driver)
    if np.std(rt) == 0 or np.std(rd) == 0:
        return np.nan, g.sum()
    return float(np.corrcoef(rt, rd)[0, 1]), g.sum()

drivers = {"log Mdot_BH": _log(mdot_qt), "log E_AGN": _log(Eagn)}
if HAS_ENV:
    drivers["log (1+d)"] = _log(overdens[:, _sij[ANCHOR_B]])
print(f"partial correlation of P({LBL_A}) with each driver, controlling for logM* AND logM_halo:")
for k, v in drivers.items():
    r, n = _partial_corr(is_fast.astype(float), v, [xb, yb])
    print(f"  {k:14s}: r_partial = {r:+.3f}   (n={n})")
print("  [+ve => stronger feeding/denser env predicts FAST even at fixed both masses]")


### V.12 (§8c) · The causal test — event-aligned stacks and the ordering env → AGN → quench

§8a–b break the *mass* degeneracy but are static. The hypothesis is fundamentally about
**timing**: environment entry triggers AGN ignition triggers the rapid quench. We test it by
re-zeroing every track on its candidate trigger instead of on cosmic time.

- `t_AGN` = first rise of the BH accretion to half its pre-quench peak.
- `t_env` = **environment entry**. Default `ENV_TRIGGER="infall"` uses the §7L cluster-infall
  time `t_infall` (first *persistent* satellite inside a halo > `CLUSTER_LOGMH`) — the cleaner,
  physical definition. Set `ENV_TRIGGER="overdensity"` to fall back to the first crossing of a
  density threshold on the `§8c·pre` overdensity history (looser, but defined for more galaxies).

> **N tradeoff.** The infall clock is only defined for galaxies that actually became persistent
> satellites of a `>CLUSTER_LOGMH` halo. In this cluster-poor box that is a minority, so the
> infall-aligned stacks use fewer galaxies than the AGN-aligned ones — lower `CLUSTER_LOGMH` in
> §7L (e.g. 12.5 for groups) to trade purity for sample size. `§8c·pre` is only needed for the
> `overdensity` fallback.

We stack `sSFR` and `Ṁ_BH` around each trigger and plot the **lead-lag distributions**
`t_QT − t_AGN` and `t_AGN − t_env`. The signature you want: for fast quenchers both lags are
short and positive (infall leads AGN leads quench), the sSFR drops within ~one snapshot of
`t_AGN`, and — split by the §8a SHMR branch — this is carried by the **under-massive-halo** galaxies.


In [ ]:
# 8c · the causal test — re-zero every track on its trigger, then stack fast vs slow.
#   t_AGN : first upward crossing of IGN_FRAC×(peak BHAR) at/before QT  (AGN ignition onset)
#   t_env : ENVIRONMENT entry. Default = the §7L cluster-infall time `t_infall` (clean & physical:
#           first persistent satellite in a halo > CLUSTER_LOGMH). Set ENV_TRIGGER="overdensity"
#           to instead use the first crossing of DELTA_TH on the §8c·pre overdensity history
#           (looser proxy, but defined for more galaxies in a cluster-poor box).
# Hypothesis: for FAST quenchers  t_env <~ t_AGN <~ t_QT  with short lags, sSFR collapses within
# ~one snapshot of t_AGN; for SLOW the AGN onset is weak / late / decoupled.
ENV_TRIGGER = "infall"            # "infall" (§7L t_infall)  |  "overdensity" (§8c·pre overdens_hist)
IGN_FRAC, IGN_FLOOR, DELTA_TH, GYR = 0.5, 1e-3, 2.0, 1e9

_ord  = np.argsort(t_cosmic_yr)                 # rows -> increasing cosmic time
t_inc = t_cosmic_yr[_ord]
ssfr_hist = np.where(P["masses.stellar"] > 0, P["sfr"] / P["masses.stellar"], np.nan)

def _track(arr2d, col):
    """per-galaxy time series for a (n_snap, n_*) array, sorted to increasing cosmic time."""
    return arr2d[_ord, col].astype(float)

# --- t_AGN : AGN ignition onset (first rise to half the pre-quench BHAR peak) ---
t_agn = np.full(len(records), np.nan)
for i, r in enumerate(records):
    col, tqt = r["col"], r["t_qt"]
    before = t_inc <= tqt + 0.05 * GYR
    md = _track(BH["bh_mdot"], col)
    m = before & np.isfinite(md) & (md > IGN_FLOOR)
    if m.sum() >= 2:
        thr = max(IGN_FRAC * np.nanmax(md[m]), IGN_FLOOR)
        cr = np.where(before & np.isfinite(md) & (md >= thr))[0]
        if len(cr): t_agn[i] = t_inc[cr[0]]

# --- t_env : environment entry ---
if ENV_TRIGGER == "infall":
    assert "t_infall" in globals(), "run §7L first (defines t_infall), or set ENV_TRIGGER='overdensity'"
    t_env = np.asarray(t_infall, float).copy()          # §7L cluster-infall time (cosmic yr)
    _env_desc = f"§7L infall (CLUSTER_LOGMH={globals().get('CLUSTER_LOGMH', '?')})"
else:
    assert "overdens_hist" in globals(), "run §8c·pre first (defines overdens_hist), or set ENV_TRIGGER='infall'"
    t_env = np.full(len(records), np.nan)
    for i, r in enumerate(records):
        before = t_inc <= r["t_qt"] + 0.05 * GYR
        od = _track(overdens_hist, i)
        me = before & np.isfinite(od) & (od >= DELTA_TH)
        if me.any(): t_env[i] = t_inc[np.where(me)[0][0]]
    _env_desc = f"overdensity > {DELTA_TH}"
print(f"trigger times — t_AGN: {np.isfinite(t_agn).sum()}/{len(records)}  |  "
      f"t_env [{_env_desc}]: {np.isfinite(t_env).sum()}/{len(records)}")

# ---------- (1,2) event-aligned median stacks of sSFR and BHAR ----------
def _estack(ax, zero_t, arr_of, ylab, logy=True, xwin=2.0, nb=12, band="iqr", ref0=False, split=None):
    """Median (line) + galaxy-to-galaxy scatter (shaded band) of arr_of around zero_t.
       band='iqr' -> 25-75th pct across galaxies; 'sem' -> bootstrap std-error of the median;
       None -> no band. ref0=True subtracts each galaxy's OWN value at the trigger (x≈0) first,
       so the curve shows relative change and is immune to the changing-cohort composition offset.
       split: list of (mask, color, label) groups to compare; defaults to fast vs slow."""
    xe = np.linspace(-xwin, xwin, nb + 1); xc = 0.5 * (xe[:-1] + xe[1:])
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    for msk, c, lab in groups:
        ii = np.where(msk & np.isfinite(zero_t))[0]
        prof = np.full((len(ii), len(xc)), np.nan)
        for k, i in enumerate(ii):
            tt = (t_inc - zero_t[i]) / GYR
            vv = arr_of(i)
            with np.errstate(divide="ignore", invalid="ignore"):
                vv = np.where(vv > 0, np.log10(vv), np.nan) if logy else vv
            for b in range(len(xc)):
                sel = (tt >= xe[b]) & (tt < xe[b + 1]) & np.isfinite(vv)
                if sel.any(): prof[k, b] = np.nanmedian(vv[sel])
        if not len(ii):
            continue
        if ref0:                                                 # subtract each galaxy's value at x≈0
            b0 = int(np.clip(np.searchsorted(xe, 0.0) - 1, 0, len(xc) - 1))
            for k in range(prof.shape[0]):
                row = prof[k]; fin = np.where(np.isfinite(row))[0]
                if not len(fin): continue
                bref = b0 if np.isfinite(row[b0]) else fin[np.argmin(np.abs(fin - b0))]
                prof[k] = row - row[bref]
        with np.errstate(all="ignore"):
            med = np.nanmedian(prof, axis=0)
            nbin = np.sum(np.isfinite(prof), axis=0)
            if band == "iqr":
                lo = np.nanpercentile(prof, 25, axis=0); hi = np.nanpercentile(prof, 75, axis=0)
            elif band == "sem":                                  # 1.253·σ/√N ≈ SE of the median
                se = 1.2533 * np.nanstd(prof, axis=0) / np.sqrt(np.maximum(nbin, 1))
                lo, hi = med - se, med + se
            else:
                lo = hi = None
        med = np.where(nbin >= 3, med, np.nan)                    # hide bins with <3 galaxies
        ax.plot(xc, med, "-o", color=c, label=f"{lab} (n={len(ii)})")
        if lo is not None:
            ax.fill_between(xc, np.where(nbin >= 3, lo, np.nan), np.where(nbin >= 3, hi, np.nan),
                            color=c, alpha=0.15, lw=0)
    ax.axvline(0, ls="--", c="k"); ax.set_ylabel(ylab); ax.legend(fontsize=8)

fig, ax = plt.subplots(2, 2, figsize=(12, 9))
_estack(ax[0, 0], t_agn, lambda i: _track(ssfr_hist, records[i]["col"]), r"med $\log$ sSFR")
ax[0, 0].set_title("sSFR aligned on AGN ignition"); ax[0, 0].set_xlabel(r"$t-t_{\rm AGN}$ [Gyr]")
_estack(ax[0, 1], t_agn, lambda i: _track(BH["bh_mdot"], records[i]["col"]), r"med $\log\dot M_{\rm BH}$")
ax[0, 1].set_title("BHAR aligned on AGN ignition"); ax[0, 1].set_xlabel(r"$t-t_{\rm AGN}$ [Gyr]")
_estack(ax[1, 0], t_env, lambda i: _track(ssfr_hist, records[i]["col"]), r"med $\log$ sSFR")
ax[1, 0].set_title(f"sSFR aligned on environment entry [{_env_desc}]"); ax[1, 0].set_xlabel(r"$t-t_{\rm env}$ [Gyr]")
_estack(ax[1, 1], t_env, lambda i: _track(BH["bh_mdot"], records[i]["col"]), r"med $\log\dot M_{\rm BH}$")
ax[1, 1].set_title("BHAR aligned on environment entry"); ax[1, 1].set_xlabel(r"$t-t_{\rm env}$ [Gyr]")
fig.suptitle("§8c · event-aligned stacks — does AGN ignition (and infall) precede the sSFR drop?", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "event_aligned_stacks.png"), dpi=130, bbox_inches="tight"); plt.show()

# ---------- (3) the causal chain: lead-lag distributions, fast vs slow ----------
lag_q = (np.array([r["t_qt"] for r in records]) - t_agn) / GYR   # t_QT - t_AGN  (>0: AGN leads quench)
lag_e = (t_agn - t_env) / GYR                                    # t_AGN - t_env (>0: infall leads AGN)
fig, axx = plt.subplots(1, 2, figsize=(11, 4))
for arr, axi, name in [(lag_q, axx[0], r"$t_{\rm QT}-t_{\rm AGN}$"), (lag_e, axx[1], r"$t_{\rm AGN}-t_{\rm env}$")]:
    for msk, c, lab in [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]:
        v = arr[msk]; v = v[np.isfinite(v)]
        if len(v) < 3: continue
        axi.hist(v, bins=15, color=c, alpha=.5, density=True, label=f"{lab} (med {np.median(v):+.2f})")
    axi.axvline(0, ls=":", c="k"); axi.set_xlabel(name + " [Gyr]"); axi.legend(fontsize=8)
    f = arr[is_fast]; s = arr[is_slow]; f = f[np.isfinite(f)]; s = s[np.isfinite(s)]
    if len(f) > 2 and len(s) > 2:
        axi.set_title(f"MW-p={mannwhitneyu(f, s).pvalue:.2g}")
fig.suptitle(f"§8c · causal ordering  env -> AGN -> quench  (env = {_env_desc};  +lag = cause leads)", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "causal_leadlag.png"), dpi=130, bbox_inches="tight"); plt.show()

# ---------- (4) split the lead-lag by SHMR branch: is it the under-massive-halo galaxies? ----------
branch = np.full(len(records), "undefined", dtype=object)            # NaN residual -> neither branch
branch[np.isfinite(dMh["qt"]) & (dMh["qt"] < 0)]  = "under-massive halo"
branch[np.isfinite(dMh["qt"]) & (dMh["qt"] >= 0)] = "over-massive halo"
print(f"median (t_QT - t_AGN) [Gyr] by class x SHMR branch  (hypothesis: short & positive for {LBL_A}/under-massive):")
for cls, msk in [(LBL_A, is_fast), (LBL_B, is_slow)]:
    for br in ("under-massive halo", "over-massive halo"):
        v = lag_q[msk & (branch == br)]; v = v[np.isfinite(v)]
        if len(v): print(f"  {cls:4s} / {br:18s}: {np.median(v):+.2f}  (n={len(v)})")

### V.13 (§8d) · Halo-mass trajectory around the triggers — is AGN ignition tied to a halo-mass scale?

The same event-aligned stacking as §8c, but tracking **log M_halo**. Read the **value at x = 0**
as the typical halo mass *at* AGN ignition (left) / *at* environment entry (right); the **slope**
is halo growth across the trigger. Bands are 25–75th percentile across galaxies.

**Top row — raw median.** A caveat lives here: an event-aligned *median* averages a **different
set of galaxies in every x-bin** (each track only covers part of the window), so the line can
drift non-physically — e.g. the post-infall median can *dip* even though no individual halo
shrinks. **Bottom row — per-galaxy-normalized** (`ref0=True`): each track is shifted to its own
halo mass at the trigger, so "halo stays put after infall" is a flat line at 0 and only a *real*
departure (tidal stripping / backsplash de-association) bends it. The printout also counts how
many galaxies revert to "central" after `t_infall` (backsplash/flicker → assigned M_halo drops back).

**Prediction (your hypothesis):** fast quenchers (the under-massive-halo branch from §8a) ignite
their AGN at a *lower* M_halo than slow quenchers, near the infall-driven jump in halo mass.


In [ ]:
# 8d · halo-mass trajectory around the triggers (needs §8c: t_agn, t_env, _estack, _track, t_inc).
#   Top: raw median logM_halo.  Bottom: per-galaxy-normalized ΔlogM_halo about the trigger
#   (ref0=True) — immune to the changing-cohort offset, so a flat line == halo unchanged after entry.
mhalo_hist = P["masses.total"]      # (n_snap, n_gal), Msun (catalogue-validated units, as in §7)

fig, ax = plt.subplots(2, 2, figsize=(12, 9), sharex="col")
for ci, (zt, xl, tag) in enumerate([(t_agn, r"$t-t_{\rm AGN}$", "AGN ignition"),
                                    (t_env, r"$t-t_{\rm env}$", f"environment entry [{_env_desc}]")]):
    _estack(ax[0, ci], zt, lambda i: _track(mhalo_hist, records[i]["col"]), r"med $\log M_{\rm halo}$")
    ax[0, ci].set_title(f"M_halo aligned on {tag}")
    _estack(ax[1, ci], zt, lambda i: _track(mhalo_hist, records[i]["col"]),
           r"med $\Delta\log M_{\rm halo}$ (rel. trigger)", ref0=True)
    ax[1, ci].axhline(0, ls=":", c="grey"); ax[1, ci].set_xlabel(xl)
fig.suptitle("§8d · halo mass around the triggers — raw (top) vs per-galaxy-normalized (bottom); "
             f"{LBL_A} vs {LBL_B}, 25-75% bands", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "mhalo_event_aligned.png"), dpi=130, bbox_inches="tight"); plt.show()

# halo mass AT the trigger (interpolate each track to t=trigger), median fast vs slow + MW-p
print(f"log10 M_halo at the trigger (interpolated to t=t_AGN / t=t_env), {SPLIT_DESC}:")
for zt, name in [(t_agn, "t_AGN"), (t_env, "t_env")]:
    grp = {}
    for msk, lab in [(is_fast, LBL_A), (is_slow, LBL_B)]:
        vals = []
        for i in np.where(msk & np.isfinite(zt))[0]:
            mh = _track(mhalo_hist, records[i]["col"]); g = np.isfinite(mh) & (mh > 0)
            if g.sum() >= 2:
                vals.append(np.interp(zt[i], t_inc[g], np.log10(mh[g])))
        grp[lab] = np.array(vals)
        if len(vals): print(f"  {name:6s} {lab:4s}: median logM_halo = {np.median(vals):.2f}  (n={len(vals)})")
    if len(grp.get(LBL_A, [])) > 2 and len(grp.get(LBL_B, [])) > 2:
        print(f"         MW-p({SPLIT_DESC}) = {mannwhitneyu(grp[LBL_A], grp[LBL_B]).pvalue:.2g}")

# why the post-infall median can DIP: backsplash / satellite->central reversion (assigned M_halo
# drops back to the galaxy's own small halo). Quantify it from the §7L central/satellite history.
if "cen_full" in globals():
    nrev = nchk = 0
    for i in np.where(np.isfinite(t_env))[0]:
        post = (t_cosmic_yr > t_env[i]) & np.isfinite(cen_full[i])
        if post.sum() >= 2:
            nchk += 1
            if np.any(cen_full[i][post] == 1.0): nrev += 1
    if nchk:
        print(f"post-infall backsplash: {nrev}/{nchk} galaxies become 'central' again at least once "
              f"after t_infall (their assigned M_halo reverts downward).")
else:
    print("(run §7L to populate cen_full for the backsplash diagnostic)")


### V.14 (§8e) · Event-aligned stacks for *any* property — one registry, one call

§8c/§8d hard-wired sSFR, BHAR and M_halo. This generalizes them: a `PROP_REGISTRY` maps a name
to a per-galaxy history (the raw `P`/`BH` quantities, the derived ratios from §4, and
`overdensity` if §8c·pre has run), and `event_stack(props, triggers, ref0=...)` produces the
fast-vs-slow aligned grid for any combination — rows = properties, columns = triggers.

```python
event_stack("dust/M*", "t_AGN")                       # one panel
event_stack(["BHAR", "sSFR", "M_H2"], ["t_AGN", "t_env"])   # 3×2 grid
event_stack("M_halo", "t_env", ref0=True)             # per-galaxy ΔlogX about the trigger
```

Triggers come from `TRIGGERS` (`t_AGN`, `t_env`, `t_QT`, `t_SFT`); add your own by inserting a
per-record time array. To register a new quantity:
`PROP_REGISTRY["x"] = dict(fn=lambda i: _track(my_arr, records[i]["col"]), logy=True, label="x")`.


In [ ]:
# 8e · event-aligned stacks for ANY property. Register once, then call event_stack().
#   Needs §8c (t_agn, t_env, _estack, _track, t_inc, ssfr_hist). The 'overdensity' entry appears
#   only if §8c·pre has been run (defines overdens_hist).

def _ratio_hist(a, b):
    """(n_snap, n_gal) ratio history a/b, NaN where b<=0; None if either field is absent."""
    A, B = P.get(a), P.get(b)
    if A is None or B is None:
        return None
    with np.errstate(all="ignore"):
        return np.where(B > 0, A / np.where(B > 0, B, np.nan), np.nan)

# name -> dict(fn = track(i)->increasing-time series, logy, axis label)
PROP_REGISTRY = {}
def _reg(name, arr, logy, label, by="gal"):
    if arr is None:
        return
    if by == "gal":                                   # (n_snap, n_gal), indexed by galaxy column
        fn = (lambda i, a=arr: _track(a, records[i]["col"]))
    else:                                             # 'rec' : (n_snap, n_records), indexed by record
        fn = (lambda i, a=arr: _track(a, i))
    PROP_REGISTRY[name] = dict(fn=fn, logy=logy, label=label)

_reg("sSFR",       ssfr_hist,                     True,  r"sSFR")
_reg("SFR",        P.get("sfr"),                  True,  r"SFR")
_reg("BHAR",       BH["bh_mdot"],                 True,  r"$\dot M_{\rm BH}$")
_reg("f_Edd",      BH["bh_fedd"],                 True,  r"$f_{\rm Edd}$")
_reg("M_BH",       BH["bh_mass"],                 True,  r"$M_{\rm BH}$")
_reg("M_halo",     P.get("masses.total"),         True,  r"$M_{\rm halo}$")
_reg("M_star",     P.get("masses.stellar"),       True,  r"$M_*$")
_reg("M_gas",      P.get("masses.gas"),           True,  r"$M_{\rm gas}$")
_reg("M_dust",     P.get("masses.dust"),          True,  r"$M_{\rm dust}$")
_reg("M_H2",       P.get("masses.H2"),            True,  r"$M_{\rm H_2}$")
_reg("M_HI",       P.get("masses.HI"),            True,  r"$M_{\rm HI}$")
_reg("dust/M*",    _ratio_hist("masses.dust", "masses.stellar"),       True,  r"$M_{\rm dust}/M_*$")
_reg("H2/M*",      _ratio_hist("masses.H2", "masses.stellar"),         True,  r"$M_{\rm H_2}/M_*$")
_reg("HI/M*",      _ratio_hist("masses.HI", "masses.stellar"),         True,  r"$M_{\rm HI}/M_*$")
_reg("dust/gas",   _ratio_hist("masses.dust", "masses.gas"),           True,  r"$M_{\rm dust}/M_{\rm gas}$")
_reg("HI/H2",      _ratio_hist("masses.HI", "masses.H2"),              True,  r"HI/H$_2$")
_reg("Rgas/Rstar", _ratio_hist("radii.gas_half_mass", "radii.stellar_half_mass"), False, r"$R_{\rm gas}/R_*$")
_reg("Rgas",       P.get("radii.gas_half_mass"),  True,  r"$R_{\rm gas}$")
_reg("Age_s",       P.get("ages.mass_weighted"),  True,  r"$Age_{\star}$")
if "overdens_hist" in globals():
    _reg("overdensity", overdens_hist, True, r"$1+\delta$", by="rec")

# triggers to align on (add your own per-record time array here)
TRIGGERS = {"t_AGN": t_agn, "t_env": t_env,
            "t_QT":     np.array([r["t_qt"]      for r in records], float),
            "t_SFT":    np.array([r["t_sft"]     for r in records], float),
            "t_SFpeak": np.array([r["t_sf_peak"] for r in records], float),
            "t_PostQuench": np.array([r["t_post_quench"] for r in records], float),
            "t_GasMin": np.array([r["t_gas_min"] for r in records], float)
            }

def event_stack(props, triggers=("t_AGN",), ref0=False, band="iqr", xwin=2.0, nb=12,
                fname=None, split=None):
    """Event-aligned stacks for any registered property, comparing two (or more) galaxy groups.
       props/triggers: a name or list of names (rows × columns). ref0=True -> per-galaxy ΔX
       about the trigger (composition-immune). split: list of (mask, color, label) groups
       (default fast vs slow) — e.g. a dust-fraction split from §8g. Returns the Figure."""
    props    = [props] if isinstance(props, str) else list(props)
    triggers = [triggers] if isinstance(triggers, str) else list(triggers)
    bad = [p for p in props if p not in PROP_REGISTRY]
    if bad:
        print("unknown prop(s):", bad, "| available:", list(PROP_REGISTRY))
        props = [p for p in props if p in PROP_REGISTRY]
    bt = [t for t in triggers if t not in TRIGGERS]
    if bt:
        print("unknown trigger(s):", bt, "| available:", list(TRIGGERS))
        triggers = [t for t in triggers if t in TRIGGERS]
    if not props or not triggers:
        return None
    nr, nc = len(props), len(triggers)
    fig, axs = plt.subplots(nr, nc, figsize=(5.6 * nc, 3.5 * nr), squeeze=False, sharex="col", sharey="row")
    for ri, pn in enumerate(props):
        spec = PROP_REGISTRY[pn]
        ylab = (r"$\Delta$" if ref0 else "med ") + ("log " if spec["logy"] else "") + spec["label"]
        for ci, tn in enumerate(triggers):
            ax = axs[ri][ci]
            _estack(ax, TRIGGERS[tn], spec["fn"], ylab, logy=spec["logy"],
                   xwin=xwin, nb=nb, band=band, ref0=ref0, split=split)
            if ref0: ax.axhline(0, ls=":", c="grey")
            if ri == nr - 1: ax.set_xlabel(f"t − {tn}  [Gyr]")
            if ri == 0:      ax.set_title(tn)
    _grp = "  (" + " vs ".join(g[2] for g in split) + ")" if split else f"  ({SPLIT_DESC})"
    fig.suptitle(f"§8e · event-aligned {'Δ-' if ref0 else ''}stacks{_grp}, 25–75% bands", y=1.0)
    plt.tight_layout()
    if fname: plt.savefig(figpath( fname), dpi=130, bbox_inches="tight")
    plt.show()
    return fig

print("§8e ready.")
print("  properties:", list(PROP_REGISTRY))
print("  triggers  :", list(TRIGGERS))
# --- example (edit freely): four ISM/AGN quantities aligned on AGN ignition and on infall ---
event_stack(["BHAR", "sSFR", "dust/M*", "M_H2"], triggers=["t_env", "t_AGN", "t_QT"], fname="event_stack_demo.png")

### V.15 (§8f) · Registering T_cgm and the dust-size ratios into §8e

Three requested quantities, three different data situations — register each appropriately so it
flows through `event_stack` like any other property:

- **T_cgm** — a *full* per-snapshot history (`P["halo_data/dicts/temperatures.mass_weighted_cgm"]`,
  the catalogue `Tcgm` from §7b). Registers as a normal continuous track: `event_stack("T_cgm", "t_AGN")`.
- **Rdust/Rstar** — the dust half-light size ratio exists **only at the §5 dust-profile stage
  anchors** (`DP_RATIO`, shape `n_rec × n_stage`), not per snapshot. We register it as a
  *stage-sampled* track: each galaxy's ≤7 anchor values are dropped onto their snapshot rows, so
  `event_stack` shows them as sparse points binned by `t − t_trigger`. **Use a wider window**
  (e.g. `xwin=5`) since the early anchors (SF-peak/SFT) sit Gyr from `t_AGN`.
- **Rdust/Rgas** — **not in the product**: the §5 builder stores `R50_dust/HI/H2/star` but no
  total-gas `R50`. We register the available proxies `Rdust/RH2` and `Rdust/RHI`. For a *true*
  total-gas size add one line to `build_profiles_job.py` — `R50_gas=_renc(R, m_gas, 0.5)`
  in `_profile_one` (m_gas is already computed) and store it — then rebuild §5a.

Run §8e first (defines `PROP_REGISTRY`/`event_stack`); these append to that registry.


In [ ]:
# 8f · add T_cgm (continuous) and the dust-size ratios (stage-sampled) to the §8e registry.
#   Needs §8e (PROP_REGISTRY, event_stack, _track, _ord). T_cgm needs the halo history (§1b/§7b);
#   the R-ratios need the §5 dust-profile product DP (run §5a/§5b first).

# --- (1) T_cgm: full per-snapshot catalogue history -> register like any gal-indexed quantity ---
_TKEY = "halo_data/dicts/temperatures.mass_weighted_cgm"
if _TKEY in P:
    _reg("T_cgm", P[_TKEY], True, r"$T_{\rm CGM}$ [K]")
    print("registered continuous: T_cgm")
else:
    print(f"T_cgm skipped — '{_TKEY}' not in history; rebuild §1b with it in PROPS['halo_data'].")

# --- (2) stage-sampled adapter: drop each galaxy's per-stage values onto their snapshot rows ---
def _reg_stage(name, stage_arr, stage_list, logy, label):
    """Register a quantity defined only at the critical-point anchors (shape n_rec × n_stage).
       The track is NaN except at each stage's snapshot row -> event_stack bins it as sparse points."""
    if stage_arr is None:
        return False
    def fn(i, A=stage_arr, SL=stage_list):
        full = np.full(n_snap, np.nan)
        for si, st in enumerate(SL):
            if si >= A.shape[1]:
                break
            row = records[i].get(f"row_{st}", -1)
            if isinstance(row, (int, np.integer)) and row >= 0:
                full[row] = A[i, si]
        return full[_ord]
    PROP_REGISTRY[name] = dict(fn=fn, logy=logy, label=label, sparse=True)
    return True

if ("DP" in globals()) and globals().get("DP_OK") and (DP is not None):
    _SL = DP_STAGES
    done = []
    if _reg_stage("Rdust/Rstar", DP_RATIO, _SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\star}$"):
        done.append("Rdust/Rstar")
    with np.errstate(all="ignore"):
        if "R50_H2" in DP and _reg_stage(
                "Rdust/RH2", np.where(DP["R50_H2"] > 0, DP["R50_dust"] / DP["R50_H2"], np.nan),
                _SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\rm H_2}$"):
            done.append("Rdust/RH2")
        if "R50_HI" in DP and _reg_stage(
                "Rdust/RHI", np.where(DP["R50_HI"] > 0, DP["R50_dust"] / DP["R50_HI"], np.nan),
                _SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\rm HI}$"):
            done.append("Rdust/RHI")
        if "R50_gas" in DP and _reg_stage(                       # only if you rebuilt §5a with R50_gas
                "Rdust/Rgas", np.where(DP["R50_gas"] > 0, DP["R50_dust"] / DP["R50_gas"], np.nan),
                _SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\rm gas}$"):
            done.append("Rdust/Rgas")
    _reg_stage("R50_dust", DP["R50_dust"], _SL, True, r"$R_{50}^{\rm dust}$ [kpc]"); done.append("R50_dust")
    print("registered stage-sampled:", done,
          "\n  (no true Rdust/Rgas yet — add R50_gas to build_profiles_job.py and rebuild §5a)"
          if "Rdust/Rgas" not in done else "")
else:
    print("R-ratios skipped — run §5a/§5b to build the dust-profile product DP first.")

# --- (3) CGM phase fractions (+ stage T/M_CGM) from the §7g temperature product (CT) ---
#   Stage-sampled, same shape as the dust R-ratios: n_rec x n_stage, dropped onto snapshot rows.
if globals().get("CGMT_OK") and ("CT" in globals()):
    _SLT = CT_STAGES
    cdone = []
    if _reg_stage("f_hot",  CT["f_hot"],  _SLT, False, r"$f_{\rm hot}^{\rm CGM}\;(T>10^5\,$K$)$"):
        cdone.append("f_hot")
    if _reg_stage("f_warm", CT["f_warm"], _SLT, False, r"$f_{\rm warm}^{\rm CGM}\;(10^4$-$10^5\,$K$)$"):
        cdone.append("f_warm")
    if _reg_stage("f_cold", CT["f_cold"], _SLT, False, r"$f_{\rm cold}^{\rm CGM}\;(T<10^4\,$K$)$"):
        cdone.append("f_cold")
    # companions from the same product, for completeness next to the continuous T_cgm
    if _reg_stage("T_CGM_s", CT["T_mw"],  _SLT, True,  r"$T_{\rm CGM}$ [K] (stage-sampled)"):
        cdone.append("T_CGM_s")
    if _reg_stage("M_CGM_T", CT["M_cgm"], _SLT, True,  r"$M_{\rm CGM}$ [M$_\odot$]"):
        cdone.append("M_CGM_T")
    print("registered stage-sampled CGM phases:", cdone)
else:
    print("CGM phase fractions skipped — run §7g (load cgm_temperature_allcrit.hdf5) first.")

print("\nPROP_REGISTRY now:", list(PROP_REGISTRY))
# --- example: continuous T_cgm + CGM phase fractions + dust size, on AGN ignition / infall / SF-peak ---
event_stack(["T_cgm", "f_hot", "f_warm", "f_cold", "Rdust/Rstar"],
            triggers=["t_env", "t_AGN", "t_QT"],
            xwin=5.0, nb=10, fname="event_stack_cgm_phases.png")

### V.16 (§8g) · Split by post-quench dust fraction instead of fast/slow

A different question than §8c–8f: rather than *how fast* a galaxy quenched, split on *whether it
kept its dust* — **dusty** if `log10(M_dust/M_*) > −5` at the `post_quench` anchor, **non-dusty**
otherwise. `dust_split(thresh, stage)` returns a `split` spec (`[(mask, color, label), …]`) that
plugs straight into the generalized `event_stack` / `_stack` from §8e:

```python
event_stack(["BHAR", "sSFR", "dust/M*"], ["t_AGN", "t_env"], split=SPLIT_DUST)
event_stack(["T_cgm", "Rdust/Rstar"], "t_AGN", split=SPLIT_DUST, xwin=5)   # §8f props too
```

The same `split=` argument works for any registered property (§8e/§8f) and any trigger. The cell
also prints the **dusty × fast/slow cross-tab** — the direct test of whether "dusty" is just a
relabelling of "slow", or a partly independent axis (the interesting case for your hypothesis:
non-dusty *slow* quenchers, or dusty *fast* ones).


In [ ]:
# 8g · split by post-quench dust fraction (dusty vs non-dusty) and feed it to §8e/§8f event_stack.
#   Needs §8e (event_stack), §4 (DIAGS). Works on every registered property / trigger via split=.
def dust_split(thresh=-5.0, stage="post_quench", colors=("C2", "0.5")):
    """Return a split spec [(mask,color,label),...] on log10(M_dust/M_*) at `stage`.
       group A = dusty (log10 dust/M* > thresh); group B = non-dusty (finite and <= thresh).
       Galaxies with an undefined dust fraction at `stage` (NaN row / M_*<=0) are in neither."""
    dfr  = DIAGS[stage]["dust/M*"]
    defi = np.isfinite(dfr)
    dusty = defi & (dfr > 10.0 ** thresh)
    nond  = defi & ~(dfr > 10.0 ** thresh)
    return [(dusty, colors[0], f"dusty (log Md/M*>{thresh:g} @{stage})"),
            (nond,  colors[1], "non-dusty")]

SPLIT_DUST = dust_split(-5.0, "post_quench")
_d, _nd = SPLIT_DUST[0][0], SPLIT_DUST[1][0]
print(f"post-quench dust split:  dusty={_d.sum()}  non-dusty={_nd.sum()}  "
      f"undefined={len(records) - int(_d.sum()) - int(_nd.sum())}")
print(f"cross-tab vs {SPLIT_DESC} (is 'dusty' just group B?):")
print(f"  dusty   : {LBL_A} {int(np.sum(_d & is_fast)):3d}   {LBL_B} {int(np.sum(_d & is_slow)):3d}")
print(f"  non-dusty: {LBL_A} {int(np.sum(_nd & is_fast)):3d}   {LBL_B} {int(np.sum(_nd & is_slow)):3d}")

# --- examples: same registry & triggers as §8e/§8f, but grouped by dust instead of speed ---
event_stack(["BHAR", "SFR", "sSFR", "dust/M*"], triggers=["t_env", "t_AGN", "t_QT"],
            split=SPLIT_DUST, fname="event_stack_dustsplit.png")
event_stack(["M_star", "H2/M*", "HI/M*", "M_halo", "Age_s"], triggers=["t_env", "t_AGN", "t_QT"],
            split=SPLIT_DUST, fname="event_stack_dustsplit_b.png")
event_stack(["T_cgm", "Rdust/Rstar"], triggers=["t_env", "t_AGN", "t_QT"],
            split=SPLIT_DUST, xwin=5.0, nb=10, fname="event_stack_dustsplit_T_R.png")

### V.16b (§8g·shape) · Whole-range stacked evolution on a movable critical-point clock

Same population splits as §8g, but instead of a narrow event window we re-zero every
galaxy's **entire** history on a chosen critical point and stack it. The question: do the
populations (fast/slow, or dusty/non-dusty) have *characteristic shapes*, or do they only
differ locally around the event? `sfh_shape()` works on any quantity in the `_SFH_QTY`
registry — SFR, sSFR, and the **gas / dust / H$_2$ / HI fractions** — reusing the §4b-iv
`_stack` idiom (interp each track onto a `t − t_anchor` grid, median + IQR) over a wide
grid. `sfh_shape_ks()` is the companion two-sample KS test per lag bin. Needs §3
(`records`, `t_<stage>`, `is_fast/is_slow`, `mbin`), §1 (`P`, `t_cosmic_yr`); the
dust-split demos also need §8g (`dust_split`).

In [ ]:
# 8g·shape — stack the WHOLE evolution (SFH or any reservoir fraction) on a movable
#   critical-point clock. Complements §8c (a ±2 Gyr window): the full rise-and-fall is kept,
#   just re-zeroed on t_<anchor>, so characteristic shapes (if any) show across populations.

# quantity registry: kind -> (field,) or (numerator, denominator); everything shown in log10.
_SFH_QTY = {
    "sfr":      ("sfr",),
    "ssfr":     ("sfr", "masses.stellar"),
    "gas/M*":   ("masses.gas", "masses.stellar"),
    "dust/M*":  ("masses.dust", "masses.stellar"),
    "H2/M*":    ("masses.H2", "masses.stellar"),
    "HI/M*":    ("masses.HI", "masses.stellar"),
    "dust/gas": ("masses.dust", "masses.gas"),
}
_SFH_YLAB = {
    "sfr":      r"$\log_{10}$ SFR [$M_\odot$/yr]",
    "ssfr":     r"$\log_{10}$ sSFR [1/yr]",
    "gas/M*":   r"$\log_{10}\,M_{\rm gas}/M_\star$",
    "dust/M*":  r"$\log_{10}\,M_{\rm dust}/M_\star$",
    "H2/M*":    r"$\log_{10}\,M_{\rm H_2}/M_\star$",
    "HI/M*":    r"$\log_{10}\,M_{\rm HI}/M_\star$",
    "dust/gas": r"$\log_{10}\,M_{\rm dust}/M_{\rm gas}$",
}
_SFH_NAME = {"sfr": "SFR", "ssfr": "sSFR", "gas/M*": "gas frac", "dust/M*": "dust frac",
             "H2/M*": "H2 frac", "HI/M*": "HI frac", "dust/gas": "dust/gas"}

def _sfh_track(col, kind="sfr"):
    """(t_sorted [yr], log10 q) for one galaxy; q from _SFH_QTY, non-positive/NaN dropped."""
    spec = _SFH_QTY[kind]
    num = P[spec[0]][:, col].astype(float)
    if len(spec) == 1:
        q = num
    else:
        den = P[spec[1]][:, col].astype(float)
        q = np.where(den > 0, num / den, np.nan)
    good = np.isfinite(q) & (q > 0) & np.isfinite(t_cosmic_yr)
    if good.sum() < 3:
        return None, None
    ts, ys = t_cosmic_yr[good], np.log10(q[good])
    o = np.argsort(ts)
    return ts[o], ys[o]

def sfh_shape(anchors=("sf_peak", "sft", "qt"), kind="sfr", split=None,
              xwin=(-12.0, 10.0), nx=61, min_n=5, facet_mass=False, fname=None):
    """Median (+IQR band) log-quantity stacked on each critical-point clock, per population.
       x = t - t_<anchor> [Gyr] over the WHOLE history; one column per anchor.
       kind  : key of _SFH_QTY ('sfr','ssfr','gas/M*','dust/M*','H2/M*','HI/M*','dust/gas').
       split : [(mask, color, label), ...]; default = the active §3 fast/slow split.
       facet_mass=True -> one row per stellar-mass bin (else a single pooled row).
       Bins with < min_n contributing galaxies are blanked so ragged tails don't mislead."""
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    xg = np.linspace(xwin[0], xwin[1], nx)
    rows = list(range(NBINS)) if facet_mass else [None]
    ylab = "med " + _SFH_YLAB[kind]
    fig, axes = plt.subplots(len(rows), len(anchors), squeeze=False,
                             figsize=(4.6 * len(anchors), 3.6 * len(rows)),
                             sharex=True, sharey="row")
    for ri, bi in enumerate(rows):
        binmask = np.ones(len(records), bool) if bi is None else (mbin == bi)
        for ci, st in enumerate(anchors):
            ax = axes[ri][ci]; tkey = f"t_{st}"
            for msk, c, lab in groups:
                acc = []
                for i in np.where(msk & binmask)[0]:
                    t0 = records[i].get(tkey, np.nan)
                    if not np.isfinite(t0):
                        continue
                    ts, ys = _sfh_track(records[i]["col"], kind)
                    if ts is None:
                        continue
                    acc.append(np.interp(xg, (ts - t0) / 1e9, ys, left=np.nan, right=np.nan))
                if not acc:
                    continue
                M = np.asarray(acc); nbin = np.sum(np.isfinite(M), axis=0)
                with np.errstate(all="ignore"):
                    med = np.nanmedian(M, axis=0)
                    lo  = np.nanpercentile(M, 25, axis=0)
                    hi  = np.nanpercentile(M, 75, axis=0)
                keep = nbin >= min_n
                ax.plot(xg, np.where(keep, med, np.nan), "-", color=c, lw=2,
                        label=f"{lab} (n={len(acc)})")
                ax.fill_between(xg, np.where(keep, lo, np.nan), np.where(keep, hi, np.nan),
                                color=c, alpha=0.15, lw=0)
            ax.axvline(0, ls=":", c="k", lw=1)
            if ri == 0:
                ax.set_title(f"aligned on {st}")
            if ri == len(rows) - 1:
                ax.set_xlabel(rf"$t - t_{{\rm {st}}}$ [Gyr]")
            if ci == 0:
                ax.set_ylabel(((f"logM* {MASS_BIN_LABELS[bi]}\n" if bi is not None else "") + ylab))
            ax.legend(fontsize=7)
    sub = " vs ".join(g[2] for g in groups)
    fig.suptitle(f"Stacked evolution on the critical-point clock — {_SFH_NAME[kind]};  "
                 f"split: {sub}", y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

# (a) SFR shape re-zeroed on three clocks, active split (fast/slow by default)
sfh_shape(("sf_peak", "sft", "qt"), kind="sfr", fname="sfh_shape_sfr.png")
# (b) specific-SFR, split by post-quench dust instead of speed (the §8g population cut)
sfh_shape(("sf_peak", "sft", "qt"), kind="ssfr", split=dust_split(-5.0, "post_quench"),
          fname="sfh_shape_ssfr_dustsplit.png")
# (c) gas fraction — full evolution, incl. its own gas-min clock (is the trough real?)
sfh_shape(("sf_peak", "qt", "gas_min"), kind="gas/M*", fname="sfh_shape_gasfrac.png")
# (d) dust fraction — full evolution, incl. the dust-peak clock
sfh_shape(("sf_peak", "qt", "dust_peak"), kind="dust/M*", fname="sfh_shape_dustfrac.png")


In [ ]:
# 8g·shape·ks — is the stacked shape *statistically* different between two populations?
#   Per-lag-bin two-sample KS on the stacked log-quantity (group A vs B) for each anchor: the
#   band plot above shows the medians; this shows WHERE on the clock the full per-galaxy
#   distributions diverge. Reuses _sfh_track/_SFH_* (prev cell). Mirrors §8c's ks_2samp use.
#   Caveat: adjacent bins share galaxies (smooth tracks) -> the p-curve is autocorrelated, so
#   read runs of p<0.05, not each bin as an independent test.
from scipy.stats import ks_2samp

def _stack_matrix(group_mask, binmask, tkey, kind, xg):
    """rows = galaxies in group, cols = lag bins; log-quantity interpolated onto xg (NaN outside)."""
    acc = []
    for i in np.where(group_mask & binmask)[0]:
        t0 = records[i].get(tkey, np.nan)
        if not np.isfinite(t0):
            continue
        ts, ys = _sfh_track(records[i]["col"], kind)
        if ts is None:
            continue
        acc.append(np.interp(xg, (ts - t0) / 1e9, ys, left=np.nan, right=np.nan))
    return np.asarray(acc) if acc else np.empty((0, len(xg)))

def sfh_shape_ks(anchors=("sf_peak", "sft", "qt"), kind="sfr", split=None,
                 xwin=(-12.0, 10.0), nx=61, min_n=8, facet_mass=False, fname=None):
    """Per-lag-bin two-sided KS test of the stacked log-quantity between EXACTLY two groups.
       Plots the KS p-value vs clock time (one column per anchor); shades p<0.05 (distinct).
       Prints, per anchor, the divergent-bin fraction and the most-separated bin (max KS D)."""
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    assert len(groups) == 2, "KS companion compares exactly two groups"
    (mA, _cA, lA), (mB, _cB, lB) = groups
    xg = np.linspace(xwin[0], xwin[1], nx)
    rows = list(range(NBINS)) if facet_mass else [None]
    fig, axes = plt.subplots(len(rows), len(anchors), squeeze=False,
                             figsize=(4.6 * len(anchors), 3.4 * len(rows)),
                             sharex=True, sharey=True)
    print(f"KS  {lA}  vs  {lB}   [{_SFH_NAME[kind]}; bins need n>={min_n}/group]")
    for ri, bi in enumerate(rows):
        binmask = np.ones(len(records), bool) if bi is None else (mbin == bi)
        tag = "pooled" if bi is None else f"logM* {MASS_BIN_LABELS[bi]}"
        for ci, st in enumerate(anchors):
            ax = axes[ri][ci]; tkey = f"t_{st}"
            A = _stack_matrix(mA, binmask, tkey, kind, xg)
            B = _stack_matrix(mB, binmask, tkey, kind, xg)
            D = np.full(nx, np.nan); pv = np.full(nx, np.nan)
            for b in range(nx):
                a = A[:, b]; a = a[np.isfinite(a)]
                d = B[:, b]; d = d[np.isfinite(d)]
                if len(a) >= min_n and len(d) >= min_n:
                    ks = ks_2samp(a, d); D[b], pv[b] = ks.statistic, ks.pvalue
            ax.plot(xg, pv, "-o", ms=3, color="0.2")
            ax.axhline(0.05, ls="--", c="C3", lw=1); ax.axvline(0, ls=":", c="k", lw=1)
            ax.fill_between(xg, 1e-4, 1.0, where=np.isfinite(pv) & (pv < 0.05),
                            color="C3", alpha=0.12, lw=0)
            ax.set_yscale("log"); ax.set_ylim(1e-4, 1.5)
            if ri == 0: ax.set_title(f"aligned on {st}")
            if ri == len(rows) - 1: ax.set_xlabel(rf"$t - t_{{\rm {st}}}$ [Gyr]")
            if ci == 0: ax.set_ylabel((f"{tag}\n" if bi is not None else "") + "KS p-value")
            tested = np.isfinite(pv)
            if tested.any():
                frac = float(np.mean(pv[tested] < 0.05))
                jb = int(np.nanargmax(np.where(tested, D, -np.inf)))
                print(f"  {tag:>14s} | {st:>11s}: {int(tested.sum()):2d} bins, "
                      f"{frac * 100:3.0f}% with p<0.05;  max D={D[jb]:.2f} "
                      f"(p={pv[jb]:.1e}) at t-t_{st}={xg[jb]:+.1f} Gyr")
            else:
                print(f"  {tag:>14s} | {st:>11s}: no bins with n>={min_n} in both groups")
    fig.suptitle(f"Shape KS test ({_SFH_NAME[kind]}) — {lA} vs {lB};  "
                 f"shaded = distinct (p<0.05)", y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

# companions to the four sfh_shape() demos above
sfh_shape_ks(("sf_peak", "sft", "qt"), kind="sfr", fname="sfh_shape_ks_sfr.png")
sfh_shape_ks(("sf_peak", "sft", "qt"), kind="ssfr", split=dust_split(-5.0, "post_quench"),
             fname="sfh_shape_ks_ssfr_dustsplit.png")
sfh_shape_ks(("sf_peak", "qt", "gas_min"), kind="gas/M*", fname="sfh_shape_ks_gasfrac.png")
sfh_shape_ks(("sf_peak", "qt", "dust_peak"), kind="dust/M*", fname="sfh_shape_ks_dustfrac.png")


### V.17 (§8h) · Dust-to-metal ratio, central/satellite & $N_{\rm sat}$ — event-aligned, split by dust at SF-peak

Three more quantities registered into the §8e engine and stacked on **t_AGN** and **t_env**, this
time grouping galaxies by their **dust content at the SF-peak** (`dust_split(-5, "sf_peak")`) instead
of fast/slow:

- **DTM** — dust-to-metal ratio (De Vis 2019), the formula from `simbanator/utils/conversions.py`:
  $\mathrm{DTM}=M_{\rm dust}/\big(M_{\rm dust}+27.36\cdot10^{\,\mathrm{O/H}-12}\,M_{\rm H_2}\big)$,
  with $12+\log(\mathrm{O/H})$ from the gas mass-weighted metallicity (`Z_to_OH12`). Needs
  `metallicities.mass_weighted` in the history — already added to `PROPS['galaxy_data']` (§1b);
  **rerun §1b once with `BUILD_HISTORY=True`** if `P` does not have it yet.
- **central** — central(1)/satellite(0) fraction, full per-snapshot history from §7L `cen_full`.
- **N_sat** — number of halo companions, full history from §7L `nsat_full` (falls back to the
  stage-sampled §7a `SAT['Nsat']` if §7L has not been rerun with the `nsat_full` addition).

Run order: §8e (registry/`event_stack`), §4 (`DIAGS`), §7L (`cen_full`/`nsat_full`), §8g (`dust_split`).

In [ ]:
# 8h · register DTM + central/satellite + N_sat, then event-align (t_AGN, t_env) split by dust @ sf_peak.
#   Needs §8e (PROP_REGISTRY, _reg, _reg_stage, event_stack), §4 (DIAGS), §7L (cen_full[, nsat_full]), §8g (dust_split).

# (1) dust-to-metal ratio history (De Vis 2019) — dimensionless, (n_snap, n_gal)
_ZKEY = "metallicities.mass_weighted"
if _ZKEY in P:
    with np.errstate(all="ignore"):
        _OH = Z_to_OH12(np.where(P[_ZKEY] > 0, P[_ZKEY], np.nan))          # 12 + log(O/H) from gas Z
        dtm_hist = Dust_to_Metal(P["masses.dust"], P["masses.H2"], _OH)    # M_dust / (M_dust + f_z·M_H2)
    _reg("DTM", dtm_hist, False, "dust-to-metal (De Vis+19)")
    print(f"registered DTM  (finite median = {np.nanmedian(dtm_hist):.3f})")
else:
    print(f"DTM skipped — '{_ZKEY}' not in P. It IS in PROPS['galaxy_data'] now; rerun §1b once with "
          f"BUILD_HISTORY=True to rebuild the cached history, then re-run this cell.")

# (2) central/satellite — full per-snapshot history from §7L (record-indexed -> transpose for _reg by='rec')
if "cen_full" in globals():
    _reg("central", cen_full.T, False, "central frac (1=cen, 0=sat)", by="rec")
    print("registered central")
else:
    print("central skipped — run §7L first (defines cen_full).")

# (3) N_sat — prefer the full history (§7L nsat_full); else stage-sampled SAT['Nsat'] (§7a) via _reg_stage
if "nsat_full" in globals():
    _reg("N_sat", nsat_full.T, False, r"$N_{\rm sat}$", by="rec")
    print("registered N_sat (full history)")
elif "SAT" in globals():
    _reg_stage("N_sat", SAT["Nsat"], STAGES, False, r"$N_{\rm sat}$")
    print("registered N_sat (stage-sampled — rerun §7L for the dense full-history version)")
else:
    print("N_sat skipped — run §7a (SAT) or rerun §7L (nsat_full).")

# (4) split by dust content at SF-peak, then event-align on AGN ignition and environment entry
SPLIT_DUST_SFP = dust_split(-5.0, "sf_peak")
_da, _db = SPLIT_DUST_SFP[0][0], SPLIT_DUST_SFP[1][0]
print(f"dust split @ sf_peak:  dusty={int(_da.sum())}  non-dusty={int(_db.sum())}  "
      f"undefined={len(records) - int(_da.sum()) - int(_db.sum())}")
print("cross-tab vs the active split:")
print(f"  dusty    : {LBL_A} {int(np.sum(_da & is_fast)):3d}   {LBL_B} {int(np.sum(_da & is_slow)):3d}")
print(f"  non-dusty: {LBL_A} {int(np.sum(_db & is_fast)):3d}   {LBL_B} {int(np.sum(_db & is_slow)):3d}")

event_stack([p for p in ["DTM", "central", "N_sat"] if p in PROP_REGISTRY],
            triggers=["t_env", "t_AGN", "t_QT"], split=SPLIT_DUST_SFP,
            fname="event_stack_dustsplit_sfpeak_DTM_cen_Nsat.png")

### V.18 (§8i) · Kinematic support (disc vs pressure, stars & gas) + clocked stacks faceted by stellar-mass bin

`kappa_rot` ≈ fraction of kinetic energy in ordered rotation: **≳0.5 → disc / rotation-supported**, lower → dispersion / **pressure-supported**. We pull a full per-component kinematic set (kappa_rot, sigma, B/T, specific angular momentum, gas-star spin misalignment) for stars & gas, plus halo virial quantities, straight from the CAESAR `rotation` dict and clock them on the event triggers, split by **stellar-mass bin** (the answer to "clocked properties for different mass bins instead of dust"). If the probe below shows only `total_kappa_rot`, this build has no per-component κ in the catalogue — compute it from particles instead (extend `build_profiles_job.py` with `Velocities`).


In [ ]:
# 8i · (a) register disc/pressure-support kinematics for stars & gas (kappa_rot, sigma, B/T,
#      specific angular momentum, gas-star spin misalignment) + halo virial quantities into the
#      §8e registry; (b) clock them split by STELLAR-MASS BIN (clocked properties by mass bin).
#   Needs §8e (event_stack,_reg,_track,PROP_REGISTRY), §0/§3 (mbin,MASS_BIN_LABELS,NBINS).
KAPPA_DISC = 0.5   # kappa_rot above this ~ disc-supported (reference; stacks show the continuous value)

# --- probe: which rotation/dispersion/virial keys this build exposes (galaxy & halo dicts) ---
for _grp in ("galaxy_data/dicts", "halo_data/dicts"):
    try:
        with h5py.File(sim.get_caesar_file(int(snaps_arr[0])), "r") as _f:
            _av = sorted(k for k in _f[_grp].keys()
                         if any(t in k for t in ("rotation", "kappa", "velocity_disp", "virial")))
        print(f"{_grp}: {_av}")
    except Exception as e:
        print(f"  (probe {_grp} failed: {e!r})")
print("rotation.* already in history P:", sorted(k for k in P if k.startswith("rotation.")))

# --- continuous per-component kinematics (auto-skipped where the key is absent from P) ---
_reg("kappa_star",  P.get("rotation.stellar_kappa_rot"),   False, r"$\kappa_{\rm rot}^{\star}$")
_reg("kappa_gas",   P.get("rotation.gas_kappa_rot"),       False, r"$\kappa_{\rm rot}^{\rm gas}$")
_reg("kappa_tot",   P.get("rotation.total_kappa_rot"),     False, r"$\kappa_{\rm rot}^{\rm tot}$")
_reg("BoverT_star", P.get("rotation.stellar_BoverT"),      False, r"$(B/T)_{\star}$")
_reg("BoverT_gas",  P.get("rotation.gas_BoverT"),          False, r"$(B/T)_{\rm gas}$")
_reg("sigma_star",  P.get("velocity_dispersions.stellar"), False, r"$\sigma_{\star}$ [km/s]")
_reg("sigma_gas",   P.get("velocity_dispersions.gas"),     False, r"$\sigma_{\rm gas}$ [km/s]")
_reg("M200c",       P.get("halo_data/dicts/virial_quantities.m200c"),      True,  r"$M_{\rm 200c}$ [M$_\odot$]")
_reg("spin_param",  P.get("halo_data/dicts/virial_quantities.spin_param"), True,  r"halo spin $\lambda$")

# --- gas-star spin MISALIGNMENT + specific angular momentum (robust to L-vector vs ALPHA/BETA) ---
def _spin_unit(comp):
    """Unit angular-momentum vector (n_snap, n_gal, 3) for `comp`: from the L 3-vector if stored,
       else reconstructed from the (ALPHA, BETA) spherical angles (assumed RADIANS; BETA polar from
       z, ALPHA azimuth — verify against your CAESAR build). Returns None if neither is available."""
    L = P.get(f"rotation.{comp}_L")
    if L is not None and L.ndim == 3 and L.shape[-1] == 3:
        nrm = np.linalg.norm(L, axis=-1, keepdims=True)
        return np.where(nrm > 0, L / nrm, np.nan)
    a = P.get(f"rotation.{comp}_ALPHA"); b = P.get(f"rotation.{comp}_BETA")
    if a is None or b is None:
        return None
    sb = np.sin(b)
    return np.stack([sb * np.cos(a), sb * np.sin(a), np.cos(b)], axis=-1)

_ug, _us = _spin_unit("gas"), _spin_unit("stellar")
if _ug is not None and _us is not None:
    with np.errstate(invalid="ignore"):
        _cos = np.clip(np.sum(_ug * _us, axis=-1), -1.0, 1.0)
        misalign_gas_star = np.degrees(np.arccos(_cos))          # (n_snap, n_gal), degrees in [0,180]
    _reg("misalign_gas_star", misalign_gas_star, False, r"gas$-\star$ spin misalignment [deg]")
    print(f"registered gas-star misalignment (overall median = {np.nanmedian(misalign_gas_star):.1f} deg)")
else:
    print("misalignment skipped — rotation.{gas,stellar}_L / ALPHA / BETA not in history yet (rebuild §1b)")

# specific angular momentum j = |L| / M (handles L stored as a 3-vector OR a scalar magnitude)
_JLAB = {"stellar": (r"$j_{\star}=|L|/M_{\star}$", "masses.stellar", "jspec_star"),
         "gas":     (r"$j_{\rm gas}=|L|/M_{\rm gas}$", "masses.gas",  "jspec_gas")}
for _c, (_lab, _mk, _name) in _JLAB.items():
    _L = P.get(f"rotation.{_c}_L"); _M = P.get(_mk)
    if _L is not None and _M is not None:
        _Lmag = np.linalg.norm(_L, axis=-1) if _L.ndim == 3 else _L
        with np.errstate(divide="ignore", invalid="ignore"):
            _j = np.where(_M > 0, _Lmag / _M, np.nan)
        _reg(_name, _j, True, _lab)

_kin = [k for k in PROP_REGISTRY if any(k.startswith(p) for p in
        ("kappa", "BoverT", "sigma", "jspec", "misalign", "M200c", "spin"))]
print("kinematic props registered:", _kin or "(none — rebuild §1b history with the new keys first)")

# --- stellar-mass bin AT a chosen critical point (default split uses the fixed z=0 mbin) ---
def _massbin_at(stage):
    """Per-record stellar-mass bin index from logM* AT `stage` anchor (e.g. 'sft','qt','z0').
       Same edges as §0; default mass_split() uses the fixed z=0 `mbin` instead."""
    lm = np.log10(np.where(gather_stage("masses.stellar", stage) > 0,
                           gather_stage("masses.stellar", stage), np.nan))
    b = np.full(len(records), -1, int)
    for k in range(NBINS):
        b[(lm >= MASS_BIN_EDGES[k]) & (lm < MASS_BIN_EDGES[k + 1])] = k
    return b

# --- split spec by stellar-mass bin: a drop-in for event_stack's split= argument ---
def mass_split(stage=None, binarr=None, labels=None, cmap="plasma"):
    """Split spec [(mask, color, label), ...] by stellar-mass bin, for event_stack.
       stage=None -> fixed z=0 bins (§0 mbin, default); stage='sft'/'qt'/... -> bin by logM*
       AT that anchor (mass when quenching). Or pass binarr= directly to override."""
    if binarr is None:
        binarr = mbin if stage is None else _massbin_at(stage)
    labels = MASS_BIN_LABELS if labels is None else labels
    cols = [plt.get_cmap(cmap)(x) for x in np.linspace(0.1, 0.85, len(labels))]
    return [((binarr == bi), cols[bi], f"logM* {labels[bi]}") for bi in range(len(labels))]

print("mass-bin split sizes:", {MASS_BIN_LABELS[b]: int((mbin == b).sum()) for b in range(NBINS)})

# --- clocked stacks BY MASS BIN: kinematic support next to the usual ISM/AGN tracers ---
_MASS_TRIGS = ["t_env", "t_AGN", "t_QT"]
event_stack(["kappa_star", "kappa_gas", "sSFR", "dust/M*", "M_H2", "BHAR"],
            triggers=_MASS_TRIGS, split=mass_split(), fname="event_stack_massbins.png")
# disc/pressure diagnostics on a wider window (kinematics evolve slowly), per mass bin
event_stack([p for p in ["kappa_star", "kappa_gas", "BoverT_star", "sigma_gas",
                         "misalign_gas_star", "jspec_star"] if p in PROP_REGISTRY],
            triggers=["t_SFpeak"] + _MASS_TRIGS, split=mass_split(), xwin=5.0, nb=10,
            fname="event_stack_massbins_kinematics.png")
# same split, but by stellar mass AT QUENCH (logM* at QT) rather than the fixed z=0 bins
event_stack([p for p in ["kappa_star", "kappa_gas", "sSFR", "M_H2"] if p in PROP_REGISTRY],
            triggers=_MASS_TRIGS, split=mass_split(stage="qt"),
            fname="event_stack_massbins_atQT.png")


### V.19 (§8j) · Direct test of the working scenario — overmassive halos, CGM→HI→H$_2$ funnel, AGN–ISM coupling

The refined hypothesis (see intro): galaxies with $10.25<\log M_\star<11.25$ in **overmassive
halos** ($\Delta\log M_{\rm halo}>0$ at fixed $M_\star$, §8a) funnel HI from the CGM into a large
rotating H$_2$ disk that feeds the BH; once jets ignite the galaxy quenches fast, and the extended
dust survives **unless** the AGN couples to the ISM — in SIMBA that is the **X-ray feedback**
channel, active only when the BH is in jet mode *and* $f_{\rm gas}=M_{\rm gas}/M_\star<0.2$.

This cell wires the missing pieces into the §8e engine:

1. `f_gas` history and an `xray_mode` indicator (jet-mode BH **and** $f_{\rm gas}<0.2$) — the
   ISM-coupling proxy;
2. `shmr_split()` — over- vs under-massive halo at fixed $M_\star$, restricted to the hypothesis
   mass window;
3. `xcoupling_split()` — galaxies that did vs did not pass through X-ray mode after AGN ignition.

**Predicted signatures if the scenario is right** (all on the $t-t_{\rm AGN}$ clock, overmassive
branch, in-window): (a) cold CGM falls while HI/M$_*$ then H$_2$/M$_*$ rise *before* $t_{\rm AGN}$;
(b) high gas $\kappa_{\rm rot}$ / large $R_{50}^{\rm H_2}$ during the funnel; (c) BHAR rise →
jet onset → fast sSFR collapse; (d) after QT, $R_{50}^{\rm dust}/R_{50}^\star$ stays $\gtrsim$1
for the never-coupled subset and collapses for the X-ray-coupled one.

In [ ]:
# 8j · operationalize the scenario: mass window + SHMR-branch split + AGN-ISM coupling.
#   Needs §8·setup (dMh, logMs), §8c/§8e (t_agn, t_inc, _track, _reg, event_stack), §7b (JET_*),
#   §4b (BH), §8f (Rdust/Rstar), §8i (kappa_gas). Run order: after §8i.
MASS_WIN        = (10.8, 11.5)   # hypothesis stellar-mass window [log10 Msun]
WIN_STAGE       = "qt"             # stage at which the window & SHMR branch are evaluated
SHMR_SPLIT_MODE = "tercile"        # "tercile": top vs bottom third of ΔlogMh (middle excluded)
SHMR_DEADBAND   = 0.15             # dex; used when SHMR_SPLIT_MODE == "deadband"
XRAY_FGAS_MAX   = 0.2              # X-ray feedback trigger: f_gas = Mgas/M* below this ...
XRAY_FEDD_MAX   = 0.02             # ... AND full-velocity jets f_Edd <= 0.02 (Dave+19)

# --- (1) coupling histories: f_gas, continuous jet strength, strict X-ray trigger ---
with np.errstate(all="ignore"):
    fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
    _bh_ok    = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
    _mbh_ok   = BH["bh_mass"] > 10 ** JET_LOGMBH
    # continuous jet strength: SIMBA's jet boost ~ log10(0.2/f_Edd), saturating at full
    # velocity for f_Edd <= 0.02  ->  w_jet in [0, 1]
    wjet_hist = np.where(_bh_ok,
                         np.where(_mbh_ok,
                                  np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)),
                                          0.0, 1.0), 0.0),
                         np.nan)
    # STRICT ISM-coupling trigger (the in-model X-ray feedback condition)
    xray_hist  = np.where(_bh_ok & np.isfinite(fgas_hist),
                          (_mbh_ok & (BH["bh_fedd"] < XRAY_FEDD_MAX)
                           & (fgas_hist < XRAY_FGAS_MAX)).astype(float), np.nan)
    # continuous coupling strength: jet strength gated by the gas-poor condition
    xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                          wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
_reg("f_gas",         fgas_hist,  True,  r"$f_{\rm gas}=M_{\rm gas}/M_*$")
_reg("jet_strength",  wjet_hist,  False, r"$w_{\rm jet}=\min[\log_{10}(0.2/f_{\rm Edd}),\,1]$")
_reg("xray_mode",     xray_hist,  False, r"X-ray-coupled (full jets & $f_{\rm gas}<0.2$)")
_reg("xray_strength", xcoup_hist, False, r"coupling strength $w_{\rm jet}\cdot[f_{\rm gas}<0.2]$")

# --- (2) mass window + SHMR-branch split (over- vs under-massive halo at fixed M*) ---
in_win = (np.isfinite(logMs[WIN_STAGE])
          & (logMs[WIN_STAGE] > MASS_WIN[0]) & (logMs[WIN_STAGE] < MASS_WIN[1]))

def shmr_split(stage=WIN_STAGE, window=True, mode=SHMR_SPLIT_MODE):
    """[(mask,color,label),...]: overmassive vs undermassive halo at fixed M*, from the §8·setup
       z-binned running-median residual. 'tercile' contrasts the distribution tails (top vs
       bottom third; middle excluded); 'deadband' cuts at fixed ±SHMR_DEADBAND dex."""
    base = (in_win if window else np.isfinite(logMs[stage])) & np.isfinite(dMh[stage])
    d = dMh[stage]
    if mode == "tercile":
        lo, hi = np.nanquantile(d[base], [1.0 / 3.0, 2.0 / 3.0])
    else:
        lo, hi = -SHMR_DEADBAND, SHMR_DEADBAND
    over, under = base & (d >= hi), base & (d <= lo)
    w = f", {MASS_WIN[0]}<logM*<{MASS_WIN[1]}" if window else ""
    return [(over,  "C1", f"overmassive halo (ΔlogMh≥{hi:+.2f}) @{stage}{w}"),
            (under, "C0", f"undermassive halo (ΔlogMh≤{lo:+.2f})")]

SPLIT_SHMR = shmr_split()
_ov, _un = SPLIT_SHMR[0][0], SPLIT_SHMR[1][0]
print(f"mass window {MASS_WIN}: {int(in_win.sum())} galaxies | mode={SHMR_SPLIT_MODE!r}: "
      f"overmassive {int(_ov.sum())} (med ΔlogMh {np.nanmedian(dMh[WIN_STAGE][_ov]):+.2f}), "
      f"undermassive {int(_un.sum())} (med {np.nanmedian(dMh[WIN_STAGE][_un]):+.2f}), "
      f"excluded mid {int(in_win.sum()) - int(_ov.sum()) - int(_un.sum())}")

# --- (3) AGN-ISM coupling after ignition: strict binary AND continuous strength ---
xfrac_post = np.full(len(records), np.nan)   # fraction of post-t_AGN snaps STRICTLY coupled
xstr_post  = np.full(len(records), np.nan)   # mean continuous coupling strength after t_AGN
for i, r in enumerate(records):
    if not np.isfinite(t_agn[i]):
        continue
    post = t_inc >= t_agn[i]
    xs = _track(xray_hist, r["col"]);  ok = post & np.isfinite(xs)
    if ok.sum() >= 2:
        xfrac_post[i] = np.nanmean(xs[ok])
    cs = _track(xcoup_hist, r["col"]); ok = post & np.isfinite(cs)
    if ok.sum() >= 2:
        xstr_post[i] = np.nanmean(cs[ok])

def xcoupling_split(window=True, mode="strict"):
    """'strict': ever passed the X-ray trigger after t_AGN vs never (the in-model condition).
       'strength': top vs bottom tercile of the mean continuous coupling strength."""
    base = in_win if window else np.ones(len(records), bool)
    if mode == "strict":
        b = base & np.isfinite(xfrac_post)
        return [(b & (xfrac_post > 0),  "C3", "ISM-coupled (X-ray trigger after t_AGN)"),
                (b & (xfrac_post == 0), "C0", "never coupled")]
    b = base & np.isfinite(xstr_post)
    lo, hi = np.nanquantile(xstr_post[b], [1.0 / 3.0, 2.0 / 3.0])
    return [(b & (xstr_post >= hi), "C3", f"strong coupling (≥{hi:.2f})"),
            (b & (xstr_post <= lo), "C0", f"weak coupling (≤{lo:.2f})")]

SPLIT_X  = xcoupling_split()                   # strict in-model trigger
SPLIT_XS = xcoupling_split(mode="strength")    # continuous-strength terciles
print(f"X-ray coupling (post-t_AGN, in window): strict {int(SPLIT_X[0][0].sum())} coupled / "
      f"{int(SPLIT_X[1][0].sum())} never | strength terciles "
      f"{int(SPLIT_XS[0][0].sum())} strong / {int(SPLIT_XS[1][0].sum())} weak")

# --- the scenario, link by link, on the AGN-ignition clock ---
# (a,b) the funnel: HI -> H2 build-up + gas fraction + disc-ness, by SHMR branch
event_stack(["HI/M*", "H2/M*", "f_gas"] + (["kappa_gas"] if "kappa_gas" in PROP_REGISTRY else []),
            triggers=["t_AGN"], split=SPLIT_SHMR, xwin=3.0, fname="scenario_funnel_shmr.png")
# (c) trigger & quench: BHAR -> jet spin-up (w_jet -> 1) -> sSFR collapse, by SHMR branch
event_stack(["BHAR", "jet_strength", "sSFR"], triggers=["t_AGN"], split=SPLIT_SHMR, xwin=3.0,
            fname="scenario_trigger_shmr.png")
# (d) the dust verdict, twice: strict in-model trigger and continuous coupling strength
event_stack(["dust/M*"] + (["Rdust/Rstar"] if "Rdust/Rstar" in PROP_REGISTRY else []),
            triggers=["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench"], split=SPLIT_X, xwin=2.0, nb=10,
            fname="scenario_dust_xcoupling.png")

_reg("H2/M*",      _ratio_hist("masses.H2", "masses.stellar"),   True,  r"$M_{\rm H_2}/M_*$")
_reg("R20g/R80g",      _ratio_hist("radii.gas_r20", "radii.gas_r80"),   True,  r"$R20/R80, gas$")
event_stack(["dust/M*", "H2/M*", "R20g/R80g"] + (["Rdust/Rstar"]),
            triggers=["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench"], split=SPLIT_XS, xwin=2.0, nb=10,
            fname="scenario_dust_xstrength.png")


### V.20 (§8k) · Strangulation vs mechanical removal — the ISM supply-and-sink ledger

A quiescent galaxy's ISM can decline through three pathways, and they leave different
fingerprints in the per-galaxy ledger between snapshots:

$$\dot M_{\rm gas} = \dot M_{\rm accr} - (1{-}R)\,{\rm SFR} - \dot M_{\rm out}
\;\;\Rightarrow\;\;
\underbrace{-\dot M_{\rm gas} - (1{-}R)\,{\rm SFR}}_{\texttt{removal\_gas}} = \dot M_{\rm out} - \dot M_{\rm accr}$$

| pathway | gas ledger | dust | D/G |
|---|---|---|---|
| **strangulation** (supply cut; SIMBA's canonical jet quenching) | declines at $\approx$ the astration rate $(1{-}R)\,$SFR; `removal_gas` $\approx 0$ | locked into stars with the gas, gently | preserved |
| **ejective / mechanical** (winds push ISM out) | `removal_gas` $> 0$ | leaves *with* the gas | preserved |
| **destruction** (X-ray heating → sputtering; §8j coupling) | little direct effect | `removal_dust` $\gg$ `removal_gas` | **drops** |

Strangulation is therefore not a separate rate but a **regime**: the post-QT gas decline is
classified per galaxy by comparing the integrated decline $\Delta M_{\rm gas}$ with the
integrated astration $A$ over `BUDGET_T_GYR` —
**strangled** ($|\Delta M_{\rm gas}-A| <$ `MECH_FRAC`$\,\Delta M_{\rm gas}$: consumption
without resupply), **ejective** (decline clearly exceeds astration), or
**replenished/growing** (decline falls short of astration, or gas grows: not strangled).

The cell also measures strangulation from the **supply side**: the cold CGM reservoir history
$M^{\rm cold}_{\rm CGM} =$ (halo HI+H$_2$) $-$ (ISM HI+H$_2$), with a per-galaxy
**strangulation clock** `t_Strang` = first persistent drop below `STRANG_FRAC` of its own
peak (anti-flicker, `STRANG_PERSIST` snaps) — added to `TRIGGERS`, so anything can be stacked
on the supply-cut moment. The ordering test prints $t_{\rm Strang}-t_{\rm AGN}$ (jets cut the
supply ⇒ positive) and $t_{\rm QT}-t_{\rm Strang}$ (strangulation precedes full quenching ⇒
positive).

**For the working scenario:** the overmassive branch should be **strangled**-dominated
(jets starve the halo; dust survives, D/G flat) with the **destroyed** cases concentrated in
the strongly coupled tercile; a large *ejective* population at weak coupling would instead
point to stellar/radiative winds and weaken the AGN-centric story.

> **Caveats.** `removal_gas` is a *net* rate (outflow exactly balanced by inflow reads as 0 —
> small effect post-quench in starved halos, which is the regime of interest); snapshot
> cadence (~100–250 Myr) smooths bursts; mergers appear as accretion spikes (negative
> removal) — the stacks use medians, robust to them; $R=0.43$ (instantaneous recycling,
> Chabrier) is a config knob.


In [ ]:
# 8k · strangulation vs mechanical removal: ledger rates, post-QT regimes, supply clock.
#   removal_gas  = -dM_gas/dt  - (1-R)·SFR        = outflow - accretion   [net mechanical sink]
#   removal_dust = -dM_dust/dt - (1-R)·SFR·(D/G)  = destruction + ejection - production
#   Strangled regime: gas declines at ~the astration rate with removal ~ 0 (supply cut).
#   Needs §8e (_reg, _track, _ord, t_inc, TRIGGERS), §8j (wjet_hist, SPLIT_SHMR, SPLIT_XS).
RETURN_FRAC    = 0.43    # instantaneous return fraction (Chabrier IMF)
BUDGET_T_GYR   = 3.0     # post-QT window for the regime classification
MECH_FRAC      = 0.25    # |decline - astration| below this fraction of the decline -> strangled
STRANG_FRAC    = 0.1     # supply 'cut' when cold CGM < this x its own peak ...
STRANG_PERSIST = 2       # ... for at least this many consecutive snapshots (anti-flicker)

_inv_ord = np.argsort(_ord)
def _ddt_hist(arr):
    """d(arr)/dt [units/yr] along each galaxy track (original (n_snap, n_gal) row order)."""
    a = np.asarray(arr, float)[_ord, :]
    with np.errstate(all="ignore"):
        d = np.gradient(a, t_inc, axis=0)
    return d[_inv_ord, :]

# --- (1) ledger rates, registered as specific rates [Gyr^-1] (>0 = net sink) ---
with np.errstate(all="ignore"):
    _gas, _dust, _sfr = P["masses.gas"], P["masses.dust"], P["sfr"]
    _dgr   = np.where(_gas > 0, _dust / _gas, np.nan)             # D/G
    astr_g = (1.0 - RETURN_FRAC) * _sfr                           # gas lock-up rate [Msun/yr]
    astr_d = astr_g * _dgr                                        # dust lock-up rate
    rem_g  = -_ddt_hist(_gas)  - astr_g                           # net mechanical gas sink
    rem_d  = -_ddt_hist(_dust) - astr_d                           # dust destr.+eject.-production
    sast_g = np.where(_gas  > 0, astr_g / _gas,  np.nan) * 1e9
    srem_g = np.where(_gas  > 0, rem_g  / _gas,  np.nan) * 1e9
    sast_d = np.where(_dust > 0, astr_d / _dust, np.nan) * 1e9
    srem_d = np.where(_dust > 0, rem_d  / _dust, np.nan) * 1e9
_reg("astration_gas",  sast_g, False, r"$(1{-}R)\,{\rm SFR}/M_{\rm gas}$ [Gyr$^{-1}$]")
_reg("removal_gas",    srem_g, False, r"net mech. gas sink $/M_{\rm gas}$ [Gyr$^{-1}$]")
_reg("astration_dust", sast_d, False, r"dust astration $/M_{\rm dust}$ [Gyr$^{-1}$]")
_reg("removal_dust",   srem_d, False, r"dust destr.$+$eject. $/M_{\rm dust}$ [Gyr$^{-1}$]")

# --- (2) post-QT regime: strangled vs ejective vs replenished, per galaxy ---
regime = np.full(len(records), "undefined", dtype=object)
f_astr = np.full(len(records), np.nan)        # astration / decline (1 = pure strangulation)
for i, r in enumerate(records):
    g = _track(P["masses.gas"], r["col"]); s = _track(P["sfr"], r["col"])
    m = (t_inc >= r["t_qt"]) & (t_inc <= r["t_qt"] + BUDGET_T_GYR * 1e9) & np.isfinite(g)
    if m.sum() < 3:
        continue
    dG = g[m][0] - g[m][-1]                                  # gas decline over the window
    A  = np.trapz((1.0 - RETURN_FRAC) * np.nan_to_num(s[m]), t_inc[m])
    if dG <= 0:
        regime[i] = "replenished/growing"; continue
    f_astr[i] = np.clip(A / dG, 0.0, 1.5)
    Mmech = dG - A
    if Mmech > MECH_FRAC * dG:
        regime[i] = "ejective"
    elif Mmech < -MECH_FRAC * dG:
        regime[i] = "replenished/growing"
    else:
        regime[i] = "strangled"

_REGS = ["strangled", "ejective", "replenished/growing"]
_groups = ([("all", np.ones(len(records), bool)), (LBL_A, is_fast), (LBL_B, is_slow)]
           + [(lab, mk) for mk, _c, lab in SPLIT_SHMR] + [(lab, mk) for mk, _c, lab in SPLIT_XS])
print(f"post-QT ({BUDGET_T_GYR:g} Gyr) ISM-decline regimes (MECH_FRAC={MECH_FRAC:g}):")
print(f"  {'group':46s}  {'strangled':>10s} {'ejective':>9s} {'replen.':>8s}   med f_astr   N")
for name, msk in _groups:
    b = msk & (regime != "undefined")
    if b.sum() < 5:
        continue
    fr = [np.mean(regime[b] == rg) for rg in _REGS]
    fa = np.nanmedian(f_astr[b & np.isfinite(f_astr)]) if np.isfinite(f_astr[b]).any() else np.nan
    print(f"  {name[:46]:46s}  {fr[0]:10.2f} {fr[1]:9.2f} {fr[2]:8.2f}   {fa:8.2f}   {int(b.sum())}")

# --- (3) supply side: cold-CGM history + strangulation clock t_Strang ---
_hHI, _hH2 = P.get("halo_data/dicts/masses.HI"), P.get("halo_data/dicts/masses.H2")
HAS_CGMCOLD = (_hHI is not None) and (_hH2 is not None)
if HAS_CGMCOLD:
    with np.errstate(all="ignore"):
        _galcold = np.nan_to_num(P["masses.HI"]) + np.nan_to_num(P["masses.H2"])
        cgm_cold_hist = np.where(np.isfinite(_hHI) & np.isfinite(_hH2),
                                 np.clip(_hHI + _hH2 - _galcold, 0.0, None), np.nan)
    _reg("CGM_cold", cgm_cold_hist, True, r"$M^{\rm cold}_{\rm CGM}$ (halo$-$ISM HI+H$_2$) [$M_\odot$]")
    t_strang = np.full(len(records), np.nan)
    for i, r in enumerate(records):
        v = _track(cgm_cold_hist, r["col"]); ok = np.isfinite(v)
        if ok.sum() < 4 or not (np.nanmax(v) > 0):
            continue
        ipk = int(np.nanargmax(v))
        below = ok & (v < STRANG_FRAC * np.nanmax(v)); below[:ipk] = False   # only after the peak
        run = 0
        for k in range(len(v)):
            run = run + 1 if below[k] else 0
            if run >= STRANG_PERSIST:
                t_strang[i] = t_inc[k - STRANG_PERSIST + 1]; break
    TRIGGERS["t_Strang"] = t_strang
    print(f"\nt_Strang (cold CGM < {STRANG_FRAC:g} x peak, persistent {STRANG_PERSIST}): "
          f"defined for {int(np.isfinite(t_strang).sum())}/{len(records)}")
    # ordering: jets cut the supply (t_Strang - t_AGN > 0); strangulation precedes full quench
    _tqt = np.array([r["t_qt"] for r in records])
    for lagname, lag in [("t_Strang - t_AGN [Gyr]", (t_strang - t_agn) / 1e9),
                         ("t_QT - t_Strang [Gyr]", (_tqt - t_strang) / 1e9)]:
        for name, msk in [(LBL_A, is_fast), (LBL_B, is_slow)] + \
                         [(lab, mk) for mk, _c, lab in SPLIT_SHMR]:
            v = lag[msk]; v = v[np.isfinite(v)]
            if len(v) >= 5:
                print(f"  {lagname:24s} {name[:38]:38s}: med {np.median(v):+.2f}  (N={len(v)})")
else:
    print("cold-CGM supply history unavailable (halo masses.HI/H2 not in P) -> no t_Strang;"
          " rerun §1b with BUILD_HISTORY=True including the halo_data keys.")

# --- (4) AGN attribution: does the mechanical sink track jet strength after quenching? ---
_rw = np.array([r["row_post_quench"] for r in records])
_remq = np.array([srem_g[_rw[i], cols[i]]    if _rw[i] >= 0 else np.nan for i in range(len(records))])
_wjq  = np.array([wjet_hist[_rw[i], cols[i]] if _rw[i] >= 0 else np.nan for i in range(len(records))])
_ok = np.isfinite(_remq) & np.isfinite(_wjq)
if _ok.sum() > 10:
    _rho, _pv = spearmanr(_remq[_ok], _wjq[_ok])
    print(f"\nSpearman(net gas removal, jet strength) at post_quench: "
          f"rho={_rho:+.2f} (p={_pv:.1g}, N={int(_ok.sum())})")

# --- (5) stacks: supply, sinks and sSFR on the AGN / quench / strangulation clocks ---
event_stack(["astration_gas", "removal_gas", "sSFR"] + (["CGM_cold"] if HAS_CGMCOLD else []),
            triggers=["t_AGN", "t_SFT", "t_QT"] + (["t_Strang"] if HAS_CGMCOLD else []),
            split=SPLIT_SHMR, xwin=4.0, fname="sinks_gas_shmr.png")
event_stack(["astration_dust", "removal_dust", "dust/M*"], triggers=["t_AGN", "t_SFT", "t_QT"],
            split=SPLIT_XS, xwin=4.0, fname="sinks_dust_xstrength.png")


### V.21 (§8l) · Who enters each branch? Population character at quenching

Before reading the §8j/§8k stacks causally, characterize *what kind of galaxy* lands in each
branch. For the three splits — **SHMR terciles** (over- vs undermassive halo), **strict X-ray
coupling** (ever triggered after $t_{\rm AGN}$ vs never) and **coupling strength** (strong vs
weak terciles) — we compare, at each galaxy's own **QT anchor**:

- **mass**: $\log M_\star$, $\log M_{\rm halo}$;
- **compactness**: stellar half-mass radius $R_{50}^\star$ (physical kpc) and the effective
  surface density $\Sigma_e = (M_\star/2)/(\pi R_{50}^2)$ (Barro-style compactness; note the
  catalogue $R_{50}$ is a 3D half-mass radius, so $\Sigma_e$ is consistent *internally* but
  ~25% different from a projected $R_e$ definition);
- **star formation**: sSFR at SFT (the last main-sequence state) and SFR at the SF peak;
- **metallicity**: gas-phase $12+\log({\rm O/H})$ (from `Z_to_OH12`) and stellar
  $Z_\star/Z_\odot$;
- **dust**: $M_{\rm dust}/M_\star$ for context.

Each panel: overlapped distributions, median lines, Mann–Whitney *p*; the printed table gives
the median per group.

> **Read with selection in mind.** Some contrasts are *built into* the split definitions and
> are not discoveries: the coupling splits condition on $f_{\rm gas}<0.2$ after ignition
> (coupled galaxies are gas-poor by construction, and plausibly dust-poor at z=0), the SHMR
> split fixes the $M_{\rm halo}$ contrast at fixed $M_\star$, and all branches live inside the
> 10.25–11.25 window (so $M_\star$ can only vary within it). The *informative* axes are the
> ones the splits do **not** condition on: compactness, pre-quench SF, and metallicity. E.g.
> if X-ray-coupled galaxies are compact and metal-rich at QT, that is the classic
> dissipative/compaction route; if the overmassive branch is metal-rich at fixed $M_\star$,
> that points to earlier formation in bigger halos.


In [ ]:
# 8l · population character at the QT anchor, compared across the three §8j splits.
#   Needs §8j (SPLIT_SHMR, SPLIT_X, SPLIT_XS), §4 (gather_stage), §0 (Z_to_OH12, mannwhitneyu).
RADII_COMOVING = True    # CAESAR half-mass radii assumed comoving kpc -> converted to physical
ZSUN           = 0.0134  # Asplund+09 solar metal mass fraction

def _z_at(stage):
    """redshift of each record's `stage` anchor row (NaN if the anchor is undefined)."""
    return np.array([redshift[r[f"row_{stage}"]] if r[f"row_{stage}"] >= 0 else np.nan
                     for r in records])

collect_stage='sft'
def _dp_conc(comp):  # R20/R80 size-ratio for `comp` at collect_stage from the §5 particle product (NaN if absent)
    if not (DP_OK and collect_stage in DP_STAGES):
        return np.full(len(records), np.nan)
    return DP_C2080[comp][:, DP_STAGES.index(collect_stage)]
with np.errstate(all="ignore"):
    _ms_qt  = gather_stage("masses.stellar", collect_stage)
    _r20    = gather_stage("radii.stellar_r20", collect_stage)
    _r50    = gather_stage("radii.stellar_half_mass", collect_stage)
    _r80    = gather_stage("radii.stellar_r80", collect_stage)
    _age    = gather_stage("ages.mass_weighted", collect_stage)
    if RADII_COMOVING:
        _r50 = _r50 / (1.0 + _z_at(collect_stage))                     # comoving kpc -> physical kpc
    _ms_sft = gather_stage("masses.stellar", collect_stage)
    _Zg     = gather_stage("metallicities.mass_weighted", collect_stage)
    CHAR = {
        r"$\log M_\star$ @{collect_stage}":                      _log(_ms_qt),
        r"$\log M_{\rm halo}$ @{collect_stage}":                 _log(gather_stage("masses.total", collect_stage)),
        r"$R20/R50$ @{collect_stage}":                 _r20/_r50,
        r"$R20/R80$ @{collect_stage}":                 _r20/_r80,
        r"$R20/R80\,{\rm H_2}$ @{collect_stage}":       _dp_conc("H2"),
        r"$R20/R80\,{\rm dust}$ @{collect_stage}":      _dp_conc("dust"),
        r"$Ages":                         _age,
        r"$\log R_{50}^\star$ [pkpc] @{collect_stage}":          _log(_r50),
        r"$\log \Sigma_e$ [$M_\odot$pkpc$^{-2}$] @{collect_stage}":
            _log(0.5 * _ms_qt / (np.pi * np.where(_r50 > 0, _r50, np.nan) ** 2)),
        r"$\log$ sSFR [yr$^{-1}$] @SFT":
            _log(np.where(_ms_sft > 0, gather_stage("sfr", "sft") / _ms_sft, np.nan)),
        r"$\log$ SFR [$M_\odot$yr$^{-1}$] @SF-peak": _log(gather_stage("sfr", "sf_peak")),
        r"12+log(O/H) gas @{collect_stage}":                     np.where(_Zg > 0, Z_to_OH12(_Zg), np.nan),
        r"$\log Z_\star/Z_\odot$ @{collect_stage}":              _log(gather_stage("metallicities.stellar", collect_stage) / ZSUN),
        r"$\log M_{\rm dust}/M_\star$ @{collect_stage}":
            _log(gather_stage("masses.dust", collect_stage) / gather_stage("masses.stellar", collect_stage)),
        r"$\log M_{\rm dust}/M_gas$ @{collect_stage}":
            _log(np.where(_ms_qt > 0, gather_stage("masses.dust", collect_stage) / gather_stage("masses.H2", collect_stage), np.nan)),
    }
    CHAR = {k.replace("{collect_stage}", collect_stage): v for k, v in CHAR.items()}  # interpolate @stage into labels

SPLITS_8L = [("SHMR terciles", SPLIT_SHMR), ("X-ray strict", SPLIT_X),
             ("coupling strength", SPLIT_XS)]
nr, nc = len(CHAR), len(SPLITS_8L)
fig, axes = plt.subplots(nr, nc, figsize=(4.8 * nc, 2.7 * nr), squeeze=False)
print(f"median per group (A | B) and MW p, per split:")
print(f"{'quantity':34s}" + "".join(f" | {nm:>26s}" for nm, _ in SPLITS_8L))
for ri, (lab, vals) in enumerate(CHAR.items()):
    fin = vals[np.isfinite(vals)]
    if len(fin) < 10:
        for ci in range(nc):
            axes[ri, ci].set_axis_off()
        continue
    bins = np.linspace(*np.nanpercentile(fin, [1, 99]), 22)
    row = f"{lab[:34]:34s}"
    for ci, (sname, sp) in enumerate(SPLITS_8L):
        ax = axes[ri, ci]
        for mk, c, glab in sp:
            v = vals[mk]; v = v[np.isfinite(v)]
            if len(v) < 3:
                continue
            ax.hist(v, bins=bins, density=True, histtype="stepfilled", alpha=0.45, color=c,
                    label=f"{glab.split('(')[0].strip()[:20]} ({len(v)})")
            ax.axvline(np.median(v), color=c, ls="--", lw=1.3)
        a = vals[sp[0][0]]; b = vals[sp[1][0]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if len(a) >= 3 and len(b) >= 3:
            p = mannwhitneyu(a, b, alternative="two-sided").pvalue
            ax.text(0.97, 0.95, f"p={p:.1g}", transform=ax.transAxes, ha="right", va="top",
                    fontsize=8, bbox=dict(fc="w", ec="0.7", alpha=0.7, pad=1))
            row += f" | {np.median(a):+7.2f} {np.median(b):+7.2f} {p:8.1g}"
        else:
            row += f" | {'--':>25s}"
        if ri == 0:
            ax.set_title(sname, fontsize=10)
        if ci == 0:
            ax.set_ylabel(lab, fontsize=8)
        ax.legend(fontsize=6)
    print(row)
fig.suptitle("§8l · who enters each branch? population character at quenching "
             "(A = over/coupled/strong, B = under/never/weak)", y=1.003)
plt.tight_layout()
plt.savefig(figpath( "branch_character_qt.png"), dpi=130, bbox_inches="tight")
plt.show()


### V.22 (§8m) · Sanity check — what do $\dot M_{\rm BH}$ and $f_{\rm Edd}$ do when the gas runs out?

The jet classification used in §7j/§8j/§10d is $M_{\rm BH}>10^{7.5}$ **and** $f_{\rm Edd}<0.2$.
That condition is also satisfied by a BH whose accretion has simply **died** — and in SIMBA the
jet *power* scales with $\dot M_{\rm BH}$, so a $\dot M\!\to\!0$ "jet-mode" BH launches nothing:
no feedback, no radio jet, **no AGN in any observable sense** (observed LERGs have
$\lambda\sim10^{-4}$–$10^{-2}$, not zero). If the catalogue records $f_{\rm Edd}=0$ (finite)
for gas-starved BHs, dead BHs contaminate the jet class; if it records NaN they are silently
excluded. This cell settles it empirically:

1. **per-galaxy extraction** — $M_{\rm gas}$, $M_{\rm H_2}$ (left axis) and
   $\dot M_{\rm BH}$, $f_{\rm Edd}$ (right axis) histories for `N_EX` quenched, jet-eligible
   galaxies with the deepest post-QT gas exhaustion (mixed fast/slow). Exact zeros are drawn
   as **down-triangles at the plot floor** (log axes cannot show 0); the $f_{\rm Edd}=0.2$ and
   $0.02$ thresholds and QT are marked;
2. **census** — among all (galaxy, snapshot) pairs in the jet-mode class (and post-QT only):
   the fraction with $\dot M_{\rm BH}\le0$/NaN and with $L_{\rm bol}<10^{40,41,42}$ erg s$^{-1}$
   — i.e. "jet AGN" with essentially no accretion power.

**If the census fractions are significant**, the jet flag and the continuous $w_{\rm jet}$
(§8j) and the §10d jet overlay need an **activity floor** (e.g. $L_{\rm bol}>10^{41}$ erg/s,
or $\lambda>10^{-4}$ to match observed LERGs) — otherwise "jet AGN" partly means "dead BH".


In [ ]:
# 8m · sanity check: bhmdot / f_Edd behaviour at gas exhaustion + jet-class contamination.
#   Needs §4b (BH), §3 (records, cols, classes, is_fast), §8c (_track, t_inc; local fallback).
N_EX      = 6
GAS_EMPTY = 1e8                  # reference 'gas finished' level [Msun] (for the printout)
_c2yr = (const.c ** 2 * u.Msun / u.yr).to(u.erg / u.s).value

if "_track" in globals() and "t_inc" in globals():
    _trk, _tg8 = _track, t_inc
else:
    _o8 = np.argsort(t_cosmic_yr); _tg8 = t_cosmic_yr[_o8]
    def _trk(arr, col): return arr[_o8, col].astype(float)

# --- pick examples: jet-eligible at z~0, deepest post-QT gas exhaustion, mixed fast/slow ---
mbh0 = BH["bh_mass"][0, cols]
gmin_post = np.full(len(records), np.nan)
for i, r in enumerate(records):
    g = _trk(P["masses.gas"], r["col"]); m = (_tg8 >= r["t_qt"]) & np.isfinite(g)
    if m.sum() >= 3:
        gmin_post[i] = float(np.min(np.where(g[m] > 0, g[m], 0.0)))
elig = np.isfinite(mbh0) & (mbh0 > 10 ** 7.5) & np.isfinite(gmin_post)
order8 = np.argsort(np.where(elig, gmin_post, np.inf))
ex = []
for want_fast in (True, False):
    ex += [int(i) for i in order8 if elig[i] and bool(is_fast[i]) == want_fast][:N_EX // 2]
ex = (ex + [int(i) for i in order8 if elig[i] and int(i) not in ex])[:N_EX]
print(f"examples (jet-eligible at z~0; deepest post-QT gas exhaustion; GAS_EMPTY={GAS_EMPTY:.0e}):")
for i in ex:
    print(f"  gid {records[i]['gid']:8d}  {str(classes[i]):>6s}  min post-QT M_gas = {gmin_post[i]:.2e}")

def _plot0(ax, x, y, floor, **kw):
    """log-plot a track; finite zeros/non-positives drawn as down-triangles at `floor`."""
    y = np.asarray(y, float)
    pos = np.isfinite(y) & (y > 0); zer = np.isfinite(y) & (y <= 0)
    ax.plot(x[pos], y[pos], **kw)
    if zer.any():
        ax.scatter(x[zer], np.full(int(zer.sum()), floor), marker="v", s=32,
                   color=kw.get("color", "k"), zorder=6)

fig, axes = plt.subplots(2, 3, figsize=(18, 9.5), sharex=True)
for k, i in enumerate(ex):
    ax = axes.ravel()[k]; r = records[i]; col = r["col"]
    tt = _tg8 / 1e9
    _plot0(ax, tt, _trk(P["masses.gas"], col), 1.5e6, color="k", lw=2, label=r"$M_{\rm gas}$")
    _plot0(ax, tt, _trk(P["masses.H2"], col),  1.5e6, color="C2", lw=2, label=r"$M_{\rm H_2}$")
    ax.set_yscale("log"); ax.set_ylim(1e6, None)
    ax.axhline(GAS_EMPTY, color="0.7", lw=0.8, ls=":")
    ax.axvline(r["t_qt"] / 1e9, color="0.4", ls=":", lw=1.5)
    axr = ax.twinx(); axr.set_yscale("log"); axr.set_ylim(1e-8, 30)
    _plot0(axr, tt, _trk(BH["bh_mdot"], col), 2e-8, color="C3", lw=1.8, label=r"$\dot M_{\rm BH}$")
    _plot0(axr, tt, _trk(BH["bh_fedd"], col), 2e-8, color="C0", lw=1.8, ls="--", label=r"$f_{\rm Edd}$")
    axr.axhline(0.2, color="C0", lw=0.8, ls=":"); axr.axhline(0.02, color="C0", lw=0.8, ls=":")
    ax.set_title(f"gid {r['gid']} ({classes[i]});  dotted vline = QT", fontsize=11)
    if k == 0:
        ax.legend(fontsize=9, loc="lower left"); axr.legend(fontsize=9, loc="upper right")
    if k >= 3:
        ax.set_xlabel("cosmic time t [Gyr]")
    if k % 3 == 0:
        ax.set_ylabel(r"$M$ [$M_\odot$]")
    if k % 3 == 2:
        axr.set_ylabel(r"$\dot M_{\rm BH}$ [$M_\odot$ yr$^{-1}$],  $f_{\rm Edd}$", fontsize=10)
fig.suptitle("Do BH accretion & f_Edd survive gas exhaustion? "
             "(down-triangles at the floor = exact zeros in the catalogue)", y=1.0)
plt.tight_layout()
plt.savefig(figpath( "bh_fedd_gas_exhaustion_check.png"), dpi=130, bbox_inches="tight")
plt.show()

# --- census: how much of the jet-mode class carries (essentially) no accretion power? ---
bh_m = BH["bh_mass"][:, cols]; bh_f = BH["bh_fedd"][:, cols]; bh_d = BH["bh_mdot"][:, cols]
with np.errstate(all="ignore"):
    Lb = 0.1 * np.where(bh_d > 0, bh_d, np.nan) * _c2yr           # erg/s; NaN where Mdot<=0
jetm = np.isfinite(bh_m) & np.isfinite(bh_f) & (bh_m > 10 ** 7.5) & (bh_f < 0.2)
post = t_cosmic_yr[:, None] >= np.array([r["t_qt"] for r in records])[None, :]
print(f"\njet-mode class (M_BH>10^7.5 & f_Edd<0.2) — fraction that is accretion-dead:")
for name, mm in [("all snapshots", jetm), ("post-QT only ", jetm & post)]:
    n = int(mm.sum())
    if n == 0:
        continue
    z_md = float(np.mean(~(bh_d[mm] > 0)))                        # Mdot <= 0 or NaN
    z_fe = float(np.mean(bh_f[mm] <= 0))                          # f_Edd recorded as exact 0
    fr = {t: float(np.mean(~(Lb[mm] > t))) for t in (1e40, 1e41, 1e42)}
    print(f"  {name}: N={n:7d} | Mdot<=0/NaN: {z_md:.2f} | f_Edd==0: {z_fe:.2f} | "
          + " ".join(f"L_bol<{t:.0e}: {fr[t]:.2f}" for t in fr))
print("\nverdict guide: these fractions are 'jet AGN' with no accretion power — dead BHs, not")
print("AGN (SIMBA jet power ~ Mdot_BH; observed LERGs have lambda ~ 1e-4..1e-2). If significant,")
print("add an activity floor to is_jet / w_jet in §8j and the §10d jet overlay, e.g.")
print("  is_jet_active = is_jet & (Lbol > 1e41)   # or fedd > 1e-4")


### V.23 (§8n) · Post-QT AGN regime — gas-removing vs CGM-heating, with a merger-safe integrated $f_{\rm Edd}$

**The question.** Far into the quenching stage (every epoch after QT where the galaxy is still
under the sSFR limit), was the AGN **consistently in the ISM-coupled X-ray regime** — the only
SIMBA channel that acts on the ISM, i.e. *removes/heats the residual gas* (jets at full velocity,
$f_{\rm Edd}<0.02$, **and** $f_{\rm gas}<0.2$) — or was it **only heating the CGM** (jet mode
without the gas-poor coupling condition)? One verdict per galaxy; no full activation history.

**Why two estimators.** AGN accretion flickers on $10^4$–$10^5$ yr — two to four orders of
magnitude below the snapshot cadence (printed exactly below from `t_cosmic_yr`) — so a single
catalogue $f_{\rm Edd}$ is one random draw from a distribution that swings between snapshots.
The instantaneous value stays **primary** (it is what SIMBA itself used to set the feedback
mode), and the classification is repeated with the time-integrated
$\langle\dot M_{\rm BH}\rangle = [M_{\rm BH}(t_{i+1})-M_{\rm BH}(t_{i-1})]/(t_{i+1}-t_{i-1})$
as a **sensitivity test**: if the verdicts agree, the regime assignment is not single-snapshot
noise; where they flip, that is itself the result.

**Merger masking without merger flags.** BH-merger mass jumps masquerade as huge accretion.
Instead of the merger catalogue, an interval is *merger-suspect* from the stellar-mass budget
alone: the jump $\Delta M_\star$ cannot be explained by star formation if constant SFR from the
previous snapshot contributes $<20\%$ of it (`MERGER_SF_FRAC`); a minimum relative jump
(`MERGER_DM_MIN`, 2%) keeps noise-level wiggles in SFR$\approx$0 galaxies from being flagged
(set it to 0 to apply the SF criterion alone). Both intervals bracketing the centered difference
must be clean. Residual caveat: a BH merger delivered by a satellite too small to leave a >2%
stellar jump can slip through — rare, and it biases toward *over*-estimating accretion, i.e.
against the jet/X-ray classification, so the verdict is conservative.


In [ ]:
# 8n · post-QT AGN regime: gas-removing (X-ray-coupled) vs CGM-heating only — instantaneous
#   f_Edd vs a merger-safe time-INTEGRATED <f_Edd> from BH mass growth (sensitivity test).
#   Needs §4b (BH), §3 (records, cols, is_fast/is_slow), §7j (JET_LOGMBH, JET_FEDD),
#   §8j (XRAY_FEDD_MAX, XRAY_FGAS_MAX), §0 (PASSIVE_FACTOR, EPS_R). Optional: §8g SPLIT_DUST.
MERGER_SF_FRAC = 0.20    # merger-suspect: constant SFR from the previous snap explains < this
MERGER_DM_MIN  = 0.02    #   fraction of the M* jump, AND the jump is > this fraction of M*
CONSIST_FRAC   = 0.75    # 'consistently in a regime' = >= this fraction of valid post-QT epochs

# --- ascending-time views + exact snapshot cadence ---
_o8n = np.argsort(t_cosmic_yr)
_t8  = t_cosmic_yr[_o8n]                                  # [yr] ascending
_dt8 = np.diff(_t8)
print(f"snapshot spacing: median {np.median(_dt8):.3e} yr (min {_dt8.min():.2e}, max {_dt8.max():.2e})"
      f" -> a 1e4-1e5 yr AGN flicker is ~{np.median(_dt8)/1e5:.0f}-{np.median(_dt8)/1e4:.0f}x below cadence")

Ms8  = P["masses.stellar"][:, cols][_o8n, :]              # (n_snap, n_rec)
SFR8 = P["sfr"][:, cols][_o8n, :]
Mg8  = P["masses.gas"][:, cols][_o8n, :]
Mbh8 = BH["bh_mass"][:, cols][_o8n, :]
Fe8  = BH["bh_fedd"][:, cols][_o8n, :]
with np.errstate(all="ignore"):
    fgas8 = np.where(Ms8 > 0, Mg8 / Ms8, np.nan)
    ssfr8 = np.where(Ms8 > 0, SFR8 / Ms8, np.nan)

# --- (1) merger-suspect intervals from the M* budget (no merger flags) ---
with np.errstate(all="ignore"):
    dMs8    = Ms8[1:] - Ms8[:-1]                          # growth across interval i-1 -> i
    sf_exp8 = SFR8[:-1] * _dt8[:, None]                   # constant-SFR mass formed in the interval
    merger_int8 = (np.isfinite(dMs8) & np.isfinite(sf_exp8)
                   & (dMs8 > MERGER_DM_MIN * Ms8[:-1])
                   & (sf_exp8 < MERGER_SF_FRAC * dMs8))   # (n_snap-1, n_rec)
print(f"merger-suspect intervals: {int(merger_int8.sum())} "
      f"({100 * merger_int8.sum() / merger_int8.size:.1f}% of all; "
      f"median {np.median(merger_int8.sum(axis=0)):.0f} per galaxy)")

# --- (2) time-integrated accretion from BH mass growth (centered; merger-masked) ---
#     <Mdot_acc> = dM_BH/dt / (1-eps_r)   (SIMBA adds (1-eps_r) of the accreted mass to the BH)
#     Mdot_Edd   = 4 pi G M_BH m_p / (eps_r sigma_T c)  ->  <f_Edd> = <Mdot_acc> / Mdot_Edd
with np.errstate(all="ignore"):
    dMbh_c8 = Mbh8[2:] - Mbh8[:-2]
    _bad8   = merger_int8[:-1] | merger_int8[1:] | ~np.isfinite(dMbh_c8) | (dMbh_c8 < 0)
    mdot_c8 = np.where(_bad8, np.nan, dMbh_c8) / (_t8[2:] - _t8[:-2])[:, None] / (1 - EPS_R)
    _kEdd = float((4 * np.pi * const.G * const.m_p
                   / (EPS_R * const.sigma_T * const.c)).to(1 / u.yr).value)  # Mdot_Edd = _kEdd*M_BH
    Fe8_int = np.full_like(Fe8, np.nan)
    Fe8_int[1:-1] = mdot_c8 / (_kEdd * Mbh8[1:-1])

# --- (3) 'far into the quenching stage': after QT and still under the sSFR limit ---
_tqt8n = np.array([r["t_qt"] for r in records])
with np.errstate(all="ignore"):
    postq8 = (_t8[:, None] > _tqt8n[None, :]) & (ssfr8 < (PASSIVE_FACTOR / _t8)[:, None])

# --- (4) regime per epoch, same in-model thresholds as §8j ---
def _regime8n(fe):
    """3 = X-ray-coupled (gas-removing): jets & f_Edd<XRAY_FEDD_MAX & f_gas<XRAY_FGAS_MAX
       2 = jet mode, CGM-heating only    1 = no jets (radiative or M_BH below cut)
       NaN = BH/gas state undefined at this epoch"""
    ok  = np.isfinite(Mbh8) & np.isfinite(fe) & np.isfinite(fgas8)
    jet = ok & (Mbh8 > 10 ** JET_LOGMBH) & (fe < JET_FEDD)
    xr  = jet & (fe < XRAY_FEDD_MAX) & (fgas8 < XRAY_FGAS_MAX)
    out = np.where(ok, 1.0, np.nan); out[jet] = 2.0; out[xr] = 3.0
    return out

REG8N = {"instantaneous": _regime8n(Fe8), "integrated": _regime8n(Fe8_int)}

VERD_LAB = {3: "gas-removing (X-ray-coupled)", 2: "CGM-heating only (jets)",
            0: "mixed / no consistent regime", -1: "unclassified (<3 valid epochs)"}
def _verdict8n(R):
    valid = postq8 & np.isfinite(R)
    nv = valid.sum(axis=0).astype(float)
    fx = np.where(nv > 0, ((R == 3) & valid).sum(axis=0) / nv, np.nan)    # frac X-ray-coupled
    fj = np.where(nv > 0, ((R == 2) & valid).sum(axis=0) / nv, np.nan)    # frac jet-only
    v = np.full(len(records), -1, int)
    v[nv >= 3] = 0
    v[(nv >= 3) & (fj >= CONSIST_FRAC)] = 2
    v[(nv >= 3) & (fx >= CONSIST_FRAC)] = 3
    return v, fx, fj, nv

VER8N = {k: _verdict8n(R) for k, R in REG8N.items()}

print(f"\nper-galaxy verdict over post-QT epochs (consistent = >={CONSIST_FRAC:.0%} of valid epochs):")
print(f"  {'':36s}{'instantaneous':>16s}{'integrated':>13s}")
for code in (3, 2, 0, -1):
    print(f"  {VERD_LAB[code]:36s}{int((VER8N['instantaneous'][0] == code).sum()):>16d}"
          f"{int((VER8N['integrated'][0] == code).sum()):>13d}")

# sensitivity: does the verdict survive the integrated estimator?
_vi, _vg = VER8N["instantaneous"][0], VER8N["integrated"][0]
_bothv = (_vi >= 0) & (_vg >= 0)
print(f"\nverdict agreement (both classifiable, N={int(_bothv.sum())}): "
      f"{100 * np.mean(_vi[_bothv] == _vg[_bothv]):.0f}% identical")
for a in (3, 2, 0):
    m = _bothv & (_vi == a)
    if m.sum():
        moved = ", ".join(f"{VERD_LAB[b].split(' (')[0]}: {int(np.sum(_vg[m] == b))}" for b in (3, 2, 0))
        print(f"  inst '{VERD_LAB[a].split(' (')[0]}' (N={int(m.sum())}) -> integrated: {moved}")
_ep = postq8 & np.isfinite(REG8N["instantaneous"]) & np.isfinite(REG8N["integrated"])
print(f"per-epoch regime agreement: "
      f"{100 * np.mean(REG8N['instantaneous'][_ep] == REG8N['integrated'][_ep]):.0f}%"
      f"  (N={int(_ep.sum())} post-QT epochs)")

# --- (5) figure: verdict mix by split (both estimators) + inst-vs-integrated f_Edd ---
_splits8n = [(is_fast, LBL_A), (is_slow, LBL_B)]
if "SPLIT_DUST" in globals():
    _splits8n += [(mk_, lb_.split(" (")[0]) for mk_, _c_, lb_ in SPLIT_DUST]

fig, (axb, axs) = plt.subplots(1, 2, figsize=(15.5, 4.2 + 0.4 * len(_splits8n)),
                               gridspec_kw={"width_ratios": [1.6, 1.0]})
_vcols = {3: "C3", 2: "C0", 0: "0.55", -1: "0.85"}
ylabs = []
for ei, est in enumerate(("instantaneous", "integrated")):
    v = VER8N[est][0]
    for mk_, lb_ in _splits8n:
        n = int(mk_.sum()); left = 0.0; y = len(ylabs)
        for code in (3, 2, 0, -1):
            f = np.sum(mk_ & (v == code)) / max(n, 1)
            axb.barh(y, f, left=left, color=_vcols[code], edgecolor="w", height=0.75)
            left += f
        ylabs.append(f"{lb_} — {est} (n={n})")
axb.axhline(len(_splits8n) - 0.5, color="k", lw=0.8)
axb.set_yticks(range(len(ylabs))); axb.set_yticklabels(ylabs, fontsize=9)
axb.invert_yaxis(); axb.set_xlim(0, 1); axb.set_xlabel("fraction of galaxies")
axb.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=_vcols[c]) for c in (3, 2, 0, -1)],
           labels=[VERD_LAB[c] for c in (3, 2, 0, -1)], fontsize=8, ncol=2, loc="upper center",
           bbox_to_anchor=(0.5, -0.18))
axb.set_title("post-QT AGN verdict: removing gas or heating the CGM?")

_msc = _ep & np.isfinite(Fe8) & np.isfinite(Fe8_int) & (Fe8 > 0) & (Fe8_int > 0)
axs.scatter(np.log10(Fe8[_msc]), np.log10(Fe8_int[_msc]), s=3, alpha=0.2, c="k", rasterized=True)
for thr in (JET_FEDD, XRAY_FEDD_MAX):
    axs.axvline(np.log10(thr), ls=":", c="C3", lw=1)
    axs.axhline(np.log10(thr), ls=":", c="C3", lw=1)
_lim = axs.get_xlim()
axs.plot(_lim, _lim, lw=0.8, c="0.5"); axs.set_xlim(_lim)
axs.set_xlabel(r"$\log_{10} f_{\rm Edd}$ (instantaneous)")
axs.set_ylabel(r"$\log_{10}\,\langle f_{\rm Edd}\rangle$ ($\Delta M_{\rm BH}/\Delta t$, merger-masked)")
axs.set_title(f"post-QT epochs (N={int(_msc.sum())}; zeros excluded)")
plt.tight_layout()
plt.savefig(figpath( "postqt_agn_regime_sensitivity.png"), dpi=130, bbox_inches="tight")
plt.show()


### Which axis organizes the population? fast/slow vs halo mass vs AGN (jet) mode

For each key outcome (residual dust, quench timescale, concentration) we measure the
**correlation ratio $\eta^2$** — the fraction of the outcome's variance explained by each
candidate grouping. The axis with the highest $\eta^2$ across outcomes is the one the
population is really organized by. Pure-numpy (no `statsmodels`). *Caveat:* $\eta^2$ rises
weakly with the number of groups, so the binary axes (fast/slow, jet) are conservative
relative to the 4-bin halo-mass axis.

In [ ]:
# primary-subdivision test via the correlation ratio eta^2. Needs §3 (is_fast/is_slow, tau_n),
#   §7·setup (hmbin), §4 (DIAGS), §4b (BH) for the jet axis.
def _eta2(values, groups):
    """Correlation ratio eta^2 of `values` over integer `groups` (negative = excluded)."""
    v = np.asarray(values, float); g = np.asarray(groups)
    ok = np.isfinite(v) & (g >= 0); v, g = v[ok], g[ok]
    if len(v) < 6 or len(np.unique(g)) < 2:
        return np.nan
    grand = v.mean(); ss_tot = np.sum((v - grand) ** 2)
    if ss_tot <= 0:
        return np.nan
    ss_bet = sum(len(v[g == k]) * (v[g == k].mean() - grand) ** 2 for k in np.unique(g))
    return ss_bet / ss_tot

ax_fs = np.where(is_fast, 0, np.where(is_slow, 1, -1))      # fast/slow axis
ax_hm = hmbin.copy()                                        # halo-mass-bin axis
ax_jet = np.full(len(records), -1)                          # jet/AGN-mode axis at z=0
if "BH" in globals() and isinstance(BH, dict) and "bh_mass" in BH and "bh_fedd" in BH:
    _rows_z0 = np.array([r["row_z0"] for r in records])
    def _bh_z0(key):
        out = np.full(len(records), np.nan); a = BH.get(key)
        if a is None:
            return out
        for i, (rw, cl) in enumerate(zip(_rows_z0, cols)):
            if rw >= 0:
                out[i] = a[rw, cl]
        return out
    _mbh0, _fed0 = _bh_z0("bh_mass"), _bh_z0("bh_fedd")
    _ok = np.isfinite(_mbh0) & np.isfinite(_fed0)
    _jet = (_mbh0 > 10 ** JET_LOGMBH) & (_fed0 < JET_FEDD)
    ax_jet = np.where(_ok, _jet.astype(int), -1)
else:
    print("note: BH history absent -> jet axis skipped (run §4b first).")
AXES = [("fast/slow", ax_fs), ("halo-mass bin", ax_hm), ("jet mode", ax_jet)]

with np.errstate(all="ignore"):
    _h2z0 = (DP_C2080["H2"][:, DP_STAGES.index("z0")]
             if (DP_OK and "z0" in DP_STAGES) else np.full(len(records), np.nan))
    OUTC = {
        r"$\log$ dust/$M_\star$ @z0": np.log10(DIAGS["z0"]["dust/M*"]),
        r"$\log\,\tau_q/t_H$":        np.log10(tau_n),
        r"$R_{20}/R_{80}$ stars @z0": DIAGS["z0"]["C20/80*"],
        r"$R_{20}/R_{80}$ H$_2$ @z0": _h2z0,
    }

labels = list(OUTC)
E = np.full((len(labels), len(AXES)), np.nan)
for li, lab in enumerate(labels):
    for ai, (_nm, grp) in enumerate(AXES):
        E[li, ai] = _eta2(OUTC[lab], grp)
print(f"{'outcome':30s}" + "".join(f" | {nm:>14s}" for nm, _ in AXES))
for li, lab in enumerate(labels):
    print(f"{lab[:30]:30s}" + "".join(f" | {E[li, ai]:14.3f}" for ai in range(len(AXES))))
print(f"{'MEAN eta^2':30s}" + "".join(f" | {np.nanmean(E[:, ai]):14.3f}" for ai in range(len(AXES))))

fig, ax = plt.subplots(figsize=(2.4 * len(labels) + 2, 4.2)); width = 0.26
for ai, (nm, _) in enumerate(AXES):
    ax.bar(np.arange(len(labels)) + ai * width, np.nan_to_num(E[:, ai]), width, label=nm)
ax.set_xticks(np.arange(len(labels)) + width); ax.set_xticklabels([l[:20] for l in labels], rotation=20, ha="right")
ax.set_ylabel(r"$\eta^2$ (variance explained)"); ax.legend()
ax.set_title("Which axis organizes the population? (higher = more primary)")
plt.tight_layout(); plt.show()

# Part 10 — Interpretation

Reading the result against the ISM-equilibrium hypothesis.

## VI (§9) · Interpretation — reading the result against the hypothesis

The "ISM equilibrium at large radii" picture makes concrete, falsifiable predictions
that the figures above test directly:

| Diagnostic | Where | Hypothesis prediction (dust survives) |
|---|---|---|
| $M_{\rm dust}/M_\star$, $M_{H_2}/M_\star$ at `post-quench` & `z=0` | §4a–b | **higher / gently declining** for slow quenchers |
| $R_{\rm gas,1/2}/R_{\star,1/2}$ through `QT` | §4c | **stays $\gtrsim 1$** (extended) for slow; **collapses** for fast |
| dust/gas, HI/H₂ | §4d | surviving ISM stays cold/atomic-dominated, not destroyed |
| $R_{50}^{\rm dust}$ vs stage | §5c–§5d | **roughly constant / large** for slow; **shrinks** for fast |
| $R_{50}^{\rm dust}/R_{50}^{\star}$ | §5c–§5d | **$>1$** for slow (dust at large radii, away from the AGN) |
| $E_{\rm AGN}$, $\Delta M_{\rm BH}$, jet-mode fraction over SFT$\to$QT | §4b | **low** for slow / dust-rich (gas never fed the AGN) |
| residual dust vs $E_{\rm AGN}$ / jet-mode time | §4b | **anti-correlated** (more feeding → more destruction) |

**Logic of the test.** If slow quenchers keep their dust *extended* and *not* funnelled
to the centre, the gas never reaches the densities needed to feed the AGN strongly —
so there is no central heating event to sputter/destroy the grains, and the residual
dust persists to z=0. Fast quenchers, by contrast, should show centrally concentrated
gas at `SFT`→`QT` (small $R_{50}^{\rm dust}$, ratio $<1$), consistent with an inflow
that both quenches *and* destroys the dust.

**Mass dependence.** Every comparison above is split into three z=0 stellar-mass bins
($\log_{10}M_\star$ in 9.5–10.5, 10.5–11, $\geq$11; binned columns/rows in each figure).
This matters because AGN jet-mode feedback in SIMBA switches on around
$M_{\rm BH}\!\sim\!10^{7.5}M_\odot$ (massive galaxies), so the dust-destruction channel
should be strongest in the top bin and the "equilibrium-survival" route most visible at
lower mass. Watch whether the fast/slow degeneracy in the BH histories (§4b-iii/iv)
breaks in a *particular* mass bin even when it holds for the pooled sample.

**Caveats / things to tighten on the cluster:**
- Confirm the snapshot `PartType0` fields used by the dust-profile builder (§5 helpers):
  `Dust_Masses`, `Metallicity`, `NeutralHydrogenAbundance`, and a molecular fraction
  (`FractionH2`/`fH2`/`GrackleH2`). If no per-particle molecular fraction exists, the H₂
  profile is unavailable and HI falls back to the neutral-H proxy; dust is unaffected.
- The reduced dust-profile product (§5) is built **once** with `BUILD_DUST_PROFILES=True`,
  then cached to `dust_profiles_allcrit.hdf5`; it covers the whole sample at all critical
  points. The build groups work by snapshot (each opened once) and reads only each galaxy's
  `glist`/`slist` particles — it can be parallelised per snapshot if needed.
- The fast/slow boundary (`TAU_SPLIT_LOG`: $\log_{10}(\tau_q/t_H(z_{QT})) < -1.5$) and the gas floor (`NGAS_MIN`) are config knobs at
  the top — re-run §3–§5 after changing them.
- For the AGN cross-check (§4b), confirm `BH_CANDIDATES` resolved to real datasets
  (the build cell prints them) and that `bhmdot`/`masses.bh` units match the assumed
  $M_\odot\,{\rm yr}^{-1}$ / $M_\odot$. For a fully independent check, the particle-level
  BH accretion (`PartType5` `BH_Mass`/`BH_Mdot`) and wind/jet tagging used in
  `agn_feedback_view.ipynb` can be added to the §5 reduced-product builder.

**Next step → §10.** The mechanisms above become survey predictions in §10: full-box dusty-quiescent abundances $n(z)$ and modified-blackbody sub-mm number counts.

# Part 11 — Observables & number counts

From the simulation to observables: dusty-quiescent abundances, sub-mm/IR number counts, the quiescent-galaxy grids vs redshift, the CO–AGN connection and the jet-phase steadiness test.

## VII (§10) · From simulation to observables — dusty-quiescent abundances & sub-mm/IR number counts

The end goal: **how many dusty quiescent galaxies should a sub-mm / FIR survey detect**, and
which observable predicts the cold-ISM content of an observed quiescent galaxy?

- **§10a — full-box census** (catalogue-only, one pass over all snapshots, cached). The §3
  sample contains only the *progenitors of z=0 passive* galaxies, so it **cannot** give
  abundances at z>0 — it misses galaxies that were quiescent at z but later rejuvenated or
  merged away. The census instead re-selects quiescent galaxies
  (`sSFR < PASSIVE_FACTOR/t_H(z)`, `logM* > MASS_FLOOR`) **independently at every snapshot**
  and stores their $M_{\rm dust}$, $M_\star$, SFR.
- **§10b — abundances & counts.** (i) Comoving number density $n(z)$ of quiescent and
  dusty-quiescent galaxies for several $\log(M_{\rm dust}/M_\star)$ cuts, plus the
  dusty fraction $P({\rm dusty}\,|\,{\rm quiescent})(z)$ — the full-box version of §3i,
  free of the z=0-progenitor selection. (ii) Each quiescent galaxy's observed-frame flux
  from a single-temperature, optically-thin **modified blackbody**,
  $S_\nu = (1+z)\,\kappa(\nu_{\rm rest})\,B_\nu(\nu_{\rm rest},T_d)\,M_{\rm dust}/d_L^2$.
  (iii) Cumulative counts $N(>S)$ per deg$^2$ by weighting each snapshot shell with
  $dV_c/dz/d\Omega$.

> **Caveats.**
> 1. Counts are instantaneous box statistics on the snapshot grid, not a lightcone — no
>    clustering / field-to-field variance.
> 2. $T_{\rm dust}$ is assumed (20–25 K bracket; quiescent dust is cold), not derived from
>    radiative transfer — for the headline numbers, cross-check a few galaxies against the
>    powderday SEDs (`mock_SED.ipynb` / `sed_critical_epochs.ipynb`).
> 3. The $M_{\rm dust}$ normalization inherits SIMBA's dust model (Li, Narayanan & Davé 2019).
> 4. The 100 Mpc/h box under-samples bright/rare sources — trust the counts faintward of the
>    knee, and treat the bright end as a lower limit.


In [ ]:
# §10·pre — Part VII bootstrap: snapshot <-> redshift/time axes WITHOUT the progenitor history.
#   Part VII (the census + observables, §10a-§10e) reads per-snapshot catalogues only and is
#   INDEPENDENT of the §1 progenitor history (which is slow to build). It just needs the
#   snap/z/time axes; define them from `sim` here if the §1c history-load cell was skipped.
#   snaps_arr keeps the same consecutive START_SNAP->END_SNAP ordering the census "row"
#   indexes, so a census built with or without the history in memory agrees. Needs §0 (sim,
#   START_SNAP, END_SNAP, COSMO).
if not all(k in globals() for k in ("snaps_arr", "redshift", "t_cosmic_yr")):
    snaps_arr   = np.arange(int(START_SNAP), int(END_SNAP) - 1, -1)
    redshift    = np.array([float(sim.get_z_from_snap(int(s))) for s in snaps_arr])
    t_cosmic_yr = COSMO.age(redshift).value * 1e9
    print(f"[Part VII] snap/z/time axes from sim (history not loaded): {len(snaps_arr)} "
          f"snapshots, z {redshift.min():.2f}-{redshift.max():.2f}")
else:
    print(f"[Part VII] using snap/z axes already in memory: {len(snaps_arr)} snapshots, "
          f"z {redshift.min():.2f}-{redshift.max():.2f}")

In [ ]:
# 10a · full-box quiescent census — one catalogue read per snapshot (light), cached.
#   Selection is INDEPENDENT per snapshot (unlike the §3 z=0-progenitor sample):
#   quiescent = sSFR < PASSIVE_FACTOR/t_H(z)  and  logM* > MASS_FLOOR.
#   Stores (M*, SFR, M_dust) of every quiescent galaxy, concatenated with the history
#   row index, so abundances/counts at any z come straight from this one file.
DQ_CENSUS_PATH  = os.path.join(SFHDIR, "dq_census_allsnaps.hdf5")
BUILD_DQ_CENSUS = not os.path.exists(DQ_CENSUS_PATH)

def _gal_field(f, name):
    '''galaxy_data field, trying the dicts/ and flat layouts (same resolver idea as §4b).'''
    for p in (f"galaxy_data/dicts/{name}", f"galaxy_data/{name}"):
        if p in f:
            return np.asarray(f[p][:], dtype=np.float64)
    return None

if BUILD_DQ_CENSUS:
    _ms, _sf, _md, _irow, _isnap = [], [], [], [], []
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                ms = _gal_field(f, "masses.stellar")
                sf = _gal_field(f, "sfr")
                md = _gal_field(f, "masses.dust")
        except (OSError, KeyError):
            print(f"  [skip] snapshot {snap}"); CORRUPT_SNAPS.add(snap); continue
        if ms is None or sf is None or md is None:
            print(f"  [skip] snapshot {snap}: missing galaxy field"); continue
        with np.errstate(divide="ignore", invalid="ignore"):
            ssfr = np.where(ms > 0, sf / ms, np.nan)
        qui = (np.log10(np.where(ms > 0, ms, np.nan)) > MASS_FLOOR) \
              & np.isfinite(ssfr) & (ssfr < PASSIVE_FACTOR / t_cosmic_yr[ri])
        _ms.append(ms[qui]); _sf.append(sf[qui]); _md.append(md[qui])
        _irow.append(np.full(int(qui.sum()), ri, dtype=np.int32))
        _isnap.append(np.full(int(qui.sum()), snap, dtype=np.int32))
    assert _ms, "census build read no snapshots — run on the cluster (catalogs not reachable here)"
    with h5py.File(DQ_CENSUS_PATH, "w") as out:
        out.attrs["passive_factor"] = PASSIVE_FACTOR
        out.attrs["mass_floor"]     = MASS_FLOOR
        out.create_dataset("row",   data=np.concatenate(_irow))   # row index into snaps_arr/redshift
        out.create_dataset("snap",  data=np.concatenate(_isnap))  # snapshot number (self-describing)
        out.create_dataset("mstar", data=np.concatenate(_ms))
        out.create_dataset("sfr",   data=np.concatenate(_sf))
        out.create_dataset("mdust", data=np.concatenate(_md))
    print("census written:", DQ_CENSUS_PATH)

if os.path.exists(DQ_CENSUS_PATH):
    with h5py.File(DQ_CENSUS_PATH, "r") as f:
        DQ = {k: f[k][:] for k in f.keys()}
    DQ_OK = True
    _zc = (np.array([float(sim.get_z_from_snap(int(s))) for s in DQ["snap"]])
           if "snap" in DQ else redshift[DQ["row"]])   # self-describing if rebuilt
    print(f"census loaded: {len(DQ['mstar'])} quiescent (galaxy,snapshot) rows; "
          f"z range {_zc.min():.2f}-{_zc.max():.2f}")
else:
    DQ, DQ_OK = None, False
    print("no census yet -> run this cell on the cluster (one catalogue read per snapshot).")


In [ ]:
# 10b · dusty-quiescent abundances and sub-mm number counts from the §10a census.
#   (i)  n(z): comoving density of quiescent / dusty-quiescent galaxies + dusty fraction.
#   (ii) single-T optically-thin modified blackbody per galaxy ->
#   (iii) cumulative counts N(>S) per deg^2 over the snapshot shells (dV_c/dz/dOmega weights).
assert DQ_OK, "run §10a first (builds/loads the census)"

V_BOX        = (BOX / COSMO.h) ** 3                # comoving box volume [Mpc^3]; BOX=100 Mpc/h
                                                   # (COSMO.h=0.674 vs SIMBA h=0.68: <3% in volume)
DFRAC_THRESH = [-5.0, -4.0, -3.0]                  # 'dusty' cuts on log10(Mdust/M*)
T_DUST_LIST  = [20.0, 25.0]                        # cold-dust bracket [K] for quiescent ISM
BETA_MBB     = 1.8                                 # emissivity index
LAM_OBS_UM   = {"850um": 850.0, "1.2mm": 1200.0}   # observed-frame bands
KAPPA0       = 0.077 * u.m**2 / u.kg               # James+2002 opacity at 850um (rest)
NU0          = (const.c / (850.0 * u.um)).to(u.Hz)
ZMIN_COUNTS  = 0.1                                 # drop local shells (d_L -> 0 blow-up)

with np.errstate(divide="ignore", invalid="ignore"):
    ldfrac = np.log10(np.where((DQ["mdust"] > 0) & (DQ["mstar"] > 0),
                               DQ["mdust"] / DQ["mstar"], np.nan))
rows = np.unique(DQ["row"]); zs = redshift[rows]
o = np.argsort(zs); rows, zs = rows[o], zs[o]

# ---- (i) comoving number densities + dusty fraction vs z ----
n_q = np.array([np.sum(DQ["row"] == r) for r in rows]) / V_BOX
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(zs, n_q, "k-", lw=2, label="all quiescent")
for th, c in zip(DFRAC_THRESH, ["C0", "C2", "C3"]):
    n_dq = np.array([np.sum((DQ["row"] == r) & (ldfrac > th)) for r in rows]) / V_BOX
    ax[0].plot(zs, n_dq, color=c, lw=2, label=f"dusty (log Md/M*>{th:g})")
    with np.errstate(invalid="ignore"):
        ax[1].plot(zs, np.where(n_q > 0, n_dq / n_q, np.nan), color=c, lw=2,
                   label=f"log Md/M*>{th:g}")
ax[0].set_yscale("log"); ax[0].set_xlabel("redshift z"); ax[0].set_ylabel(r"n [cMpc$^{-3}$]")
ax[0].set_title(f"quiescent & dusty-quiescent density (logM*>{MASS_FLOOR})"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("redshift z"); ax[1].set_ylabel("dusty fraction of quiescent"); ax[1].set_ylim(0, 1)
ax[1].set_title("P(dusty | quiescent) — full box (no z=0-progenitor selection)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(figpath( "dq_density_vs_z.png"), dpi=130, bbox_inches="tight"); plt.show()

# ---- (ii) observed-frame flux: single-T optically-thin modified blackbody ----
def mbb_flux_mjy(md_msun, z, lam_obs_um, Tdust):
    '''S_nu(obs) = (1+z) kappa(nu_rest) B_nu(nu_rest, T_d) M_dust / d_L^2  ->  [mJy].'''
    nu_rest = (const.c / (lam_obs_um * u.um)).to(u.Hz) * (1.0 + z)
    kappa   = KAPPA0 * (nu_rest / NU0) ** BETA_MBB
    x       = (const.h * nu_rest / (const.k_B * Tdust * u.K)).to(u.dimensionless_unscaled)
    Bnu     = (2.0 * const.h * nu_rest**3 / const.c**2) / np.expm1(x.value)   # W m^-2 Hz^-1 (per sr)
    dL      = COSMO.luminosity_distance(z)
    return ((1.0 + z) * (np.asarray(md_msun) * u.Msun) * kappa * Bnu / dL**2).to(u.mJy).value

# ---- (iii) cumulative counts N(>S) per deg^2 over the snapshot shells ----
dz   = np.gradient(zs)                                                     # shell widths in z
dVdO = COSMO.differential_comoving_volume(zs).to(u.Mpc**3 / u.sr).value    # dV_c/dz/dOmega
SR_PER_DEG2 = (np.pi / 180.0) ** 2
Sgrid = np.logspace(-3, 1.5, 40)                                           # mJy

def counts_per_deg2(lam_obs_um, Tdust, Sthr):
    '''N(>S) [deg^-2] of quiescent galaxies, summing n(>S, z_i)*dV/dz/dOmega*dz_i over shells.'''
    out = np.zeros_like(np.atleast_1d(Sthr), dtype=float)
    for r, z, dzi, dv in zip(rows, zs, dz, dVdO):
        if z < ZMIN_COUNTS:
            continue
        sel = (DQ["row"] == r) & (DQ["mdust"] > 0)
        if not sel.any():
            continue
        S = mbb_flux_mjy(DQ["mdust"][sel], z, lam_obs_um, Tdust)
        out += np.array([np.sum(S > s) for s in np.atleast_1d(Sthr)]) / V_BOX * dv * dzi * SR_PER_DEG2
    return out

fig, ax = plt.subplots(1, len(LAM_OBS_UM), figsize=(6.0 * len(LAM_OBS_UM), 4.6), squeeze=False)
for bi, (band, lam) in enumerate(LAM_OBS_UM.items()):
    for Td, ls in zip(T_DUST_LIST, ["--", "-"]):
        ax[0, bi].plot(Sgrid, counts_per_deg2(lam, Td, Sgrid), ls=ls, lw=2, color="C3",
                       label=f"$T_d$={Td:g} K")
    ax[0, bi].set_xscale("log"); ax[0, bi].set_yscale("log")
    ax[0, bi].set_xlabel(f"S({band}) [mJy]"); ax[0, bi].set_title(f"quiescent counts at {band}")
    ax[0, bi].legend(fontsize=8)
ax[0, 0].set_ylabel(rf"N(>S) [deg$^{{-2}}$]  (z > {ZMIN_COUNTS:g})")
fig.suptitle("Predicted sub-mm number counts of quiescent galaxies (MBB on SIMBA $M_{\\rm dust}$)", y=1.02)
plt.tight_layout(); plt.savefig(figpath( "dq_number_counts.png"), dpi=130, bbox_inches="tight"); plt.show()

# headline numbers for the write-up
print(f"cumulative quiescent counts (z > {ZMIN_COUNTS:g}), T_dust = {T_DUST_LIST[-1]:g} K:")
for band, lam in LAM_OBS_UM.items():
    for Sref in (0.1, 1.0):
        n = counts_per_deg2(lam, T_DUST_LIST[-1], Sref)[0]
        print(f"  N(>{Sref:g} mJy, {band}) ~ {n:10.2f} deg^-2")


### VII.1 (§10c) · Quiescent-galaxy grid at z ≈ `Z_TARGET` — dust, gas & depletion time vs $M_\star$, age, sSFR, in field / groups / clusters

Standalone population diagnostic: at the snapshot nearest `Z_TARGET` we select **every**
quiescent galaxy in the box (`sSFR < 0.2/t_H(z)`, `logM* > MASS_FLOOR`). Rows:
$M_{\rm dust}/M_\star$ · $M_{\rm gas}/M_\star$ · $t_{\rm dep}=M_{\rm gas}/{\rm SFR}$ [Gyr]
(log axes); columns: vs $M_\star$ · age · sSFR [Gyr$^{-1}$]. **Field** ($M_h<10^{12.5}$) is
the small open black background population; **groups** ($10^{13}$–$10^{14.5}$, matching the
§10d CO sample) orange triangles; **clusters** ($\geq10^{14.5}$, likely ~empty in this box)
green squares; running medians with 25–75% bands; MAD-sigma-clipped frames. Each panel is
annotated with $f_{\rm det}$ — the percentage of that class with the **row quantity** defined
(dust > 0 / gas > 0 / SFR > 0), consistent across the columns of a row and with the printed
table. The $10^{12.5}$–$10^{13}$ range is deliberately excluded; satellites carry their host's
halo mass; $t_{\rm dep}$ vs sSFR is a consistency view (same numbers on both axes).


In [ ]:
# 10c · 3x3 grid, observational-paper style: (dust frac | gas frac | t_dep) vs (M* | age | sSFR),
#       field (open black background) / groups (orange triangles) / clusters (green squares).
#   ALL quiescent galaxies at z~Z_TARGET, one catalogue read. Groups = logMh 13-14.5 (== §10d).
#   Needs §0 (sim, PASSIVE_FACTOR, MASS_FLOOR), §1c (snaps_arr, redshift, t_cosmic_yr).
Z_TARGET = 0.3
HALO_CLASSES = [("field",    0.0,  12.5, "k",       "o"),   # background population
                ("groups",   12.5, 14., "#FF8C00", "^"),   # dark orange triangles
                ("clusters", 14., np.inf, "#138A4E", "s")] # green squares (~empty in this box)
FS_LAB, FS_TICK, FS_LEG, FS_TITLE = 26, 20, 18, 24

_it = int(np.argmin(np.abs(redshift - Z_TARGET)))
SNAP_T, Z_T, tH_T = int(snaps_arr[_it]), float(redshift[_it]), float(t_cosmic_yr[_it])
print(f"snapshot {SNAP_T} at z={Z_T:.3f} (target {Z_TARGET}); t_H={tH_T/1e9:.2f} Gyr")

def _gfield(f, name, grp="galaxy_data"):
    for p in (f"{grp}/dicts/{name}", f"{grp}/{name}"):
        if p in f:
            return np.asarray(f[p][:], dtype=np.float64)
    return None

with h5py.File(sim.get_caesar_file(SNAP_T), "r") as f:
    ms   = _gfield(f, "masses.stellar"); sf = _gfield(f, "sfr")
    mg   = _gfield(f, "masses.H2")    
    md = _gfield(f, "masses.dust")
    age  = _gfield(f, "ages.mass_weighted")
    par  = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
    mh_h = _gfield(f, "masses.total", grp="halo_data")
assert all(a is not None for a in (ms, sf, mg, md, mh_h)), "missing catalogue fields"
if age is None:
    print("WARNING: ages.mass_weighted absent -> age column will be empty")
    age = np.full_like(ms, np.nan)

mh = np.where(par >= 0, mh_h[np.clip(par, 0, len(mh_h) - 1)], np.nan)
with np.errstate(all="ignore"):
    ssfr_yr = np.where(ms > 0, sf / ms, np.nan)
    lm      = np.log10(np.where(ms > 0, ms, np.nan))
    lmh     = np.log10(np.where(mh > 0, mh, np.nan))
    fdust_l = np.where((ms > 0) & (md > 0), md / ms, np.nan)
    fgas_l  = np.where((ms > 0) & (mg > 0), mg / ms, np.nan)
    ssfr_l  = np.where(ssfr_yr > 0, ssfr_yr * 1e9, np.nan)             # [Gyr^-1]
    tdep_l  = np.where((sf > 0) & (mg > 0), mg / sf / 1e9, np.nan)     # [Gyr]
    ms_l    = np.where(ms > 0, ms, np.nan)
qui = np.isfinite(ssfr_yr) & (ssfr_yr < PASSIVE_FACTOR / tH_T) & (lm > MASS_FLOOR)
CLS = [(nm, qui & (lmh >= lo) & (lmh < hi), col, mk) for nm, lo, hi, col, mk in HALO_CLASSES]
print(f"quiescent (full box, logM*>{MASS_FLOOR}): {int(qui.sum())} | " +
      ", ".join(f"{nm}={int(m.sum())}" for nm, m, _c, _k in CLS))
if CLS[2][1].sum() == 0:
    print("  (no >=14.5 halos in this box -> 'groups' is the densest available environment)")

XCOLS = [(ms_l,   r"$M_\star\;[M_\odot]$",            "log"),
         (age,    r"mass-weighted age $\rm[Gyr]$",    "linear"),
         (ssfr_l, r"sSFR $\rm[Gyr^{-1}]$",            "log")]
YROWS = [(fdust_l, r"$M_{\rm dust}/M_\star$",                 "log"),
         (fgas_l,  r"$M_{\rm H2}/M_\star$",                  "log"),
         (tdep_l,  r"$t_{\rm dep}=M_{\rm H2}/{\rm SFR}\;\rm[Gyr]$", "log")]

def _tolog(v, scale):
    with np.errstate(all="ignore"):
        return np.log10(np.where(v > 0, v, np.nan)) if scale == "log" else v

def _med_band_x(x, y, nbins=8, nmin=8):
    """running median + 25-75% of y in quantile bins of x (both already in plot space)."""
    g = np.isfinite(x) & np.isfinite(y)
    if g.sum() < 2 * nmin:
        return None
    e = np.nanquantile(x[g], np.linspace(0, 1, nbins + 1))
    rows = []
    for b in range(nbins):
        m = g & (x >= e[b]) & (x <= e[b + 1])
        if m.sum() >= nmin:
            rows.append((np.median(x[m]), *np.percentile(y[m], [50, 25, 75])))
    return np.array(rows).T if len(rows) >= 3 else None

def _cliplim(v, scale, nsig=3.5, pad=0.08):
    """MAD-sigma-clipped limits computed in plot space; returned in data units."""
    v = _tolog(v, scale); v = v[np.isfinite(v)]
    if v.size < 10:
        return None
    med = np.median(v); mad = 1.4826 * np.median(np.abs(v - med))
    keep = v[np.abs(v - med) < nsig * max(mad, 1e-12)]
    if keep.size < 5:
        keep = v
    lo, hi = float(keep.min()), float(keep.max()); span = max(hi - lo, 1e-6)
    lo, hi = lo - pad * span, hi + pad * span
    return (10 ** lo, 10 ** hi) if scale == "log" else (lo, hi)

_inall = np.zeros(len(ms), bool)
for _nm, m, _c, _k in CLS:
    _inall |= m

_PSTYLE = {"font.family": "STIXGeneral", "mathtext.fontset": "stix",
           "axes.linewidth": 1.6,
           "xtick.direction": "in",  "ytick.direction": "in",
           "xtick.top": True,        "ytick.right": True,
           "xtick.minor.visible": True, "ytick.minor.visible": True,
           "xtick.major.size": 9, "ytick.major.size": 9,
           "xtick.minor.size": 4.5, "ytick.minor.size": 4.5,
           "xtick.major.width": 1.4, "ytick.major.width": 1.4,
           "xtick.minor.width": 1.1, "ytick.minor.width": 1.1}
with plt.rc_context(_PSTYLE):
    fig, axes = plt.subplots(3, 3, figsize=(17, 14.5), squeeze=False, sharex="col", sharey="row")
    for ri, (yv, ylab, ysc) in enumerate(YROWS):
        for ci, (xv, xlab, xsc) in enumerate(XCOLS):
            ax = axes[ri, ci]
            ax.set_xscale(xsc); ax.set_yscale(ysc)
            for nm, m, col, mk in CLS:
                if nm == "field":          # background population: small open black symbols
                    ax.scatter(xv[m], yv[m], s=16, marker=mk, facecolors="none",
                               edgecolors="k", linewidths=0.6, alpha=0.5, rasterized=True)
                else:                      # highlighted classes: open colored symbols
                    ax.scatter(xv[m], yv[m], s=55, marker=mk, facecolors="none",
                               edgecolors=col, linewidths=1.3, alpha=0.85, rasterized=True)
                mb = _med_band_x(_tolog(xv[m], xsc), _tolog(yv[m], ysc))
                if mb is None:
                    continue
                xc, med, q1, q3 = mb
                if xsc == "log": xc = 10 ** xc
                if ysc == "log": med, q1, q3 = 10 ** med, 10 ** q1, 10 ** q3
                _ls = "--" if nm == "field" else "-"
                ax.plot(xc, med, ls=_ls, color=col, lw=2.6, marker=mk, ms=14,
                        mfc=("w" if nm == "field" else col), mec="k", mew=1.4,
                        label=f"{nm} (N={int(m.sum())})", zorder=5)
                ax.fill_between(xc, q1, q3, color=col, alpha=0.13, lw=0)
            # fraction of each class with the plotted pair defined ("detected" here)
            # _yt = 0.03
            # for nm, m, col, _mk in CLS:
            #     if m.sum() < 5:
            #         continue
            #     _fd = float(np.mean(np.isfinite(_tolog(xv[m], xsc))
            #                         & np.isfinite(_tolog(yv[m], ysc))))
            #     ax.text(0.03, _yt, f"$f_{{\\rm det}}$ {nm}: {_fd:.2f}",
            #             transform=ax.transAxes, ha="left", va="bottom", fontsize=28,
            #             color=("k" if nm == "field" else col), zorder=6)
            #     _yt += 0.07
            # fraction of each class with a non-zero (plottable) y-value on this log panel.
            # Based on the y-quantity ONLY, so it's consistent across the three x-columns
            # of a row and matches the 'dust-free'/'SFR=0' summary printed below.
            # (Folding in x-definedness would spuriously depress the sSFR column, where
            #  truly-quenched SFR=0 galaxies have no defined sSFR.)
            _yt = 0.03
            for nm, m, col, _mk in CLS:
                if m.sum() < 5:
                    continue
                _ok = np.isfinite(_tolog(yv[m], ysc))
                _fd = 100*float(np.mean(_ok)) if _ok.size else np.nan
                ax.text(0.03, _yt, f"$f_{{\\rm det}}$ {nm}: {_fd:.0f}%",
                        transform=ax.transAxes, ha="left", va="bottom", fontsize=26, fontweight="bold",
                        color=("k" if nm == "field" else col), zorder=6,
                        bbox=dict(boxstyle="round,pad=0.25",
                                  fc="white", ec=("k" if nm == "field" else col), lw=1.3, alpha=0.9))
                _yt += 0.095
            ax.tick_params(labelsize=FS_TICK, which="both")
            if ri == 2:
                ax.set_xlabel(xlab, fontsize=FS_LAB)
            if ci == 0:
                ax.set_ylabel(ylab, fontsize=FS_LAB)
    for ci, (xv, _l, xsc) in enumerate(XCOLS):
        lim = _cliplim(xv[_inall], xsc)
        if lim:
            axes[0, ci].set_xlim(*lim)
    for ri, (yv, _l, ysc) in enumerate(YROWS):
        lim = _cliplim(yv[_inall], ysc)
        if lim:
            axes[ri, 0].set_ylim(*lim)
    axes[0, 0].legend(fontsize=FS_LEG, loc="best", framealpha=0.95, edgecolor="k")
    fig.suptitle(f"Quiescent galaxies at z={Z_T:.2f} (full box; sSFR$<0.2/t_H$, "
                 f"$\\log M_\\star>${MASS_FLOOR})", y=1.0, fontsize=FS_TITLE)
    plt.tight_layout()
    _ztag = f"{Z_T:.2f}".replace(".", "p")
    plt.savefig(figpath( f"quiescent_grid_z{_ztag}.png"), dpi=160, bbox_inches="tight")
    plt.show()

# summary: medians + populations invisible on the log panels (dust-free, SFR=0)
print(f"{'class':10s} {'logMh':>11s} {'N':>6s} {'med logM*':>10s} {'med age':>8s} "
      f"{'med lfgas':>10s} {'med lfdust':>11s} {'med ltdep':>10s} {'dust-free':>10s} {'SFR=0':>7s}")
for (nm, lo, hi, _c, _k), (_n2, m, _c2, _k2) in zip(HALO_CLASSES, CLS):
    if m.sum() < 5:
        print(f"{nm:10s} {'':>11s} {int(m.sum()):6d}   (too few to summarize)")
        continue
    rng = f"[{lo:g},{hi:g})" if np.isfinite(hi) else f">={lo:g}"
    with np.errstate(all="ignore"):
        print(f"{nm:10s} {rng:>11s} {int(m.sum()):6d} {np.nanmedian(lm[m]):10.2f} "
              f"{np.nanmedian(age[m]):8.2f} {np.nanmedian(np.log10(fgas_l[m])):10.2f} "
              f"{np.nanmedian(np.log10(fdust_l[m])):11.2f} "
              f"{np.nanmedian(np.log10(tdep_l[m])):10.2f} "
              f"{np.mean(md[m] <= 0):10.2f} {np.mean(sf[m] <= 0):7.2f}")

plt.savefig(figpath('Environment_dependent_dust_gas.png'))

In [ ]:
# 10c-t · depletion time vs M*, MS | quiescent, across Z_LIST — BOTH definitions:
#   solid  t_dep(H2)  = M_H2/SFR   (CO-equivalent; SIMBA's SF law makes this ~Gyr by design)
#   dashed t_dep(gas) = M_gas/SFR  (total bound ISM; what §10c plotted)
#   Diagnoses the ~2 dex sim-vs-obs offset. Needs §10c (_gfield, _PSTYLE, FS_*), §0/§1c.
TDEP_Z    = globals().get("Z_LIST", [0.0, 0.3, 0.4, 0.6, 0.7])
MS_FACTOR = 1.0      # MS: sSFR > this / t_H(z); quiescent: sSFR < PASSIVE_FACTOR / t_H(z)
_tdcols = [plt.get_cmap("coolwarm")(x) for x in np.linspace(0.05, 0.95, len(TDEP_Z))]
_tdmks  = ["o", "s", "^", "D", "v", "P"][:len(TDEP_Z)]

def _load_tdep(zt):
    it = int(np.argmin(np.abs(redshift - zt)))
    snap, zs, tH = int(snaps_arr[it]), float(redshift[it]), float(t_cosmic_yr[it])
    with h5py.File(sim.get_caesar_file(snap), "r") as f:
        ms = _gfield(f, "masses.stellar"); sf = _gfield(f, "sfr")
        mg = _gfield(f, "masses.gas");    m2 = _gfield(f, "masses.H2")
    with np.errstate(all="ignore"):
        ssfr = np.where(ms > 0, sf / ms, np.nan)
        lm   = np.log10(np.where(ms > 0, ms, np.nan))
        td2  = np.where((sf > 0) & (m2 > 0), m2 / sf / 1e9, np.nan)   # Gyr, CO-equivalent
        tdg  = np.where((sf > 0) & (mg > 0), mg / sf / 1e9, np.nan)   # Gyr, total bound ISM
        fH2  = np.where((mg > 0) & (m2 > 0), m2 / mg, np.nan)
    okm = lm > MASS_FLOOR
    return zs, dict(lm=lm, td2=td2, tdg=tdg, fH2=fH2, sf0=okm & (sf <= 0),
                    ms=okm & np.isfinite(ssfr) & (ssfr > MS_FACTOR / tH),
                    q =okm & np.isfinite(ssfr) & (ssfr < PASSIVE_FACTOR / tH))

def _medx(x, y, nbins=6, nmin=10):
    """median + 25-75% of log10(y) in quantile bins of x; arrays in (x_lin-log10M*, y_Gyr)."""
    g = np.isfinite(x) & np.isfinite(y) & (y > 0)
    if g.sum() < 2 * nmin:
        return None
    ly = np.log10(y)
    e = np.nanquantile(x[g], np.linspace(0, 1, nbins + 1))
    rows = []
    for b in range(nbins):
        m = g & (x >= e[b]) & (x <= e[b + 1])
        if m.sum() >= nmin:
            rows.append((np.median(x[m]), *np.percentile(ly[m], [50, 16, 86])))
    return np.array(rows).T if len(rows) >= 3 else None

TD = [_load_tdep(zt) for zt in TDEP_Z]
for zs, D in TD:
    print(f"z={zs:.2f}: MS={int(D['ms'].sum())}, quiescent={int(D['q'].sum())}")

with plt.rc_context(_PSTYLE):
    fig, axes = plt.subplots(1, 2, figsize=(16.5, 7.6), sharey=True)
    _ext = {0: [], 1: []}
    for pi, (cls, ptitle) in enumerate([("ms", "main sequence"), ("q", "quiescent")]):
        ax = axes[pi]; ax.set_yscale("log")
        for (zs, D), col, mk in zip(TD, _tdcols, _tdmks):
            m = D[cls]
            for tkey, ls, mfc in [("td2", "-", col), ("tdg", "--", "w")]:
                mb = _medx(D["lm"][m], D[tkey][m])
                if mb is None:
                    continue
                xc, med, q1, q3 = mb
                lab = f"z={zs:.2f} (N={int(m.sum())})" if tkey == "td2" else None
                ax.plot(xc, 10 ** med, ls, color=col, lw=2.6, marker=mk, ms=10,
                        mfc=mfc, mec="k", mew=1.1, label=lab, zorder=5)
                if tkey == "td2":
                    ax.fill_between(xc, 10 ** q1, 10 ** q3, color=col, alpha=0.10, lw=0)
                _ext[pi] += [float(q1.min()), float(q3.max())]
        if cls == "ms":   # observed molecular depletion band for MS galaxies
            ax.axhspan(0.5, 2.0, color="0.75", alpha=0.30, zorder=0)
            ax.text(0.02, 0.04, r"obs. MS $t_{\rm dep}({\rm H_2})$ ~ 0.5-2 Gyr (Tacconi+18)",
                    transform=ax.transAxes, fontsize=FS_LEG - 3, color="0.3")
        ax.set_title(ptitle, fontsize=FS_LAB - 2)
        ax.set_xlabel(r"$\log_{10} M_\star\;[M_\odot]$", fontsize=FS_LAB - 2)
        ax.tick_params(labelsize=FS_TICK - 2, which="both")
    lo = min(min(v) for v in _ext.values()); hi = max(max(v) for v in _ext.values())
    sp = max(hi - lo, 1e-6)
    axes[0].set_ylim(10 ** (lo - 0.06 * sp), 10 ** (hi + 0.06 * sp))
    axes[0].set_ylabel(r"$t_{\rm dep}$ [Gyr]   (solid: $M_{\rm H_2}$/SFR, dashed: $M_{\rm gas}$/SFR)",
                       fontsize=FS_LAB - 4)
    axes[0].legend(fontsize=FS_LEG - 3, loc="upper left", framealpha=0.95, edgecolor="k")
    fig.suptitle("Depletion time across redshift — CO-equivalent vs total-ISM definition "
                 f"(all galaxies, logM*>{MASS_FLOOR})", y=1.0, fontsize=FS_LAB - 2)
    plt.tight_layout()
    plt.savefig(figpath( "tdep_ms_vs_quiescent_zevol.png"), dpi=160,
                bbox_inches="tight")
    plt.show()

# diagnosis table: the definitional gap, the molecular fraction, and the invisible SFR=0 pop
print(f"\n{'z':>5s} {'class':>10s} {'N':>6s} {'med tdep(H2)':>13s} {'med tdep(gas)':>14s} "
      f"{'gap [dex]':>10s} {'med f_H2':>9s} {'SFR=0':>6s}")
for zs, D in TD:
    for cls, cname in [("ms", "MS"), ("q", "quiescent")]:
        m = D[cls]
        if m.sum() < 10:
            continue
        with np.errstate(all="ignore"):
            t2 = np.nanmedian(D["td2"][m]); tg = np.nanmedian(D["tdg"][m])
            gap = np.log10(tg / t2) if (t2 > 0 and tg > 0) else np.nan
            f2 = np.nanmedian(D["fH2"][m])
            s0 = np.mean(D["sf0"][m]) if m.sum() else np.nan
        print(f"{zs:5.2f} {cname:>10s} {int(m.sum()):6d} {t2:13.2f} {tg:14.2f} "
              f"{gap:10.2f} {f2:9.3f} {s0:6.2f}")
print("\n  -> if the solid MS curves sit in the grey band while the dashed ones are ~1-2 dex"
      "\n     higher, the sim-vs-obs offset is the M_gas-vs-M_H2 definition, not SIMBA.")


### VII.1b (§10c·z) · Group quiescent galaxies — median relations across redshift

The §10c grid, **medians only** (25–75% bands), for the group range ($\log M_h=13$–$14.5$)
overlaid at **z = 0, 0.3, 0.4, 0.6, 0.7** (one catalogue read per epoch). Frames are fit
tightly around the plotted median bands. The printed table gives the per-epoch fractions with
dust > 0 / gas > 0 / SFR > 0. **Run §10c first** (reuses its loader and style helpers).


In [ ]:
# 10c-z · group-range (logMh 13-14.5) median relations at z = 0, 0.3, 0.4, 0.6, 0.7.
#   Medians only, same paper style. Needs §10c (_gfield, _med_band_x, _tolog, _cliplim,
#   _PSTYLE, FS_*), §0/§1c (sim, PASSIVE_FACTOR, MASS_FLOOR, snaps_arr, redshift, t_cosmic_yr).
Z_LIST    = [0.0, 0.3, 0.4, 0.6, 0.7]
GRP_LOGMH = (13.0, 14.5)
_zcols = [plt.get_cmap("coolwarm")(x) for x in np.linspace(0.05, 0.95, len(Z_LIST))]
_zmks  = ["o", "s", "^", "D", "v"]

def _load_grid_data(zt):
    """quiescent GROUP galaxies at the snapshot nearest zt -> (z_snap, mask, quantity dict)."""
    it = int(np.argmin(np.abs(redshift - zt)))
    snap, zs, tH = int(snaps_arr[it]), float(redshift[it]), float(t_cosmic_yr[it])
    with h5py.File(sim.get_caesar_file(snap), "r") as f:
        ms = _gfield(f, "masses.stellar"); sf = _gfield(f, "sfr")
        mg = _gfield(f, "masses.H2");    md = _gfield(f, "masses.dust")
        ag = _gfield(f, "ages.mass_weighted")
        pr = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
        mhh = _gfield(f, "masses.total", grp="halo_data")
    mh = np.where(pr >= 0, mhh[np.clip(pr, 0, len(mhh) - 1)], np.nan)
    with np.errstate(all="ignore"):
        ssfr = np.where(ms > 0, sf / ms, np.nan)
        lm   = np.log10(np.where(ms > 1e10, ms, np.nan))
        lmh  = np.log10(np.where(mh > 0, mh, np.nan))
        D = dict(ms    = np.where(ms > 0, ms, np.nan),
                 age   = (ag if ag is not None else np.full_like(ms, np.nan)),
                 ssfr  = np.where(ssfr > 0, ssfr * 1e9, np.nan),
                 fdust = np.where((ms > 0) & (md > 0), md / ms, np.nan),
                 MH2   = np.where((ms > 0) & (md > 0), mg, np.nan),
                 fH2  = np.where((ms > 0) & (mg > 0), mg / ms, np.nan),
                 tdep  = np.where((sf > 0) & (mg > 0), mg / sf / 1e9, np.nan))
    sel = (np.isfinite(ssfr) & (ssfr < PASSIVE_FACTOR / tH) & (lm > MASS_FLOOR)
           & (lmh >= GRP_LOGMH[0]) & (lmh < GRP_LOGMH[1]))
    return zs, sel, D

XSPEC = [("ms",   r"$M_\star\;[M_\odot]$",                          "log"),
         ("age",  r"mass-weighted age $\rm[Gyr]$",                  "linear"),
         ("ssfr", r"sSFR $\rm[Gyr^{-1}]$",                          "log")]
YSPEC = [("MH2", r"$M_{\rm H_2} [M_{\odot}]$",                       "log"),
         ("fH2",  r"$M_{\rm H_2}/M_\star$",                        "log"),
         ("tdep",  r"$t_{\rm dep}=M_{\rm H_2}/{\rm SFR}\;\rm[Gyr]$", "log"),
        ("fdust", r"$M_{\rm dust}/M_\star$",                       "log")]
zorder = 5
DATA = []
for zt in Z_LIST:
    zs, sel, D = _load_grid_data(zt)
    DATA.append((zs, sel, D))
    print(f"z={zs:.2f}: {int(sel.sum())} quiescent group galaxies")

with plt.rc_context(_PSTYLE):
    fig, axes = plt.subplots(4, 3, figsize=(17, 18.5), squeeze=False, sharex="col", sharey="row")
    _xext = {ci: [] for ci in range(len(XSPEC))}; _yext = {ri: [] for ri in range(len(YSPEC))}
    for ri, (yk, ylab, ysc) in enumerate(YSPEC):
        for ci, (xk, xlab, xsc) in enumerate(XSPEC):
            ax = axes[ri, ci]
            ax.set_xscale(xsc); ax.set_yscale(ysc)
            for (zs, sel, D), col, mk in zip(DATA, _zcols, _zmks):
                mb = _med_band_x(_tolog(D[xk][sel], xsc), _tolog(D[yk][sel], ysc),
                                 nbins=8, nmin=2)
                if mb is None:
                    continue
                xc, med, q1, q3 = mb
                _xext[ci] += [float(np.min(xc)), float(np.max(xc))]      # plot-space extents
                _yext[ri] += [float(np.nanmin(q1)), float(np.nanmax(q3))]
                if xsc == "log":
                    xc = 10 ** xc
                if ysc == "log":
                    med, q1, q3 = 10 ** med, 10 ** q1, 10 ** q3
                if mk=="^":
                    zorder =50
                ax.plot(xc, med, "-", color=col, lw=2.8, marker=mk, ms=12, mfc=col,
                        mec="k", mew=1.2, label=f"z={zs:.2f} (N={int(sel.sum())})", zorder=zorder)
                ax.fill_between(xc, q1, q3, color=col, alpha=0.10, lw=0)
            ax.tick_params(labelsize=FS_TICK, which="both")
            if ri == 3:
                ax.set_xlabel(xlab, fontsize=FS_LAB)
            if ci == 0:
                ax.set_ylabel(ylab, fontsize=FS_LAB)
    # frames tight around the plotted median bands (medians-only figure -> no scatter margin)
    PAD = 0.06
    for ci, (_xk, _l, xsc) in enumerate(XSPEC):
        if _xext[ci]:
            lo, hi = min(_xext[ci]), max(_xext[ci]); sp = max(hi - lo, 1e-6)
            lo, hi = lo - PAD * sp, hi + PAD * sp
            axes[0, ci].set_xlim(*((10 ** lo, 10 ** hi) if xsc == "log" else (lo, hi)))
    for ri, (_yk, _l, ysc) in enumerate(YSPEC):
        if _yext[ri]:
            lo, hi = min(_yext[ri]), max(_yext[ri]); sp = max(hi - lo, 1e-6)
            lo, hi = lo - PAD * sp, hi + PAD * sp
            axes[ri, 0].set_ylim(*((10 ** lo, 10 ** hi) if ysc == "log" else (lo, hi)))
    axes[0, 0].legend(fontsize=FS_LEG - 2, loc="best", framealpha=0.95, edgecolor="k")
    fig.suptitle(f"Quiescent group galaxies ($\\log M_h$ {GRP_LOGMH[0]:g}-{GRP_LOGMH[1]:g})"
                 " — median relations vs redshift", y=1.0, fontsize=FS_TITLE)
    plt.tight_layout()
    plt.savefig(figpath( "quiescent_grid_groups_zevol.png"), dpi=160,
                bbox_inches="tight")
    plt.show()

print("\nfraction of quiescent group galaxies with the row quantity defined:")
print(f"{'z':>5s} {'N':>6s} {'dust>0':>7s} {'gas>0':>6s} {'SFR>0':>6s}")
for zs, sel, D in DATA:
    print(f"{zs:5.2f} {int(sel.sum()):6d} {np.mean(np.isfinite(D['fdust'][sel])):7.2f} "
          f"{np.mean(np.isfinite(D['fH2'][sel])):6.2f} "
          f"{np.mean(np.isfinite(D['tdep'][sel])):6.2f}")

plt.savefig(figpath('Redshift_dependent_dust_gas_in_groups.png'))

### VII.2 (§10d) · Is molecular gas connected to AGN in quiescent group galaxies? — observational analog test

**Comparison target.** Group environments ($\log M_h=13$–$14.5$), $\log M_\star=10$–$12$,
just below the MS or $\log({\rm sSFR\,[Gyr^{-1}]})=-2$ to $-1$ (weak SED SFRs → possibly
lower). CO detected in 5; **4/5 CO-detected passive galaxies show high NII/Hα** (AGN/shocks).

**Dual BH classification, paired panels (same colors, exclusive & exhaustive classes):**
- **Eddington-ratio classes (primary, mode-based):** inefficient ($\lambda<0.01$, the
  LERG/jet/LINER regime), efficient ($0.01\le\lambda<0.1$), high ($\lambda\ge0.1$).
  $\lambda=0.01$ is the HERG/LERG dichotomy (**Best & Heckman 2012**); SIMBA's own transitions
  at $f_{\rm Edd}=0.2$/$0.02$ (Davé et al. 2019) bracket the same physics (full-jet line
  marked on the $f_{\rm Edd}$ panel).
- **$L_{\rm bol}=0.1\,\dot M_{\rm BH}c^2$ classes (observable-facing):** inactive
  ($<10^{42}$), moderate ($10^{42}$–$10^{44}$), strong ($>10^{44}$ erg s$^{-1}$). The
  $10^{42}$ anchor descends from the X-ray convention $L_X>10^{42}$ (Ranalli et al. 2003;
  Brandt & Alexander 2015; $k_{\rm bol}\sim10$–20, Duras et al. 2020 → our bolometric cut is
  the permissive version); $10^{44}$ is the conventional Seyfert/quasar-scale divide
  (Schmidt & Green 1983; Hickox & Alexander 2018). Thomas et al. (2019) for $L_{\rm bol}$ in
  SIMBA. The **jet-mode flag** overlaps both schemes (mode, not luminosity) → dashed, outside
  the sums.

> **Inclination & attenuation:** $L_{\rm bol}$ and $\lambda$ are *intrinsic* (no radiative
> transfer) → class boundaries are viewing-angle/attenuation-free by construction. NII/Hα is a
> close pair (ratio ≈ reddening-free) but line detectability and $L_{\rm bol}$ proxies are
> obscuration-dependent, and obscuration correlates with the gas/dust content under test →
> observed AGN fractions among CO-rich galaxies are **lower limits**; SIMBA's intrinsic
> fractions are the upper envelope. LINER-like ratios can also be post-AGB/shock-powered
> (WHAN; Cid Fernandes et al. 2011) — degenerate with the *inefficient* class, which is the
> right comparison line for the 4/5 statistic.

The M$_{\rm H_2}$–sSFR panel carries an inset with the per-class median tracks (25–75% bands),
framed tightly around the tracks with tick numbers inside.


In [ ]:
# 10d · CO-AGN connection in quiescent group galaxies — dual classification, paired panels.
#   Eddington classes (robust, mode-based; Best & Heckman 2012 lambda=0.01) PRIMARY;
#   L_bol classes (observable-facing; X-ray 1e42 convention / Seyfert-quasar 1e44) TWIN.
#   Same colors in both -> the two mix panels and the two detection panels compare 1:1.
#   Needs §0 (sim, const, u, fisher_exact, mannwhitneyu, spearmanr), §1c, §10c (_PSTYLE, FS_*).
Z_TARGET_OBS = 0.3
OBS_LOGM     = (10.0, 12.0)
OBS_LOGMH    = (13.0, 14.5)
OBS_LSSFR_W  = (-2.0, -1.0)         # target log sSFR [Gyr^-1] window
MH2_DET      = 1e9                  # CO-detection proxy [Msun]
FEDD_EFF     = 0.01                 # rad. efficient boundary (HERG/LERG, Best & Heckman 2012)
FEDD_HIGH    = 0.1                  # 'high' accretion (conventional luminous-AGN scale)
LBOL_MOD     = 1e42                 # observable-facing anchors [erg/s] (see markdown refs)
LBOL_STRONG  = 1e44
JET_MBH, JET_FE, JET_FULL = 10**7.5, 0.2, 0.02
C_LOW, C_MID, C_HI, C_JET = "0.45", "#FF8C00", "#C2185B", "#0173B2"
_PSTYLE = globals().get("_PSTYLE") or {
    "font.family": "STIXGeneral", "mathtext.fontset": "stix", "axes.linewidth": 1.6,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "xtick.major.size": 9, "ytick.major.size": 9, "xtick.minor.size": 4.5, "ytick.minor.size": 4.5}
FS_LAB, FS_TICK, FS_LEG = (globals().get("FS_LAB", 28), globals().get("FS_TICK", 26),
                           globals().get("FS_LEG", 22))

_it = int(np.argmin(np.abs(redshift - Z_TARGET_OBS)))
SNAP_O, Z_O, tH_O = int(snaps_arr[_it]), float(redshift[_it]), float(t_cosmic_yr[_it])
print(f"snapshot {SNAP_O} at z={Z_O:.3f} (target {Z_TARGET_OBS})")

def _gfield2(f, names, grp="galaxy_data"):
    for nm in ([names] if isinstance(names, str) else names):
        for p in (f"{grp}/dicts/{nm}", f"{grp}/{nm}"):
            if p in f:
                return np.asarray(f[p][:], dtype=np.float64)
    return None

with h5py.File(sim.get_caesar_file(SNAP_O), "r") as f:
    ms   = _gfield2(f, "masses.stellar"); sf  = _gfield2(f, "sfr")
    mh2  = _gfield2(f, "masses.H2")
    bhmd = _gfield2(f, ["bhmdot", "bh_mdot"])
    bhfe = _gfield2(f, ["bh_fedd", "bhfedd", "fedd"])
    bhm  = _gfield2(f, ["masses.bh", "masses.bh_mass", "bhmass"])
    par  = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
    mh_h = _gfield2(f, "masses.total", grp="halo_data")
assert all(a is not None for a in (ms, sf, mh2, mh_h)), "missing catalogue fields"
assert all(a is not None for a in (bhmd, bhfe, bhm)), "BH fields not found"

mh = np.where(par >= 0, mh_h[np.clip(par, 0, len(mh_h) - 1)], np.nan)
_c2 = (const.c ** 2 * u.Msun / u.yr).to(u.erg / u.s).value
with np.errstate(all="ignore"):
    lm     = np.log10(np.where(ms > 0, ms, np.nan))
    lmh    = np.log10(np.where(mh > 0, mh, np.nan))
    lssfr  = np.log10(np.where((ms > 0) & (sf > 0), sf / ms * 1e9, np.nan))
    ssfr_l = np.where((ms > 0) & (sf > 0), sf / ms * 1e9, np.nan)
    Lbol   = 0.1 * np.where(bhmd > 0, bhmd, np.nan) * _c2
    fedd   = np.where(bhfe > 0, bhfe, np.nan)

base    = ((lm >= OBS_LOGM[0]) & (lm <= OBS_LOGM[1])
           & (lmh >= OBS_LOGMH[0]) & (lmh <= OBS_LOGMH[1]))
belowMS = base & (np.isnan(lssfr) | (lssfr < OBS_LSSFR_W[1]))
inwin   = base & np.isfinite(lssfr) & (lssfr >= OBS_LSSFR_W[0]) & (lssfr < OBS_LSSFR_W[1])
co_det  = mh2 > MH2_DET
is_jet  = np.isfinite(bhm) & np.isfinite(bhfe) & (bhm > JET_MBH) & (bhfe < JET_FE)
# Eddington classes (exclusive & exhaustive; BH-less -> inefficient)
e_low = ~np.isfinite(fedd) | (fedd < FEDD_EFF)
e_mid = np.isfinite(fedd) & (fedd >= FEDD_EFF) & (fedd < FEDD_HIGH)
e_hi  = np.isfinite(fedd) & (fedd >= FEDD_HIGH)
# L_bol classes (exclusive & exhaustive)
l_low = ~np.isfinite(Lbol) | (Lbol < LBOL_MOD)
l_mid = np.isfinite(Lbol) & (Lbol >= LBOL_MOD) & (Lbol < LBOL_STRONG)
l_hi  = np.isfinite(Lbol) & (Lbol >= LBOL_STRONG)
ECLS = [(r"inefficient ($\lambda<0.01$)", e_low, C_LOW, "o"),
        (r"efficient ($0.01$–$0.1$)",     e_mid, C_MID, "^"),
        (r"high ($\lambda\geq0.1$)",      e_hi,  C_HI,  "D")]
LCLS = [(r"inactive ($<10^{42}$)",        l_low, C_LOW, "o"),
        (r"moderate ($10^{42}$–$10^{44}$)", l_mid, C_MID, "^"),
        (r"strong ($>10^{44}$)",          l_hi,  C_HI,  "D")]
print(f"analog sample (below MS): {int(belowMS.sum())} | in window: {int(inwin.sum())}"
      f" | CO-det (M_H2>{MH2_DET:.0e}): {int((belowMS & co_det).sum())}")

# _wilson (binomial CI): canonical definition in §7·setup (Part II)
for sampname, samp in [("below-MS (full analog)", belowMS), ("sSFR window [-2,-1]", inwin)]:
    print(f"\n--- {sampname}: N={int(samp.sum())} ---")
    for aname, aflag in [(f"rad. efficient (fEdd>{FEDD_EFF:g})", e_mid | e_hi),
                         (f"AGN L>{LBOL_MOD:.0e}", l_mid | l_hi),
                         (f"strong L>{LBOL_STRONG:.0e}", l_hi), ("jet mode", is_jet)]:
        a = int(np.sum(samp & co_det & aflag));  b = int(np.sum(samp & co_det & ~aflag))
        c = int(np.sum(samp & ~co_det & aflag)); d = int(np.sum(samp & ~co_det & ~aflag))
        if min(a + b, c + d) == 0:
            print(f"  {aname:28s}: contingency degenerate"); continue
        _, pf = fisher_exact([[a, b], [c, d]])
        p1 = _wilson(a, a + b); p2 = _wilson(c, c + d)
        print(f"  {aname:28s}: P(AGN|CO-det)={p1[0]:.2f} [{p1[1]:.2f},{p1[2]:.2f}] (N={a+b})"
              f"  P(AGN|no CO)={p2[0]:.2f} [{p2[1]:.2f},{p2[2]:.2f}] (N={c+d})  Fisher p={pf:.2g}")
_g = belowMS & (mh2 > 0) & np.isfinite(Lbol)
if _g.sum() > 10:
    rho, pv = spearmanr(np.log10(mh2[_g]), np.log10(Lbol[_g]))
    print(f"\nSpearman(log M_H2, log L_bol) below MS, both >0: rho={rho:+.2f} "
          f"(p={pv:.1g}, N={int(_g.sum())})")

def _lim10(v, nsig=3.5, pad=0.08):
    v = np.log10(v[np.isfinite(v) & (v > 0)])
    if v.size < 10: return None
    med = np.median(v); mad = 1.4826 * np.median(np.abs(v - med))
    keep = v[np.abs(v - med) < nsig * max(mad, 1e-12)]
    if keep.size < 5: keep = v
    lo, hi = keep.min(), keep.max(); span = max(hi - lo, 1e-6)
    return 10 ** (lo - pad * span), 10 ** (hi + pad * span)

def _mbx(x, y, nbins=5, nmin=6, band=(25, 75)):
    """running median + percentile band (log space in, log space out)."""
    g = np.isfinite(x) & np.isfinite(y)
    if g.sum() < 2 * nmin: return None
    e = np.nanquantile(x[g], np.linspace(0, 1, nbins + 1))
    rows = []
    for b in range(nbins):
        m = g & (x >= e[b]) & (x <= e[b + 1])
        if m.sum() >= nmin:
            rows.append((np.median(x[m]), *np.percentile(y[m], [50, band[0], band[1]])))
    return np.array(rows).T if len(rows) >= 3 else None

def _mix_panel(ax, classes, xtitle):
    """fraction of each (exclusive) class per M_H2 bin (solid, sums to 1) + dashed jet."""
    h2e = [0, 1e7, 1e8, MH2_DET, 1e10, np.inf]
    for lab, flag, col, mk, ls in ([(l, f, c, m, "-") for l, f, c, m in classes]
                                   + [("jet mode (overlaps)", is_jet, C_JET, "s", "--")]):
        xc, fr, lo_, hi_ = [], [], [], []
        for b in range(len(h2e) - 1):
            m = belowMS & (mh2 >= h2e[b]) & (mh2 < h2e[b + 1])
            if m.sum() < 8: continue
            xc.append(max(np.median(mh2[m]), 10 ** 6.3))
            p, l, h = _wilson(int(np.sum(m & flag)), int(m.sum()))
            fr.append(p); lo_.append(l); hi_.append(h)
        if xc:
            ax.errorbar(xc, fr, yerr=[np.array(fr) - lo_, np.array(hi_) - fr], fmt=ls + mk,
                        color=col, ms=11, mec="k", mew=1.1, capsize=4, lw=2.4, label=lab)
    ax.set_xscale("log"); ax.axvline(MH2_DET, ls="--", c="k", lw=1.3); ax.set_ylim(0, 1)
    ax.set_xlabel(r"$M_{\rm H_2}\;[M_\odot]$ (leftmost = H$_2$-free)", fontsize=FS_LAB - 2)
    ax.set_title(xtitle, fontsize=FS_LAB - 2)
    ax.legend(fontsize=FS_LEG - 4, framealpha=0.95, edgecolor="k")

def _det_panel(ax, q, edges, xlab, spans, vline=None):
    """CO-detected fraction vs q (first point = no/undetected BH)."""
    for slo, shi, scol in spans:
        ax.axvspan(slo, shi, color=scol, alpha=0.10)
    xs, fd, flo, fhi = [], [], [], []
    m = belowMS & (~np.isfinite(q) | (q < edges[0]))
    if m.sum() >= 8:
        p, l, h = _wilson(int(np.sum(m & co_det)), int(m.sum()))
        xs.append(edges[0] / 10); fd.append(p); flo.append(l); fhi.append(h)
    for b in range(len(edges) - 1):
        m = belowMS & np.isfinite(q) & (q >= edges[b]) & (q < edges[b + 1])
        if m.sum() < 8: continue
        xs.append(10 ** (0.5 * (np.log10(edges[b]) + np.log10(edges[b + 1]))))
        p, l, h = _wilson(int(np.sum(m & co_det)), int(m.sum()))
        fd.append(p); flo.append(l); fhi.append(h)
    ax.errorbar(xs, fd, yerr=[np.array(fd) - flo, np.array(fhi) - fd], fmt="-o",
                color="#029E73", ms=11, mec="k", mew=1.1, capsize=4, lw=2.4)
    if vline is not None:
        ax.axvline(vline, ls="--", c=C_JET, lw=1.8)
    ax.set_xscale("log"); ax.set_ylim(0, 1)
    ax.set_xlabel(xlab, fontsize=FS_LAB - 2)
    ax.set_ylabel(f"CO-detected fraction", fontsize=FS_LAB - 2)

with plt.rc_context(_PSTYLE):
    fig, ax = plt.subplots(2, 3, figsize=(20, 12.5))
    # (a) M_H2 vs sSFR by Eddington class + median-track INSET
    m0 = belowMS & (mh2 > 0)
    for lab, flag, col, mk in ECLS:
        mm = m0 & flag
        if lab.startswith("inefficient"):
            ax[0, 0].scatter(ssfr_l[mm], mh2[mm], s=16, marker=mk, facecolors="none",
                             edgecolors="k", linewidths=0.6, alpha=0.45, rasterized=True)
        else:
            ax[0, 0].scatter(ssfr_l[mm], mh2[mm], s=100, marker=mk, color=col,
                             edgecolors="k", linewidths=1.1, alpha=0.9, label=f"{lab} (N={int(mm.sum())})")
    jm = m0 & is_jet
    ax[0, 0].scatter(ssfr_l[jm], mh2[jm], s=110, marker="s", facecolors="none",
                     edgecolors=C_JET, linewidths=1.5, label=f"jet mode (N={int(jm.sum())})")
    ax[0, 0].set_xscale("log"); ax[0, 0].set_yscale("log")
    ax[0, 0].axvspan(10 ** OBS_LSSFR_W[0], 10 ** OBS_LSSFR_W[1], color="gold", alpha=0.12)
    ax[0, 0].axhline(MH2_DET, ls="--", c="k", lw=1.3)
    xl = _lim10(ssfr_l[m0]); yl = _lim10(mh2[m0])
    if xl: ax[0, 0].set_xlim(*xl)
    if yl: ax[0, 0].set_ylim(*yl)
    ax[0, 0].set_xlabel(r"sSFR $\rm[Gyr^{-1}]$", fontsize=FS_LAB - 2)
    ax[0, 0].set_ylabel(r"$M_{\rm H_2}\;[M_\odot]$", fontsize=FS_LAB - 2)
    ax[0, 0].legend(fontsize=FS_LEG - 4, loc="upper left", framealpha=0.95, edgecolor="k")
    axin = ax[0, 0].inset_axes([0.56, 0.05, 0.43, 0.40])
    axin.set_xscale("log"); axin.set_yscale("log")
    _ixe, _iye = [], []
    for lab, flag, col, _mk in ECLS:
        mm = m0 & flag
        mb = _mbx(np.log10(ssfr_l[mm]), np.log10(mh2[mm]))
        if mb is None: continue
        _ixe += [float(mb[0].min()), float(mb[0].max())]
        _iye += [float(mb[2].min()), float(mb[3].max())]
        xc, med, q1, q3 = (10 ** v for v in mb)
        axin.plot(xc, med, "-", color=("k" if col == C_LOW else col), lw=2.4)
        axin.fill_between(xc, q1, q3, color=("k" if col == C_LOW else col), alpha=0.15, lw=0)
    if _ixe:                       # frame tight around the tracks, not the full scatter
        sp = max(max(_ixe) - min(_ixe), 1e-6)
        axin.set_xlim(10 ** (min(_ixe) - 0.08 * sp), 10 ** (max(_ixe) + 0.08 * sp))
        sp = max(max(_iye) - min(_iye), 1e-6)
        axin.set_ylim(10 ** (min(_iye) - 0.10 * sp), 10 ** (max(_iye) + 0.10 * sp))
    axin.tick_params(which="both", direction="in", labelsize=15)
    axin.tick_params(axis="x", pad=-24)            # tick numbers INSIDE the inset
    axin.tick_params(axis="y", pad=-10)
    for _t in axin.get_yticklabels():
        _t.set_ha("left")
    axin.xaxis.set_minor_formatter(plt.NullFormatter())
    axin.yaxis.set_minor_formatter(plt.NullFormatter())
    axin.set_title("medians ± IQR", fontsize=14)
    # (b) M_H2 distributions per Eddington class + jet
    bins = np.linspace(7.5, 10.8, 23)
    for lab, flag, col, _mk in ECLS:
        v = np.log10(mh2[belowMS & flag & (mh2 > 0)])
        if len(v) < 3:
            continue
        if col != C_LOW:           # filled low-alpha face under the solid colored lines
            ax[0, 1].hist(v, bins=bins, density=True, histtype="stepfilled",
                          facecolor=col, alpha=0.22, edgecolor="none")
        ax[0, 1].hist(v, bins=bins, density=True, histtype="step", lw=2.8,
                      color=("k" if col == C_LOW else col), label=f"{lab} (N={len(v)})")
    vj = np.log10(mh2[belowMS & is_jet & (mh2 > 0)])
    if len(vj) >= 3:
        ax[0, 1].hist(vj, bins=bins, density=True, histtype="step", lw=2, ls="--",
                      color=C_JET, label=f"jet mode (N={len(vj)})")
    ax[0, 1].axvline(np.log10(MH2_DET), ls="--", c="k", lw=1.3)
    ax[0, 1].set_xlabel(r"$\log_{10} M_{\rm H_2}\;[M_\odot]$", fontsize=FS_LAB - 2)
    ax[0, 1].set_ylabel("pdf", fontsize=FS_LAB - 2)
    ax[0, 1].legend(fontsize=FS_LEG - 4, framealpha=0.95, edgecolor="k")
    # (c) summary text panel slot -> use for sSFR-window-only mix? keep simple: window mix
    _mix_panel(ax[0, 2], ECLS, "BH-state mix — Eddington classes")
    _mix_panel(ax[1, 0], LCLS, r"BH-state mix — $L_{\rm bol}$ classes")
    for a_ in (ax[0, 2], ax[1, 0]):
        a_.set_ylabel("fraction (solid: sum = 1)", fontsize=FS_LAB - 2)
    # (d,e) CO-detected fraction vs f_Edd (full-jet line marked) and vs L_bol (regions shaded)
    _det_panel(ax[1, 1], fedd, [1e-4, FEDD_EFF, FEDD_HIGH, 1.0],
               r"$f_{\rm Edd}$ (leftmost = no/undetected BH)",
               [(FEDD_EFF, FEDD_HIGH, C_MID), (FEDD_HIGH, 1.0, C_HI)], vline=JET_FULL)
    ax[1, 1].text(JET_FULL, 0.93, " full jets", color=C_JET, fontsize=FS_LEG - 2, ha="left")
    _det_panel(ax[1, 2], Lbol, [1e40, LBOL_MOD, 1e43, LBOL_STRONG, 1e46],
               r"$L_{\rm bol}=0.1\,\dot M_{\rm BH}c^2$ $\rm[erg\,s^{-1}]$ (intrinsic)",
               [(LBOL_MOD, LBOL_STRONG, C_MID), (LBOL_STRONG, 1e47, C_HI)])
    for a_ in ax.ravel():
        a_.tick_params(labelsize=FS_TICK - 2, which="both")
    fig.suptitle(f"Molecular gas vs AGN in quiescent group galaxies at z={Z_O:.2f} "
                 f"(logM* {OBS_LOGM}, logMh {OBS_LOGMH}, below MS) — "
                 r"Eddington (robust) vs $L_{\rm bol}$ (observable) classifications",
                 y=1.0, fontsize=FS_LAB - 2)
    plt.tight_layout()
    plt.savefig(figpath( "co_agn_groups.png"), dpi=160, bbox_inches="tight")
    plt.show()

plt.savefig(figpath('AGN_fraction_with_detection.png'))

In [ ]:
# 10d(a) · standalone: M_H2 vs sSFR in quiescent group galaxies, colored by Eddington class,
#          with a medians±IQR inset. Enlarged labels/numbers for a single-panel figure.
#   Needs §0 (sim), §1c (snaps_arr, redshift, t_cosmic_yr).

Z_TARGET_OBS = 0.3
OBS_LOGM     = (10.0, 12.0)
OBS_LOGMH    = (13.0, 14.5)
OBS_LSSFR_W  = (-2.0, -1.0)
MH2_DET      = 1e9
FEDD_EFF, FEDD_HIGH = 0.01, 0.1
JET_MBH, JET_FE     = 10**7.5, 0.2
DORM_LBOL = 1e40                    # 'not acting': L_bol = 0.1*Mdot*c^2 below this [erg/s]
C_LOW, C_MID, C_HI, C_JET = "0.45", "#FF8C00", "#C2185B", "#0173B2"
C_DORM = "#7570B3"

# generous fonts for a single big panel
FS_LAB, FS_TICK, FS_LEG = 30, 24, 21
FS_IN_TICK, FS_IN_TITLE = 17, 18

_PSTYLE = globals().get("_PSTYLE") or {
    "font.family": "STIXGeneral", "mathtext.fontset": "stix", "axes.linewidth": 1.6,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "xtick.major.size": 9, "ytick.major.size": 9, "xtick.minor.size": 4.5, "ytick.minor.size": 4.5}

_it = int(np.argmin(np.abs(redshift - Z_TARGET_OBS)))
SNAP_O, Z_O = int(snaps_arr[_it]), float(redshift[_it])
print(f"snapshot {SNAP_O} at z={Z_O:.3f} (target {Z_TARGET_OBS})")

# _gfield2 / _lim10 / _mbx: defined in the §10d cell above (this cell reuses them)
with h5py.File(sim.get_caesar_file(SNAP_O), "r") as f:
    ms   = _gfield2(f, "masses.stellar"); sf = _gfield2(f, "sfr")
    mh2  = _gfield2(f, "masses.H2")
    bhfe = _gfield2(f, ["bh_fedd", "bhfedd", "fedd"])
    bhmd = _gfield2(f, ["bhmdot", "bh_mdot"])
    bhm  = _gfield2(f, ["masses.bh", "masses.bh_mass", "bhmass"])
    par  = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
    mh_h = _gfield2(f, "masses.total", grp="halo_data")
assert all(a is not None for a in (ms, sf, mh2, mh_h, bhfe, bhm, bhmd)), "missing catalogue fields"

mh = np.where(par >= 0, mh_h[np.clip(par, 0, len(mh_h) - 1)], np.nan)
with np.errstate(all="ignore"):
    lm     = np.log10(np.where(ms > 0, ms, np.nan))
    lmh    = np.log10(np.where(mh > 0, mh, np.nan))
    lssfr  = np.log10(np.where((ms > 0) & (sf > 0), sf / ms * 1e9, np.nan))
    ssfr_l = np.where((ms > 0) & (sf > 0), sf / ms * 1e9, np.nan)
    fedd   = np.where(bhfe > 0, bhfe, np.nan)
    Lbol_a = 0.1 * np.where(bhmd > 0, bhmd, np.nan) * (const.c ** 2 * u.Msun / u.yr).to(u.erg / u.s).value

base    = ((lm >= OBS_LOGM[0]) & (lm <= OBS_LOGM[1])
           & (lmh >= OBS_LOGMH[0]) & (lmh <= OBS_LOGMH[1]))
belowMS = base & (np.isnan(lssfr) | (lssfr < OBS_LSSFR_W[1]))
dormant = np.isfinite(bhm) & (bhm > 0) & ~(Lbol_a > DORM_LBOL)   # BH present but not acting
is_jet  = np.isfinite(bhm) & np.isfinite(bhfe) & (bhm > JET_MBH) & (bhfe < JET_FE)
e_low = ~dormant & (~np.isfinite(fedd) | (fedd < FEDD_EFF))      # weakly accreting or no BH
e_mid = np.isfinite(fedd) & (fedd >= FEDD_EFF) & (fedd < FEDD_HIGH)
e_hi  = np.isfinite(fedd) & (fedd >= FEDD_HIGH)
ECLS = [(r"dormant ($L_{\rm bol}<10^{40}$)", dormant, C_DORM, "x"),
        (r"inefficient ($\lambda<0.01$)", e_low, C_LOW, "o"),
        (r"efficient ($0.01 < \lambda < 0.1$)",     e_mid, C_MID, "^"),
        (r"high ($\lambda\geq0.1$)",      e_hi,  C_HI,  "D")]
print(f"below-MS analog: N={int(belowMS.sum())} | dormant: {int((belowMS & dormant).sum())} | "
      f"jet-eligible: {int((belowMS & is_jet).sum())} "
      f"(accreting {int((belowMS & is_jet & ~dormant).sum())}, "
      f"dormant {int((belowMS & is_jet & dormant).sum())})")


def _agn_frac_profile(ax, vals, sample, groups, nbins=7, nmin=10, horizontal=False, span_edges=False):
    """Fraction of `sample` in each (label, mask, color) group, in log-quantile bins of
       `vals` (Wilson 68% bars). horizontal=True puts the fraction on x and `vals` on y.
       span_edges=True anchors the first/last bin markers at the data extremes so the
       fraction lines reach the axis edges instead of stopping at the inner bin medians."""
    with np.errstate(all="ignore"):
        g = sample & np.isfinite(vals) & (vals > 0)
        lv = np.where(g, np.log10(np.where(vals > 0, vals, np.nan)), np.nan)
    eb = np.nanquantile(lv[g], np.linspace(0, 1, nbins + 1))
    for lab, gm, col in groups:
        xc, pp, lo, hi = [], [], [], []
        for b in range(nbins):
            mb_ = g & (lv >= eb[b]) & ((lv < eb[b + 1]) if b < nbins - 1 else (lv <= eb[b + 1]))
            n = int(mb_.sum())
            if n < nmin:
                continue
            p, l, h = _wilson(int(np.sum(mb_ & gm)), n)
            x = (eb[0] if (span_edges and b == 0)
                 else eb[nbins] if (span_edges and b == nbins - 1)
                 else np.median(lv[mb_]))
            xc.append(x); pp.append(p); lo.append(p - l); hi.append(h - p)
        if not xc:
            continue
        xv = 10.0 ** np.asarray(xc)
        if horizontal:
            ax.errorbar(pp, xv, xerr=[lo, hi], fmt="o-", ms=5, lw=1.8, capsize=2.5,
                        color=col, label=lab)
        else:
            ax.errorbar(xv, pp, yerr=[lo, hi], fmt="o-", ms=5, lw=1.8, capsize=2.5,
                        color=col, label=lab)

with plt.rc_context(_PSTYLE):
    fig = plt.figure(figsize=(13.5, 11.5))
    _gsA = fig.add_gridspec(2, 2, width_ratios=[4.0, 1.2], height_ratios=[1.2, 4.0],
                            hspace=0.05, wspace=0.05)
    ax0 = fig.add_subplot(_gsA[1, 0])
    axt = fig.add_subplot(_gsA[0, 0], sharex=ax0)   # fractions vs sSFR
    axr = fig.add_subplot(_gsA[1, 1], sharey=ax0)   # fractions vs M_H2
    m0 = belowMS & (mh2 > 0)
    for lab, flag, col, mk in ECLS:
        mm = m0 & flag
        if mk == "x":                      # dormant: BH present, no feedback acting
            ax0.scatter(ssfr_l[mm], mh2[mm], s=70, marker=mk, color=col,
                        linewidths=1.6, alpha=0.85, label=f"{lab} (N={int(mm.sum())})")
        elif lab.startswith("inefficient"):
            ax0.scatter(ssfr_l[mm], mh2[mm], s=24, marker=mk, facecolors="none",
                        edgecolors="k", linewidths=0.7, alpha=0.45, rasterized=True,
                       label=f"Inefficient ($\lambda < 0.01$) (N={int(mm.sum())})")
        else:
            ax0.scatter(ssfr_l[mm], mh2[mm], s=140, marker=mk, color=col,
                        edgecolors="k", linewidths=1.3, alpha=0.9,
                        label=f"{lab} (N={int(mm.sum())})")
    jm = m0 & is_jet & ~dormant
    ax0.scatter(ssfr_l[jm], mh2[jm], s=160, marker="s", facecolors="none",
                edgecolors=C_JET, linewidths=1.9, label=f"jet mode, accreting (N={int(jm.sum())})")
    ax0.set_xscale("log"); ax0.set_yscale("log")
    ax0.axvspan(10 ** OBS_LSSFR_W[0], 10 ** OBS_LSSFR_W[1], color="gold", alpha=0.12)
    ax0.axhline(MH2_DET, ls="--", c="k", lw=1.4)
    xl = _lim10(ssfr_l[m0]); yl = _lim10(mh2[m0])
    if xl: ax0.set_xlim(*xl)
    if yl: ax0.set_ylim(*yl)
    ax0.set_xlabel(r"sSFR $\rm[Gyr^{-1}]$", fontsize=FS_LAB)
    ax0.set_ylabel(r"$M_{\rm H_2}\;[M_\odot]$", fontsize=FS_LAB)
    ax0.tick_params(labelsize=FS_TICK, which="both")
    ax0.legend(fontsize=FS_LEG, loc="upper left", framealpha=0.95, edgecolor="k")

    axin = ax0.inset_axes([0.56, 0.05, 0.43, 0.40])
    axin.set_xscale("log"); axin.set_yscale("log")
    _ixe, _iye = [], []
    for lab, flag, col, _mk in ECLS:
        mm = m0 & flag
        mb = _mbx(np.log10(ssfr_l[mm]), np.log10(mh2[mm]), band=(16, 86))
        if mb is None: continue
        _ixe += [float(mb[0].min()), float(mb[0].max())]
        _iye += [float(mb[2].min()), float(mb[3].max())]
        xc, med, q1, q3 = (10 ** v for v in mb)
        axin.plot(xc, med, "-", color=("k" if col == C_LOW else col), lw=2.8)
        #axin.axhline(MH2_DET, ls="--", c="k", lw=1.4)
        axin.fill_between(xc, q1, q3, color=("k" if col == C_LOW else col), alpha=0.15, lw=0)
    if _ixe:
        sp = max(max(_ixe) - min(_ixe), 1e-6)
        axin.set_xlim(10 ** (min(_ixe) - 0.08 * sp), 10 ** (max(_ixe) + 0.08 * sp))
        sp = max(max(_iye) - min(_iye), 1e-6)
        axin.set_ylim(10 ** (min(_iye) - 0.10 * sp), 10 ** (max(_iye) + 0.10 * sp))
    axin.tick_params(which="both", direction="in", labelsize=FS_IN_TICK)
    axin.tick_params(axis="x", pad=-30)            # tick numbers INSIDE the inset
    axin.tick_params(axis="y", pad=-14)
    for _t in axin.get_yticklabels():
        _t.set_ha("left")
    axin.xaxis.set_minor_formatter(plt.NullFormatter())
    axin.yaxis.set_minor_formatter(plt.NullFormatter())
    axin.set_title("medians ± IQR", fontsize=FS_IN_TITLE)

    # marginal AGN-population fractions along each axis (overlapping classes -> lines)
    GROUPS_FR = [("jets (radio-loud)",    is_jet & ~dormant & np.isfinite(fedd) & ~(fedd >= FEDD_EFF),                                  C_JET),
                 ("rad. efficient (BPT)", np.isfinite(fedd) & (fedd >= FEDD_EFF) & ~dormant,  C_MID),
                 ("dormant",              dormant,                                            C_DORM)]
    _agn_frac_profile(axt, ssfr_l, m0, GROUPS_FR)
    axt.set_ylim(0, 1); axt.set_yticks([0, 0.5, 1])
    axt.set_ylabel("fraction", fontsize=FS_LEG)
    axt.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axt.get_xticklabels(), visible=False)
    axt.legend(fontsize=FS_IN_TICK, loc="upper left", framealpha=0.9)
    _agn_frac_profile(axr, mh2, m0, GROUPS_FR, horizontal=True)
    axr.set_xlim(0, 1); axr.set_xticks([0, 0.5, 1])
    axr.set_xlabel("fraction", fontsize=FS_LEG)
    axr.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axr.get_yticklabels(), visible=False)

    fig.suptitle(f"Molecular gas vs sSFR — quiescent group galaxies at z={Z_O:.2f}",
                 fontsize=FS_LAB - 6, y=0.985)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.savefig(figpath( "co_agn_groups_panelA.png"), dpi=160, bbox_inches="tight")
    plt.show()

### VII.3 (§10e) · Is the z ≈ 0.3 jet phase *steady*? AGN-regime persistence around the observational snapshot

§10d classifies the quiescent group sample by a **single-snapshot** $f_{\rm Edd}$ — one random
draw from an accretion state that flickers far below the snapshot cadence. And the per-galaxy
post-QT interval (§8n) is the wrong frame here: its length varies wildly across the sample and
can be very short. So this cell asks the §8n question in a **fixed observer-frame window**
around the §10d snapshot (`SNAP_O`, z ≈ 0.3): each quiescent group galaxy is tracked
±`W_BACK/W_FWD` snapshots through the caesar tree (`progen_galaxy_star`: backward directly,
forward by inverting it), and its **duty fraction** in each regime is measured over the window:

- **jet mode** ($M_{\rm BH}>10^{7.5}$, $f_{\rm Edd}<0.2$) — CGM heating;
- **X-ray-coupled** (full jets $f_{\rm Edd}<0.02$ **and** $f_{\rm gas}<0.2$) — the only
  ISM/gas-removing channel.

A galaxy that is jet-classified at `SNAP_O` but has a low window duty fraction is in a
**temporary jet phase** — no steady CGM heating despite its snapshot label. Duty classes:
steady (≥ `STEADY_FRAC`), intermittent, temporary (< `TEMP_FRAC`). Both estimators are used:
instantaneous $f_{\rm Edd}$ and the merger-safe integrated $\langle f_{\rm Edd}\rangle$ from BH
mass growth (§8n; merger intervals masked by the M*-budget jump criterion, no merger flags).
The cell also re-tests the §10d CO–AGN contingency with the *steady-jet* label in place of the
single-snapshot one — if the trend survives, it is not single-snapshot noise.

Caveat: the window (~1 Gyr) still cannot see the $10^4$–$10^5$ yr flicker; the duty fraction is
itself estimated from ~9 draws. But 9 draws distinguish "almost always in jet mode" from "caught
once in a transient dip", which is exactly the distinction the CO–AGN test needs.


In [ ]:
# 10e · is the z~0.3 jet phase steady? AGN-regime persistence in a window around SNAP_O.
#   Tracks the §10d group sample (quiescent subset via GQUI) over SNAP_O-W_BACK .. SNAP_O+W_FWD via the caesar
#   tree, then measures jet / X-ray-coupled duty fractions — instantaneous f_Edd AND the
#   merger-safe integrated <f_Edd> (§8n estimator: M*-budget merger proxy, no merger flags).
#   Needs §10d (SNAP_O, OBS_*, MH2_DET, JET_MBH/JET_FE/JET_FULL, _gfield2), §0, §1c, §1 (EPS_R).
W_BACK, W_FWD = 4, 4
MIN_VALID_W   = 5            # min valid window epochs to classify a galaxy
STEADY_FRAC   = 0.75         # duty >= this -> steady ; < TEMP_FRAC -> temporary
TEMP_FRAC     = 0.25
XRAY_FGAS_W      = globals().get("XRAY_FGAS_MAX", 0.2)      # §8j value if Part V ran
MERGER_SF_FRAC_W = globals().get("MERGER_SF_FRAC", 0.20)    # §8n values if Part V ran
MERGER_DM_MIN_W  = globals().get("MERGER_DM_MIN", 0.02)

# --- window snapshots + per-snap catalogue columns (light read) ---
WSNAPS, CATW = [], {}
for _s in range(SNAP_O - W_BACK, SNAP_O + W_FWD + 1):
    _fn = sim.get_caesar_file(_s)
    if _s in CORRUPT_SNAPS or not os.path.exists(_fn):
        print(f"  [10e] snap {_s}: no usable catalogue -> skipped"); continue
    with h5py.File(_fn, "r") as f:
        CATW[_s] = dict(
            ms=_gfield2(f, "masses.stellar"), sf=_gfield2(f, "sfr"), mg=_gfield2(f, "masses.gas"),
            mbh=_gfield2(f, ["masses.bh", "masses.bh_mass", "bhmass"]),
            fe=_gfield2(f, ["bh_fedd", "bhfedd", "fedd"]),
            pro=(np.asarray(f["tree_data/progen_galaxy_star"][:, 0]) if "tree_data" in f else None))
        if _s == SNAP_O:
            CATW[_s]["par"]  = np.asarray(f["galaxy_data/parent_halo_index"][:], dtype=np.int64)
            CATW[_s]["mh_h"] = _gfield2(f, "masses.total", grp="halo_data")
            CATW[_s]["mh2"]  = _gfield2(f, "masses.H2")
    WSNAPS.append(_s)
WSNAPS = np.asarray(WSNAPS)
_zw  = np.array([float(sim.get_z_from_snap(int(s))) for s in WSNAPS])
TWYR = COSMO.age(_zw).value * 1e9
_i0  = int(np.where(WSNAPS == SNAP_O)[0][0])
print(f"window: snaps {WSNAPS[0]}-{WSNAPS[-1]} (z {_zw[0]:.3f}-{_zw[-1]:.3f}), "
      f"span {(TWYR[-1] - TWYR[0]) / 1e9:.2f} Gyr, median dt {np.median(np.diff(TWYR)):.2e} yr")

# --- the §10d group sample (full mass/halo window), recomputed locally at SNAP_O ---
_c0 = CATW[SNAP_O]
with np.errstate(all="ignore"):
    _lm  = np.log10(np.where(_c0["ms"] > 0, _c0["ms"], np.nan))
    _mhw = np.where(_c0["par"] >= 0, _c0["mh_h"][np.clip(_c0["par"], 0, len(_c0["mh_h"]) - 1)], np.nan)
    _lmh = np.log10(np.where(_mhw > 0, _mhw, np.nan))
    _lss = np.log10(np.where((_c0["ms"] > 0) & (_c0["sf"] > 0), _c0["sf"] / _c0["ms"] * 1e9, np.nan))
# Track the FULL group sample (mass/halo window) through the window so the integrated <f_Edd>
# exists for EVERY group galaxy (10e(b) flags persistent jets at any sSFR, not just below 1/t_H).
# GQUI is the below-MS (quiescent) subset and is re-applied to every quiescent-specific result
# below, so this cell's duty / CO-AGN analysis is identical to the GSEL=quiescent version.
_grp_w = ((_lm >= OBS_LOGM[0]) & (_lm <= OBS_LOGM[1])
          & (_lmh >= OBS_LOGMH[0]) & (_lmh <= OBS_LOGMH[1]))       # full group sample (no sSFR cut)
GSEL  = np.where(_grp_w)[0]
GQUI  = np.isnan(_lss[GSEL]) | (_lss[GSEL] < OBS_LSSFR_W[1])       # below-MS (quiescent) subset of GSEL
CO_W  = _c0["mh2"][GSEL] > MH2_DET

# --- track the sample through the window (backward: direct; forward: invert progen) ---
IDXW = np.full((len(WSNAPS), len(GSEL)), -1, dtype=np.int64)
IDXW[_i0] = GSEL
for k in range(_i0, 0, -1):                          # backward  WSNAPS[k] -> WSNAPS[k-1]
    pro = CATW[int(WSNAPS[k])]["pro"]
    if pro is None or WSNAPS[k] - WSNAPS[k - 1] != 1:
        print(f"  [10e] no tree link {WSNAPS[k]}->{WSNAPS[k-1]}; backward chain stops"); break
    cur = IDXW[k]
    IDXW[k - 1] = np.where(cur >= 0, pro[np.clip(cur, 0, len(pro) - 1)], -1)
for k in range(_i0, len(WSNAPS) - 1):                # forward   WSNAPS[k] -> WSNAPS[k+1]
    _sn = int(WSNAPS[k + 1]); pro = CATW[_sn]["pro"]
    if pro is None or WSNAPS[k + 1] - WSNAPS[k] != 1:
        print(f"  [10e] no tree link {WSNAPS[k]}->{_sn}; forward chain stops"); break
    msn   = CATW[_sn]["ms"]
    order = np.argsort(pro, kind="stable"); pros = pro[order]
    cur, out = IDXW[k], np.full(len(GSEL), -1, np.int64)
    for j, c in enumerate(cur):                      # descendant: most massive galaxy at k+1
        if c < 0: continue                           #   whose most-massive progenitor is ours
        lo, hi = np.searchsorted(pros, c), np.searchsorted(pros, c, side="right")
        if hi > lo:
            cand = order[lo:hi]
            out[j] = int(cand[np.argmax(msn[cand])])
    IDXW[k + 1] = out
ntrk = (IDXW >= 0).sum(axis=0)
print(f"sample: {len(GSEL)} group galaxies at snap {SNAP_O} ({int(GQUI.sum())} below-MS/quiescent); "
      f"tracked over the full window: {int((ntrk == len(WSNAPS)).sum())} "
      f"({100 * np.mean(ntrk == len(WSNAPS)):.0f}%)")

def _gw(key):
    out = np.full(IDXW.shape, np.nan)
    for k, s in enumerate(WSNAPS):
        a = CATW[int(s)][key]; m = IDXW[k] >= 0
        out[k, m] = a[IDXW[k, m]]
    return out
MSW, SFW, MGW, BHW, FEW = (_gw(k) for k in ("ms", "sf", "mg", "mbh", "fe"))
with np.errstate(all="ignore"):
    FGW = np.where(MSW > 0, MGW / MSW, np.nan)

# --- merger-safe integrated <f_Edd> (same estimator as §8n) ---
_dtw = np.diff(TWYR)
with np.errstate(all="ignore"):
    _dms  = MSW[1:] - MSW[:-1]
    _sfex = SFW[:-1] * _dtw[:, None]
    merger_w = (np.isfinite(_dms) & np.isfinite(_sfex)
                & (_dms > MERGER_DM_MIN_W * MSW[:-1]) & (_sfex < MERGER_SF_FRAC_W * _dms))
    _dbh  = BHW[2:] - BHW[:-2]
    _badw = merger_w[:-1] | merger_w[1:] | ~np.isfinite(_dbh) | (_dbh < 0)
    _kE_w = float((4 * np.pi * const.G * const.m_p
                   / (EPS_R * const.sigma_T * const.c)).to(1 / u.yr).value)
    FEW_INT = np.full_like(FEW, np.nan)
    FEW_INT[1:-1] = (np.where(_badw, np.nan, _dbh) / (TWYR[2:] - TWYR[:-2])[:, None]
                     / (1 - EPS_R)) / (_kE_w * BHW[1:-1])
print(f"merger-suspect intervals in window: {int(merger_w.sum())} "
      f"({100 * merger_w.sum() / max(merger_w.size, 1):.1f}%)")

# --- regimes + per-galaxy duty fractions over the window ---
def _regw(fe):
    ok  = np.isfinite(BHW) & np.isfinite(fe) & np.isfinite(FGW)
    jet = ok & (BHW > JET_MBH) & (fe < JET_FE)
    xr  = jet & (fe < JET_FULL) & (FGW < XRAY_FGAS_W)
    return ok, jet, xr

DUTY = {}
for _nm, _fe in [("instantaneous", FEW), ("integrated", FEW_INT)]:
    ok, jet, xr = _regw(_fe)
    nv = ok.sum(axis=0).astype(float)
    DUTY[_nm] = dict(nv=nv, ok=ok, jetH=jet, xrH=xr,
                     jet=np.where(nv > 0, jet.sum(0) / nv, np.nan),
                     xr=np.where(nv > 0, xr.sum(0) / nv, np.nan))
jet_now = DUTY["instantaneous"]["jetH"][_i0]         # the §10d-style single-snapshot label
xr_now  = DUTY["instantaneous"]["xrH"][_i0]

print(f"\nAT snap {SNAP_O} (quiescent subset): jet-mode {int((jet_now & GQUI).sum())}/{int(GQUI.sum())}, "
      f"X-ray-coupled {int((xr_now & GQUI).sum())}/{int(GQUI.sum())} (single-snapshot labels)")
print(f"window persistence of the snap-{SNAP_O} jet class "
      f"(steady>={STEADY_FRAC:.0%}, temporary<{TEMP_FRAC:.0%} of valid epochs):")
for _nm in ("instantaneous", "integrated"):
    D = DUTY[_nm]; m = jet_now & (D["nv"] >= MIN_VALID_W) & GQUI
    if m.sum() == 0:
        print(f"  {_nm:13s}: no classifiable galaxies"); continue
    f = D["jet"][m]
    print(f"  {_nm:13s}: N={int(m.sum()):3d} | steady {np.mean(f >= STEADY_FRAC):5.0%} | "
          f"intermittent {np.mean((f >= TEMP_FRAC) & (f < STEADY_FRAC)):5.0%} | "
          f"temporary {np.mean(f < TEMP_FRAC):5.0%} | median duty {np.median(f):.2f}")
print("X-ray-coupled (gas-removing) duty over the quiescent subset:")
for _nm in ("instantaneous", "integrated"):
    D = DUTY[_nm]; m = (D["nv"] >= MIN_VALID_W) & GQUI
    print(f"  {_nm:13s}: median {np.nanmedian(D['xr'][m]):.2f} | "
          f"steady fraction {np.mean(D['xr'][m] >= STEADY_FRAC):.0%} (N={int(m.sum())})")

# --- does the §10d CO-AGN contingency survive the steady-jet label? ---
D = DUTY["instantaneous"]
_clw = np.where(D["nv"] >= MIN_VALID_W,
                np.where(D["jet"] >= STEADY_FRAC, 2, np.where(D["jet"] < TEMP_FRAC, 0, 1)), -1)
print("\nCO detection vs jet label (quiescent group sample):")
for lab, flag in [("single-snapshot jet", jet_now & (_clw >= 0)), ("steady jet (window)", _clw == 2)]:
    cl = (_clw >= 0) & GQUI
    a = int(np.sum(cl & CO_W & flag)); b = int(np.sum(cl & CO_W & ~flag))
    c = int(np.sum(cl & ~CO_W & flag)); d = int(np.sum(cl & ~CO_W & ~flag))
    if min(a + c, b + d) == 0: continue
    _, pv = fisher_exact([[a, b], [c, d]])
    print(f"  {lab:22s}: CO-det {a}/{a + c} with vs {b}/{b + d} without  (Fisher p={pv:.3f})")

# --- figure: duty histogram | population regime fraction per snapshot | CO by duty class ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4.6))
m = jet_now & (DUTY["instantaneous"]["nv"] >= MIN_VALID_W) & GQUI
_bins = np.linspace(0, 1, 10)
ax1.hist(DUTY["instantaneous"]["jet"][m], bins=_bins, color="C0", alpha=0.65, label="instantaneous")
mi = jet_now & (DUTY["integrated"]["nv"] >= MIN_VALID_W) & GQUI
ax1.hist(DUTY["integrated"]["jet"][mi], bins=_bins, histtype="step", lw=2, color="C3", label="integrated")
for thr in (TEMP_FRAC, STEADY_FRAC):
    ax1.axvline(thr, ls=":", c="k", lw=1)
ax1.set_xlabel("jet-mode duty fraction over the window")
ax1.set_ylabel("N galaxies"); ax1.legend(fontsize=9)
ax1.set_title(f"galaxies jet-classified at snap {SNAP_O}")

ok, jet, xr = _regw(FEW)                              # restrict the population fractions to GQUI
_nok = np.maximum(ok[:, GQUI].sum(axis=1), 1)
ax2.plot(_zw, jet[:, GQUI].sum(1) / _nok, "-o", c="C0", label="jet mode")
ax2.plot(_zw, xr[:, GQUI].sum(1) / _nok, "-s", c="C3", label="X-ray-coupled")
ax2.axvline(Z_O, ls=":", c="k", lw=1)
ax2.invert_xaxis(); ax2.set_xlabel("z"); ax2.set_ylabel("fraction of tracked sample")
ax2.set_ylim(0, 1); ax2.legend(fontsize=9)
ax2.set_title("population regime fraction across the window")

_dlab = ["temporary", "intermittent", "steady"]
for di, dc in enumerate((0, 1, 2)):
    mm = (_clw == dc) & GQUI
    n = int(mm.sum())
    if n == 0: continue
    p, lo, hi = _wilson(int(np.sum(mm & CO_W)), n)
    ax3.errorbar(di, p, yerr=[[p - lo], [hi - p]], fmt="o", ms=8, capsize=4, color="C2")
    ax3.annotate(f"n={n}", (di, p), textcoords="offset points", xytext=(8, 4), fontsize=9)
ax3.set_xticks(range(3)); ax3.set_xticklabels(_dlab)
ax3.set_xlim(-0.5, 2.5); ax3.set_ylim(0, 1)
ax3.set_xlabel("jet duty class over the window"); ax3.set_ylabel(f"CO-detected fraction (M_H2>{MH2_DET:.0e})")
ax3.set_title("CO detection vs jet persistence")
plt.tight_layout()
plt.savefig(figpath( "agn_persistence_z03_window.png"), dpi=130, bbox_inches="tight")
plt.show()


#### VII.3a (§10e·a) · The 10d(a) panel with the **integrated** $\langle f_{\rm Edd}\rangle$ as AGN flag

Same sample, axes and styling as 10d(a) ($M_{\rm H_2}$ vs sSFR, quiescent group galaxies at
z ≈ 0.3, medians±IQR inset) — but the Eddington classes and the jet flag now come from the
merger-safe time-integrated $\langle f_{\rm Edd}\rangle$ of §10e (BH mass growth centered on
`SNAP_O`, merger intervals masked via the M*-budget proxy) instead of the instantaneous
catalogue value. Galaxies whose $\langle f_{\rm Edd}\rangle$ is undefined (merger-masked, track
lost at a neighbouring snapshot, or no BH) fall in the *inefficient/undefined* class — the same
convention 10d(a) uses for absent BHs; the printout reports how many. The class-migration
summary (instantaneous → integrated at the same snapshot) quantifies directly how much of the
10d(a) classification was single-snapshot noise.


In [ ]:
# 10e(a) · 10d(a) figure with INTEGRATED <f_Edd> (§10e) as the AGN classification.
#   Needs §10e (CATW, GSEL, FEW, FEW_INT, BHW, _i0, SNAP_O, Z_O), §10d (_lim10, _mbx),
#   10d(a) (FEDD_EFF/FEDD_HIGH, JET_MBH/JET_FE, MH2_DET, OBS_LSSFR_W, C_*, FS_*, _PSTYLE,
#   _agn_frac_profile, DORM_LBOL).
_c0e = CATW[SNAP_O]
_mse, _sfe, _mh2e = _c0e["ms"][GSEL], _c0e["sf"][GSEL], _c0e["mh2"][GSEL]
with np.errstate(all="ignore"):
    ssfrl_e   = np.where((_mse > 0) & (_sfe > 0), _sfe / _mse * 1e9, np.nan)   # sSFR [Gyr^-1]
    feddI     = FEW_INT[_i0]                       # integrated <f_Edd>, centered on SNAP_O
    feddI_pos = np.where(feddI > 0, feddI, np.nan)
    fedd0     = np.where(FEW[_i0] > 0, FEW[_i0], np.nan)
_mbhe = BHW[_i0]

# dormant: BH present but not acting — <L_bol> = EPS_R * <Mdot_acc> * c^2 below the floor.
# <f_Edd>==0 (BH gained no mass over the bracketing ~2 snapshot intervals) is *provably*
# dormant; NaN (merger-masked / lost track) stays in the undefined/inefficient class.
DORM_LBOL_I = globals().get("DORM_LBOL", 1e40)     # same floor as 10d(a) [erg/s]
_c2i = (const.c ** 2 * u.Msun / u.yr).to(u.erg / u.s).value
with np.errstate(all="ignore"):
    LbolI = EPS_R * (feddI * _kE_w * _mbhe) * _c2i
    Lbol0 = EPS_R * (FEW[_i0] * _kE_w * _mbhe) * _c2i           # instantaneous twin, same formula
dormantI = np.isfinite(_mbhe) & (_mbhe > 0) & np.isfinite(feddI) & ~(LbolI > DORM_LBOL_I)
dormant0 = np.isfinite(_mbhe) & (_mbhe > 0) & ~(Lbol0 > DORM_LBOL_I)   # off-BH (NaN L_bol) -> dormant, matching 10d(a)/10e(b)

m0_e = (_mh2e > 0) & GQUI                          # GSEL is now the FULL group sample -> take the quiescent subset
jetI = np.isfinite(_mbhe) & np.isfinite(feddI) & (_mbhe > JET_MBH) & (feddI < JET_FE)
jet0 = np.isfinite(_mbhe) & np.isfinite(FEW[_i0]) & (_mbhe > JET_MBH) & (FEW[_i0] < JET_FE)
eI_low = ~dormantI & (~np.isfinite(feddI_pos) | (feddI_pos < FEDD_EFF))
eI_mid = np.isfinite(feddI_pos) & (feddI_pos >= FEDD_EFF) & (feddI_pos < FEDD_HIGH)
eI_hi  = np.isfinite(feddI_pos) & (feddI_pos >= FEDD_HIGH)
ECLS_I = [(r"dormant ($\langle L_{\rm bol}\rangle<10^{40}$)",      dormantI, C_DORM, "x"),
          (r"inefficient ($\langle\lambda\rangle<0.01$)",          eI_low, C_LOW, "o"),
          (r"efficient ($0.01<\langle\lambda\rangle<0.1$)",        eI_mid, C_MID, "^"),
          (r"high ($\langle\lambda\rangle\geq0.1$)",               eI_hi,  C_HI,  "D")]

# class migration: instantaneous -> integrated, same snapshot, same galaxies
def _ecls3(fe):
    return np.where(~np.isfinite(fe) | (fe < FEDD_EFF), 0, np.where(fe < FEDD_HIGH, 1, 2))
_ci, _cg = _ecls3(fedd0[m0_e]), _ecls3(feddI_pos[m0_e])
print(f"integrated <f_Edd> defined for {int(np.isfinite(feddI[m0_e]).sum())}/{int(m0_e.sum())} "
      f"plotted galaxies (NaN = merger-masked / lost track / no BH)")
print(f"Eddington class unchanged inst->int: {np.mean(_ci == _cg):.0%} | "
      f"jet flag unchanged: {np.mean(jet0[m0_e] == jetI[m0_e]):.0%} "
      f"(jet: inst {int((m0_e & jet0).sum())} -> int {int((m0_e & jetI).sum())})")
print(f"dormant: inst {int((m0_e & dormant0).sum())} -> int {int((m0_e & dormantI).sum())} | "
      f"jet-eligible & dormant: inst {int((m0_e & jet0 & dormant0).sum())} -> "
      f"int {int((m0_e & jetI & dormantI).sum())}")
for a, b, lab in [(1, 0, "efficient->inefficient"), (2, 1, "high->efficient"),
                  (0, 1, "inefficient->efficient"), (1, 2, "efficient->high")]:
    n = int(np.sum((_ci == a) & (_cg == b)))
    if n: print(f"  moved {lab}: {n}")

with plt.rc_context(_PSTYLE):
    fig = plt.figure(figsize=(13.5, 11.5))
    _gsA = fig.add_gridspec(2, 2, width_ratios=[4.0, 1.2], height_ratios=[1.2, 4.0],
                            hspace=0.05, wspace=0.05)
    ax0 = fig.add_subplot(_gsA[1, 0])
    axt = fig.add_subplot(_gsA[0, 0], sharex=ax0)   # fractions vs sSFR
    axr = fig.add_subplot(_gsA[1, 1], sharey=ax0)   # fractions vs M_H2
    for lab, flag, col, mk in ECLS_I:
        mm = m0_e & flag
        if mk == "x":                      # dormant: BH present, no feedback acting
            ax0.scatter(ssfrl_e[mm], _mh2e[mm], s=70, marker=mk, color=col,
                        linewidths=1.6, alpha=0.85, label=f"{lab} (N={int(mm.sum())})")
        elif lab.startswith("inefficient"):
            ax0.scatter(ssfrl_e[mm], _mh2e[mm], s=24, marker=mk, facecolors="none",
                        edgecolors="k", linewidths=0.7, alpha=0.45, rasterized=True,
                        label=rf"Inefficient ($\langle\lambda\rangle < 0.01$) (N={int(mm.sum())})")
        else:
            ax0.scatter(ssfrl_e[mm], _mh2e[mm], s=140, marker=mk, color=col,
                        edgecolors="k", linewidths=1.3, alpha=0.9,
                        label=f"{lab} (N={int(mm.sum())})")
    jm = m0_e & jetI & ~dormantI
    ax0.scatter(ssfrl_e[jm], _mh2e[jm], s=160, marker="s", facecolors="none",
                edgecolors=C_JET, linewidths=1.9, label=f"jet mode, accreting (N={int(jm.sum())})")
    ax0.set_xscale("log"); ax0.set_yscale("log")
    ax0.axvspan(10 ** OBS_LSSFR_W[0], 10 ** OBS_LSSFR_W[1], color="gold", alpha=0.12)
    ax0.axhline(MH2_DET, ls="--", c="k", lw=1.4)
    xl = _lim10(ssfrl_e[m0_e]); yl = _lim10(_mh2e[m0_e])
    if xl: ax0.set_xlim(*xl)
    if yl: ax0.set_ylim(*yl)
    ax0.set_xlabel(r"sSFR $\rm[Gyr^{-1}]$", fontsize=FS_LAB)
    ax0.set_ylabel(r"$M_{\rm H_2}\;[M_\odot]$", fontsize=FS_LAB)
    ax0.tick_params(labelsize=FS_TICK, which="both")
    ax0.legend(fontsize=FS_LEG, loc="upper left", framealpha=0.95, edgecolor="k")

    axin = ax0.inset_axes([0.56, 0.05, 0.43, 0.40])
    axin.set_xscale("log"); axin.set_yscale("log")
    _ixe, _iye = [], []
    for lab, flag, col, _mk in ECLS_I:
        mm = m0_e & flag
        mb = _mbx(np.log10(ssfrl_e[mm]), np.log10(_mh2e[mm]), band=(16, 86))
        if mb is None: continue
        _ixe += [float(mb[0].min()), float(mb[0].max())]
        _iye += [float(mb[2].min()), float(mb[3].max())]
        xc, med, q1, q3 = (10 ** v for v in mb)
        axin.plot(xc, med, "-", color=("k" if col == C_LOW else col), lw=2.8)
        axin.fill_between(xc, q1, q3, color=("k" if col == C_LOW else col), alpha=0.15, lw=0)
    if _ixe:
        sp = max(max(_ixe) - min(_ixe), 1e-6)
        axin.set_xlim(10 ** (min(_ixe) - 0.08 * sp), 10 ** (max(_ixe) + 0.08 * sp))
        sp = max(max(_iye) - min(_iye), 1e-6)
        axin.set_ylim(10 ** (min(_iye) - 0.10 * sp), 10 ** (max(_iye) + 0.10 * sp))
    axin.tick_params(which="both", direction="in", labelsize=FS_IN_TICK)
    axin.tick_params(axis="x", pad=-30)
    axin.tick_params(axis="y", pad=-14)
    for _t in axin.get_yticklabels():
        _t.set_ha("left")
    axin.xaxis.set_minor_formatter(plt.NullFormatter())
    axin.yaxis.set_minor_formatter(plt.NullFormatter())
    axin.set_title("medians ± IQR", fontsize=FS_IN_TITLE)

    # marginal AGN-population fractions along each axis (overlapping classes -> lines)
    GROUPS_FR_I = [("jets (radio-loud)",    jetI & ~dormantI,                                                  C_JET),
                   ("rad. efficient (BPT)", np.isfinite(feddI_pos) & (feddI_pos >= FEDD_EFF) & ~dormantI,      C_MID),
                   ("dormant",              dormantI,                                                          C_DORM)]
    _agn_frac_profile(axt, ssfrl_e, m0_e, GROUPS_FR_I)
    axt.set_ylim(0, 1); axt.set_yticks([0, 0.5, 1])
    axt.set_ylabel("fraction", fontsize=FS_LEG)
    axt.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axt.get_xticklabels(), visible=False)
    axt.legend(fontsize=FS_IN_TICK, loc="upper left", framealpha=0.9)
    _agn_frac_profile(axr, _mh2e, m0_e, GROUPS_FR_I, horizontal=True)
    axr.set_xlim(0, 1); axr.set_xticks([0, 0.5, 1])
    axr.set_xlabel("fraction", fontsize=FS_LEG)
    axr.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axr.get_yticklabels(), visible=False)

    fig.suptitle(rf"Molecular gas vs sSFR — integrated $\langle f_{{\rm Edd}}\rangle$, z={Z_O:.2f}",
                 fontsize=FS_LAB - 6, y=0.985)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.savefig(figpath( "co_agn_groups_panelA_integrated.png"),
                dpi=160, bbox_inches="tight")
    plt.show()


#### VII.3b (§10e·b) · The 10d(a) panel with a **persistent** ($\langle\lambda\rangle$) jet flag — decoupled sample + reference lines

Same layout, colours and inset as **10d(a)**: the Eddington colour classes (dormant / inefficient /
efficient / high) are the **instantaneous**, single-snapshot $f_{\rm Edd}$. The change is the
**jet-mode points** (open blue squares), now flagged from the merger-safe **time-integrated
$\langle f_{\rm Edd}\rangle$** of §10e ($M_{\rm BH}>10^{7.5}$, $\langle f_{\rm Edd}\rangle<0.2$, BH
present, not $\langle\rangle$-dormant). A galaxy counts as jet-mode only if it sits in the
low-Eddington *preventive* regime **persistently across the window**, not because one snapshot caught
a transient dip.

The cell reads the `SNAP_O` catalogue directly, so the **plotted sample is decoupled from
`OBS_LSSFR_W`** — that window now *only* shades the gold observed-range band. Selection knobs:
`SAMPLE_MODE="below_ms"` keeps galaxies below the cut (`SSFR_CUT_HAND`, default
$1/t_H$ = `MS_FACTOR`/$t_H$, the MS boundary), while `"all"` plots every group source in the
$M_\star$/$M_{\rm halo}$ window. Three vertical reference lines mark, in sSFR: the empirical
**MS ridge** (median sSFR of star-forming galaxies, $\rm sSFR>1/t_H$; or `MS_SSFR_HAND`),
**$1/t_H$** (`MS_FACTOR`/$t_H$, the SF boundary) and **$0.2/t_H$** (the passive threshold). The
x-range is widened so all three lines and the band stay visible even when only quiescent galaxies are
plotted. The integrated jet flag is only defined for galaxies tracked through the §10e window (the
below-MS sample $\subset$ GSEL), so in `"all"` mode star-forming sources simply carry no jet square.

The printout checks whether the **"inactive"** selection ($L_{\rm bol}<10^{42}$, §10d) and the
**"dormant"** selection ($L_{\rm bol}<10^{40}$, 10d(a)) are mutually consistent: dormant should be a
strict subset of inactive among BH hosts (so *dormant & not-inactive* must be 0), and
*inactive & not-dormant* isolates the $10^{40}$–$10^{42}\,{\rm erg\,s^{-1}}$ band plus BH-less
galaxies. Both instantaneous and integrated $L_{\rm bol}$ are reported.


In [ ]:
# 10e(b) · 10d(a) LAYOUT (instantaneous Eddington colour classes), but JET-MODE points come from the
#   merger-safe time-integrated <f_Edd> (§10e) -> 'jet mode' = a *persistent* low-Eddington (preventive)
#   state, not a single-snapshot dip. Self-contained: reads the SNAP_O catalogue (CATW) directly so the
#   plotted SAMPLE is DECOUPLED from OBS_LSSFR_W (which now only shades the observed-range band). The
#   sample is 'below the MS' via the 1/t_H MS boundary (or a hand value), or ALL group sources; three
#   reference lines are drawn — MS ridge, 1/t_H, 0.2/t_H. Also tests whether 'inactive' (L_bol<1e42,
#   §10d) is consistent with 'dormant' (<1e40, 10d(a)).
#   Needs §10e (CATW, GSEL, FEW_INT, _i0, SNAP_O, Z_O, _kE_w), §0 (COSMO, const, u, PASSIVE_FACTOR),
#   §1 (EPS_R), 10d(a) (_lim10, _mbx, _agn_frac_profile, FEDD_EFF/HIGH, JET_MBH/JET_FE, MH2_DET,
#   OBS_LOGM/OBS_LOGMH/OBS_LSSFR_W, C_*, FS_*, _PSTYLE, DORM_LBOL).

# ---- knobs (independent of OBS_LSSFR_W, which now ONLY shades the observed-range band) ----
SAMPLE_MODE   = "all"          # "below_ms": keep galaxies below the cut; "all": every group source
SSFR_CUT_HAND = None                # log10 sSFR [Gyr^-1] hand cut for 'below MS'; None -> MS_FACTOR/t_H (MS boundary)
MS_SSFR_HAND  = None                # log10 sSFR [Gyr^-1] hand value for the MS-ridge line; None -> empirical
PASSIVE_FACTOR = globals().get("PASSIVE_FACTOR", 0.2)   # 0.2/t_H quiescent threshold (§0)
MS_FACTOR      = globals().get("MS_FACTOR", 1.0)        # 1.0/t_H star-forming/MS boundary (§10c)
LBOL_MOD       = globals().get("LBOL_MOD", 1e42)        # 'inactive' anchor (§10d, X-ray 1e42 convention)
DORM_LBOL_B    = globals().get("DORM_LBOL", 1e40)       # 'dormant' floor (10d(a))

# ---- full SNAP_O catalogue (already loaded into CATW by §10e) ----
_c0b = CATW[SNAP_O]
_msb, _sfb, _mh2b, _mbhb, _feb = _c0b["ms"], _c0b["sf"], _c0b["mh2"], _c0b["mbh"], _c0b["fe"]
_parb, _mhhb = _c0b["par"], _c0b["mh_h"]
_c2b = (const.c ** 2 * u.Msun / u.yr).to(u.erg / u.s).value
with np.errstate(all="ignore"):
    _mhb    = np.where(_parb >= 0, _mhhb[np.clip(_parb, 0, len(_mhhb) - 1)], np.nan)
    _lmb    = np.log10(np.where(_msb > 0, _msb, np.nan))
    _lmhb   = np.log10(np.where(_mhb > 0, _mhb, np.nan))
    ssfr_b  = np.where((_msb > 0) & (_sfb > 0), _sfb / _msb * 1e9, np.nan)   # sSFR [Gyr^-1]
    fedd0_b = np.where(_feb > 0, _feb, np.nan)                              # instantaneous f_Edd

# ---- tau-based reference sSFR lines [Gyr^-1]; tau = t_H(z) = age of Universe at SNAP_O ----
print("Current redshift:", Z_O)
tH_Gyr     = float(COSMO.age(Z_O).value)
ssfr_1tau  = MS_FACTOR / tH_Gyr                         # MS / star-forming boundary
ssfr_02tau = PASSIVE_FACTOR / tH_Gyr                    # passive (quiescent) threshold
base_b = ((_lmb >= OBS_LOGM[0]) & (_lmb <= OBS_LOGM[1])
          & (_lmhb >= OBS_LOGMH[0]) & (_lmhb <= OBS_LOGMH[1]))
if MS_SSFR_HAND is not None:
    ssfr_MS = 10.0 ** MS_SSFR_HAND
else:                                                   # empirical SF main-sequence ridge
    _sfms   = base_b & np.isfinite(ssfr_b) & (ssfr_b > ssfr_1tau)
    ssfr_MS = float(np.median(ssfr_b[_sfms])) if int(_sfms.sum()) >= 5 else ssfr_1tau

# ---- sample selection (decoupled from OBS_LSSFR_W) ----
_cut_b = 10.0 ** SSFR_CUT_HAND if SSFR_CUT_HAND is not None else ssfr_1tau
if SAMPLE_MODE == "all":
    samp_b = base_b.copy()
else:                                                   # below MS: below the passive cut (NaN sSFR = quenched)
    samp_b = base_b & (~np.isfinite(ssfr_b) | (ssfr_b < _cut_b))
m0_b = samp_b & (_mh2b > 0)
print(f"snap {SNAP_O} (z={Z_O:.3f}): t_H={tH_Gyr:.2f} Gyr | sSFR lines [Gyr^-1]: "
      f"MS ridge={ssfr_MS:.3f}, 1/t_H={ssfr_1tau:.3f}, 0.2/t_H={ssfr_02tau:.3f}")
print(f"SAMPLE_MODE='{SAMPLE_MODE}' (cut sSFR<{_cut_b:.3f} Gyr^-1): N={int(samp_b.sum())} group galaxies "
      f"| {int(m0_b.sum())} with M_H2>0")

# ---- integrated <f_Edd> (§10e) mapped onto the full catalogue via GSEL ----
feddI_b = np.full(len(_msb), np.nan)
feddI_b[GSEL] = FEW_INT[_i0]
with np.errstate(all="ignore"):
    LbolI_b = EPS_R * (feddI_b * _kE_w * _mbhb) * _c2b          # integrated <L_bol>
    Lbol0_b = EPS_R * (fedd0_b * _kE_w * _mbhb) * _c2b          # instantaneous L_bol (same formula)
dormantI_b = np.isfinite(_mbhb) & (_mbhb > 0) & np.isfinite(feddI_b) & ~(LbolI_b > DORM_LBOL_B)
dormant0_b = np.isfinite(_mbhb) & (_mbhb > 0) & ~(Lbol0_b > DORM_LBOL_B)   # off BH (f_Edd<=0 ->
#   NaN L_bol -> ~(NaN>floor)=True) counts as dormant, matching 10d(a); NO isfinite(fedd) gate.
jetI_b     = np.isfinite(_mbhb) & np.isfinite(feddI_b) & (_mbhb > JET_MBH) & (feddI_b < JET_FE)

# instantaneous Eddington colour classes (EXACTLY 10d(a))
e0_low_b = ~dormant0_b & (~np.isfinite(fedd0_b) | (fedd0_b < FEDD_EFF))
e0_mid_b = np.isfinite(fedd0_b) & (fedd0_b >= FEDD_EFF) & (fedd0_b < FEDD_HIGH)
e0_hi_b  = np.isfinite(fedd0_b) & (fedd0_b >= FEDD_HIGH)
ECLS_B = [(r"dormant ($L_{\rm bol}<10^{40}$)",  dormant0_b, C_DORM, "x"),
          (r"inefficient ($\lambda<0.01$)",     e0_low_b, C_LOW, "o"),
          (r"efficient ($0.01<\lambda<0.1$)",   e0_mid_b, C_MID, "^"),
          (r"high ($\lambda\geq0.1$)",          e0_hi_b,  C_HI,  "D")]

# ---- inactive (<1e42, incl. no BH) vs dormant (BH present & <1e40): are they consistent? ----
def _consistency_b(Lb, tag):
    inact = samp_b & (~np.isfinite(Lb) | (Lb < LBOL_MOD))
    dorm  = samp_b & np.isfinite(_mbhb) & (_mbhb > 0) & ~(Lb > DORM_LBOL_B)
    nobh  = samp_b & ~(np.isfinite(_mbhb) & (_mbhb > 0))
    both, in_nod   = int((inact & dorm).sum()), int((inact & ~dorm).sum())
    nin_d, neither = int((~inact & dorm).sum()), int((~inact & ~dorm).sum())
    ntot = int(samp_b.sum())
    print(f"  [{tag}] N={ntot} | inactive(<1e42)={int(inact.sum())} dormant(<1e40)={int(dorm.sum())} "
          f"(BH-less={int(nobh.sum())})")
    print(f"    dormant&inactive={both} | inactive&not-dormant={in_nod} (1e40<L_bol<1e42 or no BH) | "
          f"dormant&not-inactive={nin_d} (must be 0)")
    print(f"    same label for {(both + neither) / max(ntot, 1):.0%} | "
          f"dormant ⊆ inactive among BH hosts: {'YES' if nin_d == 0 else 'NO (!)'}")
print("inactive vs dormant consistency on the selected sample:")
_consistency_b(Lbol0_b, "instantaneous L_bol")
_consistency_b(LbolI_b, "integrated  <L_bol>")

with plt.rc_context(_PSTYLE):
    fig = plt.figure(figsize=(13.5, 11.5))
    _gsA = fig.add_gridspec(2, 2, width_ratios=[4.0, 1.2], height_ratios=[1.2, 4.0],
                            hspace=0.05, wspace=0.05)
    ax0 = fig.add_subplot(_gsA[1, 0])
    axt = fig.add_subplot(_gsA[0, 0], sharex=ax0)   # fractions vs sSFR
    axr = fig.add_subplot(_gsA[1, 1], sharey=ax0)   # fractions vs M_H2
    for lab, flag, col, mk in ECLS_B:
        mm = m0_b & flag
        if mk == "x":                      # dormant: BH present, no feedback acting
            ax0.scatter(ssfr_b[mm], _mh2b[mm], s=70, marker=mk, color=col,
                        linewidths=1.6, alpha=0.85, label=f"{lab} (N={int(mm.sum())})")
        elif lab.startswith("inefficient"):
            ax0.scatter(ssfr_b[mm], _mh2b[mm], s=24, marker=mk, facecolors="none",
                        edgecolors="k", linewidths=0.7, alpha=0.45, rasterized=True,
                        label=f"Inefficient ($\lambda < 0.01$) (N={int(mm.sum())})")
        else:
            ax0.scatter(ssfr_b[mm], _mh2b[mm], s=140, marker=mk, color=col,
                        edgecolors="k", linewidths=1.3, alpha=0.9, label=f"{lab} (N={int(mm.sum())})")
    jm = m0_b & jetI_b & ~dormantI_b
    ax0.scatter(ssfr_b[jm], _mh2b[jm], s=160, marker="s", facecolors="none",
                edgecolors=C_JET, linewidths=1.9,
                label=r"jet mode, persistent $\langle\lambda\rangle<0.2$ " + f"(N={int(jm.sum())})")
    ax0.set_xscale("log"); ax0.set_yscale("log")
    ax0.axvspan(10 ** OBS_LSSFR_W[0], 10 ** OBS_LSSFR_W[1], color="gold", alpha=0.12)  # observed range
    ax0.axhline(MH2_DET, ls="--", c="k", lw=1.4)
    # three sSFR reference lines (annotated, not in the legend) — MS ridge | 1/t_H | 0.2/t_H
    for xv, lab, ls in [(ssfr_MS, "MS ridge", "-"), (ssfr_1tau, r"$1/t_H$", "--"),
                        (ssfr_02tau, r"$0.2/t_H$", ":")]:
        ax0.axvline(xv, color="0.35", ls=ls, lw=1.8)
        axt.axvline(xv, color="0.35", ls=ls, lw=1.2)
        ax0.text(xv, 0.985, " " + lab, transform=ax0.get_xaxis_transform(), rotation=90,
                 va="top", ha="left", fontsize=FS_LEG - 5, color="0.30")
    # x-range: data range, but always wide enough to show the band and the three reference lines
    xl = _lim10(ssfr_b[m0_b]); yl = _lim10(_mh2b[m0_b])
    _xref = [ssfr_MS, ssfr_1tau, ssfr_02tau, 10 ** OBS_LSSFR_W[0], 10 ** OBS_LSSFR_W[1]]
    if xl:
        ax0.set_xlim(min(xl[0], min(_xref) * 0.8), max(xl[1], max(_xref) * 1.25))
    if yl: ax0.set_ylim(*yl)
    ax0.set_xlabel(r"sSFR $\rm[Gyr^{-1}]$", fontsize=FS_LAB)
    ax0.set_ylabel(r"$M_{\rm H_2}\;[M_\odot]$", fontsize=FS_LAB)
    ax0.tick_params(labelsize=FS_TICK, which="both")
    ax0.legend(fontsize=FS_LEG, loc="upper left", framealpha=0.95, edgecolor="k")

    axin = ax0.inset_axes([0.56, 0.05, 0.43, 0.40])
    axin.set_xscale("log"); axin.set_yscale("log")
    _ixe, _iye = [], []
    for lab, flag, col, _mk in ECLS_B:
        mm = m0_b & flag
        mb = _mbx(np.log10(ssfr_b[mm]), np.log10(_mh2b[mm]), band=(16, 86))
        if mb is None: continue
        _ixe += [float(mb[0].min()), float(mb[0].max())]
        _iye += [float(mb[2].min()), float(mb[3].max())]
        xc, med, q1, q3 = (10 ** v for v in mb)
        axin.plot(xc, med, "-", color=("k" if col == C_LOW else col), lw=2.8)
        axin.fill_between(xc, q1, q3, color=("k" if col == C_LOW else col), alpha=0.15, lw=0)
    if _ixe:
        sp = max(max(_ixe) - min(_ixe), 1e-6)
        axin.set_xlim(10 ** (min(_ixe) - 0.08 * sp), 10 ** (max(_ixe) + 0.08 * sp))
        sp = max(max(_iye) - min(_iye), 1e-6)
        axin.set_ylim(10 ** (min(_iye) - 0.10 * sp), 10 ** (max(_iye) + 0.10 * sp))
    axin.tick_params(which="both", direction="in", labelsize=FS_IN_TICK)
    axin.tick_params(axis="x", pad=-30)
    axin.tick_params(axis="y", pad=-14)
    for _t in axin.get_yticklabels():
        _t.set_ha("left")
    axin.xaxis.set_minor_formatter(plt.NullFormatter())
    axin.yaxis.set_minor_formatter(plt.NullFormatter())
    axin.set_title("medians ± IQR", fontsize=FS_IN_TITLE)

    # marginal fractions — jets = persistent (integrated <f_Edd>), efficient/dormant = instantaneous.
    # span_edges=True extends the first/last bins to the data range so the lines reach the axis edges.
    GROUPS_FR_B = [(r"jets, persistent $\langle\lambda\rangle$", jetI_b & ~dormantI_b,                       C_JET),
                   ("rad. efficient (BPT)", np.isfinite(fedd0_b) & (fedd0_b >= FEDD_EFF) & ~dormant0_b, C_MID),
                   ("dormant",              dormant0_b,                                                C_DORM)]
    _agn_frac_profile(axt, ssfr_b, m0_b, GROUPS_FR_B, span_edges=True)
    axt.set_ylim(0, 1); axt.set_yticks([0, 0.5, 1])
    axt.set_ylabel("fraction", fontsize=FS_LEG)
    axt.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axt.get_xticklabels(), visible=False)
    axt.legend(fontsize=FS_IN_TICK, loc="upper left", framealpha=0.9)
    _agn_frac_profile(axr, _mh2b, m0_b, GROUPS_FR_B, horizontal=True, span_edges=True)
    axr.set_xlim(0, 1); axr.set_xticks([0, 0.5, 1])
    axr.set_xlabel("fraction", fontsize=FS_LEG)
    axr.tick_params(labelsize=FS_IN_TICK, which="both")
    plt.setp(axr.get_yticklabels(), visible=False)

    _smode = "below MS" if SAMPLE_MODE != "all" else "all group sources"
    fig.suptitle(rf"Molecular gas vs sSFR — instantaneous classes, persistent $\langle f_{{\rm Edd}}\rangle$ jet flag "
                 rf"({_smode}, z={Z_O:.2f})", fontsize=FS_LAB - 9, y=0.985)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.savefig(figpath( "co_agn_groups_panelA_persistentjet.png"),
                dpi=160, bbox_inches="tight")
    plt.show()


# Appendix — superseded, negative & QC material

Per-galaxy QC tracks and diagnostics that were superseded by sharper readouts: example sSFR tracks, the pooled BH histories (shown degenerate), the M*–M_halo median grids (washed out the signal), and a standalone CGM-summary utility call.

In [ ]:
# 4e. Example sSFR tracks with critical points marked — rows = mass bin, cols = class
def show_examples(n_each=2):
    fig, axes = plt.subplots(NBINS, 2, figsize=(13, 4.0 * NBINS), sharex=True)
    axes = np.atleast_2d(axes)
    for bi in range(NBINS):
        binmask = mbin == bi
        for col_i, (cmask, title) in enumerate([(is_fast & binmask, LBL_A), (is_slow & binmask, LBL_B)]):
            ax = axes[bi, col_i]
            for i in np.where(cmask)[0][:n_each]:
                r = records[i]; col = r["col"]
                mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
                ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
                ax.plot(t_cosmic_yr / 1e9, ssfr, "-", lw=1, alpha=0.8, label=f"gid {r['gid']}")
                for st, mk in [("sft", "^"), ("qt", "v"), ("dust_peak", "*")]:
                    row = r[f"row_{st}"]
                    if row >= 0:
                        ax.scatter(t_cosmic_yr[row] / 1e9, ssfr[row], marker=mk, s=70, zorder=5)
            ax.set_yscale("log"); ax.set_title(f"{title} — logM* {MASS_BIN_LABELS[bi]}"); ax.legend(fontsize=7)
            if bi == NBINS - 1:
                ax.set_xlabel("cosmic time [Gyr]")
        axes[bi, 0].set_ylabel("sSFR [1/yr]")
    plt.tight_layout(); plt.show()
show_examples()

### A.1 (§4b-iii) · Full BH accretion / Eddington-ratio history — are fast & slow degenerate?

The stage-sampled view (§4b-i) only sees the anchors. Here we plot the **entire**
$\dot M_{\rm BH}(z)$ and $f_{\rm Edd}(z)$ histories — median with 25–75% band over all
fast vs all slow quenchers, vs **redshift** with a **cosmic-time** top axis — to see
whether the two populations actually trace *different* BH-growth paths or whether the
accretion history is **degenerate** between them. A KS test on the pooled per-snapshot
values and on the total integrated $E_{\rm AGN}$ quantifies any difference.

In [ ]:
def _med_band(arr2d, colsel, positive=True):
    sub = arr2d[:, colsel].astype(float)
    if positive:
        sub = np.where(sub > 0, sub, np.nan)
    with np.errstate(all="ignore"):
        return (np.nanmedian(sub, axis=1),
                np.nanpercentile(sub, 25, axis=1),
                np.nanpercentile(sub, 75, axis=1))

def _pooled(arr2d, colsel):
    v = arr2d[:, colsel].ravel()
    return v[np.isfinite(v) & (v > 0)]

def _total_Eagn(col):
    t = t_cosmic_yr[::-1]; md = BH['bh_mdot'][::-1, col]
    good = np.isfinite(md) & (md >= 0)
    if good.sum() < 2:
        return np.nan
    return EPS_R * np.trapz(md[good], t[good]) * c2_erg_per_Msun

# rows = [bh_mdot, f_Edd], cols = mass bins; fast vs slow median+IQR vs redshift
keys = [("bh_mdot", r"$\dot M_{\rm BH}$ [$M_\odot$ yr$^{-1}$]"), ("bh_fedd", r"$f_{\rm Edd}$")]
fig, axes = plt.subplots(2, NBINS, figsize=(5.0 * NBINS, 9), squeeze=False)
for ki, (key, lab) in enumerate(keys):
    for bi in range(NBINS):
        ax = axes[ki, bi]; binmask = mbin == bi
        for cls_mask, c, name in [(is_fast & binmask, COL_A, LBL_A), (is_slow & binmask, COL_B, LBL_B)]:
            colsel = cols[cls_mask]
            if len(colsel) == 0:
                continue
            med, lo, hi = _med_band(BH[key], colsel)
            ax.plot(redshift, med, "-", color=c, lw=2, label=f"{name} (n={len(colsel)})")
            ax.fill_between(redshift, lo, hi, color=c, alpha=0.15)
        ax.set_yscale("log"); ax.invert_xaxis(); ax.legend(fontsize=7)
        if ki == 0:
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}")
            secax = ax.secondary_xaxis("top", functions=(_z_to_t, _t_to_z)); secax.set_xlabel("t [Gyr]", fontsize=8)
        if ki == 1:
            ax.set_xlabel("redshift z")
    axes[ki, 0].set_ylabel("median " + lab)
fig.suptitle(f"BH accretion / Eddington history — {SPLIT_DESC}, by mass bin", y=1.01)
plt.tight_layout(); plt.savefig(figpath( "bh_accretion_history_fast_slow.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── quantify degeneracy, per mass bin ──────────────────────────────────────
for bi in range(NBINS):
    binmask = mbin == bi
    fast_cols = cols[is_fast & binmask]; slow_cols = cols[(is_slow) & binmask]
    print(f"--- logM* {MASS_BIN_LABELS[bi]} ({LBL_A} {len(fast_cols)}, {LBL_B} {len(slow_cols)}) ---")
    for key, lab in [("bh_mdot", "Mdot"), ("bh_fedd", "f_Edd")]:
        a = np.log10(_pooled(BH[key], fast_cols)); b = np.log10(_pooled(BH[key], slow_cols))
        if len(a) > 5 and len(b) > 5:
            ks = ks_2samp(a, b)
            print(f"  KS pooled log {lab:6s}: D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
                  f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")
    E_fast = np.array([_total_Eagn(c) for c in fast_cols]); E_fast = E_fast[np.isfinite(E_fast) & (E_fast > 0)]
    E_slow = np.array([_total_Eagn(c) for c in slow_cols]); E_slow = E_slow[np.isfinite(E_slow) & (E_slow > 0)]
    if len(E_fast) > 5 and len(E_slow) > 5:
        ks = ks_2samp(np.log10(E_fast), np.log10(E_slow))
        print(f"  KS total log E_AGN : D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
              f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")

### A.2 (§7k) · M\*–M_halo grids of dust fraction & BH accretion, fast vs slow

For each critical point listed in `GRID_STAGES` we make one figure with two side-by-side
subplots — **fast** and **slow** — that share the same $\log_{10} M_*$ (x) and
$\log_{10} M_{\rm halo}$ (y) ranges and the same colour scale. Every square marker is one
(M\*, M_halo) bin: its **colour** is the median quantity in the bin and its **size** scales
with the number of galaxies in the bin. Edit `GRID_STAGES` to choose which critical points to
show. Run once for dust fraction ($M_{\rm dust}/M_*$) and once for BH accretion ($\dot M_{\rm BH}$).

In [ ]:
# 7k. M*-Mhalo binned grids, fast vs slow as SEPARATE subplots (shared x/y ranges and
#     colour scale), one figure per chosen critical point. Colour = median quantity in each
#     (logM*, logMhalo) bin; marker size ∝ number of sources in the bin.

# ---- choose what to show -------------------------------------------------------------
GRID_STAGES = ["sft", "qt", "z0"]      # any subset of STAGES, in display order
NX = NY = 7                            # number of bins per axis
# --------------------------------------------------------------------------------------

def _stage_xy(st):
    """log10 M* (x) and log10 M_halo (y) at the stage anchor, aligned to records."""
    ms = DIAGS[st]["Mstar"]; mh = gather_stage("masses.total", st)
    return (np.log10(np.where(ms > 0, ms, np.nan)),
            np.log10(np.where(mh > 0, mh, np.nan)))

def _bin_grid(st, value_of, xe, ye, mask, log_color):
    """median value + source count per (M*, Mhalo) bin for the galaxies in `mask`."""
    x, y = _stage_xy(st)
    v = value_of(st).astype(float)
    v = np.where(v > 0, np.log10(v), np.nan) if log_color else v
    ix = np.digitize(x, xe) - 1; iy = np.digitize(y, ye) - 1
    good = (mask & np.isfinite(x) & np.isfinite(y) & np.isfinite(v)
            & (ix >= 0) & (ix < len(xe) - 1) & (iy >= 0) & (iy < len(ye) - 1))
    buck = defaultdict(list)
    for k in np.where(good)[0]:
        buck[(ix[k], iy[k])].append(v[k])
    med = {key: np.median(vals) for key, vals in buck.items()}
    cnt = {key: len(vals) for key, vals in buck.items()}
    return med, cnt

def mstar_mhalo_grids(value_of, vlabel, fname_prefix=None,
                      stages=None, nx=NX, ny=NY, cmap="viridis",
                      smin=24, smax=340, log_color=True):
    """One figure per chosen critical point; each figure = fast | slow subplots sharing the
       same x/y ranges and colour scale. value_of(stage) -> per-record array."""
    stages = stages or GRID_STAGES
    # shared bin edges + colour scale across the chosen stages and both classes
    xa = np.concatenate([_stage_xy(st)[0] for st in stages])
    ya = np.concatenate([_stage_xy(st)[1] for st in stages])
    xe = np.linspace(*np.nanpercentile(xa, [1, 99]), nx + 1)
    ye = np.linspace(*np.nanpercentile(ya, [1, 99]), ny + 1)
    xc = 0.5 * (xe[:-1] + xe[1:]); yc = 0.5 * (ye[:-1] + ye[1:])

    grids, all_med, cmax = {}, [], 1
    for st in stages:
        for lab, msk in [(LBL_A, is_fast), (LBL_B, is_slow)]:
            med, cnt = _bin_grid(st, value_of, xe, ye, msk, log_color)
            grids[(st, lab)] = (med, cnt)
            all_med += list(med.values())
            cmax = max(cmax, max(cnt.values(), default=1))
    all_med = np.array(all_med) if all_med else np.array([0.0, 1.0])
    vmin, vmax = np.percentile(all_med, [5, 95])
    sm = cm.ScalarMappable(Normalize(vmin, vmax), cmap)

    for st in stages:
        fig, axs = plt.subplots(1, 2, figsize=(9.2, 4.3), sharex=True, sharey=True)
        for ax, lab in zip(axs, [LBL_A, LBL_B]):
            med, cnt = grids[(st, lab)]
            if med:
                keys = list(med)
                xs = [xc[k[0]] for k in keys]; ys = [yc[k[1]] for k in keys]
                cs = sm.to_rgba(np.array([med[k] for k in keys]))
                ss = [smin + (cnt[k] / cmax) * (smax - smin) for k in keys]
                ax.scatter(xs, ys, s=ss, c=cs, marker="s", edgecolors="k", linewidths=0.3)
            ax.set_xlim(xe[0], xe[-1]); ax.set_ylim(ye[0], ye[-1])
            ax.set_xlabel(r"$\log_{10} M_*$")
            ax.set_title(f"{lab}  (n={int(sum(cnt.values()))})")
        axs[0].set_ylabel(r"$\log_{10} M_{\rm halo}$")
        cbar = fig.colorbar(sm, ax=axs, shrink=0.9, pad=0.02); cbar.set_label(vlabel)
        for q in (0.25, 0.5, 1.0):                       # marker-size legend (source counts)
            axs[1].scatter([], [], s=smin + q * (smax - smin), c="0.7", marker="s",
                           edgecolors="k", linewidths=0.3, label=f"N≈{int(round(q * cmax))}")
        axs[1].legend(title="bin count", fontsize=7, loc="upper left", framealpha=0.9)
        fig.suptitle(f"{vlabel}   —   critical point: {st}", y=1.02)
        if fname_prefix:
            plt.savefig(figpath( f"{fname_prefix}_{st}.png"), dpi=130, bbox_inches="tight")
        plt.show()

# dust fraction = Mdust / M*
mstar_mhalo_grids(lambda st: DIAGS[st]["dust/M*"],
                  r"median $\log_{10}(M_{\rm dust}/M_*)$", fname_prefix="grid_dustfrac")
# BH accretion rate
if _have_bh:
    mstar_mhalo_grids(lambda st: DIAGS_CGM[st]["bh_mdot"],
                      r"median $\log_{10}\dot M_{\rm BH}$ [$M_\odot$ yr$^{-1}$]",
                      fname_prefix="grid_bhmdot")
else:
    print("BH history missing -> run §4b before the Mdot grid.")


In [ ]:
# (optional) regenerate the standalone CGM summary figures from the merged cluster
# products — unrelated to the §8e registry above; safe to skip.
!python plot_cgm_stats.py